# Chapter 18: Lexer

> *“The lexical analyser is the gatekeeper of the compiler. It
> transforms an unruly, unstructured torrent of source characters into a
> disciplined, discrete stream of symbolic tokens.”*  
> — **Alfred V. Aho, Monica S. Lam, Ravi Sethi, and Jeffrey D. Ullman**,
> *Compilers: Principles, Techniques, and Tools*

------------------------------------------------------------------------

### 18.1 The Compiler Pipeline: From Text to Meaning

To a human programmer, source code is an expressive, structured medium:
expressions compute values, functions encapsulate procedural
abstractions, and type annotations enforce semantic correctness.

To an operating system kernel and a standard text editor, however,
source code is merely an unstructured array of ASCII or UTF-8 encoded
bytes sitting in a file buffer. It has no intrinsic hierarchy, no notion
of arithmetic precedence, and no concept of variable scope.

<figure id="fig:code-transformation">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzNjhwdCIgaGVpZ2h0PSIzODRwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzNjguMDAgMzg0LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDM4
MCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gc291cmNlIC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNz
PSJub2RlIj4KPHRpdGxlPnNvdXJjZTwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIHBvaW50cz0iMzYwLC0zNzYgMCwtMzc2IDAsLTM0MCAzNjAsLTM0MCAzNjAsLTM3
NiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii0zNjEiIGZvbnQtZmFtaWx5
PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+U291cmNlIEZp
bGUgKCZxdW90O3Byb2dyYW0uZmFiJnF1b3Q7KTo8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJz
dGFydCIgeD0iOCIgeT0iLTM1MCIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVk
aXVtIiBmb250LXNpemU9IjEwLjAwIj5sZXQgc3F1YXJlID0gZm4gKHg6IEludCk6IEludCA9Jmd0
OyB4ICogeDs8L3RleHQ+CjwvZz4KPCEtLSBsZXhpY2FsIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNz
PSJub2RlIj4KPHRpdGxlPmxleGljYWw8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJv
a2U9ImJsYWNrIiBwb2ludHM9IjM2MCwtMzEzIDAsLTMxMyAwLC0yNDggMzYwLC0yNDggMzYwLC0z
MTMiLz4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSI4IiB5PSItMjk0LjUiIGZvbnQtZmFt
aWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+U1RBR0Ug
MTogTGV4aWNhbCBBbmFseXNpcyAoU2Nhbm5pbmcgLyBUb2tlbmlzYXRpb24pICYjNDU7IFRoaXMg
Q2hhcHRlcjwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSI4IiB5PSItMjgzLjUi
IGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4w
MCI+JiM0NTsgVHJhbnNsYXRlcyBjaGFyYWN0ZXJzIHRvIGRpc2NyZXRlIHRva2VuczwvdGV4dD4K
PHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSI4IiB5PSItMjcyLjUiIGZvbnQtZmFtaWx5PSJJ
b3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+JiM0NTsgU3RyaXBz
IHRyaXZpYSAod2hpdGVzcGFjZSwgY29tbWVudHMpPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0i
c3RhcnQiIHg9IjgiIHk9Ii0yNjEuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8g
TWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4mIzQ1OyBSZWNvcmRzIHNvdXJjZSBjb29yZGluYXRl
cyBmb3IgZGlhZ25vc3RpY3M8L3RleHQ+CjwvZz4KPCEtLSBzb3VyY2UmIzQ1OyZndDtsZXhpY2Fs
IC0tPgo8ZyBpZD0iZWRnZTEiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnNvdXJjZSYjNDU7Jmd0O2xl
eGljYWw8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTgwLC0z
MzkuODdDMTgwLC0zMzQuOTIgMTgwLC0zMjkuMjkgMTgwLC0zMjMuNTIiLz4KPHBvbHlnb24gZmls
bD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTgzLjUsLTMyMy4xMiAxODAsLTMxMy4x
MiAxNzYuNSwtMzIzLjEyIDE4My41LC0zMjMuMTIiLz4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0
IiB4PSIxODAiIHk9Ii0zMjQiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1
bSIgZm9udC1zaXplPSIxMC4wMCI+ICYjMTYwOyhSYXcgQ2hhcmFjdGVyIFN0cmVhbSk8L3RleHQ+
CjwvZz4KPCEtLSBzeW50YWN0aWMgLS0+CjxnIGlkPSJub2RlMyIgY2xhc3M9Im5vZGUiPgo8dGl0
bGU+c3ludGFjdGljPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
cG9pbnRzPSIzNjAsLTIyMSAwLC0yMjEgMCwtMTcxIDM2MCwtMTcxIDM2MCwtMjIxIi8+Cjx0ZXh0
IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTIwNC41IiBmb250LWZhbWlseT0iSW9zZXZr
YSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlNUQUdFIDI6IFN5bnRhY3Rp
YyBBbmFseXNpcyAoUGFyc2luZykgJiM0NTsgQ2hhcHRlciAxOTwvdGV4dD4KPHRleHQgdGV4dC1h
bmNob3I9InN0YXJ0IiB4PSI4IiB5PSItMTkzLjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJv
biBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+JiM0NTsgRXZhbHVhdGVzIENvbnRleHQm
IzQ1O0ZyZWUgR3JhbW1hciAoQ0ZHKSB2aWEgUmVjdXJzaXZlIERlc2NlbnQ8L3RleHQ+Cjx0ZXh0
IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTE4Mi41IiBmb250LWZhbWlseT0iSW9zZXZr
YSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPiYjNDU7IEJ1aWxkcyB0aGUg
QWJzdHJhY3QgU3ludGF4IFRyZWUgKEFTVCk8L3RleHQ+CjwvZz4KPCEtLSBsZXhpY2FsJiM0NTsm
Z3Q7c3ludGFjdGljIC0tPgo8ZyBpZD0iZWRnZTIiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmxleGlj
YWwmIzQ1OyZndDtzeW50YWN0aWM8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBkPSJNMTgwLC0yNDcuODRDMTgwLC0yNDIuNDEgMTgwLC0yMzYuNzUgMTgwLC0yMzEuMjYi
Lz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTgzLjUsLTIz
MS4wMSAxODAsLTIyMS4wMSAxNzYuNSwtMjMxLjAxIDE4My41LC0yMzEuMDEiLz4KPHRleHQgdGV4
dC1hbmNob3I9InN0YXJ0IiB4PSIxODAiIHk9Ii0yMzIiIGZvbnQtZmFtaWx5PSJJb3NldmthIENo
YXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+ICYjMTYwO3N0ZDo6dmVjdG9yJmx0
O1Rva2VuJmd0OzwvdGV4dD4KPC9nPgo8IS0tIHNlbWFudGljIC0tPgo8ZyBpZD0ibm9kZTQiIGNs
YXNzPSJub2RlIj4KPHRpdGxlPnNlbWFudGljPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIg
c3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzNjAsLTE0NCAwLC0xNDQgMCwtOTQgMzYwLC05NCAzNjAs
LTE0NCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii0xMjcuNSIgZm9udC1m
YW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5TVEFH
RSAzOiBTZW1hbnRpYyBBbmFseXNpcyAoVHlwZSBDaGVja2luZykgJiM0NTsgQ2hhcHRlciAyMDwv
dGV4dD4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSI4IiB5PSItMTE2LjUiIGZvbnQtZmFt
aWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+JiM0NTsg
RW5mb3JjZXMgc3RhdGljIHR5cGUgcnVsZXMgYW5kIHNjb3BlIHJlc29sdXRpb248L3RleHQ+Cjx0
ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTEwNS41IiBmb250LWZhbWlseT0iSW9z
ZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPiYjNDU7IERlY29yYXRl
cyBBU1Qgd2l0aCB2ZXJpZmllZCB0eXBlczwvdGV4dD4KPC9nPgo8IS0tIHN5bnRhY3RpYyYjNDU7
Jmd0O3NlbWFudGljIC0tPgo8ZyBpZD0iZWRnZTMiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnN5bnRh
Y3RpYyYjNDU7Jmd0O3NlbWFudGljPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJi
bGFjayIgZD0iTTE4MCwtMTcwLjg2QzE4MCwtMTY1LjY0IDE4MCwtMTYwLjAyIDE4MCwtMTU0LjQ5
Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE4My41LC0x
NTQuMSAxODAsLTE0NC4xIDE3Ni41LC0xNTQuMSAxODMuNSwtMTU0LjEiLz4KPHRleHQgdGV4dC1h
bmNob3I9InN0YXJ0IiB4PSIxODAiIHk9Ii0xNTUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJv
biBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+ICYjMTYwO0FTVCAoTm9kZXMsIEV4cHJl
c3Npb25zKTwvdGV4dD4KPC9nPgo8IS0tIGVtaXNzaW9uIC0tPgo8ZyBpZD0ibm9kZTUiIGNsYXNz
PSJub2RlIj4KPHRpdGxlPmVtaXNzaW9uPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgcG9pbnRzPSIzNjAsLTY3IDAsLTY3IDAsLTMxIDM2MCwtMzEgMzYwLC02NyIv
Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii01MiIgZm9udC1mYW1pbHk9Iklv
c2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5TVEFHRSA0OiBDb2Rl
IEdlbmVyYXRpb24gKEVtaXNzaW9uKSAmIzQ1OyBDaGFwdGVyIDIxPC90ZXh0Pgo8dGV4dCB0ZXh0
LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii00MSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9u
IE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4mIzQ1OyBUcmF2ZXJzZXMgQVNUIGFuZCBl
bWl0cyB0YXJnZXQgTW9kZXJuIEMrKyBjb2RlPC90ZXh0Pgo8L2c+CjwhLS0gc2VtYW50aWMmIzQ1
OyZndDtlbWlzc2lvbiAtLT4KPGcgaWQ9ImVkZ2U0IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5zZW1h
bnRpYyYjNDU7Jmd0O2VtaXNzaW9uPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJi
bGFjayIgZD0iTTE4MCwtOTMuODNDMTgwLC04OC40NCAxODAsLTgyLjY4IDE4MCwtNzcuMTciLz4K
PHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTgzLjUsLTc3LjA0
IDE4MCwtNjcuMDQgMTc2LjUsLTc3LjA0IDE4My41LC03Ny4wNCIvPgo8dGV4dCB0ZXh0LWFuY2hv
cj0ic3RhcnQiIHg9IjE4MCIgeT0iLTc4IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9u
byBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPiAmIzE2MDtUeXBlZCBhbmQgVmVyaWZpZWQgQVNU
PC90ZXh0Pgo8L2c+CjwhLS0gdGFyZ2V0IC0tPgo8ZyBpZD0ibm9kZTYiIGNsYXNzPSJub2RlIj4K
PHRpdGxlPnRhcmdldDwvdGl0bGU+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE4MCIg
eT0iLTQuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNp
emU9IjEwLjAwIj5UYXJnZXQgQXJ0aWZhY3QgKCZxdW90O3Byb2dyYW0uY3BwJnF1b3Q7KTwvdGV4
dD4KPC9nPgo8IS0tIGVtaXNzaW9uJiM0NTsmZ3Q7dGFyZ2V0IC0tPgo8ZyBpZD0iZWRnZTUiIGNs
YXNzPSJlZGdlIj4KPHRpdGxlPmVtaXNzaW9uJiM0NTsmZ3Q7dGFyZ2V0PC90aXRsZT4KPHBhdGgg
ZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE4MCwtMzAuOEMxODAsLTI4LjY3IDE4MCwt
MjYuNSAxODAsLTI0LjM4Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBw
b2ludHM9IjE4My41LC0yNC4xNSAxODAsLTE0LjE1IDE3Ni41LC0yNC4xNSAxODMuNSwtMjQuMTUi
Lz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 1: Faberge to C++ Code Transformation</figcaption>
</figure>

The transformation of raw human-readable text into an executable (or in
our case, compilable) artifact occurs across four foundational stages
(as shown in fig. 1):

1.  **Lexical Analysis (Scanning / Tokenisation):** The character stream
    is scanned, classified, and converted into a sequence of atomic
    units called **Tokens**.
2.  **Syntactic Analysis (Parsing):** The token sequence is matched
    against the rules of a **Context-Free Grammar (CFG)** to construct a
    hierarchical **Abstract Syntax Tree (AST)**.
3.  **Semantic Analysis (Type Checking):** The AST is analysed against
    the language’s static type system and lexical scoping rules,
    verifying that identifiers exist and operand types match.
4.  **Code Emission:** The verified AST is lowered into target code—in
    our project, clean, functional **Modern C++**.

In this chapter, we commence our four-stage compiler project. We will
design and construct **Stage 1**: a high-throughput, hand-crafted
**Lexical Analyser** in Modern C++23 for our functional programming
language, **Faberge**. This language is named after the Imperial Russian
Fabergé company of the 19th century which produced luxury surprise
Easter Eggs—a term used in software to describe hidden features.

### 18.2 Formal Language Theory: Alphabets, Regular Expressions, and Grammars

Before writing a single line of scanning code, we must ground our work
in formal language theory.

#### 18.2.1 The Chomsky Hierarchy

In 1956, linguist and cognitive scientist Noam Chomsky categorised
formal grammars into four distinct nested levels based on their
expressive power (see fig. 2).

<figure id="fig:formal-grammars">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MzAuNzg0IiBo
ZWlnaHQ9IjE5OS41NTIiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNTMwLjc4NCAxOTku
NTUyIj4KICA8c3R5bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xl
LCAuc3ZnYm9iIHJlY3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJv
a2Utd2lkdGg6IDEuNTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAg
c3Ryb2tlLWxpbmVjYXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdi
b2IgdGV4dCB7CiAgd2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWls
eTogSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0sIG1vbm9zcGFjZTsKICBmb250LXNpemU6IDE0
cHg7Cn0KCi5zdmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6IHRy
YW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7Cn0K
Ci5zdmdib2IgLmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxlZCB7
CiAgZmlsbDogdHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5ub2Zp
bGwgewogIGZpbGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93IHsK
ICBtYXJrZXItZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fycm93
IHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0X21h
cmtlZF9kaWFtb25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2Ig
LmVuZF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLnN0YXJ0X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUpOwp9
Cgouc3ZnYm9iIC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI29w
ZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJr
ZXItc3RhcnQ6IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2JpZ19v
cGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgouc3Zn
Ym9iIC5zdGFydF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgj
YmlnX29wZW5fY2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9ImFy
cm93IiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3
IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBv
bHlnb24gcG9pbnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgog
ICAgPG1hcmtlciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZ
PSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQt
cmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+PC9w
b2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9IjAg
MCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9Ijci
IG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQi
IHI9IjIiIGNsYXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2Vy
IGlkPSJvcGVuX2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFy
a2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2Ui
PgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9j
aXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIgdmll
d0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhl
aWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0
IiBjeT0iNCIgcj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4K
ICA8L2RlZnM+CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9IjUz
MC43ODQiIGhlaWdodD0iMTk5LjU1MiI+PC9yZWN0PgogIDxyZWN0IHg9IjMxLjQyODAwMSIgeT0i
Ni45ODQiIHdpZHRoPSI0ODguODgwMDQiIGhlaWdodD0iMTY3LjYxNjAxIiBjbGFzcz0ic29saWQg
bm9maWxsIiByeD0iMCI+PC9yZWN0PgogIDxyZWN0IHg9IjU5LjM2NDAwMiIgeT0iMzQuOTIwMDAy
IiB3aWR0aD0iNDQ2Ljk3NiIgaGVpZ2h0PSIxMjUuNzEyMDA2IiBjbGFzcz0ic29saWQgbm9maWxs
IiByeD0iMCI+PC9yZWN0PgogIDxyZWN0IHg9Ijg3LjMiIHk9IjYyLjg1NjAwMyIgd2lkdGg9IjQw
NS4wNzIwMiIgaGVpZ2h0PSI4My44MDgiIGNsYXNzPSJzb2xpZCBub2ZpbGwiIHJ4PSIwIj48L3Jl
Y3Q+CiAgPHJlY3QgeD0iMTE1LjIzNiIgeT0iOTAuNzkyIiB3aWR0aD0iMzYzLjE2ODAzIiBoZWln
aHQ9IjQxLjkwNCIgY2xhc3M9InNvbGlkIG5vZmlsbCIgcng9IjAiPjwvcmVjdD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMjcuNDU4IiB5PSIxMDguMjUyMDEiID5UeXBlLTM6IFJl
Z3VsYXIgR3JhbW1hcnMgKEZpbml0ZSBBdXRvbWF0YSk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iMTY5LjM2MiIgeT0iMTIyLjIyIiA+KioqIExFWElDQUwgQU5BTFlTSVMg
RE9NQUlOICoqKjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5OS41MjIi
IHk9IjgwLjMxNiIgPlR5cGUtMjogQ29udGV4dC1GcmVlIEdyYW1tYXJzIChQdXNoZG93biBBdXRv
bWF0YSk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzEuNTg2IiB5PSI1
Mi4zOCIgPlR5cGUtMTogQ29udGV4dC1TZW5zaXRpdmUgR3JhbW1hcnMgKExpbmVhci1Cb3VuZGVk
IEF1dG9tYXRhKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0My42NSIg
eT0iMjQuNDQ0IiA+VHlwZS0wOiBVbnJlc3RyaWN0ZWQgR3JhbW1hcnMgKFR1cmluZyBNYWNoaW5l
cyk8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 2: Chomsky Hierarchy of Formal Grammars</figcaption>
</figure>

Of interest to compiler-writers are:

- **Type-3 (Regular Languages):** Recognisable by **Finite State
  Automata** without auxiliary memory. This is the exact domain of
  lexical tokens: identifiers, numbers, and operators can all be
  recognised using fixed, finite state transitions.
- **Type-2 (Context-Free Languages):** Recognisable by **Pushdown
  Automata** (state machines augmented with a last-in, first-out stack).
  This is the domain of nested syntax: matched parentheses, balanced
  code blocks, and recursive expressions.

#### 18.2.2 Mathematical Foundations of Regular Languages

Let us formalise our definitions:

- **Alphabet ($\Sigma$):** A finite, non-empty set of symbols (e.g., the
  ASCII character set, where
  $\Sigma = \{ \text{`a'}, \text{`b'}, \dots, \text{`0'}, \dots, \text{`;'} \}$).
- **String ($w$):** A finite sequence of symbols chosen from $\Sigma$.
  The length of a string is denoted $|w|$. The empty string is denoted
  $\epsilon$ ($|\epsilon| = 0$).
- **Language ($L$):** A set of strings over a fixed alphabet
  ($L \subseteq \Sigma^*$, where $\Sigma^*$ is the Kleene closure of
  $\Sigma$).

A language is **Regular** if and only if it can be defined by a
**Regular Expression (Regex)**. The base axioms of regular expressions
are:

1.  **Empty Set ($\emptyset$):** Denotes the language containing no
    strings.
2.  **Epsilon ($\epsilon$):** Denotes the language containing only the
    empty string $\{\epsilon\}$.
3.  **Literal Symbol ($a \in \Sigma$):** Denotes the singleton language
    $\{a\}$.

Given regular expressions $R$ and $S$ representing languages $L(R)$ and
$L(S)$, three operations form inductive closures:

- **Alternation ($R \mid S$):** $L(R \mid S) = L(R) \cup L(S)$
- **Concatenation ($R \cdot S$ or $RS$):**
  $L(RS) = \{ uv \mid u \in L(R) \land v \in L(S) \}$
- **Kleene Star ($R^*$):** $L(R^*) = \bigcup_{i=0}^{\infty} L(R)^i$
  (Zero or more repetitions).

From these primitives, we construct common shorthand notations:

- Positive Closure ($R^+$): $R \cdot R^*$ (One or more repetitions).
- Optionality ($R?$): $(R \mid \epsilon)$ (Zero or one occurrence).
- Character Ranges ($[a-z]$): $(a \mid b \mid c \mid \dots \mid z)$.

### 18.3 Finite State Automata: NFAs and DFAs

While regular expressions provide a declarative notation for humans,
computers execute regular languages using **Finite State Automata
(FSA)**.

An automaton is a mathematical machine consisting of a finite set of
internal states, an input character tape, and a transition function that
guides state changes as characters are consumed.

#### 18.3.1 Deterministic Finite Automata (DFA)

A **Deterministic Finite Automaton** is formally defined as a 5-tuple:

$$M = (Q, \Sigma, \delta, q_0, F)$$

Where:

- $Q$ is a finite set of states.
- $\Sigma$ is the input alphabet.
- $\delta: Q \times \Sigma \to Q$ is the **deterministic transition
  function** (for every state and every character, there is at most
  *one* valid target state).
- $q_0 \in Q$ is the unique initial starting state.
- $F \subseteq Q$ is the set of accepting (or final) states.

See fig. 3 for a representation of a DFA with seven states (five
accepting states).

<figure id="fig:dfa-states">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzMzZwdCIgaGVpZ2h0PSIyMzRwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzMzYuMDAgMjM0LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIz
MCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gcTAgLS0+CjxnIGlkPSJub2RlMSIgY2xhc3M9Im5v
ZGUiPgo8dGl0bGU+cTA8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNr
IiBjeD0iMTgiIGN5PSItODQiIHJ4PSIxOCIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjE4IiB5PSItODciIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1l
ZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+cTA8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRk
bGUiIHg9IjE4IiB5PSItNzYiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1
bSIgZm9udC1zaXplPSIxMC4wMCI+U3RhcnQ8L3RleHQ+CjwvZz4KPCEtLSBxMSAtLT4KPGcgaWQ9
Im5vZGUyIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5xMTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5v
bmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxMjgiIGN5PSItMTc1IiByeD0iMTgiIHJ5PSIxOCIvPgo8
ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMTI4IiBjeT0iLTE3NSIgcng9
IjIyIiByeT0iMjIiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTI4IiB5PSItMTc4
IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAu
MDAiPnExPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxMjgiIHk9Ii0xNjci
IGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4w
MCI+SUQ8L3RleHQ+CjwvZz4KPCEtLSBxMCYjNDU7Jmd0O3ExIC0tPgo8ZyBpZD0iZWRnZTEiIGNs
YXNzPSJlZGdlIj4KPHRpdGxlPnEwJiM0NTsmZ3Q7cTE8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25l
IiBzdHJva2U9ImJsYWNrIiBkPSJNMzIuMjQsLTk1LjE2QzQ5LjcyLC0xMDkuODkgODAuODMsLTEz
Ni4xIDEwMi42MiwtMTU0LjQ2Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNr
IiBwb2ludHM9IjEwMC41NiwtMTU3LjMgMTEwLjQ2LC0xNjEuMDcgMTA1LjA3LC0xNTEuOTUgMTAw
LjU2LC0xNTcuMyIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI3MSIgeT0iLTE0NSIg
Zm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAw
Ij5bYSYjNDU7ekEmIzQ1O1pfXTwvdGV4dD4KPC9nPgo8IS0tIHEyIC0tPgo8ZyBpZD0ibm9kZTMi
IGNsYXNzPSJub2RlIj4KPHRpdGxlPnEyPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgY3g9IjEyOCIgY3k9Ii04NCIgcng9IjE4IiByeT0iMTgiLz4KPGVsbGlwc2Ug
ZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjEyOCIgY3k9Ii04NCIgcng9IjIyIiByeT0i
MjIiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTI4IiB5PSItODciIGZvbnQtZmFt
aWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+cTI8L3Rl
eHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjEyOCIgeT0iLTc2IiBmb250LWZhbWls
eT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPklOVDwvdGV4
dD4KPC9nPgo8IS0tIHEwJiM0NTsmZ3Q7cTIgLS0+CjxnIGlkPSJlZGdlMyIgY2xhc3M9ImVkZ2Ui
Pgo8dGl0bGU+cTAmIzQ1OyZndDtxMjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGQ9Ik0zNi4zNSwtODRDNTIuMzUsLTg0IDc2LjQ4LC04NCA5NS44MiwtODQiLz4KPHBv
bHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iOTUuOTksLTg3LjUgMTA1
Ljk5LC04NCA5NS45OSwtODAuNSA5NS45OSwtODcuNSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIiB4PSI3MSIgeT0iLTg3IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRp
dW0iIGZvbnQtc2l6ZT0iMTAuMDAiPjAmIzQ1Ozk8L3RleHQ+CjwvZz4KPCEtLSBxNSAtLT4KPGcg
aWQ9Im5vZGU2IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5xNTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9
Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxMjgiIGN5PSItMjIiIHJ4PSIxOCIgcnk9IjE4Ii8+
CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxMjgiIGN5PSItMjIiIHJ4
PSIyMiIgcnk9IjIyIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjEyOCIgeT0iLTI1
IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAu
MDAiPnE1PC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxMjgiIHk9Ii0xNCIg
Zm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAw
Ij4mcXVvdDsmZ3Q7JnF1b3Q7PC90ZXh0Pgo8L2c+CjwhLS0gcTAmIzQ1OyZndDtxNSAtLT4KPGcg
aWQ9ImVkZ2U4IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5xMCYjNDU7Jmd0O3E1PC90aXRsZT4KPHBh
dGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTM0LjAyLC03NS4zOEM1MC45OCwtNjUu
NjQgNzguOTQsLTQ5LjU5IDk5Ljc3LC0zNy42MyIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ry
b2tlPSJibGFjayIgcG9pbnRzPSIxMDEuNTcsLTQwLjY0IDEwOC41LC0zMi42MiA5OC4wOCwtMzQu
NTcgMTAxLjU3LC00MC42NCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI3MSIgeT0i
LTY3IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0i
MTAuMDAiPiZxdW90OyZndDsmcXVvdDs8L3RleHQ+CjwvZz4KPCEtLSBxMSYjNDU7Jmd0O3ExIC0t
Pgo8ZyBpZD0iZWRnZTIiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnExJiM0NTsmZ3Q7cTE8L3RpdGxl
Pgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTIwLjI2LC0yMDUuODRDMTIx
LjI5LC0yMTEuMjQgMTIzLjg3LC0yMTUgMTI4LC0yMTUgMTM0LjQ1LC0yMTUgMTM3LjExLC0yMDUu
ODQgMTM2LjAxLC0xOTUuNTgiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2si
IHBvaW50cz0iMTIzLjc1LC0yMDUuNDkgMTE5Ljk5LC0xOTUuNTggMTE2Ljc2LC0yMDUuNjcgMTIz
Ljc1LC0yMDUuNDkiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTI4IiB5PSItMjE4
IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAu
MDAiPlthJiM0NTt6QSYjNDU7WjAmIzQ1OzldXzwvdGV4dD4KPC9nPgo8IS0tIHEyJiM0NTsmZ3Q7
cTIgLS0+CjxnIGlkPSJlZGdlNCIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+cTImIzQ1OyZndDtxMjwv
dGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xMjAuMjYsLTExNC44
NEMxMjEuMjksLTEyMC4yNCAxMjMuODcsLTEyNCAxMjgsLTEyNCAxMzQuNDUsLTEyNCAxMzcuMTEs
LTExNC44NCAxMzYuMDEsLTEwNC41OCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJi
bGFjayIgcG9pbnRzPSIxMjMuNzUsLTExNC40OSAxMTkuOTksLTEwNC41OCAxMTYuNzYsLTExNC42
NyAxMjMuNzUsLTExNC40OSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxMjgiIHk9
Ii0xMjciIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXpl
PSIxMC4wMCI+MCYjNDU7OTwvdGV4dD4KPC9nPgo8IS0tIHEzIC0tPgo8ZyBpZD0ibm9kZTQiIGNs
YXNzPSJub2RlIj4KPHRpdGxlPnEzPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tl
PSJibGFjayIgY3g9IjIxNiIgY3k9Ii04NCIgcng9IjE4IiByeT0iMTgiLz4KPHRleHQgdGV4dC1h
bmNob3I9Im1pZGRsZSIgeD0iMjE2IiB5PSItODEuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hh
cm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5xMzwvdGV4dD4KPC9nPgo8IS0tIHEy
JiM0NTsmZ3Q7cTMgLS0+CjxnIGlkPSJlZGdlNSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+cTImIzQ1
OyZndDtxMzwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xNTAu
MjYsLTg0QzE2MS41NywtODQgMTc1LjYsLTg0IDE4Ny43MSwtODQiLz4KPHBvbHlnb24gZmlsbD0i
YmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTg3Ljc0LC04Ny41IDE5Ny43NCwtODQgMTg3
Ljc0LC04MC41IDE4Ny43NCwtODcuNSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIx
NzIiIHk9Ii04NyIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250
LXNpemU9IjEwLjAwIj4mcXVvdDsuJnF1b3Q7PC90ZXh0Pgo8L2c+CjwhLS0gcTQgLS0+CjxnIGlk
PSJub2RlNSIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+cTQ8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJu
b25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMzA2IiBjeT0iLTg0IiByeD0iMTgiIHJ5PSIxOCIvPgo8
ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMzA2IiBjeT0iLTg0IiByeD0i
MjIiIHJ5PSIyMiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIzMDYiIHk9Ii04NyIg
Zm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAw
Ij5xNDwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMzA2IiB5PSItNzYiIGZv
bnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+
RkxPQVQ8L3RleHQ+CjwvZz4KPCEtLSBxMyYjNDU7Jmd0O3E0IC0tPgo8ZyBpZD0iZWRnZTYiIGNs
YXNzPSJlZGdlIj4KPHRpdGxlPnEzJiM0NTsmZ3Q7cTQ8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25l
IiBzdHJva2U9ImJsYWNrIiBkPSJNMjM0LjM5LC04NEMyNDUuNjEsLTg0IDI2MC41NCwtODQgMjcz
Ljc3LC04NCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIy
NzMuOTMsLTg3LjUgMjgzLjkzLC04NCAyNzMuOTMsLTgwLjUgMjczLjkzLC04Ny41Ii8+Cjx0ZXh0
IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjI2MSIgeT0iLTg3IiBmb250LWZhbWlseT0iSW9zZXZr
YSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPjAmIzQ1Ozk8L3RleHQ+Cjwv
Zz4KPCEtLSBxNCYjNDU7Jmd0O3E0IC0tPgo8ZyBpZD0iZWRnZTciIGNsYXNzPSJlZGdlIj4KPHRp
dGxlPnE0JiM0NTsmZ3Q7cTQ8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNr
IiBkPSJNMjk4LjI2LC0xMTQuODRDMjk5LjI5LC0xMjAuMjQgMzAxLjg3LC0xMjQgMzA2LC0xMjQg
MzEyLjQ1LC0xMjQgMzE1LjExLC0xMTQuODQgMzE0LjAxLC0xMDQuNTgiLz4KPHBvbHlnb24gZmls
bD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMzAxLjc1LC0xMTQuNDkgMjk3Ljk5LC0x
MDQuNTggMjk0Ljc2LC0xMTQuNjcgMzAxLjc1LC0xMTQuNDkiLz4KPHRleHQgdGV4dC1hbmNob3I9
Im1pZGRsZSIgeD0iMzA2IiB5PSItMTI3IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9u
byBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPjAmIzQ1Ozk8L3RleHQ+CjwvZz4KPCEtLSBxNiAt
LT4KPGcgaWQ9Im5vZGU3IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5xNjwvdGl0bGU+CjxlbGxpcHNl
IGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIyMTYiIGN5PSItMjIiIHJ4PSIxOCIgcnk9
IjE4Ii8+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIyMTYiIGN5PSIt
MjIiIHJ4PSIyMiIgcnk9IjIyIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjIxNiIg
eT0iLTI1IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6
ZT0iMTAuMDAiPnE2PC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIyMTYiIHk9
Ii0xNCIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9
IjEwLjAwIj4mcXVvdDsmZ3Q7PSZxdW90OzwvdGV4dD4KPC9nPgo8IS0tIHE1JiM0NTsmZ3Q7cTYg
LS0+CjxnIGlkPSJlZGdlOSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+cTUmIzQ1OyZndDtxNjwvdGl0
bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xNTAuMjYsLTIyQzE2MC4z
NiwtMjIgMTcyLjYzLC0yMiAxODMuNzcsLTIyIi8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJv
a2U9ImJsYWNrIiBwb2ludHM9IjE4My45OSwtMjUuNSAxOTMuOTksLTIyIDE4My45OSwtMTguNSAx
ODMuOTksLTI1LjUiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTcyIiB5PSItMjUi
IGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4w
MCI+JnF1b3Q7PSZxdW90OzwvdGV4dD4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 3: Deterministic Finite Automaton (DFA) state diagram
matching identifiers, integers, floats, and operators</figcaption>
</figure>

#### 18.3.2 From Regular Expressions to DFAs

Industrial scanner generators (such as Lex, Flex, or Ragel) translate
declarative regular expressions into optimized executable DFAs through a
three-step mathematical pipeline:

1.  **Thompson’s Construction (Regex $\to$ NFA):** Transforms the
    regular expression into a Non-Deterministic Finite Automaton
    containing $\epsilon$-transitions (free jumps requiring no character
    input).
2.  **The Powerset Construction (Subset Construction, NFA $\to$ DFA):**
    Eliminates non-determinism and $\epsilon$-transitions by
    constructing states that represent sets of NFA states.
3.  **Hopcroft’s Algorithm (DFA State Minimisation):** Collapses
    indistinguishable equivalent states to produce the minimal DFA
    possessing the minimum possible number of states.

### 18.4 Anatomy and Engineering of a Hand-Crafted Lexer

While automated tools like Flex generate state transition tables, almost
all modern production compilers—including **Clang/LLVM (C++)**, **rustc
(Rust)**, **tsc (TypeScript)**, and **V8 (JavaScript)**—use
**hand-written, recursive-descent scanners**.

Why write a hand-crafted lexer?

1.  **Rich Diagnostic Reporting:** When a scanner encounters an error
    (e.g., an unclosed string or an invalid escape character), a
    hand-crafted lexer can inspect surrounding context, emit source code
    snippet underlines, and offer compiler fix-it hints.
2.  **Performance and Cache Locality:** Hand-crafted scanners avoid
    giant table lookup indirection, enabling the compiler to scan
    characters directly out of hot CPU cache lines using modern CPU
    branch predictors.
3.  **Complex Language Grammar Interop:** Context-dependent tokenisation
    (such as string interpolation `f"Hello {name}"` or
    indentation-sensitive scoping) is straightforward to express in
    procedural code but notoriously painful in table-driven generators.

#### 18.4.1 The Two Golden Scanning Rules

When transforming an ambiguous character stream into tokens, our lexer
strictly obeys two universal disambiguation rules:

**The Principle of Maximal Munch (Longest Match Rule):**

The scanner must always consume the longest possible valid prefix of
characters that constitutes a legal token.

- Given the character sequence `==`, the lexer must emit the single
  equality operator token `TokenType::EqualEqual`, rather than emitting
  two consecutive assignment tokens `TokenType::Equal`,
  `TokenType::Equal`.
- Given `identifier_123`, the lexer must emit a single identifier token
  rather than `identifier_` followed by an integer `123`.

**The Keyword Precedence Rule:**

Keywords are syntactically identical to identifiers: they both match the
regular expression `[a-zA-Z_][a-zA-Z0-9_]*`. When an identifier token is
scanned, the lexer performs a lookup against the language’s reserved
keyword table. If matched, the token type is specialised (e.g., `let`
becomes `TokenType::Keyword_Let`). fig. 4 shows how the string
`let count = 42` is tokenised.

<figure id="fig:keyword-precedence">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI3MTIuMzY4MDQi
IGhlaWdodD0iMTI5LjcxMiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA3MTIuMzY4MDQg
MTI5LjcxMiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNp
cmNsZSwgLnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAg
c3Ryb2tlLXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAx
OwogIHN0cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgou
c3ZnYm9iIHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1m
YW1pbHk6IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXpl
OiAxNHB4Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxs
OiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4
Owp9Cgouc3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxs
ZWQgewogIGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAu
bm9maWxsIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJv
dyB7CiAgbWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9h
cnJvdyB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtl
ZF9kaWFtb25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3Zn
Ym9iIC5lbmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgou
c3ZnYm9iIC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJs
KCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAg
bWFya2VyLXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9i
aWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1
cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlk
PSJhcnJvdyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxwb2x5Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtl
cj4KICAgIDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIg
cmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0
YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIi
PjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94
PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0
PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5
PSI0IiByPSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1h
cmtlciBpZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQi
IG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQi
PjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUi
IHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJr
ZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBj
eD0iNCIgY3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJr
ZXI+CiAgPC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRo
PSI3MTIuMzY4MDQiIGhlaWdodD0iMTI5LjcxMiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+SW5wdXQ8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTAuNjM0MDAzIiB5PSIxMC40NzYiID5TdHJlYW06PC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEyMC40NzQwMSIgeT0iMTAuNDc2
IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMzQuNDQyIiB5PSIx
MC40NzYiID5sPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE0OC40MSIg
eT0iMTAuNDc2IiA+XTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNjIu
Mzc4IiB5PSIxMC40NzYiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjE3Ni4zNDYwMSIgeT0iMTAuNDc2IiA+ZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIxOTAuMzE0MDEiIHk9IjEwLjQ3NiIgPl08L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iMjA0LjI4MjAxIiB5PSIxMC40NzYiID5bPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjIxOC4yNSIgeT0iMTAuNDc2IiA+dDwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMzIuMjE4IiB5PSIxMC40NzYiID5dPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI0Ni4xODYiIHk9IjEwLjQ3NiIgPls8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjc0LjEyMiIgeT0iMTAuNDc2IiA+
XTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyODguMDkiIHk9IjEwLjQ3
NiIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzAyLjA1OCIgeT0i
MTAuNDc2IiA+YzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMTYuMDI2
IiB5PSIxMC40NzYiID5dPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMy
OS45OTQwMiIgeT0iMTAuNDc2IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIzNDMuOTYyIiB5PSIxMC40NzYiID5vPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjM1Ny45MzAwMiIgeT0iMTAuNDc2IiA+XTwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSIzNzEuODk4IiB5PSIxMC40NzYiID5bPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjM4NS44NjYiIHk9IjEwLjQ3NiIgPnU8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzk5LjgzNCIgeT0iMTAuNDc2IiA+XTwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MTMuODAyIiB5PSIxMC40NzYiID5bPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQyNy43NzAwMiIgeT0iMTAuNDc2
IiA+bjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NDEuNzM4IiB5PSIx
MC40NzYiID5dPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ1NS43MDYw
MiIgeT0iMTAuNDc2IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0
NjkuNjc0IiB5PSIxMC40NzYiID50PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjQ4My42NDIwMyIgeT0iMTAuNDc2IiA+XTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSI0OTcuNjEwMDIiIHk9IjEwLjQ3NiIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iNTI1LjU0NiIgeT0iMTAuNDc2IiA+XTwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1MzkuNTE0MDQiIHk9IjEwLjQ3NiIgPls8L3RleHQ+CiAg
PGxpbmUgeDE9IjU1MS43MzYiIHkxPSI1LjIzOCIgeDI9IjU1OC43MjAwMyIgeTI9IjUuMjM4IiBj
bGFzcz0ic29saWQiPjwvbGluZT4KICA8bGluZSB4MT0iNTUxLjczNiIgeTE9IjguNzMwMDAwNSIg
eDI9IjU1OC43MjAwMyIgeTI9IjguNzMwMDAwNSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTY3LjQ1IiB5PSIxMC40NzYiID5dPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjU4MS40MTgiIHk9IjEwLjQ3NiIgPls8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNjA5LjM1NCIgeT0iMTAuNDc2IiA+
XTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2MjMuMzIyIiB5PSIxMC40
NzYiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjYzNy4yOTAwNCIg
eT0iMTAuNDc2IiA+NDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2NTEu
MjU4IiB5PSIxMC40NzYiID5dPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjY2NS4yMjYiIHk9IjEwLjQ3NiIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iNjc5LjE5NDAzIiB5PSIxMC40NzYiID4yPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjY5My4xNjIwNSIgeT0iMTAuNDc2IiA+XTwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSI4LjczMDAwMDUiIHk9IjUyLjM4IiA+VG9rZW48L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTAuNjM0MDAzIiB5PSI1Mi4zOCIgPlNsaWNl
czo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTYyLjM3OCIgeT0iNTIu
MzgiID4mIzM5O2xldCYjMzk7PC90ZXh0PgogIDxsaW5lIHgxPSIxNzguMDkyMDEiIHkxPSI1NS44
NzIiIHgyPSIxNzguMDkyMDEiIHkyPSI3My4zMzIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxw
b2x5Z29uIHBvaW50cz0iMTc0LjYsNzMuMzMyIDE4MS41ODQsNzMuMzMyIDE3OC4wOTIwMSw4My44
MDgiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIzNjQuOTE0IiB5PSI1Mi4zOCIgPiYjMzk7Y291bnQmIzM5OzwvdGV4dD4KICA8bGluZSB4
MT0iMzg3LjYxMiIgeTE9IjU1Ljg3MiIgeDI9IjM4Ny42MTIiIHkyPSI3My4zMzIiIGNsYXNzPSJz
b2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzg0LjEyMDAzLDczLjMzMiAzOTEuMTA0
LDczLjMzMiAzODcuNjEyLDgzLjgwOCIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjU0Ni40OTgiIHk9IjUyLjM4IiA+JiMzOTs8L3RleHQ+
CiAgPGxpbmUgeDE9IjU1MS43MzYiIHkxPSI0Ny4xNDIwMDIiIHgyPSI1NTguNzIwMDMiIHkyPSI0
Ny4xNDIwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxsaW5lIHgxPSI1NTEuNzM2IiB5MT0i
NTAuNjM0MDAzIiB4Mj0iNTU4LjcyMDAzIiB5Mj0iNTAuNjM0MDAzIiBjbGFzcz0ic29saWQiPjwv
bGluZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1NjAuNDY2IiB5PSI1Mi4zOCIg
PiYjMzk7PC90ZXh0PgogIDxsaW5lIHgxPSI1NTUuMjI4IiB5MT0iNTUuODcyIiB4Mj0iNTU1LjIy
OCIgeTI9IjczLjMzMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSI1
NTEuNzM2LDczLjMzMiA1NTguNzIwMDMsNzMuMzMyIDU1NS4yMjgsODMuODA4IiBjbGFzcz0iZmls
bGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNjUxLjI1OCIg
eT0iNTIuMzgiID4mIzM5OzQyJiMzOTs8L3RleHQ+CiAgPGxpbmUgeDE9IjY2Ni45NzIwNSIgeTE9
IjU1Ljg3MiIgeDI9IjY2Ni45NzIwNSIgeTI9IjczLjMzMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+
CiAgPHBvbHlnb24gcG9pbnRzPSI2NjMuNDgwMDQsNzMuMzMyIDY3MC40NjQsNzMuMzMyIDY2Ni45
NzIwNSw4My44MDgiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSI4LjczMDAwMDUiIHk9IjEwOC4yNTIwMSIgPkNsYXNzaWZpZWQ6PC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM1MC45NDYiIHk9IjEwOC4yNTIwMSIg
PklkZW50aWZpZXI8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTM5LjUx
NDA0IiB5PSIxMDguMjUyMDEiID5FcXVhbDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIxNDEuNDI2MDEiIHk9IjEwOC4yNTIwMSIgPktleXdvcmRfTGV0PC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjYzMC4zMDYiIHk9IjEwOC4yNTIwMSIgPkxpdGVy
YWxfSW50PC90ZXh0PgogIDxnPgogICAgPGxpbmUgeDE9IjEyNS43MTIwMDYiIHkxPSIxMy45Njgi
IHgyPSIxMzIuNjk2IiB5Mj0iMjcuOTM2IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5l
IHgxPSIxMzIuNjk2IiB5MT0iMjcuOTM2IiB4Mj0iMjIzLjQ4OCIgeTI9IjI3LjkzNiIgY2xhc3M9
InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMjMwLjQ3MiIgeTE9IjEzLjk2OCIgeDI9IjIy
My40ODgiIHkyPSIyNy45MzYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwvZz4KICA8Zz4KICAg
IDxsaW5lIHgxPSIyOTMuMzI4IiB5MT0iMTMuOTY4IiB4Mj0iMzAwLjMxMiIgeTI9IjI3LjkzNiIg
Y2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMzAwLjMxMiIgeTE9IjI3LjkzNiIg
eDI9IjQ3NC45MTIwMiIgeTI9IjI3LjkzNiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGlu
ZSB4MT0iNDgxLjg5NjAzIiB5MT0iMTMuOTY4IiB4Mj0iNDc0LjkxMjAyIiB5Mj0iMjcuOTM2IiBj
bGFzcz0ic29saWQiPjwvbGluZT4KICA8L2c+CiAgPGc+CiAgICA8bGluZSB4MT0iNTQ0Ljc1MiIg
eTE9IjEzLjk2OCIgeDI9IjU1MS43MzYiIHkyPSIyNy45MzYiIGNsYXNzPSJzb2xpZCI+PC9saW5l
PgogICAgPGxpbmUgeDE9IjU1MS43MzYiIHkxPSIyNy45MzYiIHgyPSI1NTguNzIwMDMiIHkyPSIy
Ny45MzYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjU2NS43MDQwNCIgeTE9
IjEzLjk2OCIgeDI9IjU1OC43MjAwMyIgeTI9IjI3LjkzNiIgY2xhc3M9InNvbGlkIj48L2xpbmU+
CiAgPC9nPgogIDxnPgogICAgPGxpbmUgeDE9IjYyOC41NiIgeTE9IjEzLjk2OCIgeDI9IjYzNS41
NDQiIHkyPSIyNy45MzYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjYzNS41
NDQiIHkxPSIyNy45MzYiIHgyPSI2ODQuNDMyIiB5Mj0iMjcuOTM2IiBjbGFzcz0ic29saWQiPjwv
bGluZT4KICAgIDxsaW5lIHgxPSI2OTEuNDE2IiB5MT0iMTMuOTY4IiB4Mj0iNjg0LjQzMiIgeTI9
IjI3LjkzNiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 4: Precedence of Keyword Tokenisation</figcaption>
</figure>

### 18.5 Specification of the Faberge Language

Before writing the implementation, we define the complete lexical
specification for our functional language, **Faberge**.

#### 18.5.1 Keywords

```
let     mut     fn      if      then    else    
match   with    type    true    false   return
```

#### 18.5.2 Operators and Punctuation

```
+       -       *       /       %               (Arithmetic)
==      !=      <       <=      >       >=      (Comparison)
&&      ||      !                               (Logical)
=       =>      ->      ::      |               (Structural / Functional)
(       )       {       }       [       ]       (Delimiters)
,       :       ;       .                       (Separators)
```

#### 18.5.3 Literals and Identifiers

- **Integer Literals:** Contiguous decimal digits `[0-9]+` or
  hexadecimal prefixed literals `0x[0-9a-fA-F]+`.
- **Floating-Point Literals:** Decimal numbers with a fractional
  component: `[0-9]+\.[0-9]+`.
- **String Literals:** Double-quoted character sequences supporting
  escape characters: `\"`, `\\`, `\n`, `\t`, `\r`.
- **Identifiers:** Sequences matching `[a-zA-Z_][a-zA-Z0-9_]*`.

#### 18.5.4 Trivia (Comments and Whitespace)

- Whitespace (``, `\t`, `\r`, `\n`) separates tokens and increments
  source coordinates, but is discarded.
- Single-line comments begin with `//` and extend to the end of the
  line.
- Multi-line comments begin with `/*` and extend to `*/`, supporting
  nested block comments (`/* /* nested */ */`).

### 18.6 Modern C++ Lexer Architecture

Let us architect our lexical analyser (as outlined in fig. 5) using
idiomatic Modern C++20 features:

- `std::string_view` for zero-allocation token string slicing.
- A strongly typed `SourceLocation` coordinate system to anchor tokens
  precisely to their source line and column.
- A high-fidelity `DiagnosticEngine` capable of rendering compiler
  errors with visual carets (`^~~~`).
- A clean, single-pass `Lexer` class employing `peek()`, `advance()`,
  and `match()` abstractions.

<figure id="fig:lexer-subsystem">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MjMuOCIgaGVp
Z2h0PSIyOTcuMzI4IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDUyMy44IDI5Ny4zMjgi
PgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5z
dmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13
aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJv
a2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0
ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBJ
b3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTRweDsK
fQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNw
YXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2
Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBm
aWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7
CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewog
IG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9u
ZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2Vk
X2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5k
X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5z
dmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9j
aXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1z
dGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5f
Y2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2Ig
LnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdf
b3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ci
IHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1h
cmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdv
biBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8
bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIi
IG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHln
b24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDgg
OCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3Jp
ZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0i
MiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9
Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJX
aWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAg
ICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNs
ZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94
PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0
PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5
PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwv
ZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNTIzLjgi
IGhlaWdodD0iMjk3LjMyOCI+PC9yZWN0PgogIDxyZWN0IHg9IjEwLjQ3NiIgeT0iMTMyLjY5NiIg
d2lkdGg9IjIwOS41MiIgaGVpZ2h0PSI4My44MDgwMTQiIGNsYXNzPSJzb2xpZCBub2ZpbGwiIHJ4
PSIwIj48L3JlY3Q+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDMuNjUiIHk9IjE1
MC4xNTYiID5Ub2tlbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI4NS41
NTQiIHk9IjE1MC4xNTYiID5DbGFzc2lmaWNhdGlvbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIyOS42ODIwMDEiIHk9IjE2NC4xMjQwMSIgPi0gc2Nhbl9pZGVudGlmaWVy
X29yX2t3KCk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjkuNjgyMDAx
IiB5PSIxNzguMDkyMDEiID4tIHNjYW5fbnVtYmVyX2xpdGVyYWwoKTwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOS42ODIwMDEiIHk9IjE5Mi4wNjAwMSIgPi0gc2Nhbl9z
dHJpbmdfbGl0ZXJhbCgpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI5
LjY4MjAwMSIgeT0iMjA2LjAyOCIgPi0gc2Nhbl9vcGVyYXRvcl9zeW1ib2woKTwvdGV4dD4KICA8
cmVjdCB4PSIzMTAuNzg4IiB5PSIxMzIuNjk2IiB3aWR0aD0iMjAyLjUzNjA0IiBoZWlnaHQ9IjY5
Ljg0MDAxIiBjbGFzcz0ic29saWQgbm9maWxsIiByeD0iMCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjM1MC45NDYiIHk9IjE1MC4xNTYiID5EaWFnbm9zdGljPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQyNy43NzAwMiIgeT0iMTUwLjE1NiIg
PkVuZ2luZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjkuOTk0MDIi
IHk9IjE2NC4xMjQwMSIgPi0gU291cmNlIGxpbmUgZm9ybWF0dGluZzwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjkuOTk0MDIiIHk9IjE3OC4wOTIwMSIgPi0gQU5TSSBj
b2xvdXIgZm9ybWF0dGluZzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIz
MjkuOTk0MDIiIHk9IjE5Mi4wNjAwMSIgPi0gQ2FyZXQgdW5kZXJsaW5lcyAoXn5+KTwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMjUuMjM0MDEiIHk9IjI0LjQ0NCIgPkxl
eGVyPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI2Ny4xMzgiIHk9IjI0
LjQ0NCIgPlN1YnN5c3RlbTwvdGV4dD4KICA8cG9seWdvbiBwb2ludHM9Ijk3Ljc3NiwxMTUuMjM2
IDEwNC43NiwxMTUuMjM2IDEwMS4yNjgwMDUsMTI1LjcxMjAwNiIgY2xhc3M9ImZpbGxlZCI+PC9w
b2x5Z29uPgogIDxwb2x5Z29uIHBvaW50cz0iMzk4LjA4OCwxMTUuMjM2IDQwNS4wNzIwMiwxMTUu
MjM2IDQwMS41ODAwMiwxMjUuNzEyMDA2IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxp
bmUgeDE9IjEwMS4yNjgwMDUiIHkxPSIyMjMuNDg4IiB4Mj0iMTAxLjI2ODAwNSIgeTI9IjI0MC45
NDgwMSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSI5Ny43NzYsMjQw
Ljk0ODAxIDEwNC43NiwyNDAuOTQ4MDEgMTAxLjI2ODAwNSwyNTEuNDI0MDEiIGNsYXNzPSJmaWxs
ZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOS42ODIwMDEi
IHk9IjUyLjM4IiA+U291cmNlIENvZGU6IHN0ZDo6c3RyaW5nX3ZpZXc8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjkuNjgyMDAxIiB5PSI2Ni4zNDgiID5DdXJyZW50IEN1
cnNvcjogc2l6ZV90IG1fY3Vyc29yPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjI5LjY4MjAwMSIgeT0iODAuMzE2IiA+U291cmNlIENvb3JkaW5hdGVzOiBtX2xpbmUsIG1f
Y29sdW1uPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM2LjY2NiIgeT0i
Mjc1Ljg2OCIgPnN0ZDo6dmVjdG9yJmx0O1Rva2VuJmd0OzwvdGV4dD4KICA8Zz4KICAgIDxsaW5l
IHgxPSIxMC40NzYiIHkxPSI2Ljk4NCIgeDI9IjUxMy4zMjQwNCIgeTI9IjYuOTg0IiBjbGFzcz0i
c29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSIxMC40NzYiIHkxPSI2Ljk4NCIgeDI9IjEwLjQ3
NiIgeTI9IjkwLjc5MiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iNTEzLjMy
NDA0IiB5MT0iNi45ODQiIHgyPSI1MTMuMzI0MDQiIHkyPSI5MC43OTIiIGNsYXNzPSJzb2xpZCI+
PC9saW5lPgogICAgPGxpbmUgeDE9IjEwLjQ3NiIgeTE9IjM0LjkyMDAwMiIgeDI9IjUxMy4zMjQw
NCIgeTI9IjM0LjkyMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMTAu
NDc2IiB5MT0iOTAuNzkyIiB4Mj0iNTEzLjMyNDA0IiB5Mj0iOTAuNzkyIiBjbGFzcz0ic29saWQi
PjwvbGluZT4KICAgIDxsaW5lIHgxPSIxMDEuMjY4MDA1IiB5MT0iOTAuNzkyIiB4Mj0iMTAxLjI2
ODAwNSIgeTI9IjExNS4yMzYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjQw
MS41ODAwMiIgeTE9IjkwLjc5MiIgeDI9IjQwMS41ODAwMiIgeTI9IjExNS4yMzYiIGNsYXNzPSJz
b2xpZCI+PC9saW5lPgogIDwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 5: Token Classification and Diagnostic
Engine</figcaption>
</figure>

### 18.7 Experimental Implementation in Modern C++: The Core Lexical Analyser

The listings below are a feature-complete lexer implementation for all
of the language features described in this chapter:

- lst. 1 provides precise source tracking, which is the hallmark of a
  professional compiler. Every token must know exactly where it was
  born.
- lst. 2 lists every token type available in the Faberge language, and
  provides a way to convert these into textual format.
- lst. 3 defines an error-reporting function and a way of extracting the
  sourcecode line from an associated `SourceLocation` object. A great
  compiler does not merely report that a scan failed; it prints the
  exact source line and visually pinpoints the offending column with
  ANSI colour formatting.
- lst. 4 implements the core scanner. It maintains zero heap allocations
  per token by slicing views over the source buffer.
- lst. 5 creates a comprehensive test suite to demonstrate this lexer:
  1.  Scanning an expressive **Faberge** functional script.
  2.  Formatting and inspecting generated tokens.
  3.  Exercising our diagnostic engine on intentionally malformed source
      strings.

Listing 1: Tracking Source Coordinates (SourceLocation.hpp)

In [ ]:
#ifndef SOURCE_LOCATION_HPP
#define SOURCE_LOCATION_HPP

#include <string>
#include <string_view>
#include <cstdint>
#include <ostream>

namespace Faberge {

struct SourceLocation {
    std::string_view filename{"<anonymous>"};
    uint32_t line{1};
    uint32_t column{1};
    uint32_t offset{0};

    friend std::ostream& operator<<(std::ostream& os, const SourceLocation& loc) {
        os << loc.filename << ":" << loc.line << ":" << loc.column;
        return os;
    }
};

struct SourceSpan {
    SourceLocation start;
    SourceLocation end;
    std::string_view lexeme;
};

} // namespace Faberge

#endif // SOURCE_LOCATION_HPP

Listing 2: Strongly Typed Lexical Tokens (Token.hpp)

In [ ]:
#ifndef TOKEN_HPP
#define TOKEN_HPP

// #include "SourceLocation.hpp"
#include <string_view>
#include <string>

namespace Faberge {

enum class TokenType {
    // End of Input
    EndOfFile,

    // Identifiers and Literals
    Identifier,
    Literal_Integer,
    Literal_Float,
    Literal_String,
    Literal_Boolean,

    // Keywords
    Keyword_Let,
    Keyword_Mut,
    Keyword_Fn,
    Keyword_If,
    Keyword_Then,
    Keyword_Else,
    Keyword_Match,
    Keyword_With,
    Keyword_Type,
    Keyword_Return,

    // Arithmetic Operators
    Plus,           // +
    Minus,          // -
    Asterisk,       // *
    Slash,          // /
    Percent,        // %

    // Comparison Operators
    EqualEqual,     // ==
    ExclamationEqual,// !=
    Less,           // <
    LessEqual,      // <=
    Greater,        // >
    GreaterEqual,   // >=

    // Logical Operators
    AmpersandAmpersand, // &&
    PipePipe,           // ||
    Exclamation,        // !

    // Assignment and Structural Symbols
    Equal,          // =
    FatArrow,       // =>
    ThinArrow,      // ->
    ColonColon,     // ::
    Pipe,           // |
    Colon,          // :
    Semicolon,      // ;
    Comma,          // ,
    Dot,            // .

    // Delimiters
    LeftParen,      // (
    RightParen,     // )
    LeftBrace,      // {
    RightBrace,     // }
    LeftBracket,    // [
    RightBracket,   // ]

    // Lexical Error Sentinel
    Error
};

struct Token {
    TokenType type{TokenType::EndOfFile};
    std::string_view lexeme{};
    SourceLocation location{};

    [[nodiscard]] bool is(TokenType expected) const noexcept { return type == expected; }
    [[nodiscard]] bool is_not(TokenType expected) const noexcept { return type != expected; }

    [[nodiscard]] static std::string_view to_string(TokenType t) noexcept {
        switch (t) {
            case TokenType::EndOfFile:          return "EOF";
            case TokenType::Identifier:         return "Identifier";
            case TokenType::Literal_Integer:    return "Literal_Integer";
            case TokenType::Literal_Float:      return "Literal_Float";
            case TokenType::Literal_String:     return "Literal_String";
            case TokenType::Literal_Boolean:    return "Literal_Boolean";
            case TokenType::Keyword_Let:        return "let";
            case TokenType::Keyword_Mut:        return "mut";
            case TokenType::Keyword_Fn:         return "fn";
            case TokenType::Keyword_If:         return "if";
            case TokenType::Keyword_Then:       return "then";
            case TokenType::Keyword_Else:       return "else";
            case TokenType::Keyword_Match:      return "match";
            case TokenType::Keyword_With:       return "with";
            case TokenType::Keyword_Type:       return "type";
            case TokenType::Keyword_Return:     return "return";
            case TokenType::Plus:               return "+";
            case TokenType::Minus:              return "-";
            case TokenType::Asterisk:           return "*";
            case TokenType::Slash:              return "/";
            case TokenType::Percent:            return "%";
            case TokenType::EqualEqual:         return "==";
            case TokenType::ExclamationEqual:   return "!=";
            case TokenType::Less:               return "<";
            case TokenType::LessEqual:          return "<=";
            case TokenType::Greater:            return ">";
            case TokenType::GreaterEqual:       return ">=";
            case TokenType::AmpersandAmpersand: return "&&";
            case TokenType::PipePipe:           return "||";
            case TokenType::Exclamation:        return "!";
            case TokenType::Equal:              return "=";
            case TokenType::FatArrow:           return "=>";
            case TokenType::ThinArrow:          return "->";
            case TokenType::ColonColon:         return "::";
            case TokenType::Pipe:               return "|";
            case TokenType::Colon:              return ":";
            case TokenType::Semicolon:          return ";";
            case TokenType::Comma:              return ",";
            case TokenType::Dot:                return ".";
            case TokenType::LeftParen:          return "(";
            case TokenType::RightParen:         return ")";
            case TokenType::LeftBrace:          return "{";
            case TokenType::RightBrace:         return "}";
            case TokenType::LeftBracket:        return "[";
            case TokenType::RightBracket:       return "]";
            case TokenType::Error:              return "Error";
        }
        return "Unknown";
    }
};

} // namespace Faberge

#endif // TOKEN_HPP

Listing 3: Professional Compiler Diagnostics (Diagnostic.hpp)

In [ ]:
#ifndef DIAGNOSTIC_HPP
#define DIAGNOSTIC_HPP

// #include "SourceLocation.hpp"
#include <iostream>
#include <string_view>
#include <sstream>
#include <vector>

namespace Faberge {

class DiagnosticEngine {
public:
    static void report_error(std::string_view source, 
                             SourceLocation loc, 
                             std::string_view message) {
        // ANSI escape codes for modern terminal formatting
        constexpr std::string_view ANSI_RED_BOLD = "\033[1;31m";
        constexpr std::string_view ANSI_BOLD     = "\033[1m";
        constexpr std::string_view ANSI_BLUE     = "\033[1;34m";
        constexpr std::string_view ANSI_RESET    = "\033[0m";

        std::cerr << ANSI_RED_BOLD << "error: " << ANSI_RESET 
                  << ANSI_BOLD << message << ANSI_RESET << "\n"
                  << "  --> " << loc << "\n";

        // Extract the entire offending source line
        std::string_view line_str = extract_line(source, loc.offset);
        
        std::string line_num_str = std::to_string(loc.line);
        std::string padding(line_num_str.length(), ' ');

        std::cerr << ANSI_BLUE << padding << " |\n"
                  << line_num_str << " | " << ANSI_RESET << line_str << "\n"
                  << ANSI_BLUE << padding << " | " << ANSI_RED_BOLD;

        // Render the visual caret underline (^~~~~)
        for (uint32_t i = 1; i < loc.column; ++i) {
            std::cerr << ' ';
        }
        std::cerr << "^" << ANSI_RESET << "\n\n";
    }

private:
    static std::string_view extract_line(std::string_view source, uint32_t offset) {
        if (offset >= source.size()) offset = source.empty() ? 0 : source.size() - 1;

        // Trace backward to find line start
        size_t start = offset;
        while (start > 0 && source[start - 1] != '\n') {
            --start;
        }

        // Trace forward to find line end
        size_t end = offset;
        while (end < source.size() && source[end] != '\n' && source[end] != '\r') {
            ++end;
        }

        return source.substr(start, end - start);
    }
};

} // namespace Faberge

#endif // DIAGNOSTIC_HPP

Listing 4: The Complete State Machine Scanner (Lexer.hpp)

In [ ]:
#ifndef LEXER_HPP
#define LEXER_HPP

// #include "Token.hpp"
// #include "Diagnostic.hpp"
#include <unordered_map>
#include <vector>
#include <cctype>

namespace Faberge {

class Lexer {
public:
    explicit Lexer(std::string_view source, std::string_view filename = "<anonymous>")
        : m_source(source), m_filename(filename) {
        initialise_keywords();
    }

    std::vector<Token> tokenise() {
        std::vector<Token> tokens;

        while (!is_at_end()) {
            m_start_cursor = m_current_cursor;
            m_start_column = m_column;

            Token token = scan_token();
            if (token.type != TokenType::Error) {
                tokens.push_back(token);
            }
        }

        tokens.push_back(Token{
            .type = TokenType::EndOfFile,
            .lexeme = "",
            .location = current_location()
        });

        return tokens;
    }

    [[nodiscard]] bool has_errors() const noexcept { return m_error_count > 0; }
    [[nodiscard]] size_t error_count() const noexcept { return m_error_count; }

private:
    std::string_view m_source;
    std::string_view m_filename;
    size_t m_start_cursor{0};
    size_t m_current_cursor{0};
    uint32_t m_line{1};
    uint32_t m_column{1};
    uint32_t m_start_column{1};
    size_t m_error_count{0};

    std::unordered_map<std::string_view, TokenType> m_keywords;

    void initialise_keywords() {
        m_keywords["let"]    = TokenType::Keyword_Let;
        m_keywords["mut"]    = TokenType::Keyword_Mut;
        m_keywords["fn"]     = TokenType::Keyword_Fn;
        m_keywords["if"]     = TokenType::Keyword_If;
        m_keywords["then"]   = TokenType::Keyword_Then;
        m_keywords["else"]   = TokenType::Keyword_Else;
        m_keywords["match"]  = TokenType::Keyword_Match;
        m_keywords["with"]   = TokenType::Keyword_With;
        m_keywords["type"]   = TokenType::Keyword_Type;
        m_keywords["return"] = TokenType::Keyword_Return;
        m_keywords["true"]   = TokenType::Literal_Boolean;
        m_keywords["false"]  = TokenType::Literal_Boolean;
    }

    [[nodiscard]] bool is_at_end() const noexcept {
        return m_current_cursor >= m_source.size();
    }

    char advance() noexcept {
        if (is_at_end()) return '\0';
        char c = m_source[m_current_cursor++];
        ++m_column;
        return c;
    }

    [[nodiscard]] char peek() const noexcept {
        return is_at_end() ? '\0' : m_source[m_current_cursor];
    }

    [[nodiscard]] char peek_next() const noexcept {
        return (m_current_cursor + 1 >= m_source.size()) ? '\0' : m_source[m_current_cursor + 1];
    }

    bool match(char expected) noexcept {
        if (is_at_end() || m_source[m_current_cursor] != expected) {
            return false;
        }
        ++m_current_cursor;
        ++m_column;
        return true;
    }

    [[nodiscard]] SourceLocation current_location() const noexcept {
        return SourceLocation{
            .filename = m_filename,
            .line = m_line,
            .column = m_start_column,
            .offset = static_cast<uint32_t>(m_start_cursor)
        };
    }

    [[nodiscard]] Token make_token(TokenType type) const noexcept {
        return Token{
            .type = type,
            .lexeme = m_source.substr(m_start_cursor, m_current_cursor - m_start_cursor),
            .location = current_location()
        };
    }

    Token error_token(std::string_view message) {
        ++m_error_count;
        DiagnosticEngine::report_error(m_source, current_location(), message);
        return Token{
            .type = TokenType::Error,
            .lexeme = m_source.substr(m_start_cursor, m_current_cursor - m_start_cursor),
            .location = current_location()
        };
    }

    // -------------------------------------------------------------------------
    // Scanning Subroutines
    // -------------------------------------------------------------------------

    Token scan_token() {
        skip_trivia();
        if (is_at_end()) return make_token(TokenType::EndOfFile);

        m_start_cursor = m_current_cursor;
        m_start_column = m_column;

        char c = advance();

        // 1. Identifiers and Keywords
        if (std::isalpha(c) || c == '_') {
            return scan_identifier();
        }

        // 2. Numeric Literals (Integers, Hex, Floats)
        if (std::isdigit(c)) {
            return scan_number();
        }

        // 3. String Literals
        if (c == '"') {
            return scan_string();
        }

        // 4. Operators and Punctuation
        switch (c) {
            // Single-character delimiters
            case '(': return make_token(TokenType::LeftParen);
            case ')': return make_token(TokenType::RightParen);
            case '{': return make_token(TokenType::LeftBrace);
            case '}': return make_token(TokenType::RightBrace);
            case '[': return make_token(TokenType::LeftBracket);
            case ']': return make_token(TokenType::RightBracket);
            case ',': return make_token(TokenType::Comma);
            case ';': return make_token(TokenType::Semicolon);
            case '.': return make_token(TokenType::Dot);
            case '%': return make_token(TokenType::Percent);

            // Arithmetic and Structural Operators
            case '+': return make_token(TokenType::Plus);
            case '*': return make_token(TokenType::Asterisk);
            case '/': return make_token(TokenType::Slash);

            case '-': 
                return match('>') ? make_token(TokenType::ThinArrow) : make_token(TokenType::Minus);

            case '=':
                if (match('=')) return make_token(TokenType::EqualEqual);
                if (match('>')) return make_token(TokenType::FatArrow);
                return make_token(TokenType::Equal);

            case '!':
                return match('=') ? make_token(TokenType::ExclamationEqual) : make_token(TokenType::Exclamation);

            case '<':
                return match('=') ? make_token(TokenType::LessEqual) : make_token(TokenType::Less);

            case '>':
                return match('=') ? make_token(TokenType::GreaterEqual) : make_token(TokenType::Greater);

            case ':':
                return match(':') ? make_token(TokenType::ColonColon) : make_token(TokenType::Colon);

            case '|':
                return match('|') ? make_token(TokenType::PipePipe) : make_token(TokenType::Pipe);

            case '&':
                if (match('&')) return make_token(TokenType::AmpersandAmpersand);
                return error_token("Unexpected character '&'. Did you mean '&&'?");

            default:
                break;
        }

        return error_token(std::string("Unrecognised or illegal character '") + c + "'.");
    }

    Token scan_identifier() {
        while (std::isalnum(peek()) || peek() == '_') {
            advance();
        }

        std::string_view text = m_source.substr(m_start_cursor, m_current_cursor - m_start_cursor);
        
        // Keyword disambiguation
        auto it = m_keywords.find(text);
        if (it != m_keywords.end()) {
            return make_token(it->second);
        }

        return make_token(TokenType::Identifier);
    }

    Token scan_number() {
        bool is_float = false;

        // Check for Hexadecimal: 0x...
        if (m_source[m_start_cursor] == '0' && (peek() == 'x' || peek() == 'X')) {
            advance(); // consume 'x'
            if (!std::isxdigit(peek())) {
                return error_token("Hexadecimal literal requires at least one valid hex digit [0-9a-fA-F].");
            }
            while (std::isxdigit(peek())) {
                advance();
            }
            return make_token(TokenType::Literal_Integer);
        }

        // Standard Decimal Integer
        while (std::isdigit(peek())) {
            advance();
        }

        // Check for Fractional Float: .[0-9]+
        if (peek() == '.' && std::isdigit(peek_next())) {
            is_float = true;
            advance(); // Consume '.'
            while (std::isdigit(peek())) {
                advance();
            }
        }

        return make_token(is_float ? TokenType::Literal_Float : TokenType::Literal_Integer);
    }

    Token scan_string() {
        while (peek() != '"' && !is_at_end()) {
            if (peek() == '\n') {
                ++m_line;
                m_column = 1;
            }
            // Handle escape characters
            if (peek() == '\\') {
                advance(); // consume '\'
                if (is_at_end()) break;
                char esc = advance();
                if (esc != '"' && esc != '\\' && esc != 'n' && esc != 't' && esc != 'r') {
                    while (peek() != '"' && !is_at_end()) { // Get to closing '"'...
                        advance();
                    }
                    advance(); // ...and consume it
                    return error_token(std::string("Invalid escape sequence '\\") + esc + "' in string literal.");
                }
                continue;
            }
            advance();
        }

        if (is_at_end()) {
            return error_token("Unterminated string literal: missing closing '\"'.");
        }

        advance(); // Consume closing '"'
        return make_token(TokenType::Literal_String);
    }

    void skip_trivia() {
        while (!is_at_end()) {
            char c = peek();
            switch (c) {
                case ' ':
                case '\t':
                case '\r':
                    advance();
                    break;
                case '\n':
                    ++m_line;
                    m_column = 0; // Will be set to 1 on advance
                    advance();
                    break;

                case '/':
                    // Single-line comment: // ...
                    if (peek_next() == '/') {
                        while (peek() != '\n' && !is_at_end()) {
                            advance();
                        }
                    } 
                    // Multi-line block comment: /* ... */ (supports nesting!)
                    else if (peek_next() == '*') {
                        advance(); // consume '/'
                        advance(); // consume '*'
                        int nesting_depth = 1;

                        while (nesting_depth > 0 && !is_at_end()) {
                            if (peek() == '/' && peek_next() == '*') {
                                advance();
                                advance();
                                ++nesting_depth;
                            } else if (peek() == '*' && peek_next() == '/') {
                                advance();
                                advance();
                                --nesting_depth;
                            } else {
                                if (peek() == '\n') {
                                    ++m_line;
                                    m_column = 0;
                                }
                                advance();
                            }
                        }

                        if (nesting_depth > 0) {
                            error_token("Unterminated block comment: missing closing '*/'.");
                            return;
                        }
                    } else {
                        return; // It is a division operator '/'
                    }
                    break;

                default:
                    return;
            }
        }
    }
};

} // namespace Faberge

#endif // LEXER_HPP

Listing 5: Verification and Diagnostics Showcase (main.cpp)

In [ ]:
// #include "Lexer.hpp"
#include <iostream>
#include <iomanip>

void run_lexer_demo(std::string_view test_name, std::string_view source_code) {
    using namespace Faberge;

    std::cout << "\n====================================================\n";
    std::cout << " TEST CASE: " << test_name << "\n";
    std::cout << "====================================================\n";
    std::cout << "SOURCE CODE:\n" << source_code << "\n----------------------------------------------------\n";

    Lexer lexer(source_code, "test.fab");
    auto tokens = lexer.tokenise();

    if (lexer.has_errors()) {
        std::cout << "[Result]: Lexer encountered " << lexer.error_count() << " error(s).\n\n";
        return;
    }

    std::cout << std::left << std::setw(6)  << "LINE" 
              << std::left << std::setw(6)  << "COL" 
              << std::left << std::setw(22) << "TOKEN TYPE" 
              << "LEXEME\n";
    std::cout << "----------------------------------------------------\n";

    for (const auto& token : tokens) {
        if (token.type == TokenType::EndOfFile) break;
        std::cout << std::left << std::setw(6)  << token.location.line
                  << std::left << std::setw(6)  << token.location.column
                  << std::left << std::setw(22) << Token::to_string(token.type)
                  << "'" << token.lexeme << "'\n";
    }
    std::cout << "----------------------------------------------------\n";
    std::cout << "[Success]: Emitted " << tokens.size() << " tokens cleanly.\n\n";
}

int main() {
    // 1. Valid Functional Script: Factorial and Higher-Order Functions
    constexpr std::string_view valid_source = R"(
        /* 
           Higher-order recursive pipeline
           Demonstrating nested block comments: /* valid nested block */ 
        */
        let factorial = fn (n: Int): Int => {
            if n <= 1 then 1 else n * factorial(n - 1)
        };

        let main = fn (): Int => {
            let limit = 42;
            let pi_approx = 3.14159;
            let greeting = "Hello, Faberge Compiler!\n";
            factorial(5)
        };
    )";

    run_lexer_demo("Valid Functional Script", valid_source);

    // 2. Syntax with Errors: Invalid escape sequence and unclosed comment
    constexpr std::string_view error_source = R"(
        let bad_string = "Invalid escape: \q here";
        let unclosed_comment = /* Unfinished block comment...
        let x = 100;
    )";

    run_lexer_demo("Syntax Error Diagnostics", error_source);

    return 0;
}

### 18.9 Summary

In this first stage of our compiler project, we established the
computational foundations of lexical analysis:

- **The Chomsky Hierarchy** categorises formal grammars. Lexical
  analysis operates strictly within the domain of **Regular Languages
  (Type-3)**.
- **Deterministic Finite Automata (DFAs)** represent state machines with
  unambiguous, deterministic transition paths, executing regular
  expressions in linear time $O(N)$.
- **Maximal Munch** and **Keyword Prioritisation** resolve lexical
  ambiguities, ensuring that identifiers, multi-character operators, and
  reserved words are classified correctly.
- **Modern C++20 Value Types** (`std::string_view`, explicit coordinate
  structs) enable us to build a zero-allocation, high-throughput scanner
  with rich, compiler-grade diagnostics.

### 18.10 Programming Challenges

#### Exercise 18.1: Arbitrary Base Integer Literals (Binary, Octal, Hexadecimal)

**Context and Theory:**

Modern systems programming languages require direct expression of binary
bitmasks, memory permissions, and hardware registers using non-decimal
radix prefixes. In the Faberge language, numeric literals can be
declared using three standard base prefixes:

- **Binary (`radix = 2`):** Prefixed with `0b` or `0B`, containing
  digits in $[0, 1]$.
- **Octal (`radix = 8`):** Prefixed with `0o` or `0O`, containing digits
  in $[0, 7]$.
- **Hexadecimal (`radix = 16`):** Prefixed with `0x` or `0X`, containing
  digits in $[0, 9, \text{a}-\text{f}, \text{A}-\text{F}]$.

During scanning, each character $c$ maps to an integer digit value
$d \in [0, \text{radix}-1]$. The numerical accumulator updates via:

$$v_{i} = v_{i-1} \times \text{radix} + d$$

To avoid silent unsigned integer overflow in 64-bit systems
($2^{64} - 1$), the lexer must verify the overflow invariant before
performing multiplication and addition:

$$v_{i-1} \le \frac{\text{UINT64\_MAX} - d}{\text{radix}}$$

fig. 6 outlines the process of left-to-right (MSB first) scanning of a
string representing a binary number.

<figure id="fig:scan-overflow">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MTQuNTkyMDQi
IGhlaWdodD0iNTkuODcyIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDYxNC41OTIwNCA1
OS44NzIiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTog
MTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsK
fQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVk
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5v
ZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7
Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1h
cmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmln
X29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJs
KCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0i
YXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+
CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJl
Zlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFy
dC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48
L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0i
MCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0i
NyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0i
NCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJr
ZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBt
YXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJz
ZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48
L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2
aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2Vy
SGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9
IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2Vy
PgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0i
NjE0LjU5MjA0IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+SW5wdXQ6PC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjU3LjYxOCIgeT0iMTAuNDc2IiA+JiMzOTswYjEwMTEmIzM5
OzwvdGV4dD4KICA8bGluZSB4MT0iMTE4LjcyODAwNCIgeTE9IjYuOTg0IiB4Mj0iMTM5LjY4MDAx
IiB5Mj0iNi45ODQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMTM5
LjY4MDAxLDMuNDkyIDE0Ni42NjQsNi45ODQgMTM5LjY4MDAxLDEwLjQ3NiIgY2xhc3M9ImZpbGxl
ZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE1NS4zOTQiIHk9
IjEwLjQ3NiIgPlByZWZpeDo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MjExLjI2NiIgeT0iMTAuNDc2IiA+JiMzOTswYiYjMzk7PC90ZXh0PgogIDxwYXRoIGQ9Ik0gMjUx
LjQyNDAxLDAgQSAxMy45NjgsMTMuOTY4IDAsMCwwIDI1MS40MjQwMSwxMy45NjgiIGNsYXNzPSJu
b2ZpbGwiPjwvcGF0aD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNTMuMTcwMDEi
IHk9IjEwLjQ3NiIgPlJhZGl4PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjI5NS4wNzQiIHk9IjEwLjQ3NiIgPjI8L3RleHQ+CiAgPHBhdGggZD0iTSAzMDAuMzEyLDAgQSAx
My45NjgsMTMuOTY4IDAsMCwxIDMwMC4zMTIsMTMuOTY4IiBjbGFzcz0ibm9maWxsIj48L3BhdGg+
CiAgPGxpbmUgeDE9IjMxNC4yOCIgeTE9IjYuOTg0IiB4Mj0iMzM1LjIzMiIgeTI9IjYuOTg0IiBj
bGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjMzNS4yMzIsMy40OTIgMzQy
LjIxNiw2Ljk4NCAzMzUuMjMyLDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM1MC45NDYiIHk9IjEwLjQ3NiIgPlN0cmVhbTo8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDA2LjgxODAyIiB5PSIxMC40
NzYiID5bJiMzOTsxJiMzOTssPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjQ0OC43MjIwMiIgeT0iMTAuNDc2IiA+JiMzOTswJiMzOTssPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjQ4My42NDIwMyIgeT0iMTAuNDc2IiA+JiMzOTsxJiMzOTssPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUxOC41NjIiIHk9IjEwLjQ3NiIg
PiYjMzk7MSYjMzk7XTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI4Ljcz
MDAwMDUiIHk9IjM4LjQxMjAwMyIgPkFjY3VtdWxhdG9yOiAoMCAqIDIpICsgMSA9IDEgLSZndDsg
KDEgKiAyKSArIDAgPSAyIC0mZ3Q7ICgyICogMikgKyAxID0gNSAtJmd0OyAoNSAqIDIpICsgMSA9
IDExPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 6: Evaluation of 0b1011</figcaption>
</figure>

**Specification:**

Implement a dedicated private scanning subroutine,
`scan_prefixed_integer()`, within the Faberge `Lexer`. The method must
detect prefixes, parse the character stream, detect arithmetic overflow
against `std::numeric_limits<uint64_t>::max()`, and emit a
`TokenType::Literal_Integer` token.

``` cpp
class Lexer {
    // ... existing definitions ...
private:
    Token scan_prefixed_integer(uint64_t radix) {
        advance(); // consume base specifier ('b', 'o', 'x')
        size_t num_start = current_cursor_;
        uint64_t accumulated_value = 0;
        while (!is_at_end() && is_valid_digit_for_radix(peek(), radix)) {
            uint64_t digit = char_to_digit(advance());
            if (accumulated_value > (std::numeric_limits<uint64_t>::max() - digit) / radix) {
                return error_token("Integer literal exceeds 64-bit unsigned bounds.");
            }
            accumulated_value = accumulated_value * radix + digit;
        }
        if (current_cursor_ == num_start) {
            return error_token("Radix literal requires at least one valid digit.");
        }
        return make_token(TokenType::Literal_Integer);
    }
};
```

**Verification Tasks:**

1.  **Multi-Base Extraction:** Scan the sequence
    `0b1101 0o755 0xDEADBEEF`. Verify that the emitted tokens have
    lexemes matching the full source slices and classify as
    `TokenType::Literal_Integer`.
2.  **Missing Digit Detection:** Pass `0b` and `0x;` into the scanner.
    Verify that the lexer halts the invalid literal, advances to the
    delimiter, and reports a diagnostic indicating missing digits.
3.  **64-bit Boundary Overflow:** Scan the maximum 64-bit unsigned hex
    integer `0xFFFFFFFFFFFFFFFF`, followed by `0x10000000000000000`.
    Verify that the first scans cleanly, whereas the second triggers an
    integer overflow diagnostic.

#### Exercise 18.2: IEEE-754 Scientific Notation Floating-Point Scanner

**Context and Theory:**

In scientific computing and compiler numerical analysis, floating-point
literals often span massive or infinitesimal orders of magnitude. The
Faberge language requires full support for scientific exponential
notation conforming to IEEE-754 floating-point standards:

$$\text{Format: } [\text{Digits}]\,.\,[\text{Digits}]\,[\text{eE}][+-]?\,[\text{Digits}]$$

A Deterministic Finite Automaton (DFA) scanning scientific floats (see
fig. 7) must disambiguate complex lexical contexts:

- A literal with no leading whole number part (e.g. `.5`) vs the
  struct/tuple member access operator (`.`).
- A trailing dot operator in method chaining (e.g. `42.to_string()`) vs
  a fractional point.
- The optional signed exponent component (`e+10`, `E-4`, `e8`).

<figure id="fig:dfa-float">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI1MTNwdCIgaGVpZ2h0PSI1OXB0Igogdmlld0JveD0iMC4wMCAwLjAw
IDUxMy4wMCA1OS4wMCIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIiB4bWxuczp4
bGluaz0iaHR0cDovL3d3dy53My5vcmcvMTk5OS94bGluayI+CjxnIGlkPSJncmFwaDAiIGNsYXNz
PSJncmFwaCIgdHJhbnNmb3JtPSJzY2FsZSgxIDEpIHJvdGF0ZSgwKSB0cmFuc2xhdGUoNCA1NSki
Pgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gc2F3X2RpZ2l0cyAtLT4KPGcgaWQ9Im5vZGUxIiBjbGFz
cz0ibm9kZSI+Cjx0aXRsZT5zYXdfZGlnaXRzPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIg
c3Ryb2tlPSJibGFjayIgY3g9IjIxLjUiIGN5PSItMjUuNSIgcng9IjIxLjUiIHJ5PSIyMS41Ii8+
Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjIxLjUiIHk9Ii0yOC41IiBmb250LWZhbWls
eT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlNhdzwvdGV4
dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjEuNSIgeT0iLTE3LjUiIGZvbnQtZmFt
aWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+RGlnaXRz
PC90ZXh0Pgo8L2c+CjwhLS0gc2F3X3BvaW50IC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2Rl
Ij4KPHRpdGxlPnNhd19wb2ludDwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGN4PSIxMDUuNSIgY3k9Ii0yNS41IiByeD0iMjEuNSIgcnk9IjIxLjUiLz4KPHRleHQg
dGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTA1LjUiIHk9Ii0yOC41IiBmb250LWZhbWlseT0iSW9z
ZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlNhdzwvdGV4dD4KPHRl
eHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTA1LjUiIHk9Ii0xNy41IiBmb250LWZhbWlseT0i
SW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlBvaW50PC90ZXh0
Pgo8L2c+CjwhLS0gc2F3X2RpZ2l0cyYjNDU7Jmd0O3Nhd19wb2ludCAtLT4KPGcgaWQ9ImVkZ2Ux
IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5zYXdfZGlnaXRzJiM0NTsmZ3Q7c2F3X3BvaW50PC90aXRs
ZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTQzLjE5LC0yNS41QzUyLjQy
LC0yNS41IDYzLjQ4LC0yNS41IDczLjY1LC0yNS41Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBz
dHJva2U9ImJsYWNrIiBwb2ludHM9IjczLjgxLC0yOSA4My44MSwtMjUuNSA3My44MSwtMjIgNzMu
ODEsLTI5Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjYzLjUiIHk9Ii0yOC41IiBm
b250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAi
PiZxdW90Oy4mcXVvdDs8L3RleHQ+CjwvZz4KPCEtLSBmcmFjdGlvbiAtLT4KPGcgaWQ9Im5vZGUz
IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5mcmFjdGlvbjwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5v
bmUiIHN0cm9rZT0iYmxhY2siIGN4PSIyMDAuNSIgY3k9Ii0yNS41IiByeD0iMjEuNSIgcnk9IjIx
LjUiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjAwLjUiIHk9Ii0yMyIgZm9udC1m
YW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5GcmFj
dGlvbjwvdGV4dD4KPC9nPgo8IS0tIHNhd19wb2ludCYjNDU7Jmd0O2ZyYWN0aW9uIC0tPgo8ZyBp
ZD0iZWRnZTIiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnNhd19wb2ludCYjNDU7Jmd0O2ZyYWN0aW9u
PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTEyNy4xNCwtMjUu
NUMxMzkuMywtMjUuNSAxNTQuOTIsLTI1LjUgMTY4LjUzLC0yNS41Ii8+Cjxwb2x5Z29uIGZpbGw9
ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE2OC45MiwtMjkgMTc4LjkyLC0yNS41IDE2
OC45MiwtMjIgMTY4LjkyLC0yOSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxNTMi
IHk9Ii0yOC41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQt
c2l6ZT0iMTAuMDAiPlswJiM0NTs5XTwvdGV4dD4KPC9nPgo8IS0tIHNhd19leHAgLS0+CjxnIGlk
PSJub2RlNCIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+c2F3X2V4cDwvdGl0bGU+CjxlbGxpcHNlIGZp
bGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIyOTEuNSIgY3k9Ii0yNS41IiByeD0iMjEuNSIg
cnk9IjIxLjUiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjkxLjUiIHk9Ii0yOC41
IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAu
MDAiPlNhdzwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjkxLjUiIHk9Ii0x
Ny41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0i
MTAuMDAiPiZxdW90O2UmcXVvdDsvJnF1b3Q7RSZxdW90OzwvdGV4dD4KPC9nPgo8IS0tIGZyYWN0
aW9uJiM0NTsmZ3Q7c2F3X2V4cCAtLT4KPGcgaWQ9ImVkZ2UzIiBjbGFzcz0iZWRnZSI+Cjx0aXRs
ZT5mcmFjdGlvbiYjNDU7Jmd0O3Nhd19leHA8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJv
a2U9ImJsYWNrIiBkPSJNMjIyLjE0LC0yNS41QzIzMy4zOCwtMjUuNSAyNDcuNDksLTI1LjUgMjU5
Ljk3LC0yNS41Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9
IjI1OS45NywtMjkgMjY5Ljk3LC0yNS41IDI1OS45NywtMjIgMjU5Ljk3LC0yOSIvPgo8dGV4dCB0
ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIyNDYiIHk9Ii0yOC41IiBmb250LWZhbWlseT0iSW9zZXZr
YSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPltlRV08L3RleHQ+CjwvZz4K
PCEtLSBzYXdfZXhwX3NpZ24gLS0+CjxnIGlkPSJub2RlNSIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+
c2F3X2V4cF9zaWduPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
Y3g9IjM4MC41IiBjeT0iLTI1LjUiIHJ4PSIyMS41IiByeT0iMjEuNSIvPgo8dGV4dCB0ZXh0LWFu
Y2hvcj0ibWlkZGxlIiB4PSIzODAuNSIgeT0iLTI4LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENo
YXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+U2F3IEV4cDwvdGV4dD4KPHRleHQg
dGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMzgwLjUiIHk9Ii0xNy41IiBmb250LWZhbWlseT0iSW9z
ZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlNpZ248L3RleHQ+Cjwv
Zz4KPCEtLSBzYXdfZXhwJiM0NTsmZ3Q7c2F3X2V4cF9zaWduIC0tPgo8ZyBpZD0iZWRnZTQiIGNs
YXNzPSJlZGdlIj4KPHRpdGxlPnNhd19leHAmIzQ1OyZndDtzYXdfZXhwX3NpZ248L3RpdGxlPgo8
cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMzEzLjEyLC0yNS41QzMyMy42Niwt
MjUuNSAzMzYuNzEsLTI1LjUgMzQ4LjQ0LC0yNS41Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBz
dHJva2U9ImJsYWNrIiBwb2ludHM9IjM0OC43MywtMjkgMzU4LjczLC0yNS41IDM0OC43MywtMjIg
MzQ4LjczLC0yOSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIzMzYiIHk9Ii0yOC41
IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAu
MDAiPlsrJiM0NTtdPC90ZXh0Pgo8L2c+CjwhLS0gYWNjZXB0X2Zsb2F0IC0tPgo8ZyBpZD0ibm9k
ZTYiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmFjY2VwdF9mbG9hdDwvdGl0bGU+CjxlbGxpcHNlIGZp
bGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSI0NzkuNSIgY3k9Ii0yNS41IiByeD0iMjEuNSIg
cnk9IjIxLjUiLz4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjQ3OS41
IiBjeT0iLTI1LjUiIHJ4PSIyNS41IiByeT0iMjUuNSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIiB4PSI0NzkuNSIgeT0iLTI4LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25v
IE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+U2F3PC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0i
bWlkZGxlIiB4PSI0NzkuNSIgeT0iLTE3LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBN
b25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+RGlnaXRzPC90ZXh0Pgo8L2c+CjwhLS0gc2F3
X2V4cF9zaWduJiM0NTsmZ3Q7YWNjZXB0X2Zsb2F0IC0tPgo8ZyBpZD0iZWRnZTUiIGNsYXNzPSJl
ZGdlIj4KPHRpdGxlPnNhd19leHBfc2lnbiYjNDU7Jmd0O2FjY2VwdF9mbG9hdDwvdGl0bGU+Cjxw
YXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik00MDIuMDcsLTI1LjVDNDE0LjEzLC0y
NS41IDQyOS42NywtMjUuNSA0NDMuNTksLTI1LjUiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0
cm9rZT0iYmxhY2siIHBvaW50cz0iNDQzLjgzLC0yOSA0NTMuODMsLTI1LjUgNDQzLjgzLC0yMiA0
NDMuODMsLTI5Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjQyOCIgeT0iLTI4LjUi
IGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4w
MCI+WzAmIzQ1OzldPC90ZXh0Pgo8L2c+CjwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 7: State Diagram of a DFA Scanning Scientific
Floats</figcaption>
</figure>

**Specification:**

Extend the `scan_number()` subroutine to support signed exponents.
Implement multi-character lookahead using `peek()` and `peek_next()` to
avoid eagerly consuming member-access dots or range operators (`..`).

``` cpp
Token scan_scientific_number() {
    bool has_fractional = false;
    while (std::isdigit(peek())) advance();
    if (peek() == '.' && std::isdigit(peek_next())) {
        has_fractional = true;
        advance(); // consume '.'
        while (std::isdigit(peek())) advance();
    }
    if (peek() == 'e' || peek() == 'E') {
        char next = peek_next();
        if (std::isdigit(next) || next == '+' || next == '-') {
            advance(); // consume 'e'/'E'
            if (peek() == '+' || peek() == '-') advance();
            if (!std::isdigit(peek())) return error_token("Malformed exponent in float.");
            while (std::isdigit(peek())) advance();
            return make_token(TokenType::Literal_Float);
        }
    }
    return make_token(has_fractional ? TokenType::Literal_Float : TokenType::Literal_Integer);
}
```

**Verification Tasks:**

1.  **Scientific Exponent Permutations:** Scan the tokens `1e10`,
    `3.14159E-4`, `0.5e+12`, and `42E5`. Verify that each token is
    correctly identified as `TokenType::Literal_Float`.
2.  **Method Access Disambiguation:** Scan the string `42.toString()`.
    Verify that the scanner yields an integer literal `42`, followed by
    a dot operator `TokenType::Dot`, followed by the identifier
    `toString`.
3.  **Incomplete Exponent Recovery:** Scan `1.23e` followed by an
    operator `+ 5`. Verify that the scanner reports an error for the
    unterminated exponent without consuming the addition operator.

#### Exercise 18.3: Digit Separator Grammar and Numerical Normalisation

**Context and Theory:**

To improve readability in high-density systems code, modern languages
allow single underscores `_` as visual digit separators within integer
and floating-point literals (e.g. `1_000_000` or `0xCAFE_BABE`).

However, relaxed separator placement leads to ambiguous syntax. Faberge
strictly enforces the **Interspersed Separator Rule**:

1.  An underscore is valid if and only if it is immediately preceded and
    immediately followed by a legal digit for the literal’s base.
2.  Leading underscores (`_100`) are classified as identifiers.
3.  Trailing underscores (`100_`) or adjacent underscores (`1__000`) are
    illegal lexical errors.
4.  Underscores adjacent to radix prefixes (`0x_123`) or decimal points
    (`1._2` or `1_.2`) are illegal.

- **Valid:** `1_000_000`, `0b1011_0001`, `3.141_592`
- **Invalid:** `1__000`, `0x_FFFF`, `123_`, `1._0`

**Specification:**

Add digit separator validation to the Faberge numerical scanner.
Implement a normalisation utility, `normalise_number_lexeme()`, which
returns a clean, allocation-free string representation with all
underscores stripped out for consumption by the downstream C++ emitter.

``` cpp
struct NumericLiteralScanner {
    static bool is_valid_separator_position(std::string_view source, size_t pos) {
        if (pos == 0 || pos + 1 >= source.size()) return false;
        char prev = source[pos - 1];
        char next = source[pos + 1];
        return (std::isxdigit(prev) && std::isxdigit(next));
    }
    static std::string strip_separators(std::string_view lexeme) {
        std::string clean;
        clean.reserve(lexeme.size());
        for (char c : lexeme) {
            if (c != '_') clean.push_back(c);
        }
        return clean;
    }
};
```

**Verification Tasks:**

1.  **Valid Separator Invariance:** Scan `1_000_000`, `0xDE_AD_BE_EF`,
    and `0b10_01_11_00`. Verify that all three pass as legal tokens and
    that `strip_separators()` yields `"1000000"`, `"0xDEADBEEF"`, and
    `"0b10011100"`.
2.  **Consecutive Separator Rejection:** Scan `10__000`. Verify that the
    scanner flags the second consecutive underscore as a syntax error.
3.  **Prefix and Suffix Violation:** Scan `0x_12` and `42_`. Verify that
    both strings trigger descriptive compiler diagnostics pointing to
    the exact column of the offending underscore.

#### Exercise 18.4: Multi-Token Lexical Lookahead ($LL(k)$ Ring Buffer)

**Context and Theory:**

Recursive-descent parsers frequently require looking ahead by $k$ tokens
($k \ge 2$) to disambiguate non-terminal production rules without
mutating the state of the scanner. For example, distinguishing between:

- A parenthesised expression: `(x + 1)`
- A lambda parameter list: `(x: Int, y: Int): Int => ...`

Reading the entire source file into an array of tokens consumes memory
proportional to the size of the program ($O(N)$). Industrial compilers
use a sliding-window **Circular Ring Buffer** of size $K$. The scanner
buffers at most $K$ lookahead tokens on-demand, running in $O(K)$ space.

<figure id="fig:circular-buffer">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSIzODQuMTIwMDMi
IGhlaWdodD0iODcuODA4IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDM4NC4xMjAwMyA4
Ny44MDgiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTog
MTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsK
fQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVk
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5v
ZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7
Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1h
cmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmln
X29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJs
KCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0i
YXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+
CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJl
Zlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFy
dC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48
L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0i
MCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0i
NyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0i
NCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJr
ZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBt
YXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJz
ZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48
L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2
aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2Vy
SGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9
IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2Vy
PgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0i
Mzg0LjEyMDAzIiBoZWlnaHQ9Ijg3LjgwOCI+PC9yZWN0PgogIDxsaW5lIHgxPSIxMzYuMTg4IiB5
MT0iMjQuNDQ0IiB4Mj0iMTM2LjE4OCIgeTI9IjQ1LjM5NiIgY2xhc3M9InNvbGlkIj48L2xpbmU+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS43NDYiIHk9IjEwLjQ3NiIgPlsgVG9r
ZW4gMyBdIFsgVG9rZW4gMCAoSGVhZCkgXSBbIFRva2VuIDEgXSBbIFRva2VuIDIgXTwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Mi41MzgiIHk9IjY2LjM0OCIgPnBlZWtf
dG9rZW4oMCk8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 8: Circular Ring Buffer (K = 4, Head = 1, Size =
3)</figcaption>
</figure>

**Specification:**

Implement an $LL(k)$ `LookaheadLexer` container class wrapping the basic
Faberge `Lexer`. The class must maintain a fixed-size ring buffer using
`std::array<Token, K>`, supporting `peek_token(size_t distance)` and
`consume_token()`.

``` cpp
template <size_t K = 4>
class LookaheadLexer {
public:
    explicit LookaheadLexer(Lexer lexer) : scanner_(std::move(lexer)), head_(0), count_(0) {
        for (size_t i = 0; i < K; ++i) fetch_next();
    }
    const Token& peek_token(size_t distance = 0) const {
        if (distance >= K) throw std::out_of_range("Lookahead distance exceeds K.");
        return buffer_[(head_ + distance) % K];
    }
    Token consume_token() {
        Token front = buffer_[head_];
        head_ = (head_ + 1) % K;
        fetch_next();
        return front;
    }
private:
    Lexer scanner_;
    std::array<Token, K> buffer_;
    size_t head_{0}, count_{0};
    void fetch_next() { buffer_[(head_ + count_) % K] = scanner_.scan_token(); }
};
```

**Verification Tasks:**

1.  **Lookahead Window Invariance:** Scan a stream of distinct
    arithmetic tokens `a + b * c`. Verify that `peek_token(0)` reads
    `a`, `peek_token(1)` reads `+`, and `peek_token(2)` reads `b`
    without advancing the underlying parser cursor.
2.  **Buffer Wrap-Around Execution:** Consume $500$ sequential tokens
    through a ring buffer of capacity $K = 4$. Verify that ring-buffer
    indices wrap cleanly around zero and that all tokens match their
    expected values.
3.  **End-of-Stream Saturation:** Advance the lookahead window past the
    final statement of a source script. Verify that all subsequent
    lookahead queries return `TokenType::EndOfFile` without buffer
    underflow or out-of-bounds memory access.

#### Exercise 18.5: String Interpolation and Lexer State Stacks

**Context and Theory:**

Modern high-level languages provide interpolated string templates:

```
let greeting = f"Hello, {user.name}! Your balance is: {account.balance * 1.20}";
```

Scanning interpolated strings cannot be achieved using a single static
DFA because string templates contain embedded, arbitrarily complex
source expressions that may themselves contain strings, lambdas, or
further nested interpolations.

To handle this context-dependent syntax, the scanner must operate as a
**Pushdown Lexical Automaton** maintaining a **Mode Stack**:

- `Mode::Standard`: Scanning normal statements and operators.
- `Mode::StringLiteral`: Consuming raw string characters until an
  unescaped `{` is reached.
- `Mode::Interpolation`: Pushed upon encountering `{`; switches back to
  standard tokenisation. A brace-depth counter tracks nested code blocks
  until a matching closing `}` pops the mode back to
  `Mode::StringLiteral`.

<figure id="fig:scan-interpolated">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIxNzNwdCIgaGVpZ2h0PSIzMjBwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAxNzIuODMgMzIwLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDMx
NikiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gc3RhbmRhcmQgLS0+CjxnIGlkPSJub2RlMSIgY2xh
c3M9Im5vZGUiPgo8dGl0bGU+c3RhbmRhcmQ8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBz
dHJva2U9ImJsYWNrIiBjeD0iODIuNDEiIGN5PSItMjk0IiByeD0iMzcuODUiIHJ5PSIxOCIvPgo8
dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI4Mi40MSIgeT0iLTI5MS41IiBmb250LWZhbWls
eT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPltTdGFuZGFy
ZF08L3RleHQ+CjwvZz4KPCEtLSBwdXNoX2xpdGVyYWwgLS0+CjxnIGlkPSJub2RlMiIgY2xhc3M9
Im5vZGUiPgo8dGl0bGU+cHVzaF9saXRlcmFsPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIg
c3Ryb2tlPSJibGFjayIgY3g9IjgyLjQxIiBjeT0iLTIyNSIgcng9Ijc4Ljc2IiByeT0iMTgiLz4K
PHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iODIuNDEiIHk9Ii0yMjIuNSIgZm9udC1mYW1p
bHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5QdXNoKE1v
ZGU6OlN0cmluZ0xpdGVyYWwpPC90ZXh0Pgo8L2c+CjwhLS0gc3RhbmRhcmQmIzQ1OyZndDtwdXNo
X2xpdGVyYWwgLS0+CjxnIGlkPSJlZGdlMSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+c3RhbmRhcmQm
IzQ1OyZndDtwdXNoX2xpdGVyYWw8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBkPSJNODIuNDEsLTI3NS43NUM4Mi40MSwtMjY5IDgyLjQxLC0yNjEuMDkgODIuNDEsLTI1
My41OSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSI4NS45
MSwtMjUzLjQzIDgyLjQxLC0yNDMuNDMgNzguOTEsLTI1My40MyA4NS45MSwtMjUzLjQzIi8+Cjx0
ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iODIuNDEiIHk9Ii0yNTciIGZvbnQtZmFtaWx5PSJJ
b3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+ICYjMTYwOyYjMzk7
ZiZxdW90OyYjMzk7PC90ZXh0Pgo8L2c+CjwhLS0gcHVzaF9pbnRlcnBvbGF0aW9uIC0tPgo8ZyBp
ZD0ibm9kZTMiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnB1c2hfaW50ZXJwb2xhdGlvbjwvdGl0bGU+
CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSI4Mi40MSIgY3k9Ii0xNTYi
IHJ4PSI4Mi4zMyIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjgyLjQx
IiB5PSItMTUzLjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9u
dC1zaXplPSIxMC4wMCI+UHVzaChNb2RlOjpJbnRlcnBvbGF0aW9uKTwvdGV4dD4KPC9nPgo8IS0t
IHB1c2hfbGl0ZXJhbCYjNDU7Jmd0O3B1c2hfaW50ZXJwb2xhdGlvbiAtLT4KPGcgaWQ9ImVkZ2Uy
IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5wdXNoX2xpdGVyYWwmIzQ1OyZndDtwdXNoX2ludGVycG9s
YXRpb248L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNODIuNDEs
LTIwNi43NUM4Mi40MSwtMjAwIDgyLjQxLC0xOTIuMDkgODIuNDEsLTE4NC41OSIvPgo8cG9seWdv
biBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSI4NS45MSwtMTg0LjQzIDgyLjQx
LC0xNzQuNDMgNzguOTEsLTE4NC40MyA4NS45MSwtMTg0LjQzIi8+Cjx0ZXh0IHRleHQtYW5jaG9y
PSJzdGFydCIgeD0iODIuNDEiIHk9Ii0xODgiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBN
b25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+ICYjMTYwOyYjMzk7eyYjMzk7PC90ZXh0Pgo8
L2c+CjwhLS0gcG9wX2ludGVycG9sYXRpb24gLS0+CjxnIGlkPSJub2RlNCIgY2xhc3M9Im5vZGUi
Pgo8dGl0bGU+cG9wX2ludGVycG9sYXRpb248L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBz
dHJva2U9ImJsYWNrIiBjeD0iODIuNDEiIGN5PSItODciIHJ4PSI2NS44NiIgcnk9IjE4Ii8+Cjx0
ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjgyLjQxIiB5PSItODQuNSIgZm9udC1mYW1pbHk9
Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5bTW9kZTo6U3Ry
aW5nTGl0ZXJhbF08L3RleHQ+CjwvZz4KPCEtLSBwdXNoX2ludGVycG9sYXRpb24mIzQ1OyZndDtw
b3BfaW50ZXJwb2xhdGlvbiAtLT4KPGcgaWQ9ImVkZ2UzIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5w
dXNoX2ludGVycG9sYXRpb24mIzQ1OyZndDtwb3BfaW50ZXJwb2xhdGlvbjwvdGl0bGU+CjxwYXRo
IGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik04Mi40MSwtMTM3Ljc1QzgyLjQxLC0xMzEg
ODIuNDEsLTEyMy4wOSA4Mi40MSwtMTE1LjU5Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJv
a2U9ImJsYWNrIiBwb2ludHM9Ijg1LjkxLC0xMTUuNDMgODIuNDEsLTEwNS40MyA3OC45MSwtMTE1
LjQzIDg1LjkxLC0xMTUuNDMiLz4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSI4Mi40MSIg
eT0iLTExOSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNp
emU9IjEwLjAwIj4gJiMxNjA7UG9wKCkgLi4gJiMzOTt9JiMzOTs8L3RleHQ+CjwvZz4KPCEtLSBz
dGFuZGFyZDIgLS0+CjxnIGlkPSJub2RlNSIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+c3RhbmRhcmQy
PC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjgyLjQxIiBj
eT0iLTE4IiByeD0iMzcuODUiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4
PSI4Mi40MSIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1
bSIgZm9udC1zaXplPSIxMC4wMCI+W1N0YW5kYXJkXTwvdGV4dD4KPC9nPgo8IS0tIHBvcF9pbnRl
cnBvbGF0aW9uJiM0NTsmZ3Q7c3RhbmRhcmQyIC0tPgo8ZyBpZD0iZWRnZTQiIGNsYXNzPSJlZGdl
Ij4KPHRpdGxlPnBvcF9pbnRlcnBvbGF0aW9uJiM0NTsmZ3Q7c3RhbmRhcmQyPC90aXRsZT4KPHBh
dGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTgyLjQxLC02OC43NUM4Mi40MSwtNjIg
ODIuNDEsLTU0LjA5IDgyLjQxLC00Ni41OSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tl
PSJibGFjayIgcG9pbnRzPSI4NS45MSwtNDYuNDMgODIuNDEsLTM2LjQzIDc4LjkxLC00Ni40MyA4
NS45MSwtNDYuNDMiLz4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSI4Mi40MSIgeT0iLTUw
IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAu
MDAiPiAmIzE2MDtQb3AoKTwvdGV4dD4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 9: State Transitions for Interpolated String
Scanning</figcaption>
</figure>

**Specification:**

Extend the Faberge `Lexer` with a mode stack (`std::vector<LexerMode>`)
and brace-depth counter. When scanning an interpolated string, emit
distinct structural tokens: `TokenType::InterpolationStart`,
`TokenType::StringChunk`, `TokenType::InterpolationExprOpen`, and
`TokenType::InterpolationExprClose`.

``` cpp
enum class LexerMode { Standard, InString, InInterpolation };

class StatefulLexer {
public:
    Token scan_template_token() {
        if (current_mode() == LexerMode::InString) {
            return scan_string_chunk();
        }
        return scan_standard_token();
    }
private:
    std::vector<LexerMode> mode_stack_{LexerMode::Standard};
    std::vector<size_t> brace_depth_stack_;
    [[nodiscard]] LexerMode current_mode() const { return mode_stack_.back(); }
    void push_mode(LexerMode mode);
    void pop_mode();
    Token scan_string_chunk();
};
```

**Verification Tasks:**

1.  **Token Stream Decomposition:** Scan `f"Count: {x + 1}."`. Verify
    the emitted sequence: `InterpolationStart`,
    `StringChunk("Count: ")`, `InterpolationExprOpen`,
    `Identifier("x")`, `Plus`, `Literal_Integer("1")`,
    `InterpolationExprClose`, `StringChunk(".")`, `InterpolationEnd`.
2.  **Nested Block Disambiguation:** Scan an interpolation containing a
    nested block: `f"Value: { if a then "A" else "B" }"`. Verify that
    the mode stack tracks internal braces and string literals without
    prematurely closing the outer interpolation template.
3.  **Unterminated Interpolation Diagnostic:** Scan
    `f"Unfinished { x + 1"`. Verify that the scanner reports an unclosed
    interpolation expression with an accurate coordinate point pointing
    to the opening `{`.

#### Exercise 18.6: UTF-8 Multi-Byte Identifier Scanning (Unicode Scalar Values)

**Context and Theory:**

Mathematical and functional algorithms frequently benefit from Unicode
symbols for identifiers (e.g. $\lambda$, $\pi$, $\Sigma$, $\Delta$). The
Faberge language adheres to the Unicode Standard Annex #31 (UAX #31) for
identifiers:

$$\text{Identifier} = \text{XID\_Start} \cdot (\text{XID\_Continue})^*$$

Because source files are UTF-8 encoded byte streams, a single Unicode
code point spans between $1$ and $4$ contiguous bytes (see tbl. 1).

| Byte 1     | Byte 2     | Byte 3     | Byte 4     | Code Point Range            |
|---------------|---------------|---------------|---------------|---------------|
| `0xxxxxxx` | —          | —          | —          | `U+0000` – `U+007F` (ASCII) |
| `110xxxxx` | `10xxxxxx` | —          | —          | `U+0080` – `U+07FF`         |
| `1110xxxx` | `10xxxxxx` | `10xxxxxx` | —          | `U+0800` – `U+FFFF`         |
| `11110xxx` | `10xxxxxx` | `10xxxxxx` | `10xxxxxx` | `U+10000` – `U+10FFFF`      |

Table 1: Unicode UTF-8 Encoding {#tbl:unicode-utf8}

The scanner must decode multi-byte UTF-8 sequences into a 32-bit scalar
value (`char32_t`), classify its Unicode category, and reject ill-formed
sequences (such as surrogate halves or overlong encodings).

```
Raw Bytes: [ 0xCF, 0x80 ] -> Binary: 11001111 10000000
Decoded 32-bit Code Point: U+03C0 (Greek Small Letter Pi: 'π')
```

**Specification:**

Implement a UTF-8 decoding engine inside the Faberge lexer. Implement
`decode_next_codepoint()` and predicate functions checking whether a
decoded code point belongs to a permissible identifier class.

``` cpp
struct UnicodeScanner {
    static std::pair<char32_t, size_t> decode_utf8(std::string_view bytes, size_t cursor) {
        if (cursor >= bytes.size()) return {0, 0};
        uint8_t lead = static_cast<uint8_t>(bytes[cursor]);
        if (lead < 0x80) return {static_cast<char32_t>(lead), 1};
        if ((lead >> 5) == 0x6) {
            if (cursor + 1 >= bytes.size()) throw std::runtime_error("Truncated UTF-8");
            char32_t cp = ((lead & 0x1F) << 6) | (bytes[cursor + 1] & 0x3F);
            return {cp, 2};
        }
        // ... Handle 3-byte and 4-byte boundaries ...
        throw std::runtime_error("Invalid UTF-8 leading byte.");
    }
    static bool is_xid_start(char32_t cp);
    static bool is_xid_continue(char32_t cp);
};
```

**Verification Tasks:**

1.  **Mathematical Code Point Scanning:** Scan the string
    `let π = 3.14159; let Δ_total = 100;`. Verify that `π` and `Δ_total`
    are accepted as single valid `TokenType::Identifier` tokens.
2.  **Invalid Continuation Byte Rejection:** Inject a corrupted UTF-8
    sequence (`0xC3` followed by an ASCII space rather than a
    continuation byte `0x80..0xBF`). Verify that the scanner halts on
    the corrupted byte and logs a decoding error.
3.  **Overlong Encoding Guard:** Attempt to encode the ASCII character
    `'A'` (`0x41`) via the illegal overlong 2-byte sequence `0xC1 0x81`.
    Verify that the scanner rejects the overlong encoding as a malformed
    sequence.

#### Exercise 18.7: Pythonic Off-Side Rule (Synthesised Indent and Dedent Tokens)

**Context and Theory:**

Peter Landin’s **Off-Side Rule** underpins layout-sensitive programming
languages (e.g. Python, Haskell, F#). Rather than relying on explicit
delimiters (`{` and `}`), structural code blocks are determined by the
horizontal indentation depth of subsequent lines.

The scanner translates indentation changes into explicit structural
tokens:

1.  The scanner maintains an **Indentation Stack** of column numbers,
    initialised to `[0]`.
2.  When scanning non-whitespace characters on a new line:
    - If current indentation $> \text{stack.top()}$: push current
      indentation to stack, and emit `TokenType::Indent`.
    - If current indentation $== \text{stack.top()}$: emit no token.
    - If current indentation $< \text{stack.top()}$: pop stack until
      matching level is found. For every level popped, emit a
      `TokenType::Dedent`.
    - If current indentation does not match any existing level on the
      stack, raise an **Indentation Error**.
3.  At `EOF`, emit a `TokenType::Dedent` for each remaining indentation
    level greater than $0$.

<figure id="fig:indent-stack">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MTQuNTkyMDQi
IGhlaWdodD0iNzMuODQwMDA0IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDYxNC41OTIw
NCA3My44NDAwMDQiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2Jv
YiBjaXJjbGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNr
OwogIHN0cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0
eTogMTsKICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsK
fQoKLnN2Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZv
bnQtZmFtaWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQt
c2l6ZTogMTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJh
eTogODsKfQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdf
ZmlsbGVkIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdi
b2IgLm5vZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
YXJyb3cgewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfYXJyb3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9t
YXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsK
fQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Np
cmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6
IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUg
ewogIG1hcmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7
Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtl
ciBpZD0iYXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2Vy
V2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgog
ICAgICA8cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9t
YXJrZXI+CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9
IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0
by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQg
MCwyIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmll
d0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhl
aWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0
IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZ
PSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQt
cmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmls
bGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2ly
Y2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJj
bGUgY3g9IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwv
bWFya2VyPgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3
aWR0aD0iNjE0LjU5MjA0IiBoZWlnaHQ9IjczLjg0MDAwNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjEuNzQ2IiB5PSIxMC40NzYiID5sZXQ8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjkuNjgyMDAxIiB5PSIxMC40NzYiID5hPC90ZXh0Pgog
IDxsaW5lIHgxPSI0MS45MDQiIHkxPSI1LjIzOCIgeDI9IjQ4Ljg4OCIgeTI9IjUuMjM4IiBjbGFz
cz0ic29saWQiPjwvbGluZT4KICA8bGluZSB4MT0iNDEuOTA0IiB5MT0iOC43MzAwMDA1IiB4Mj0i
NDguODg4IiB5Mj0iOC43MzAwMDA1IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIyOS42ODIwMDEiIHk9IjI0LjQ0NCIgPmxldDwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1Ny42MTgiIHk9IjEwLjQ3NiIgPjEwPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjU3LjYxOCIgeT0iMjQuNDQ0IiA+Yjwv
dGV4dD4KICA8bGluZSB4MT0iNjkuODQwMDA0IiB5MT0iMTkuMjA2MDAxIiB4Mj0iNzYuODI0MDA1
IiB5Mj0iMTkuMjA2MDAxIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8bGluZSB4MT0iNjkuODQw
MDA0IiB5MT0iMjIuNjk4IiB4Mj0iNzYuODI0MDA1IiB5Mj0iMjIuNjk4IiBjbGFzcz0ic29saWQi
PjwvbGluZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1Ny42MTgiIHk9IjM4LjQx
MjAwMyIgPmxldDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1Ny42MTgi
IHk9IjUyLjM4IiA+NDA8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iODUu
NTU0IiB5PSIyNC40NDQiID4yMDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSI4NS41NTQiIHk9IjM4LjQxMjAwMyIgPmM8L3RleHQ+CiAgPGxpbmUgeDE9Ijk3Ljc3NiIgeTE9
IjMzLjE3NCIgeDI9IjEwNC43NiIgeTI9IjMzLjE3NCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAg
PGxpbmUgeDE9Ijk3Ljc3NiIgeTE9IjM2LjY2NiIgeDI9IjEwNC43NiIgeTI9IjM2LjY2NiIgY2xh
c3M9InNvbGlkIj48L2xpbmU+CiAgPGxpbmUgeDE9IjEzMi42OTYiIHkxPSIyMC45NTIiIHgyPSIx
NTMuNjQ4MDEiIHkyPSIyMC45NTIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBv
aW50cz0iMTUzLjY0ODAxLDE3LjQ2MDAwMSAxNjAuNjMyLDIwLjk1MiAxNTMuNjQ4MDEsMjQuNDQ0
IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjEzMi42OTYiIHkxPSIzNC45
MjAwMDIiIHgyPSIxNTMuNjQ4MDEiIHkyPSIzNC45MjAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5l
PgogIDxwb2x5Z29uIHBvaW50cz0iMTUzLjY0ODAxLDMxLjQyODAwMSAxNjAuNjMyLDM0LjkyMDAw
MiAxNTMuNjQ4MDEsMzguNDEyMDAzIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUg
eDE9IjEzMi42OTYiIHkxPSI0OC44ODgiIHgyPSIxNTMuNjQ4MDEiIHkyPSI0OC44ODgiIGNsYXNz
PSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMTUzLjY0ODAxLDQ1LjM5NiAxNjAu
NjMyLDQ4Ljg4OCAxNTMuNjQ4MDEsNTIuMzgiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMTMuNDkwMDA1IiB5PSIzOC40MTIwMDMiID4z
MDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjc0NiIgeT0iNTIuMzgi
ID5sZXQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjkuNjgyMDAxIiB5
PSI1Mi4zOCIgPmQ8L3RleHQ+CiAgPGxpbmUgeDE9IjQxLjkwNCIgeTE9IjQ3LjE0MjAwMiIgeDI9
IjQ4Ljg4OCIgeTI9IjQ3LjE0MjAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPGxpbmUgeDE9
IjQxLjkwNCIgeTE9IjUwLjYzNDAwMyIgeDI9IjQ4Ljg4OCIgeTI9IjUwLjYzNDAwMyIgY2xhc3M9
InNvbGlkIj48L2xpbmU+CiAgPGxpbmUgeDE9IjMxNC4yOCIgeTE9IjQ4Ljg4OCIgeDI9IjMyOC4y
NDgwMiIgeTI9IjQ4Ljg4OCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRz
PSIzMjguMjQ4MDIsNDUuMzk2IDMzNS4yMzIsNDguODg4IDMyOC4yNDgwMiw1Mi4zOCIgY2xhc3M9
ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSI0OTUuODY0IiB5MT0iNDguODg4IiB4Mj0i
NTA5LjgzMiIgeTI9IjQ4Ljg4OCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9p
bnRzPSI1MDkuODMyLDQ1LjM5NiA1MTYuODE2MDQsNDguODg4IDUwOS44MzIsNTIuMzgiIGNsYXNz
PSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNjku
MzYyIiB5PSIyNC40NDQiID5JbmRlbnQgJmd0OyAwOiBQdXNoKDQpLCBFbWl0KElOREVOVCk8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTY5LjM2MiIgeT0iMzguNDEyMDAz
IiA+SW5kZW50ICZndDsgNDogUHVzaCg4KSwgRW1pdChJTkRFTlQpPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjE2OS4zNjIiIHk9IjUyLjM4IiA+SW5kZW50ID09IDA6IFBv
cCg4KTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzNDMuOTYyIiB5PSI1
Mi4zOCIgPkVtaXQoREVERU5UKSwgUG9wKDQpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjUyNS41NDYiIHk9IjUyLjM4IiA+RW1pdChERURFTlQpPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 10: Indent Stack: [0]</figcaption>
</figure>

**Specification:**

Create an `IndentationFilter` class that sits between the raw character
scanner and the token consumer. It must track line-start states, compute
visual column depth (expanding tabs to 4 spaces), manage the indentation
stack, and inject synthetic `Indent` and `Dedent` tokens.

``` cpp
class IndentationFilter {
public:
    explicit IndentationFilter(Lexer scanner);
    Token next_token();
private:
    Lexer scanner_;
    std::vector<size_t> indent_stack_{0};
    std::queue<Token> pending_tokens_;
    bool at_line_start_{true};

    void process_line_start();
    size_t calculate_indentation();
};
```

**Verification Tasks:**

1.  **Nested Block Tracking:** Feed an indented script into the filter:

    ```
    let outer = fn (): Int =>
        let inner = 42
        inner
    ```

    Verify that the token stream contains an `Indent` token before
    `inner = 42`, and a `Dedent` token immediately preceding `EOF`.

2.  **Multi-Level Dedent Collapse:** Step out of three nested
    indentation levels simultaneously back to column 0. Verify that the
    filter produces exactly three consecutive `TokenType::Dedent`
    tokens.

3.  **Unaligned Indentation Rejection:** Indent a block by 4 spaces,
    then dedent the subsequent statement to 2 spaces. Verify that the
    filter raises an unaligned indentation error due to the absence of
    column 2 on the stack.

#### Exercise 18.8: SIMD-Accelerated Trivia and Whitespace Scanning

**Context and Theory:**

In massive software codebases, whitespace and comments account for
$20\%$ to $40\%$ of total source text. A standard scalar character
scanner inspects characters one-by-one in an iterative loop:

``` cpp
while (c == ' ' || c == '\t' || c == '\r' || c == '\n') advance();
```

This scalar loop triggers unpredictable branch instructions on dense
whitespace runs.

Using **SIMD (Single Instruction, Multiple Data)** intrinsics (such as
Intel AVX2 or ARM Neon), the compiler can process 32 bytes of source
text in a single CPU cycle (see fig. 11):

1.  Load 32 characters into a 256-bit vector register
    (`_mm256_loadu_si256`).
2.  Broadcast comparison masks for whitespace characters (`' '`, `\t`,
    `\r`, `\n`) using `_mm256_cmpeq_epi8`.
3.  Bitwise-OR the masks and convert the vector mask into a 32-bit
    integer scalar mask using `_mm256_movemask_epi8`.
4.  Compute the count of contiguous matching bytes in a single
    instruction using **Count Trailing Zeros** (`_tzcnt_u32` or
    `std::countr_zero`).

<figure id="fig:vector-register">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MjEuNTc2IiBo
ZWlnaHQ9IjU5Ljg3MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA2MjEuNTc2IDU5Ljg3
MiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI2MjEu
NTc2IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjEuNzQ2IiB5PSIxMC40NzYiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjE1LjcxNDAwMSIgeT0iMTAuNDc2IiA+JiMzOTs8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMjkuNjgyMDAxIiB5PSIxMC40NzYiID4mIzM5Oyw8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTAuNjM0MDAzIiB5PSIxMC40NzYiID4mIzM5
OzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2NC42MDIwMDUiIHk9IjEw
LjQ3NiIgPiYjMzk7LDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI4NS41
NTQiIHk9IjEwLjQ3NiIgPiYjMzk7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9Ijk5LjUyMiIgeT0iMTAuNDc2IiA+JiMzOTssPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjEyMC40NzQwMSIgeT0iMTAuNDc2IiA+JiMzOTs8L3RleHQ+CiAgPGxpbmUg
eDE9IjEyNS43MTIwMDYiIHkxPSIwIiB4Mj0iMTMyLjY5NiIgeTI9IjEzLjk2OCIgY2xhc3M9InNv
bGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTM0LjQ0MiIgeT0i
MTAuNDc2IiA+dCYjMzk7LDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIx
NjIuMzc4IiB5PSIxMC40NzYiID4mIzM5OzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIxNzYuMzQ2MDEiIHk9IjEwLjQ3NiIgPiYjMzk7LDwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIxOTcuMjk4IiB5PSIxMC40NzYiID4mIzM5OzwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMTEuMjY2IiB5PSIxMC40NzYiID4mIzM5Oyw8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjMyLjIxOCIgeT0iMTAuNDc2
IiA+JiMzOTtsJiMzOTssPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI2
Ny4xMzgiIHk9IjEwLjQ3NiIgPiYjMzk7ZSYjMzk7LDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIzMDIuMDU4IiB5PSIxMC40NzYiID4mIzM5O3QmIzM5Oyw8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzM2Ljk3OCIgeT0iMTAuNDc2IiA+Li4uPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuNzQ2IiB5PSIzOC40MTIwMDMi
ID5NYXNrPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM2LjY2NiIgeT0i
MzguNDEyMDAzIiA+TWF0Y2g6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
Ijg1LjU1NCIgeT0iMzguNDEyMDAzIiA+MTExMTEwMDAuLi48L3RleHQ+CiAgPGxpbmUgeDE9IjE2
Ny42MTYiIHkxPSIzNC45MjAwMDIiIHgyPSIxODEuNTg0IiB5Mj0iMzQuOTIwMDAyIiBjbGFzcz0i
c29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjE4MS41ODQsMzEuNDI4MDAxIDE4OC41
NjgwMSwzNC45MjAwMDIgMTgxLjU4NCwzOC40MTIwMDMiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdv
bj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxOTcuMjk4IiB5PSIzOC40MTIwMDMi
ID5UcmFpbGluZzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNjAuMTU0
MDIiIHk9IjM4LjQxMjAwMyIgPlplcm9zPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjMwMi4wNTgiIHk9IjM4LjQxMjAwMyIgPkNvdW50OjwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIzNTAuOTQ2IiB5PSIzOC40MTIwMDMiID42PC90ZXh0PgogIDxs
aW5lIHgxPSIzNjMuMTY4IiB5MT0iMzQuOTIwMDAyIiB4Mj0iMzc3LjEzNjAyIiB5Mj0iMzQuOTIw
MDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjM3Ny4xMzYwMiwz
MS40MjgwMDEgMzg0LjEyMDAzLDM0LjkyMDAwMiAzNzcuMTM2MDIsMzguNDEyMDAzIiBjbGFzcz0i
ZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzkyLjg1
IiB5PSIzOC40MTIwMDMiID5BZHZhbmNlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjQ0OC43MjIwMiIgeT0iMzguNDEyMDAzIiA+Y3Vyc29yPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjQ5Ny42MTAwMiIgeT0iMzguNDEyMDAzIiA+Ynk8L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTE4LjU2MiIgeT0iMzguNDEyMDAzIiA+
NjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1MzIuNTMiIHk9IjM4LjQx
MjAwMyIgPmltbWVkaWF0ZWx5ITwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIzNjQuOTE0IiB5PSIxMC40NzYiID4oMzIgY2hhcnMpIF08L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 11: 256-bit Vector Register</figcaption>
</figure>

**Specification:**

Implement an AVX2-accelerated whitespace skipping routine,
`skip_whitespace_simd()`, with automatic scalar fallback for buffer
tails smaller than 32 bytes.

``` cpp
#include <immintrin.h>

class FastTriviaScanner {
public:
    static size_t skip_spaces_avx2(std::string_view source, size_t cursor) {
        const __m256i space_mask = _mm256_set1_epi8(' ');
        const __m256i tab_mask   = _mm256_set1_epi8('\t');

        while (cursor + 32 <= source.size()) {
            __m256i chunk = _mm256_loadu_si256(reinterpret_cast<const __m256i*>(source.data() + cursor));
            __m256i is_space = _mm256_or_si256(_mm256_cmpeq_epi8(chunk, space_mask),
                                               _mm256_cmpeq_epi8(chunk, tab_mask));
            uint32_t mask = ~static_cast<uint32_t>(_mm256_movemask_epi8(is_space));
            if (mask == 0) { cursor += 32; continue; }
            return cursor + std::countr_zero(mask);
        }
        while (cursor < source.size() && (source[cursor] == ' ' || source[cursor] == '\t')) ++cursor;
        return cursor;
    }
};
```

**Verification Tasks:**

1.  **Dense Whitespace Acceleration:** Feed an artificial buffer
    containing $128$ spaces followed by the token `let`. Verify that the
    SIMD scanner advances the cursor precisely to the index of `'l'`
    without overshooting.
2.  **Unaligned Boundary Safety:** Test a buffer of 35 characters
    containing 33 spaces followed by `fn`. Verify that the routine
    transitions from vector skipping to scalar tail fallback seamlessly.
3.  **Line Tracking Consistency:** Combine horizontal whitespace
    skipping with newline tracking. Ensure line and column coordinates
    remain synchronized with the advanced cursor.

#### Exercise 18.9: Context-Aware Lexer Disambiguation (Nested Templates vs Shift Operators)

**Context and Theory:**

The Principle of Maximal Munch dictates that the scanner must always
match the longest valid token prefix. This rule creates a notorious
grammatical ambiguity in modern languages supporting generic or template
parameters:

```
let matrix: Vector<Vector<Int>> = create();
```

When scanning the character sequence `>>`:

- Under standard Maximal Munch, the scanner greedily emits a single
  bitwise right-shift operator: `TokenType::GreaterGreater` (`>>`).
- At this position in the grammar, the program actually requires two
  distinct closing template delimiters: `TokenType::Greater` (`>`)
  followed by `TokenType::Greater` (`>`).

If the scanner cannot adapt its matching strategy based on syntactic
context, the downstream parser is forced to implement convoluted token
splitting or lookahead hacks.

<figure id="fig:ambiguity-scanning">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MDAuNjI0IiBo
ZWlnaHQ9Ijg3LjgwOCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA2MDAuNjI0IDg3Ljgw
OCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI2MDAu
NjI0IiBoZWlnaHQ9Ijg3LjgwOCI+PC9yZWN0PgogIDxsaW5lIHgxPSI0MDUuMDcyMDIiIHkxPSIz
NC45MjAwMDIiIHgyPSI0MTkuMDQiIHkyPSIzNC45MjAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5l
PgogIDxwb2x5Z29uIHBvaW50cz0iNDE5LjA0LDMxLjQyODAwMSA0MjYuMDI0MDIsMzQuOTIwMDAy
IDQxOS4wNCwzOC40MTIwMDMiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSI0MzQuNzU0IiB5PSIzOC40MTIwMDMiID5TWU5UQVg8L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDgzLjY0MjAzIiB5PSIzOC40MTIwMDMi
ID5FUlJPUjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1MjUuNTQ2IiB5
PSIzOC40MTIwMDMiID5pbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1
NDYuNDk4IiB5PSIzOC40MTIwMDMiID5QYXJzZXIhPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjEuNzQ2IiB5PSIxMC40NzYiID5TdHJlYW06IC4uLiAmIzM5O0ludCZndDsm
Z3Q7JiMzOTs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS43NDYiIHk9
IjM4LjQxMjAwMyIgPk1heGltYWwgTXVuY2g6ICBbSWRlbnRpZmllcigmIzM5O0ludCYjMzk7KV0g
W1Nock9wZXJhdG9yKCYjMzk7Jmd0OyZndDsmIzM5OyldPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjEuNzQ2IiB5PSI2Ni4zNDgiID5Db250ZXh0LUF3YXJlOiAgW0lkZW50
aWZpZXIoJiMzOTtJbnQmIzM5OyldIFtUZW1wbGF0ZUNsb3NlKCYjMzk7Jmd0OyYjMzk7KV0gW1Rl
bXBsYXRlQ2xvc2UoJiMzOTsmZ3Q7JiMzOTspXTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 12: Ambiguity-Aware Scanning</figcaption>
</figure>

**Specification:**

Enhance the Faberge `Lexer` with a **Contextual Disambiguation
Channel**. The lexer exposes a state flag, `in_template_context_depth_`,
which can be incremented or decremented by the parser. When this depth
is greater than zero, the lexer disables the multi-character `>>` token
and instead emits individual `>` tokens.

``` cpp
class DisambiguatedLexer {
public:
    void enter_template_context() noexcept { ++template_depth_; }
    void exit_template_context() noexcept { if (template_depth_ > 0) --template_depth_; }

    Token scan_greater_symbol() {
        if (template_depth_ > 0) {
            return make_token(TokenType::Greater); // Disallow greedy '>>' match
        }
        if (match('>')) {
            return make_token(TokenType::GreaterGreater);
        }
        return make_token(TokenType::Greater);
    }
private:
    size_t template_depth_{0};
};
```

**Verification Tasks:**

1.  **Bitwise Shift Preservation:** In standard context
    (`template_depth_ == 0`), scan `x >> 2`. Verify that the emitted
    sequence contains the single token `TokenType::GreaterGreater`.
2.  **Template Delimiter Splitting:** Set `template_depth_ = 2`. Scan
    `Vector<Vector<Int>>`. Verify that the sequence terminates with two
    separate `TokenType::Greater` tokens, each possessing its own
    accurate source column coordinate.
3.  **Depth Re-entry:** Increment template depth to $3$, decrement twice
    to $1$, and scan `>`. Verify that it emits a single
    `TokenType::Greater`. Decrement to $0$ and scan `>>`; verify it
    emits `TokenType::GreaterGreater`.

#### Exercise 18.10: Single-Pass Lexical Comment Pruning and Source Minifier

**Context and Theory:**

Prior to transmission over networks, embedding source text into
binaries, or generating target C++ code from Faberge source, compilers
often run a **Source Minification Pass**.

Minification strips all single-line and multi-line comments, eliminates
redundant whitespace, and compacts delimiters while preserving program
semantics:

1.  Comments must be completely erased.
2.  Contiguous whitespace must be collapsed to a single space, or
    removed entirely if adjacent to non-alphanumeric punctuation
    (e.g. `{ a + b }` compresses to `{a+b}`).
3.  Whitespace **cannot** be removed between alphanumeric identifiers or
    keywords: `let a = 10` cannot compress to `leta=10`.
4.  String literals must be preserved byte-for-byte, including internal
    spaces, escape sequences, and formatting.

**Original:**

```
     // Compute total
    let   result   =   compute ( 42 ) ;
```

**Minified:**

```
    let result=compute(42);
```

**Specification:**

Implement a single-pass `SourceMinifier` class that consumes Faberge
tokens from the `Lexer` and writes a minified source string to an output
stream, enforcing whitespace boundaries between adjacent alphanumeric
identifiers.

``` cpp
class SourceMinifier {
public:
    static std::string minify(std::span<const Token> tokens) {
        std::string out;
        bool last_was_word = false;

        for (const auto& tok : tokens) {
            if (tok.is(TokenType::EndOfFile)) break;
            bool current_is_word = is_word_token(tok.type);
            if (last_was_word && current_is_word) {
                out.push_back(' '); // Mandatory space separating keywords/identifiers
            }
            out.append(tok.lexeme);
            last_was_word = current_is_word;
        }
        return out;
    }
private:
    static bool is_word_token(TokenType t);
};
```

**Verification Tasks:**

1.  **Trivia Pruning:** Minify a source file containing heavy
    single-line and nested multi-line comments. Verify that zero comment
    text appears in the emitted string.
2.  **Syntactic Boundary Preservation:** Minify
    `let count = 42; let rate = 1.5;`. Verify that the output is exactly
    `let count=42;let rate=1.5;` and that re-tokenising this string
    yields the identical token types as the original.
3.  **String Literal Preservation:** Minify
    `let msg = "Hello   World!  // Not a comment";`. Verify that
    whitespace and comment-like characters inside the string literal
    remain unmodified.

#### Exercise 18.11: Custom Diagnostic Fix-It Hints and Edit Suggestions

**Context and Theory:**

Professional compilers (e.g. Clang, GCC, rustc) go beyond identifying
syntax errors: they suggest concrete textual replacements called
**Fix-It Hints**.

When the scanner encounters an invalid character, an unrecognised
keyword typo, or a common cross-language syntax mistake, it provides an
edit suggestion:

- Typing a single `&` where Faberge requires `&&`.
- Common keyword typos (e.g. `ltt` instead of `let`, `fnction` instead
  of `fn`).

Typo suggestion algorithms calculate the **Levenshtein Distance**
$D(s_1, s_2)$ between the scanned token and all reserved keywords:

$$D(s_1, s_2) = \text{minimum number of single-character edits (insertions, deletions, substitutions)}$$

If $D \le 1$ (or $D \le 2$ for longer identifiers), the diagnostic
engine flags the typo and emits a structured fix-it suggestion with
insertion and deletion coordinates.

```
 error: unrecognised keyword 'ltt'
   --> main.fab:3:5
  3 |     ltt count = 10;
    |     ^^^
    |     help: did you mean 'let'?
```

**Specification:**

Extend the `DiagnosticEngine` with a `FixIt` data structure. Add a
keyword spellchecker to the Faberge `Lexer` using the Levenshtein
distance algorithm, automatically attaching fix-it hints to error
tokens.

``` cpp
struct FixItHint {
    SourceLocation location;
    std::string replacement_text;
    std::string explanation;
};

class IntelligentDiagnosticEngine {
public:
    static void report_error_with_fixit(std::string_view source, 
                                        SourceLocation loc, 
                                        std::string_view message, 
                                        const FixItHint& hint);
    static size_t compute_levenshtein(std::string_view a, std::string_view b);
    static std::optional<std::string_view> find_closest_keyword(std::string_view typo);
};
```

**Verification Tasks:**

1.  **Keyword Typo Suggestion:** Scan `ltt count = 10;`. Verify that the
    error reporter pinpoints `ltt` and outputs a diagnostic suggesting
    `let`.
2.  **Symbolic Error Correction:** Scan `if a & b then`. Verify that the
    scanner detects the isolated `&`, suggests the replacement `&&`, and
    emits the appropriate fix-it coordinate span.
3.  **Distance Threshold Accuracy:** Scan `supercalifragilistic`. Verify
    that the spellchecker does not match against short keywords like
    `fn` or `let` due to distance threshold clamping.

#### Exercise 18.12: Adversarial Grammar Fuzzing and Crash Resistance Engine

**Context and Theory:**

Compilers are vulnerable to **Denial of Service (DoS)** attacks, buffer
overruns, and undefined behaviour when fed maliciously crafted or
malformed input streams:

- Deeply nested comments: `/* /* /* ... */ */ */` leading to stack
  overflow in recursive scanners.
- Unterminated string literals spanning billions of bytes.
- Source streams saturated with embedded null bytes (`\0`), control
  characters, and illegal non-UTF-8 byte sequences.

A robust compiler front-end must maintain **Crash Invariance**: no input
stream, regardless of corruption, entropy, or structure, may cause the
scanner to invoke undefined behaviour, trigger a memory segmentation
fault, or enter an infinite loop (see fig. 13).

<figure id="fig:front-end-robustness">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIxODlwdCIgaGVpZ2h0PSIyMzJwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAxODkuMDIgMjMxLjk3IiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIy
Ny45NykiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gaW5wdXQgLS0+CjxnIGlkPSJub2RlMSIgY2xh
c3M9Im5vZGUiPgo8dGl0bGU+aW5wdXQ8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJv
a2U9ImJsYWNrIiBjeD0iOTAuNTEiIGN5PSItMTg3LjIiIHJ4PSI5MC41MiIgcnk9IjM2LjU0Ii8+
Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjkwLjUxIiB5PSItMjAxLjIiIGZvbnQtZmFt
aWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+UmFuZG9t
aXplZCAvPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI5MC41MSIgeT0iLTE5
MC4yIiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0i
MTAuMDAiPkNvcnJ1cHRlZCBJbnB1dCBTdHJlYW08L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjkwLjUxIiB5PSItMTc5LjIiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBN
b25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+KE51bGwgYnl0ZXMsIHVuY2xvc2VkPC90ZXh0
Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI5MC41MSIgeT0iLTE2OC4yIiBmb250LWZh
bWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPmVzY2Fw
ZXMsIG92ZXJmbG93cyk8L3RleHQ+CjwvZz4KPCEtLSBsZXhlciAtLT4KPGcgaWQ9Im5vZGUyIiBj
bGFzcz0ibm9kZSI+Cjx0aXRsZT5sZXhlcjwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0
cm9rZT0iYmxhY2siIGN4PSI5MC41MSIgY3k9Ii05Ni40MyIgcng9IjU1LjE3IiByeT0iMTgiLz4K
PHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iOTAuNTEiIHk9Ii05My45MyIgZm9udC1mYW1p
bHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5bIEZhYmVy
Z2UgTGV4ZXIgXTwvdGV4dD4KPC9nPgo8IS0tIGlucHV0JiM0NTsmZ3Q7bGV4ZXIgLS0+CjxnIGlk
PSJlZGdlMSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+aW5wdXQmIzQ1OyZndDtsZXhlcjwvdGl0bGU+
CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik05MC41MSwtMTUwLjM1QzkwLjUx
LC0xNDEuNzggOTAuNTEsLTEzMi43NyA5MC41MSwtMTI0LjY3Ii8+Cjxwb2x5Z29uIGZpbGw9ImJs
YWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9Ijk0LjAxLC0xMjQuNjUgOTAuNTEsLTExNC42NSA4
Ny4wMSwtMTI0LjY1IDk0LjAxLC0xMjQuNjUiLz4KPC9nPgo8IS0tIG91dHB1dCAtLT4KPGcgaWQ9
Im5vZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5vdXRwdXQ8L3RpdGxlPgo8ZWxsaXBzZSBmaWxs
PSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iOTAuNTEiIGN5PSItMjEuMjEiIHJ4PSI4Ni45NSIg
cnk9IjIxLjQzIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjkwLjUxIiB5PSItMjQu
MjEiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIx
MC4wMCI+R3JhY2VmdWwgRXJyb3JzIE9ubHk8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRk
bGUiIHg9IjkwLjUxIiB5PSItMTMuMjEiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25v
IE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+KFplcm8gY3Jhc2hlcywgemVybyBVQik8L3RleHQ+
CjwvZz4KPCEtLSBsZXhlciYjNDU7Jmd0O291dHB1dCAtLT4KPGcgaWQ9ImVkZ2UyIiBjbGFzcz0i
ZWRnZSI+Cjx0aXRsZT5sZXhlciYjNDU7Jmd0O291dHB1dDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5v
bmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik05MC41MSwtNzguMDhDOTAuNTEsLTcwLjUgOTAuNTEsLTYx
LjM5IDkwLjUxLC01Mi43NCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIg
cG9pbnRzPSI5NC4wMSwtNTIuNTYgOTAuNTEsLTQyLjU2IDg3LjAxLC01Mi41NiA5NC4wMSwtNTIu
NTYiLz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 13: Avoiding Undefined Behaviour in
Front-End</figcaption>
</figure>

**Specification:**

Implement an automated **Fuzzing Harness** for the Faberge `Lexer`. The
harness must generate millions of randomised byte streams (incorporating
random mutations, ASCII control bytes, and deep recursive comment
nesting) and feed them to the lexer, asserting that the scanner always
halts cleanly within a bounded execution time.

``` cpp
class LexerFuzzHarness {
public:
    struct FuzzResult {
        size_t iterations{0};
        size_t errors_caught{0};
        bool survived{true};
    };
    static FuzzResult run_fuzz_campaign(size_t iteration_count, size_t max_buffer_size) {
        FuzzResult res;
        std::mt19937_64 rng(1337);
        for (size_t i = 0; i < iteration_count; ++i) {
            std::string payload = generate_adversarial_payload(rng, max_buffer_size);
            Lexer lexer(payload, "fuzz_input.fab");
            auto tokens = lexer.tokenise(); // Must NEVER crash or hang
            res.errors_caught += lexer.error_count();
            ++res.iterations;
        }
        return res;
    }
private:
    static std::string generate_adversarial_payload(std::mt19937_64& rng, size_t max_len);
};
```

**Verification Tasks:**

1.  **Null-Byte Infiltration:** Feed a $10\text{ KB}$ buffer packed with
    alternating null bytes (`\0`), valid Faberge keywords, and unclosed
    quotes. Verify that the scanner terminates at the end of the buffer
    without freezing or crashing.
2.  **Deep Comment Nesting:** Generate a payload containing $50,000$
    opening block comment delimiters `/*`. Verify that the scanner
    terminates with an unclosed comment error rather than crashing via
    recursion-induced stack overflow.
3.  **Large Random Fuzz Run:** Execute a campaign of $10,000$ randomised
    adversarial byte arrays through `run_fuzz_campaign()`. Verify that
    $100\%$ of test cases complete with clean error reporting and zero
    memory faults.

# Chapter 19: Parser

> *“The most important asset of a software engineer is the ability to
> structure complex systems hierarchically. Top-down parsing reflects
> this principle with crystalline clarity: from a single root
> proposition, an entire universe of meaning unfolds.”*  
> — **Niklaus Wirth**, *Algorithms + Data Structures = Programs* (1976)

------------------------------------------------------------------------

### 19.1 From Linear Sequences to Hierarchical Trees

In **Chapter 18**, we constructed the compiler’s lexical front-end. Our
scanner consumed raw source text and produced a flat, linear stream of
typed tokens:

    [ Keyword_Let, Identifier("x"), Equal, Literal_Int(10), Plus, Literal_Int(20), Semicolon ]

While this linear stream classifies individual words and symbols, it
possesses no structural understanding. It does not know that `10 + 20`
forms a single arithmetic expression, that `+` binds tighter than
assignment, or that nested curly braces `{ ... }` delineate lexical
scopes.

The task of **Syntactic Analysis (Parsing)** is to transform this flat
token sequence into a rich, recursive graph called the **Abstract Syntax
Tree (AST)**.

<figure id="fig:concrete-abstract">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI1MDVwdCIgaGVpZ2h0PSIyMThwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCA1MDQuNTAgMjE4LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIx
NCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gYXNzaWdubWVudCAtLT4KPGcgaWQ9Im5vZGUxIiBj
bGFzcz0ibm9kZSI+Cjx0aXRsZT5hc3NpZ25tZW50PC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9u
ZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxNDAuNSwtMjEwIDU3LjUsLTIxMCA1Ny41LC0xNzQg
MTQwLjUsLTE3NCAxNDAuNSwtMjEwIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9Ijk5
IiB5PSItMTg5LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9u
dC1zaXplPSIxMC4wMCI+Jmx0O0Fzc2lnbm1lbnQmZ3Q7PC90ZXh0Pgo8L2c+CjwhLS0gaWQgLS0+
CjxnIGlkPSJub2RlMiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+aWQ8L3RpdGxlPgo8cG9seWdvbiBm
aWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjU0LC0xNTIgMCwtMTUyIDAsLTExNiA1
NCwtMTE2IDU0LC0xNTIiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjciIHk9Ii0x
MzEuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9
IjEwLjAwIj5JRCgmcXVvdDt4JnF1b3Q7KTwvdGV4dD4KPC9nPgo8IS0tIGFzc2lnbm1lbnQmIzQ1
OyZndDtpZCAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5hc3NpZ25tZW50
JiM0NTsmZ3Q7aWQ8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJN
NzcuMDEsLTE3My44OUM3MC43MywtMTY5LjAxIDYzLjc3LC0xNjMuNiA1Ny4wOSwtMTU4LjQxIi8+
Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjU4LjksLTE1NS4z
OCA0OC44NiwtMTUyIDU0LjYsLTE2MC45IDU4LjksLTE1NS4zOCIvPgo8L2c+CjwhLS0gZXF1YWwg
LS0+CjxnIGlkPSJub2RlMyIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+ZXF1YWw8L3RpdGxlPgo8cG9s
eWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjEyNiwtMTUyIDcyLC0xNTIg
NzIsLTExNiAxMjYsLTExNiAxMjYsLTE1MiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4
PSI5OSIgeT0iLTEzMS41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0i
IGZvbnQtc2l6ZT0iMTAuMDAiPiYjMzk7PSYjMzk7PC90ZXh0Pgo8L2c+CjwhLS0gYXNzaWdubWVu
dCYjNDU7Jmd0O2VxdWFsIC0tPgo8ZyBpZD0iZWRnZTIiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmFz
c2lnbm1lbnQmIzQ1OyZndDtlcXVhbDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGQ9Ik05OSwtMTczLjg5Qzk5LC0xNzAuMiA5OSwtMTY2LjIxIDk5LC0xNjIuMjQiLz4K
PHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTAyLjUsLTE2MiA5
OSwtMTUyIDk1LjUsLTE2MiAxMDIuNSwtMTYyIi8+CjwvZz4KPCEtLSBleHByIC0tPgo8ZyBpZD0i
bm9kZTQiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmV4cHI8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJu
b25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE5OCwtMTUyIDE0NCwtMTUyIDE0NCwtMTE2IDE5
OCwtMTE2IDE5OCwtMTUyIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE3MSIgeT0i
LTEzMS41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6
ZT0iMTAuMDAiPiZsdDtFeHByJmd0OzwvdGV4dD4KPC9nPgo8IS0tIGFzc2lnbm1lbnQmIzQ1OyZn
dDtleHByIC0tPgo8ZyBpZD0iZWRnZTMiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmFzc2lnbm1lbnQm
IzQ1OyZndDtleHByPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0i
TTEyMC45OSwtMTczLjg5QzEyNy4yNywtMTY5LjAxIDEzNC4yMywtMTYzLjYgMTQwLjkxLC0xNTgu
NDEiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTQzLjQs
LTE2MC45IDE0OS4xNCwtMTUyIDEzOS4xLC0xNTUuMzggMTQzLjQsLTE2MC45Ii8+CjwvZz4KPCEt
LSBpbnQxIC0tPgo8ZyBpZD0ibm9kZTUiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmludDE8L3RpdGxl
Pgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjEyNiwtOTQgNzIs
LTk0IDcyLC01OCAxMjYsLTU4IDEyNiwtOTQiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIg
eD0iOTkiIHk9Ii03My41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0i
IGZvbnQtc2l6ZT0iMTAuMDAiPklOVCgxKTwvdGV4dD4KPC9nPgo8IS0tIGV4cHImIzQ1OyZndDtp
bnQxIC0tPgo8ZyBpZD0iZWRnZTQiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmV4cHImIzQ1OyZndDtp
bnQxPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE0OS4wMSwt
MTE1Ljg5QzE0Mi43MywtMTExLjAxIDEzNS43NywtMTA1LjYgMTI5LjA5LC0xMDAuNDEiLz4KPHBv
bHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTMwLjksLTk3LjM4IDEy
MC44NiwtOTQgMTI2LjYsLTEwMi45IDEzMC45LC05Ny4zOCIvPgo8L2c+CjwhLS0gcGx1cyAtLT4K
PGcgaWQ9Im5vZGU2IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5wbHVzPC90aXRsZT4KPHBvbHlnb24g
ZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxOTgsLTk0IDE0NCwtOTQgMTQ0LC01
OCAxOTgsLTU4IDE5OCwtOTQiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTcxIiB5
PSItNzMuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNp
emU9IjEwLjAwIj4mIzM5OysmIzM5OzwvdGV4dD4KPC9nPgo8IS0tIGV4cHImIzQ1OyZndDtwbHVz
IC0tPgo8ZyBpZD0iZWRnZTUiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmV4cHImIzQ1OyZndDtwbHVz
PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE3MSwtMTE1Ljg5
QzE3MSwtMTEyLjIgMTcxLC0xMDguMjEgMTcxLC0xMDQuMjQiLz4KPHBvbHlnb24gZmlsbD0iYmxh
Y2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTc0LjUsLTEwNCAxNzEsLTk0IDE2Ny41LC0xMDQg
MTc0LjUsLTEwNCIvPgo8L2c+CjwhLS0gdGVybSAtLT4KPGcgaWQ9Im5vZGU3IiBjbGFzcz0ibm9k
ZSI+Cjx0aXRsZT50ZXJtPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFj
ayIgcG9pbnRzPSIyNzAsLTk0IDIxNiwtOTQgMjE2LC01OCAyNzAsLTU4IDI3MCwtOTQiLz4KPHRl
eHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjQzIiB5PSItNzMuNSIgZm9udC1mYW1pbHk9Iklv
c2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4mbHQ7VGVybSZndDs8
L3RleHQ+CjwvZz4KPCEtLSBleHByJiM0NTsmZ3Q7dGVybSAtLT4KPGcgaWQ9ImVkZ2U2IiBjbGFz
cz0iZWRnZSI+Cjx0aXRsZT5leHByJiM0NTsmZ3Q7dGVybTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5v
bmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xOTIuOTksLTExNS44OUMxOTkuMjcsLTExMS4wMSAyMDYu
MjMsLTEwNS42IDIxMi45MSwtMTAwLjQxIi8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9
ImJsYWNrIiBwb2ludHM9IjIxNS40LC0xMDIuOSAyMjEuMTQsLTk0IDIxMS4xLC05Ny4zOCAyMTUu
NCwtMTAyLjkiLz4KPC9nPgo8IS0tIGludDIgLS0+CjxnIGlkPSJub2RlOCIgY2xhc3M9Im5vZGUi
Pgo8dGl0bGU+aW50MjwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2si
IHBvaW50cz0iMTk4LC0zNiAxNDQsLTM2IDE0NCwwIDE5OCwwIDE5OCwtMzYiLz4KPHRleHQgdGV4
dC1hbmNob3I9Im1pZGRsZSIgeD0iMTcxIiB5PSItMTUuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2Eg
Q2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5JTlQoMik8L3RleHQ+CjwvZz4K
PCEtLSB0ZXJtJiM0NTsmZ3Q7aW50MiAtLT4KPGcgaWQ9ImVkZ2U3IiBjbGFzcz0iZWRnZSI+Cjx0
aXRsZT50ZXJtJiM0NTsmZ3Q7aW50MjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGQ9Ik0yMjEuMDEsLTU3Ljg5QzIxNC43MywtNTMuMDEgMjA3Ljc3LC00Ny42IDIwMS4w
OSwtNDIuNDEiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0i
MjAyLjksLTM5LjM4IDE5Mi44NiwtMzYgMTk4LjYsLTQ0LjkgMjAyLjksLTM5LjM4Ii8+CjwvZz4K
PCEtLSBzdGFyIC0tPgo8ZyBpZD0ibm9kZTkiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnN0YXI8L3Rp
dGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjI3MCwtMzYg
MjE2LC0zNiAyMTYsMCAyNzAsMCAyNzAsLTM2Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUi
IHg9IjI0MyIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1
bSIgZm9udC1zaXplPSIxMC4wMCI+JiMzOTvvvIomIzM5OzwvdGV4dD4KPC9nPgo8IS0tIHRlcm0m
IzQ1OyZndDtzdGFyIC0tPgo8ZyBpZD0iZWRnZTgiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnRlcm0m
IzQ1OyZndDtzdGFyPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0i
TTI0MywtNTcuODlDMjQzLC01NC4yIDI0MywtNTAuMjEgMjQzLC00Ni4yNCIvPgo8cG9seWdvbiBm
aWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyNDYuNSwtNDYgMjQzLC0zNiAyMzku
NSwtNDYgMjQ2LjUsLTQ2Ii8+CjwvZz4KPCEtLSBpbnQzIC0tPgo8ZyBpZD0ibm9kZTEwIiBjbGFz
cz0ibm9kZSI+Cjx0aXRsZT5pbnQzPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tl
PSJibGFjayIgcG9pbnRzPSIzNDIsLTM2IDI4OCwtMzYgMjg4LDAgMzQyLDAgMzQyLC0zNiIvPgo8
dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIzMTUiIHk9Ii0xNS41IiBmb250LWZhbWlseT0i
SW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPklOVCgzKTwvdGV4
dD4KPC9nPgo8IS0tIHRlcm0mIzQ1OyZndDtpbnQzIC0tPgo8ZyBpZD0iZWRnZTkiIGNsYXNzPSJl
ZGdlIj4KPHRpdGxlPnRlcm0mIzQ1OyZndDtpbnQzPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIg
c3Ryb2tlPSJibGFjayIgZD0iTTI2NC45OSwtNTcuODlDMjcxLjI3LC01My4wMSAyNzguMjMsLTQ3
LjYgMjg0LjkxLC00Mi40MSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIg
cG9pbnRzPSIyODcuNCwtNDQuOSAyOTMuMTQsLTM2IDI4My4xLC0zOS4zOCAyODcuNCwtNDQuOSIv
Pgo8L2c+CjwhLS0gbGV0c3RtdCAtLT4KPGcgaWQ9Im5vZGUxMSIgY2xhc3M9Im5vZGUiPgo8dGl0
bGU+bGV0c3RtdDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBv
aW50cz0iNDMwLC0yMTAgMzU2LC0yMTAgMzU2LC0xNzQgNDMwLC0xNzQgNDMwLC0yMTAiLz4KPHRl
eHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMzkzIiB5PSItMTg5LjUiIGZvbnQtZmFtaWx5PSJJ
b3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+TGV0U3RtdCAoJnF1
b3Q7eCZxdW90Oyk8L3RleHQ+CjwvZz4KPCEtLSBiaW5leHByMSAtLT4KPGcgaWQ9Im5vZGUxMiIg
Y2xhc3M9Im5vZGUiPgo8dGl0bGU+YmluZXhwcjE8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25l
IiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjQzNCwtMTUyIDM1MiwtMTUyIDM1MiwtMTE2IDQzNCwt
MTE2IDQzNCwtMTUyIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjM5MyIgeT0iLTEz
MS41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0i
MTAuMDAiPkJpbmFyeUV4cHIgKCspPC90ZXh0Pgo8L2c+CjwhLS0gbGV0c3RtdCYjNDU7Jmd0O2Jp
bmV4cHIxIC0tPgo8ZyBpZD0iZWRnZTEwIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5sZXRzdG10JiM0
NTsmZ3Q7YmluZXhwcjE8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBk
PSJNMzkzLC0xNzMuODlDMzkzLC0xNzAuMiAzOTMsLTE2Ni4yMSAzOTMsLTE2Mi4yNCIvPgo8cG9s
eWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzOTYuNSwtMTYyIDM5Mywt
MTUyIDM4OS41LC0xNjIgMzk2LjUsLTE2MiIvPgo8L2c+CjwhLS0gbGl0ZXJhbDEgLS0+CjxnIGlk
PSJub2RlMTMiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmxpdGVyYWwxPC90aXRsZT4KPHBvbHlnb24g
ZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzNzguNSwtOTQgMzE5LjUsLTk0IDMx
OS41LC01OCAzNzguNSwtNTggMzc4LjUsLTk0Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUi
IHg9IjM0OSIgeT0iLTczLjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1
bSIgZm9udC1zaXplPSIxMC4wMCI+TGl0ZXJhbCAoMSk8L3RleHQ+CjwvZz4KPCEtLSBiaW5leHBy
MSYjNDU7Jmd0O2xpdGVyYWwxIC0tPgo8ZyBpZD0iZWRnZTExIiBjbGFzcz0iZWRnZSI+Cjx0aXRs
ZT5iaW5leHByMSYjNDU7Jmd0O2xpdGVyYWwxPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgZD0iTTM3OS41NiwtMTE1Ljg5QzM3Ni4xNSwtMTExLjU1IDM3Mi40MSwtMTA2
LjggMzY4Ljc2LC0xMDIuMTQiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2si
IHBvaW50cz0iMzcxLjI5LC05OS43IDM2Mi4zNiwtOTQgMzY1Ljc4LC0xMDQuMDMgMzcxLjI5LC05
OS43Ii8+CjwvZz4KPCEtLSBiaW5leHByMiAtLT4KPGcgaWQ9Im5vZGUxNCIgY2xhc3M9Im5vZGUi
Pgo8dGl0bGU+YmluZXhwcjI8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBwb2ludHM9IjQ3Ny41LC05NCAzOTYuNSwtOTQgMzk2LjUsLTU4IDQ3Ny41LC01OCA0Nzcu
NSwtOTQiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNDM3IiB5PSItNzMuNSIgZm9u
dC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5C
aW5hcnlFeHByICgqKTwvdGV4dD4KPC9nPgo8IS0tIGJpbmV4cHIxJiM0NTsmZ3Q7YmluZXhwcjIg
LS0+CjxnIGlkPSJlZGdlMTIiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmJpbmV4cHIxJiM0NTsmZ3Q7
YmluZXhwcjI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNNDA2
LjQ0LC0xMTUuODlDNDA5Ljg1LC0xMTEuNTUgNDEzLjU5LC0xMDYuOCA0MTcuMjQsLTEwMi4xNCIv
Pgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSI0MjAuMjIsLTEw
NC4wMyA0MjMuNjQsLTk0IDQxNC43MSwtOTkuNyA0MjAuMjIsLTEwNC4wMyIvPgo8L2c+CjwhLS0g
bGl0ZXJhbDIgLS0+CjxnIGlkPSJub2RlMTUiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmxpdGVyYWwy
PC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSI0MTku
NSwtMzYgMzYwLjUsLTM2IDM2MC41LDAgNDE5LjUsMCA0MTkuNSwtMzYiLz4KPHRleHQgdGV4dC1h
bmNob3I9Im1pZGRsZSIgeD0iMzkwIiB5PSItMTUuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hh
cm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5MaXRlcmFsICgyKTwvdGV4dD4KPC9n
Pgo8IS0tIGJpbmV4cHIyJiM0NTsmZ3Q7bGl0ZXJhbDIgLS0+CjxnIGlkPSJlZGdlMTMiIGNsYXNz
PSJlZGdlIj4KPHRpdGxlPmJpbmV4cHIyJiM0NTsmZ3Q7bGl0ZXJhbDI8L3RpdGxlPgo8cGF0aCBm
aWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNNDIyLjY0LC01Ny44OUM0MTguOTEsLTUzLjQ0
IDQxNC44MSwtNDguNTYgNDEwLjgxLC00My44Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJv
a2U9ImJsYWNrIiBwb2ludHM9IjQxMy4zOCwtNDEuNDEgNDA0LjI3LC0zNiA0MDguMDIsLTQ1Ljkx
IDQxMy4zOCwtNDEuNDEiLz4KPC9nPgo8IS0tIGxpdGVyYWwzIC0tPgo8ZyBpZD0ibm9kZTE2IiBj
bGFzcz0ibm9kZSI+Cjx0aXRsZT5saXRlcmFsMzwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUi
IHN0cm9rZT0iYmxhY2siIHBvaW50cz0iNDk2LjUsLTM2IDQzNy41LC0zNiA0MzcuNSwwIDQ5Ni41
LDAgNDk2LjUsLTM2Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjQ2NyIgeT0iLTE1
LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIx
MC4wMCI+TGl0ZXJhbCAoMyk8L3RleHQ+CjwvZz4KPCEtLSBiaW5leHByMiYjNDU7Jmd0O2xpdGVy
YWwzIC0tPgo8ZyBpZD0iZWRnZTE0IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5iaW5leHByMiYjNDU7
Jmd0O2xpdGVyYWwzPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0i
TTQ0Ni4xNiwtNTcuODlDNDQ4LjM3LC01My43NyA0NTAuNzgsLTQ5LjI3IDQ1My4xNiwtNDQuODQi
Lz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iNDU2LjI2LC00
Ni40NyA0NTcuODksLTM2IDQ1MC4wOCwtNDMuMTYgNDU2LjI2LC00Ni40NyIvPgo8L2c+CjwvZz4K
PC9zdmc+Cg==
" alt="" />
<figcaption>Figure 14: Comparison of Parse Tree and AST for expression
"let x = 1 + 2 * 3;"</figcaption>
</figure>

#### 19.1.1 Concrete Syntax Trees versus Abstract Syntax Trees

- **Concrete Syntax Tree (CST / Parse Tree):** A direct 1:1 graphical
  representation of the formal grammar productions. A CST retains every
  semicolon, comma, and grouping parenthesis. While useful in
  theoretical linguistics, CSTs are bloated and tedious for compilation
  passes.
- **Abstract Syntax Tree (AST):** An optimised, structural
  representation of program semantics. Syntactic trivia (such as
  semicolons and grouping parentheses) are discarded. Operator nodes
  directly link their operand subtrees. The AST is the central data
  structure around which type checking, optimisation, and code emission
  operate.

See fig. 14 for a visual comparison of the two.

### 19.2 Context-Free Grammars and EBNF

In Chapter 18, we saw that lexical tokens are governed by **Regular
Grammars (Type-3)**. However, regular expressions cannot describe nested
or recursively balanced structures, such as arbitrarily nested
parentheses `((1 + (2 * (3))))` or nested conditional blocks.

To specify program syntax, we step up the Chomsky hierarchy to
**Context-Free Grammars (Type-2)**.

#### 19.2.1 Formal Definition of a Context-Free Grammar

A Context-Free Grammar is formally defined as a 4-tuple:

$$G = (V, \Sigma, R, S)$$

Where:

- $V$ is a finite set of **Non-Terminal symbols** (syntactic variables
  representing syntactic structures).
- $\Sigma$ is a finite set of **Terminal symbols** (the atomic lexical
  tokens emitted by our lexer, disjoint from $V$).
- $R$ is a finite set of **Production Rules** mapping a single
  non-terminal to a sequence of non-terminals and terminals:
  $$A \to \alpha \quad \text{where } A \in V \text{ and } \alpha \in (V \cup \Sigma)^*$$
- $S \in V$ is the distinguished **Start Symbol**.

#### 19.2.2 Extended Backus-Naur Form (EBNF) Notation

To define compiler grammars concisely, we use **Extended Backus-Naur
Form (EBNF)** notation:

- `A ::= B C` denotes concatenation (Rule $A$ derives $B$ followed by
  $C$).
- `A ::= B | C` denotes alternation (Rule $A$ derives either $B$ or
  $C$).
- `A ::= (B)?` denotes optionality (Zero or one occurrence of $B$).
- `A ::= (B)*` denotes zero or more repetitions (Kleene closure).
- `A ::= (B)+` denotes one or more repetitions.

#### 19.2.3 The Ambiguity Problem

A grammar is **ambiguous** if a single string of tokens can yield two or
more distinct parse trees. Consider the classic naive arithmetic
grammar:

$$E \to E + E \mid E \times E \mid \text{number}$$

For the input string `1 + 2 * 3`, this grammar permits two valid
derivations:

1.  $(1 + 2) \times 3 = 9$ (Tree grouping $+$ above $\times$).
2.  $1 + (2 \times 3) = 7$ (Tree grouping $\times$ above $+$).

In mathematics and programming, multiplication binds tighter than
addition. A compiler grammar must be completely **unambiguous**,
explicitly baking in:

1.  **Operator Precedence:** Higher precedence operators appear deeper
    in the grammar hierarchy.
2.  **Associativity:** Left-associative operators (e.g., subtraction:
    $10 - 4 - 2 \equiv (10 - 4) - 2$) grow trees downward to the left;
    right-associative operators (e.g., exponentiation:
    $2^{3^2} \equiv 2^{(3^2)}$ or assignment: $a = b = c$) grow downward
    to the right.

### 19.3 Parsing Paradigms: Top-Down vs Bottom-Up

Compiler front-ends historically fall into two major architectural
paradigms (see tbl. 2)

| Top-Down Parsers (LL)           | Bottom-Up Parsers (LR)           |
|:--------------------------------|:---------------------------------|
| Starts at Start Symbol S        | Starts at leaf tokens            |
| Predicts productions downward   | Shifts tokens into a stack       |
| LL(1), LL(k), Recursive Descent | Reduces patterns into non-terms  |
| Hand-crafted, pristine errors   | LR(1), LALR(1) (Yacc, Bison)     |
| Used by: Clang, Rust, V8, tsc   | Extremely rigid, poor error msgs |

Table 2: Parsing Paradigms {#tbl:parsing-paradigms}

#### 19.3.1 Why Production Compilers Choose Hand-Crafted Recursive Descent

While table-driven LALR(1) parser generators (such as Yacc and Bison)
were popular in the 1970s and 1980s, virtually every modern industrial
compiler (**Clang/LLVM**, **rustc**, **Go gc**, **TypeScript tsc**,
**V8**, **Swift**) has abandoned parser generators in favour of
**hand-crafted Recursive Descent**.

The reasons are pragmatic:

1.  **State-of-the-Art Error Diagnostics:** When a syntax error occurs,
    a recursive descent parser has the entire operating system call
    stack matching the grammar hierarchy. It knows precisely which
    expression or statement was expected and can emit insightful
    contextual diagnostics with pinpoint recovery.
2.  **Custom Disambiguation:** Complex contextual edge cases (e.g., C++
    template brackets `vector<vector<int>>` vs right-shift operators
    `>>`) are trivial to disambiguate procedurally with arbitrary
    lookahead, but cause shift-reduce conflicts in grammar tables.
3.  **Performance:** Direct C++ function calls with predictable control
    flow execute substantially faster on modern CPU branch predictors
    than table-driven state transitions.

### 19.4 Pratt Parsing: Top-Down Operator Precedence

Standard recursive descent handles structured statements cleanly, but
when parsing deeply nested binary expressions with 10+ levels of
operator precedence, a naive grammar requires 10+ distinct recursive
functions (`parse_expr` $\to$ `parse_equality` $\to$ `parse_relational`
$\to$ `parse_additive` $\to$ `parse_multiplicative` $\dots$). This
introduces call stack overhead and makes adding new operators tedious.

In 1973, **Vaughan Pratt** introduced **Top-Down Operator Precedence
(Pratt Parsing)**. Pratt’s insight was to associate two numerical values
with every token:

1.  **Prefix Denotation (`nud` - Null Denotation):** How a token behaves
    at the *start* of an expression (e.g., integer literals,
    identifiers, unary `-`, unary `!`, grouping `(`).
2.  **Infix Denotation (`led` - Left Denotation):** How a token behaves
    *between* two expressions (e.g., `+`, `*`, `==`, function calls `(`,
    index access `[`).
3.  **Binding Power (Precedence):** An integer defining how tightly the
    operator binds its left and right operands.

<figure>
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA4MDAgMjQwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyNDAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0iYm94U2hhZG93IiB4PSItNSUiIHk9
Ii01JSIgd2lkdGg9IjExMCUiIGhlaWdodD0iMTIwJSI+CiAgICAgIDxmZURyb3BTaGFkb3cgZHg9
IjAiIGR5PSIyIiBzdGREZXZpYXRpb249IjIiIGZsb29kLWNvbG9yPSIjMDAwIiBmbG9vZC1vcGFj
aXR5PSIwLjUiLz4KICAgIDwvZmlsdGVyPgogIDwvZGVmcz4KICA8IS0tIFRva2VucyB3aXRoIEJp
bmRpbmcgUG93ZXIgQmFkZ2VzIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDYwLCA0MCki
PgogICAgPHRleHQgeD0iMzgwIiB5PSIwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9
IjE2IiBmb250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+UHJhdHQgQmluZGlu
ZyBQb3dlciBSZXNvbHV0aW9uIGZvcjogMSArIDIgKiAzID09IDc8L3RleHQ+CiAgICA8IS0tIFRv
a2VuIDE6ICIxIiAtLT4KICAgIDxyZWN0IHg9IjAiIHk9IjMwIiB3aWR0aD0iNzAiIGhlaWdodD0i
NTAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxi
YWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiIGZpbHRlcj0i
dXJsKCNib3hTaGFkb3cpIi8+CiAgICA8dGV4dCB4PSIzNSIgeT0iNjAiIGZpbGw9InZhcigtLXN2
Zy1mZykiIGZvbnQtc2l6ZT0iMTYiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIj4xPC90ZXh0PgogICAgPCEtLSBUb2tlbiAyOiAiKyIgLS0+CiAgICA8cmVjdCB4PSIxMTAi
IHk9IjMwIiB3aWR0aD0iODAiIGhlaWdodD0iNTAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGlu
dC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIg
c3Ryb2tlLXdpZHRoPSIyIiBmaWx0ZXI9InVybCgjYm94U2hhZG93KSIvPgogICAgPHRleHQgeD0i
MTUwIiB5PSI1NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9udC13ZWln
aHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPis8L3RleHQ+CiAgICA8cmVjdCB4PSIxMjAi
IHk9IjYyIiB3aWR0aD0iNjAiIGhlaWdodD0iMTUiIHJ4PSIzIiBmaWxsPSJ2YXIoLS1zdmctdGlu
dC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiLz4KICAgIDx0ZXh0IHg9IjE1MCIgeT0i
NzMiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRk
bGUiPkJQOiA0MDwvdGV4dD4KICAgIDwhLS0gVG9rZW4gMzogIjIiIC0tPgogICAgPHJlY3QgeD0i
MjMwIiB5PSIzMCIgd2lkdGg9IjcwIiBoZWlnaHQ9IjUwIiByeD0iNiIgZmlsbD0idmFyKC0tc3Zn
LXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1m
ZykiIHN0cm9rZS13aWR0aD0iMS41IiBmaWx0ZXI9InVybCgjYm94U2hhZG93KSIvPgogICAgPHRl
eHQgeD0iMjY1IiB5PSI2MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9u
dC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPjI8L3RleHQ+CiAgICA8IS0tIFRv
a2VuIDQ6ICIqIiAtLT4KICAgIDxyZWN0IHg9IjM0MCIgeT0iMzAiIHdpZHRoPSI4MCIgaGVpZ2h0
PSI1MCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFs
bGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGZpbHRlcj0i
dXJsKCNib3hTaGFkb3cpIi8+CiAgICA8dGV4dCB4PSIzODAiIHk9IjU1IiBmaWxsPSJ2YXIoLS1z
dmctZmcpIiBmb250LXNpemU9IjE2IiBmb250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1p
ZGRsZSI+KjwvdGV4dD4KICAgIDxyZWN0IHg9IjM1MCIgeT0iNjIiIHdpZHRoPSI2MCIgaGVpZ2h0
PSIxNSIgcng9IjMiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFs
bGJhY2spKSIvPgogICAgPHRleHQgeD0iMzgwIiB5PSI3MyIgZmlsbD0idmFyKC0tc3ZnLWZnKSIg
Zm9udC1zaXplPSIxNCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+QlA6IDUwPC90ZXh0PgogICAgPCEt
LSBUb2tlbiA1OiAiMyIgLS0+CiAgICA8cmVjdCB4PSI0NjAiIHk9IjMwIiB3aWR0aD0iNzAiIGhl
aWdodD0iNTAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0x
LWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiIGZp
bHRlcj0idXJsKCNib3hTaGFkb3cpIi8+CiAgICA8dGV4dCB4PSI0OTUiIHk9IjYwIiBmaWxsPSJ2
YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE2IiBmb250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNo
b3I9Im1pZGRsZSI+MzwvdGV4dD4KICAgIDwhLS0gVG9rZW4gNjogIj09IiAtLT4KICAgIDxyZWN0
IHg9IjU3MCIgeT0iMzAiIHdpZHRoPSI4MCIgaGVpZ2h0PSI1MCIgcng9IjYiIGZpbGw9InZhcigt
LXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1z
dmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGZpbHRlcj0idXJsKCNib3hTaGFkb3cpIi8+CiAgICA8
dGV4dCB4PSI2MTAiIHk9IjU1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE2IiBm
b250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+PT08L3RleHQ+CiAgICA8cmVj
dCB4PSI1ODAiIHk9IjYyIiB3aWR0aD0iNjAiIGhlaWdodD0iMTUiIHJ4PSIzIiBmaWxsPSJ2YXIo
LS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiLz4KICAgIDx0ZXh0IHg9
IjYxMCIgeT0iNzMiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5j
aG9yPSJtaWRkbGUiPkJQOiAyMDwvdGV4dD4KICAgIDwhLS0gVG9rZW4gNzogIjciIC0tPgogICAg
PHJlY3QgeD0iNjkwIiB5PSIzMCIgd2lkdGg9IjcwIiBoZWlnaHQ9IjUwIiByeD0iNiIgZmlsbD0i
dmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZh
cigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMS41IiBmaWx0ZXI9InVybCgjYm94U2hhZG93KSIv
PgogICAgPHRleHQgeD0iNzI1IiB5PSI2MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXpl
PSIxNiIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPjc8L3RleHQ+CiAg
PC9nPgogIDwhLS0gRXhwbGFuYXRpb24gY2FsbG91dHMgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFu
c2xhdGUoNjAsIDE1MCkiPgogICAgPHJlY3QgeD0iMCIgeT0iMCIgd2lkdGg9Ijc2MCIgaGVpZ2h0
PSIxMDUiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZh
bGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHRleHQgeD0iMjAiIHk9IjI1
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCI+
QWxnb3JpdGhtIEZsb3c6PC90ZXh0PgogICAgPHRleHQgeD0iMjAiIHk9IjQ1IiBmaWxsPSJ2YXIo
LS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij4KICAgICAgMS4gUGFyc2VyIGVudGVycyB3aXRoIG1p
bl9icCA9IDAuIENvbnN1bWVzICcxJy4gU2VlcyAnKycgd2l0aCBCUCA0MCAmZ3Q7IDAuCiAgICA8
L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iNjAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQt
c2l6ZT0iMTQiPgogICAgICAyLiBSZWN1cnNpdmVseSBwYXJzZXMgUkhTIG9mICcrJyB3aXRoIG1p
bl9icCA9IDQwLiBDb25zdW1lcyAnMicuIFNlZXMgJyonIHdpdGggQlAgNTAgJmd0OyA0MC4KICAg
IDwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSI3NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9u
dC1zaXplPSIxNCI+CiAgICAgIDMuIFJlY3Vyc2l2ZWx5IHBhcnNlcyBSSFMgb2YgJyonIHdpdGgg
bWluX2JwID0gNTAuIENvbnN1bWVzICczJy4gU2VlcyAnPT0nIHdpdGggQlAgMjAgJmx0OyA1MC4K
ICAgIDwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSI5MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIg
Zm9udC1zaXplPSIxNCI+CiAgICAgICYjeGEwOyYjeGEwOyYjeGEwOyhSZXR1cm5zIGltbWVkaWF0
ZWx5ISkKICAgIDwvdGV4dD4KICA8L2c+Cjwvc3ZnPg==
" alt="Binding Power Resolution" />
<figcaption>Binding Power Resolution</figcaption>
</figure>

Pratt parsing collapses dozens of recursive grammar levels into a single
elegant loop:

``` cpp
std::unique_ptr<ExprAST> parse_expression(int min_precedence) {
    auto left = parse_prefix(); // nud: Literal, Unary, Identifier, Grouping

    while (current_token_precedence() > min_precedence) {
        left = parse_infix(std::move(left)); // led: Binary Operator, Call
    }

    return left;
}
```

### 19.5 Complete Grammar Specification for Faberge

We now formalise the complete Context-Free Grammar for **Faberge** in
EBNF:

```
Program        ::= Statement* EOF

Statement      ::= LetBinding
                 | ReturnStmt
                 | ExprStatement

LetBinding     ::= "let" Identifier (":" Type)? "=" Expression ";"
ReturnStmt     ::= "return" Expression ";"
ExprStatement  ::= Expression ";"

Type           ::= "Int" | "Float" | "String" | "Bool"
                 | "(" Type "->" Type ")"
                 | "[" Type "]"

Expression     ::= LogicalOrExpr

LogicalOrExpr  ::= LogicalAndExpr ( "||" LogicalAndExpr )*
LogicalAndExpr ::= EqualityExpr ( "&&" EqualityExpr )*
EqualityExpr   ::= RelationalExpr ( ("==" | "!=") RelationalExpr )*
RelationalExpr ::= AdditiveExpr ( ("<" | "<=" | ">" | ">=") AdditiveExpr )*
AdditiveExpr   ::= MultiplicativeExpr ( ("+" | "-") MultiplicativeExpr )*
MultiplicativeExpr ::= UnaryExpr ( ("*" | "/" | "%") UnaryExpr )*

UnaryExpr      ::= ("-" | "!") UnaryExpr
                 | PostfixExpr

PostfixExpr    ::= PrimaryExpr ( "(" ArgumentList? ")" )*

PrimaryExpr    ::= Literal_Integer
                 | Literal_Float
                 | Literal_String
                 | Literal_Boolean
                 | Identifier
                 | "(" Expression ")"
                 | BlockExpr
                 | IfExpr
                 | LambdaExpr

BlockExpr      ::= "{" Statement* (Expression)? "}"
IfExpr         ::= "if" Expression "then" Expression "else" Expression
LambdaExpr     ::= "fn" "(" ParameterList? ")" (":" Type)? "=>" Expression

ArgumentList   ::= Expression ( "," Expression )*
ParameterList  ::= Parameter ( "," Parameter )*
Parameter      ::= Identifier ":" Type
```

### 19.6 Experimental Implementation in Modern C++: Designing the AST

Let us design an object-oriented, polymorphic Abstract Syntax Tree in
Modern C++20.

#### 19.6.1 Key Architectural Decisions:

1.  **Memory Management:** Explicit ownership using
    `std::unique_ptr<Node>`. Subtrees are uniquely owned by their parent
    nodes, guaranteeing leak-free RAII destruction.
2.  **Source Span Tracking:** Every single AST node embeds a
    `SourceLocation`, enabling downstream type checkers (Chapter 20) and
    code emitters (Chapter 21) to pinpoint source locations in error
    diagnostics.
3.  **The Visitor Pattern:** We equip our AST with a double-dispatch
    `ASTVisitor` interface, separating node data structures from
    algorithmic passes (pretty-printing, type checking, and code
    emission).

<figure id="fig:ast-hierarchy">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI0ODhwdCIgaGVpZ2h0PSIxMzJwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCA0ODguMDAgMTMyLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDEy
OCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gQVNUTm9kZSAtLT4KPGcgaWQ9Im5vZGUxIiBjbGFz
cz0ibm9kZSI+Cjx0aXRsZT5BU1ROb2RlPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgcG9pbnRzPSIyMzEsLTEyNCAxNjYsLTEyNCAxNjYsLTg4IDIzMSwtODggMjMx
LC0xMjQiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTk4LjUiIHk9Ii0xMDkiIGZv
bnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+
Jmx0OyZsdDthYnN0cmFjdCZndDsmZ3Q7PC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxl
IiB4PSIxOTguNSIgeT0iLTk4IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRp
dW0iIGZvbnQtc2l6ZT0iMTAuMDAiPkFTVE5vZGU8L3RleHQ+CjwvZz4KPCEtLSBTdG10QVNUIC0t
Pgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRpdGxlPlN0bXRBU1Q8L3RpdGxlPgo8cG9s
eWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE0OCwtNjYgODMsLTY2IDgz
LC00NCAxNDgsLTQ0IDE0OCwtNjYiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTE1
LjUiIHk9Ii01Mi41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZv
bnQtc2l6ZT0iMTAuMDAiPlN0bXRBU1Q8L3RleHQ+CjwvZz4KPCEtLSBBU1ROb2RlJiM0NTsmZ3Q7
U3RtdEFTVCAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5BU1ROb2RlJiM0
NTsmZ3Q7U3RtdEFTVDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9
Ik0xNjkuNDcsLTg3Ljg2QzE1Ny4xMywtODAuNTggMTQzLjE3LC03Mi4zMyAxMzIuNTMsLTY2LjA2
Ii8+CjwvZz4KPCEtLSBFeHByQVNUIC0tPgo8ZyBpZD0ibm9kZTMiIGNsYXNzPSJub2RlIj4KPHRp
dGxlPkV4cHJBU1Q8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBw
b2ludHM9IjMxNCwtNjYgMjQ5LC02NiAyNDksLTQ0IDMxNCwtNDQgMzE0LC02NiIvPgo8dGV4dCB0
ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIyODEuNSIgeT0iLTUyLjUiIGZvbnQtZmFtaWx5PSJJb3Nl
dmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+RXhwckFTVDwvdGV4dD4K
PC9nPgo8IS0tIEFTVE5vZGUmIzQ1OyZndDtFeHByQVNUIC0tPgo8ZyBpZD0iZWRnZTIiIGNsYXNz
PSJlZGdlIj4KPHRpdGxlPkFTVE5vZGUmIzQ1OyZndDtFeHByQVNUPC90aXRsZT4KPHBhdGggZmls
bD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTIyNy41MywtODcuODZDMjM5Ljg3LC04MC41OCAy
NTMuODMsLTcyLjMzIDI2NC40NywtNjYuMDYiLz4KPC9nPgo8IS0tIExldFN0bXRBU1QgLS0+Cjxn
IGlkPSJub2RlNCIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+TGV0U3RtdEFTVDwvdGl0bGU+Cjxwb2x5
Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iNjUsLTIyIDAsLTIyIDAsMCA2
NSwwIDY1LC0yMiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIzMi41IiB5PSItOC41
IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAu
MDAiPkxldFN0bXRBU1Q8L3RleHQ+CjwvZz4KPCEtLSBTdG10QVNUJiM0NTsmZ3Q7TGV0U3RtdEFT
VCAtLT4KPGcgaWQ9ImVkZ2UzIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5TdG10QVNUJiM0NTsmZ3Q7
TGV0U3RtdEFTVDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik05
NS40MSwtNDMuODNDODIuMzYsLTM3LjIzIDY1LjQyLC0yOC42NiA1Mi40MSwtMjIuMDciLz4KPC9n
Pgo8IS0tIEV4cHJTdG10QVNUIC0tPgo8ZyBpZD0ibm9kZTUiIGNsYXNzPSJub2RlIj4KPHRpdGxl
PkV4cHJTdG10QVNUPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
cG9pbnRzPSIxNDgsLTIyIDgzLC0yMiA4MywwIDE0OCwwIDE0OCwtMjIiLz4KPHRleHQgdGV4dC1h
bmNob3I9Im1pZGRsZSIgeD0iMTE1LjUiIHk9Ii04LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENo
YXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+RXhwclN0bXRBU1Q8L3RleHQ+Cjwv
Zz4KPCEtLSBTdG10QVNUJiM0NTsmZ3Q7RXhwclN0bXRBU1QgLS0+CjxnIGlkPSJlZGdlNCIgY2xh
c3M9ImVkZ2UiPgo8dGl0bGU+U3RtdEFTVCYjNDU7Jmd0O0V4cHJTdG10QVNUPC90aXRsZT4KPHBh
dGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTExNS41LC00My44M0MxMTUuNSwtMzcu
MjMgMTE1LjUsLTI4LjY2IDExNS41LC0yMi4wNyIvPgo8L2c+CjwhLS0gQmluYXJ5RXhwciAtLT4K
PGcgaWQ9Im5vZGU2IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5CaW5hcnlFeHByPC90aXRsZT4KPHBv
bHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyMzEsLTIyIDE2NiwtMjIg
MTY2LDAgMjMxLDAgMjMxLC0yMiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxOTgu
NSIgeT0iLTguNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250
LXNpemU9IjEwLjAwIj5CaW5hcnlFeHByPC90ZXh0Pgo8L2c+CjwhLS0gRXhwckFTVCYjNDU7Jmd0
O0JpbmFyeUV4cHIgLS0+CjxnIGlkPSJlZGdlNSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+RXhwckFT
VCYjNDU7Jmd0O0JpbmFyeUV4cHI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBkPSJNMjYxLjQxLC00My44M0MyNDguMzYsLTM3LjIzIDIzMS40MiwtMjguNjYgMjE4LjQx
LC0yMi4wNyIvPgo8L2c+CjwhLS0gSWZFeHByIC0tPgo8ZyBpZD0ibm9kZTciIGNsYXNzPSJub2Rl
Ij4KPHRpdGxlPklmRXhwcjwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIHBvaW50cz0iMzE0LC0yMiAyNDksLTIyIDI0OSwwIDMxNCwwIDMxNCwtMjIiLz4KPHRleHQg
dGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjgxLjUiIHk9Ii04LjUiIGZvbnQtZmFtaWx5PSJJb3Nl
dmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+SWZFeHByPC90ZXh0Pgo8
L2c+CjwhLS0gRXhwckFTVCYjNDU7Jmd0O0lmRXhwciAtLT4KPGcgaWQ9ImVkZ2U2IiBjbGFzcz0i
ZWRnZSI+Cjx0aXRsZT5FeHByQVNUJiM0NTsmZ3Q7SWZFeHByPC90aXRsZT4KPHBhdGggZmlsbD0i
bm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTI4MS41LC00My44M0MyODEuNSwtMzcuMjMgMjgxLjUs
LTI4LjY2IDI4MS41LC0yMi4wNyIvPgo8L2c+CjwhLS0gTGFtYmRhRXhwciAtLT4KPGcgaWQ9Im5v
ZGU4IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5MYW1iZGFFeHByPC90aXRsZT4KPHBvbHlnb24gZmls
bD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzOTcsLTIyIDMzMiwtMjIgMzMyLDAgMzk3
LDAgMzk3LC0yMiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIzNjQuNSIgeT0iLTgu
NSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEw
LjAwIj5MYW1iZGFFeHByPC90ZXh0Pgo8L2c+CjwhLS0gRXhwckFTVCYjNDU7Jmd0O0xhbWJkYUV4
cHIgLS0+CjxnIGlkPSJlZGdlNyIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+RXhwckFTVCYjNDU7Jmd0
O0xhbWJkYUV4cHI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJN
MzAxLjU5LC00My44M0MzMTQuNjQsLTM3LjIzIDMzMS41OCwtMjguNjYgMzQ0LjU5LC0yMi4wNyIv
Pgo8L2c+CjwhLS0gQ2FsbEV4cHIgLS0+CjxnIGlkPSJub2RlOSIgY2xhc3M9Im5vZGUiPgo8dGl0
bGU+Q2FsbEV4cHI8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBw
b2ludHM9IjQ4MCwtMjIgNDE1LC0yMiA0MTUsMCA0ODAsMCA0ODAsLTIyIi8+Cjx0ZXh0IHRleHQt
YW5jaG9yPSJtaWRkbGUiIHg9IjQ0Ny41IiB5PSItOC41IiBmb250LWZhbWlseT0iSW9zZXZrYSBD
aGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPkNhbGxFeHByPC90ZXh0Pgo8L2c+
CjwhLS0gRXhwckFTVCYjNDU7Jmd0O0NhbGxFeHByIC0tPgo8ZyBpZD0iZWRnZTgiIGNsYXNzPSJl
ZGdlIj4KPHRpdGxlPkV4cHJBU1QmIzQ1OyZndDtDYWxsRXhwcjwvdGl0bGU+CjxwYXRoIGZpbGw9
Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0zMTQuMywtNDUuN0MzNDMuNCwtMzguMzQgMzg1LjY5
LC0yNy42NCA0MTQuNzcsLTIwLjI4Ii8+CjwvZz4KPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 15: AST Polymorphic Class Hierarchy</figcaption>
</figure>

#### 19.6.2 Implementation

The listings below implement and test the C++ classes needed to complete
the functionality outlined in this chapter:

- lst. 6 defines the foundational AST definitions using move semantics
  and binary tree classes.
- lst. 7 uses the Visitor Pattern to create a source-file pretty-printer
  for easy side-by-side analysis.
- lst. 8 constructs the core parser engine, combining structured
  recursive descent with Pratt precedence climbing and **Panic-Mode
  Error Recovery**.
- lst. 9 implements a comprehensive driver to parse an expressive
  functional **Faberge** program, construct the AST, and render the
  resulting tree hierarchy.

Listing 6: AST Node Hierarchy (AST.hpp)

In [ ]:
#ifndef AST_HPP
#define AST_HPP

#include "../chapter-18/Token.hpp"
#include <string>
#include <vector>
#include <memory>
#include <string_view>
#include <optional>

namespace Faberge {

// Forward declarations
class ASTVisitor;

// -----------------------------------------------------------------------------
// Base AST Node
// -----------------------------------------------------------------------------
class ASTNode {
public:
    explicit ASTNode(SourceLocation loc) : m_location(loc) {}
    virtual ~ASTNode() = default;

    [[nodiscard]] SourceLocation location() const noexcept { return m_location; }
    virtual void accept(ASTVisitor& visitor) const = 0;

private:
    SourceLocation m_location;
};

// -----------------------------------------------------------------------------
// Types Representation
// -----------------------------------------------------------------------------
enum class PrimitiveTypeKind {
    Int,
    Float,
    String,
    Bool,
    Void
};

class TypeNode {
public:
    enum class Kind { Primitive, Function, List };

    explicit TypeNode(PrimitiveTypeKind prim) 
        : m_kind(Kind::Primitive), m_primitive_kind(prim) {}

    static TypeNode make_function(TypeNode param, TypeNode ret) {
        TypeNode t(Kind::Function);
        t.m_param_types.push_back(std::make_shared<TypeNode>(std::move(param)));
        t.m_return_type = std::make_shared<TypeNode>(std::move(ret));
        return t;
    }

    [[nodiscard]] Kind kind() const noexcept { return m_kind; }
    [[nodiscard]] PrimitiveTypeKind primitive_kind() const noexcept { return m_primitive_kind; }

    [[nodiscard]] std::string to_string() const {
        switch (m_kind) {
            case Kind::Primitive:
                switch (m_primitive_kind) {
                    case PrimitiveTypeKind::Int:    return "Int";
                    case PrimitiveTypeKind::Float:  return "Float";
                    case PrimitiveTypeKind::String: return "String";
                    case PrimitiveTypeKind::Bool:   return "Bool";
                    case PrimitiveTypeKind::Void:   return "Void";
                }
                break;
            case Kind::Function:
                return "(" + (m_param_types.empty() ? "Void" : m_param_types[0]->to_string()) + 
                       " -> " + m_return_type->to_string() + ")";
            case Kind::List:
                return "[" + m_param_types[0]->to_string() + "]";
        }
        return "Unknown";
    }

private:
    explicit TypeNode(Kind k) : m_kind(k) {}
    Kind m_kind;
    PrimitiveTypeKind m_primitive_kind{PrimitiveTypeKind::Void};
    std::vector<std::shared_ptr<TypeNode>> m_param_types;
    std::shared_ptr<TypeNode> m_return_type;
};

// -----------------------------------------------------------------------------
// Expression Nodes
// -----------------------------------------------------------------------------
class ExprAST : public ASTNode {
public:
    using ASTNode::ASTNode;
};

class LiteralExprAST : public ExprAST {
public:
    LiteralExprAST(SourceLocation loc, TokenType type, std::string_view value)
        : ExprAST(loc), m_token_type(type), m_value(value) {}

    [[nodiscard]] TokenType token_type() const noexcept { return m_token_type; }
    [[nodiscard]] std::string_view value() const noexcept { return m_value; }

    void accept(ASTVisitor& visitor) const override;

private:
    TokenType m_token_type;
    std::string m_value;
};

class VariableExprAST : public ExprAST {
public:
    VariableExprAST(SourceLocation loc, std::string name)
        : ExprAST(loc), m_name(std::move(name)) {}

    [[nodiscard]] const std::string& name() const noexcept { return m_name; }

    void accept(ASTVisitor& visitor) const override;

private:
    std::string m_name;
};

class UnaryExprAST : public ExprAST {
public:
    UnaryExprAST(SourceLocation loc, TokenType op, std::unique_ptr<ExprAST> operand)
        : ExprAST(loc), m_op(op), m_operand(std::move(operand)) {}

    [[nodiscard]] TokenType op() const noexcept { return m_op; }
    [[nodiscard]] const ExprAST& operand() const noexcept { return *m_operand; }

    void accept(ASTVisitor& visitor) const override;

private:
    TokenType m_op;
    std::unique_ptr<ExprAST> m_operand;
};

class BinaryExprAST : public ExprAST {
public:
    BinaryExprAST(SourceLocation loc, TokenType op, 
                   std::unique_ptr<ExprAST> left, 
                   std::unique_ptr<ExprAST> right)
        : ExprAST(loc), m_op(op), m_left(std::move(left)), m_right(std::move(right)) {}

    [[nodiscard]] TokenType op() const noexcept { return m_op; }
    [[nodiscard]] const ExprAST& left() const noexcept { return *m_left; }
    [[nodiscard]] const ExprAST& right() const noexcept { return *m_right; }

    void accept(ASTVisitor& visitor) const override;

private:
    TokenType m_op;
    std::unique_ptr<ExprAST> m_left;
    std::unique_ptr<ExprAST> m_right;
};

class IfExprAST : public ExprAST {
public:
    IfExprAST(SourceLocation loc, 
               std::unique_ptr<ExprAST> condition, 
               std::unique_ptr<ExprAST> then_branch, 
               std::unique_ptr<ExprAST> else_branch)
        : ExprAST(loc), m_condition(std::move(condition)), 
          m_then_branch(std::move(then_branch)), m_else_branch(std::move(else_branch)) {}

    [[nodiscard]] const ExprAST& condition() const noexcept { return *m_condition; }
    [[nodiscard]] const ExprAST& then_branch() const noexcept { return *m_then_branch; }
    [[nodiscard]] const ExprAST& else_branch() const noexcept { return *m_else_branch; }

    void accept(ASTVisitor& visitor) const override;

private:
    std::unique_ptr<ExprAST> m_condition;
    std::unique_ptr<ExprAST> m_then_branch;
    std::unique_ptr<ExprAST> m_else_branch;
};

class CallExprAST : public ExprAST {
public:
    CallExprAST(SourceLocation loc, std::unique_ptr<ExprAST> callee, 
                 std::vector<std::unique_ptr<ExprAST>> args)
        : ExprAST(loc), m_callee(std::move(callee)), m_arguments(std::move(args)) {}

    [[nodiscard]] const ExprAST& callee() const noexcept { return *m_callee; }
    [[nodiscard]] const std::vector<std::unique_ptr<ExprAST>>& arguments() const noexcept { 
        return m_arguments; 
    }

    void accept(ASTVisitor& visitor) const override;

private:
    std::unique_ptr<ExprAST> m_callee;
    std::vector<std::unique_ptr<ExprAST>> m_arguments;
};

// Forward declaration for BlockExprAST
class StmtAST;

class BlockExprAST : public ExprAST {
public:
    BlockExprAST(SourceLocation loc, 
                 std::vector<std::unique_ptr<StmtAST>> statements,
                 std::unique_ptr<ExprAST> result_expr);
    ~BlockExprAST(); // Constructor and destructor cannot be defined inline for Clang with -std=c++23/26
    [[nodiscard]] const std::vector<std::unique_ptr<StmtAST>>& statements() const noexcept { 
        return m_statements; 
    }
    [[nodiscard]] const ExprAST* result_expr() const noexcept { return m_result_expr.get(); }

    void accept(ASTVisitor& visitor) const override;

private:
    std::vector<std::unique_ptr<StmtAST>> m_statements;
    std::unique_ptr<ExprAST> m_result_expr;
};

struct Parameter {
    std::string name;
    TypeNode type;
    SourceLocation location;
};

class LambdaExprAST : public ExprAST {
public:
    LambdaExprAST(SourceLocation loc, std::vector<Parameter> params, 
                  std::optional<TypeNode> return_type, 
                  std::unique_ptr<ExprAST> body)
        : ExprAST(loc), m_parameters(std::move(params)), 
          m_return_type(std::move(return_type)), m_body(std::move(body)) {}

    [[nodiscard]] const std::vector<Parameter>& parameters() const noexcept { return m_parameters; }
    [[nodiscard]] const std::optional<TypeNode>& return_type() const noexcept { return m_return_type; }
    [[nodiscard]] const ExprAST& body() const noexcept { return *m_body; }

    void accept(ASTVisitor& visitor) const override;

private:
    std::vector<Parameter> m_parameters;
    std::optional<TypeNode> m_return_type;
    std::unique_ptr<ExprAST> m_body;
};

// -----------------------------------------------------------------------------
// Statement Nodes
// -----------------------------------------------------------------------------
class StmtAST : public ASTNode {
public:
    using ASTNode::ASTNode;
};

BlockExprAST::BlockExprAST(SourceLocation loc, 
    std::vector<std::unique_ptr<StmtAST>> statements,
    std::unique_ptr<ExprAST> result_expr)
     : ExprAST(loc), m_statements(std::move(statements)), m_result_expr(std::move(result_expr)) {}
BlockExprAST::~BlockExprAST() = default;

class LetStmtAST : public StmtAST {
public:
    LetStmtAST(SourceLocation loc, std::string name, 
               std::optional<TypeNode> type_annot, 
               std::unique_ptr<ExprAST> initialiser)
        : StmtAST(loc), m_name(std::move(name)), 
          m_type_annot(std::move(type_annot)), m_initialiser(std::move(initialiser)) {}

    [[nodiscard]] const std::string& name() const noexcept { return m_name; }
    [[nodiscard]] const std::optional<TypeNode>& type_annotation() const noexcept { return m_type_annot; }
    [[nodiscard]] const ExprAST& initialiser() const noexcept { return *m_initialiser; }

    void accept(ASTVisitor& visitor) const override;

private:
    std::string m_name;
    std::optional<TypeNode> m_type_annot;
    std::unique_ptr<ExprAST> m_initialiser;
};

class ExprStmtAST : public StmtAST {
public:
    ExprStmtAST(SourceLocation loc, std::unique_ptr<ExprAST> expr)
        : StmtAST(loc), m_expression(std::move(expr)) {}

    [[nodiscard]] const ExprAST& expression() const noexcept { return *m_expression; }

    void accept(ASTVisitor& visitor) const override;

private:
    std::unique_ptr<ExprAST> m_expression;
};

// -----------------------------------------------------------------------------
// Program Root Node
// -----------------------------------------------------------------------------
class ProgramAST : public ASTNode {
public:
    ProgramAST(SourceLocation loc, std::vector<std::unique_ptr<StmtAST>> stmts)
        : ASTNode(loc), m_statements(std::move(stmts)) {}

    [[nodiscard]] const std::vector<std::unique_ptr<StmtAST>>& statements() const noexcept { 
        return m_statements; 
    }

    void accept(ASTVisitor& visitor) const override;

private:
    std::vector<std::unique_ptr<StmtAST>> m_statements;
};

} // namespace Faberge

#endif // AST_HPP

Listing 7: AST Inspector and Pretty-Printer (ASTVisitor.hpp)

In [ ]:
#ifndef AST_VISTOR_HPP
#define AST_VISTOR_HPP

// #include "AST.hpp"
#include <iostream>

namespace Faberge {

class ASTVisitor {
public:
    virtual ~ASTVisitor() = default;

    virtual void visit(const ProgramAST& node) = 0;
    virtual void visit(const LetStmtAST& node) = 0;
    virtual void visit(const ExprStmtAST& node) = 0;
    virtual void visit(const LiteralExprAST& node) = 0;
    virtual void visit(const VariableExprAST& node) = 0;
    virtual void visit(const UnaryExprAST& node) = 0;
    virtual void visit(const BinaryExprAST& node) = 0;
    virtual void visit(const IfExprAST& node) = 0;
    virtual void visit(const CallExprAST& node) = 0;
    virtual void visit(const BlockExprAST& node) = 0;
    virtual void visit(const LambdaExprAST& node) = 0;
};

// Inline definitions of double-dispatch accept methods
inline void ProgramAST::accept(ASTVisitor& v) const     { v.visit(*this); }
inline void LetStmtAST::accept(ASTVisitor& v) const     { v.visit(*this); }
inline void ExprStmtAST::accept(ASTVisitor& v) const    { v.visit(*this); }
inline void LiteralExprAST::accept(ASTVisitor& v) const { v.visit(*this); }
inline void VariableExprAST::accept(ASTVisitor& v) const{ v.visit(*this); }
inline void UnaryExprAST::accept(ASTVisitor& v) const   { v.visit(*this); }
inline void BinaryExprAST::accept(ASTVisitor& v) const  { v.visit(*this); }
inline void IfExprAST::accept(ASTVisitor& v) const      { v.visit(*this); }
inline void CallExprAST::accept(ASTVisitor& v) const    { v.visit(*this); }
inline void BlockExprAST::accept(ASTVisitor& v) const   { v.visit(*this); }
inline void LambdaExprAST::accept(ASTVisitor& v) const  { v.visit(*this); }

// Diagnostic AST Pretty-Printer
class ASTPrinter : public ASTVisitor {
public:
    void print(const ASTNode& node) {
        m_indent_level = 0;
        node.accept(*this);
    }

    void visit(const ProgramAST& node) override {
        print_indent();
        std::cout << "\033[1;36mProgram\033[0m\n";
        indent();
        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }
        dedent();
    }

    void visit(const LetStmtAST& node) override {
        print_indent();
        std::cout << "\033[1;32mLetStmt\033[0m (name: '\033[1m" << node.name() << "\033[0m'";
        if (node.type_annotation()) {
            std::cout << ", type: \033[33m" << node.type_annotation()->to_string() << "\033[0m";
        }
        std::cout << ")\n";
        indent();
        node.initialiser().accept(*this);
        dedent();
    }

    void visit(const ExprStmtAST& node) override {
        print_indent();
        std::cout << "\033[1;34mExprStmt\033[0m\n";
        indent();
        node.expression().accept(*this);
        dedent();
    }

    void visit(const LiteralExprAST& node) override {
        print_indent();
        std::cout << "\033[33mLiteral\033[0m (" << node.value() << ")\n";
    }

    void visit(const VariableExprAST& node) override {
        print_indent();
        std::cout << "\033[35mVariable\033[0m (" << node.name() << ")\n";
    }

    void visit(const UnaryExprAST& node) override {
        print_indent();
        std::cout << "\033[31mUnaryExpr\033[0m (op: " << Token::to_string(node.op()) << ")\n";
        indent();
        node.operand().accept(*this);
        dedent();
    }

    void visit(const BinaryExprAST& node) override {
        print_indent();
        std::cout << "\033[1;31mBinaryExpr\033[0m (op: " << Token::to_string(node.op()) << ")\n";
        indent();
        node.left().accept(*this);
        node.right().accept(*this);
        dedent();
    }

    void visit(const IfExprAST& node) override {
        print_indent();
        std::cout << "\033[1;35mIfExpr\033[0m\n";
        indent();
        node.condition().accept(*this);
        node.then_branch().accept(*this);
        node.else_branch().accept(*this);
        dedent();
    }

    void visit(const CallExprAST& node) override {
        print_indent();
        std::cout << "\033[1;34mCallExpr\033[0m\n";
        indent();
        node.callee().accept(*this);
        for (const auto& arg : node.arguments()) {
            arg->accept(*this);
        }
        dedent();
    }

    void visit(const BlockExprAST& node) override {
        print_indent();
        std::cout << "\033[1;36mBlockExpr\033[0m\n";
        indent();
        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }
        if (node.result_expr()) {
            node.result_expr()->accept(*this);
        }
        dedent();
    }

    void visit(const LambdaExprAST& node) override {
        print_indent();
        std::cout << "\033[1;32mLambdaExpr\033[0m (";
        for (size_t i = 0; i < node.parameters().size(); ++i) {
            std::cout << node.parameters()[i].name << ": " << node.parameters()[i].type.to_string();
            if (i + 1 < node.parameters().size()) std::cout << ", ";
        }
        std::cout << ")\n";
        indent();
        node.body().accept(*this);
        dedent();
    }

private:
    int m_indent_level{0};

    void indent() noexcept { ++m_indent_level; }
    void dedent() noexcept { --m_indent_level; }

    void print_indent() const {
        for (int i = 0; i < m_indent_level; ++i) {
            std::cout << "  ";
        }
    }
};

} // namespace Faberge

#endif // AST_VISITOR_HPP

Listing 8: Recursive Descent and Pratt Parser (Parser.hpp)

In [ ]:
#ifndef PARSER_HPP
#define PARSER_HPP

#include "../chapter-18/Token.hpp"
// #include "AST.hpp"
#include "../chapter-18/Diagnostic.hpp"
#include <vector>
#include <memory>
#include <stdexcept>
#include <span>

namespace Faberge {

class Parser {
public:
    explicit Parser(std::vector<Token> tokens, std::string_view source_code)
        : m_tokens(std::move(tokens)), m_source_code(source_code), m_cursor(0) {}

    std::unique_ptr<ProgramAST> parse_program() {
        std::vector<std::unique_ptr<StmtAST>> statements;
        SourceLocation start_loc = current_token().location;

        while (!is_at_end()) {
            try {
                statements.push_back(parse_statement());
            } catch (const std::runtime_error& e) {
                synchronise();
            }
        }

        return std::make_unique<ProgramAST>(start_loc, std::move(statements));
    }

    [[nodiscard]] bool has_errors() const noexcept { return m_error_count > 0; }
    [[nodiscard]] size_t error_count() const noexcept { return m_error_count; }

private:
    std::vector<Token> m_tokens;
    std::string_view m_source_code;
    size_t m_cursor{0};
    size_t m_error_count{0};

    // -------------------------------------------------------------------------
    // Token Navigation Helpers
    // -------------------------------------------------------------------------
    [[nodiscard]] bool is_at_end() const noexcept {
        return m_cursor >= m_tokens.size() || current_token().type == TokenType::EndOfFile;
    }

    [[nodiscard]] const Token& current_token() const noexcept {
        return m_tokens[m_cursor];
    }

    [[nodiscard]] const Token& previous_token() const noexcept {
        return m_tokens[m_cursor - 1];
    }

    const Token& advance() noexcept {
        if (!is_at_end()) ++m_cursor;
        return previous_token();
    }

    bool check(TokenType type) const noexcept {
        if (is_at_end()) return false;
        return current_token().type == type;
    }

    bool match(TokenType type) noexcept {
        if (check(type)) {
            advance();
            return true;
        }
        return false;
    }

    const Token& consume(TokenType expected, std::string_view err_msg) {
        if (check(expected)) return advance();
        
        ++m_error_count;
        DiagnosticEngine::report_error(m_source_code, current_token().location, err_msg);
        throw std::runtime_error(std::string(err_msg));
    }

    // -------------------------------------------------------------------------
    // Panic-Mode Error Recovery
    // -------------------------------------------------------------------------
    void synchronise() noexcept {
        advance();
        while (!is_at_end()) {
            if (previous_token().type == TokenType::Semicolon) return;

            switch (current_token().type) {
                case TokenType::Keyword_Let:
                case TokenType::Keyword_Fn:
                case TokenType::Keyword_If:
                case TokenType::Keyword_Return:
                    return;
                default:
                    advance();
                    break;
            }
        }
    }

    // -------------------------------------------------------------------------
    // Statement Parsing
    // -------------------------------------------------------------------------
    std::unique_ptr<StmtAST> parse_statement() {
        if (match(TokenType::Keyword_Let)) {
            return parse_let_statement();
        }
        return parse_expression_statement();
    }

    std::unique_ptr<StmtAST> parse_let_statement() {
        SourceLocation loc = previous_token().location;
        Token name_tok = consume(TokenType::Identifier, "Expected variable name after 'let'.");

        std::optional<TypeNode> type_annot;
        if (match(TokenType::Colon)) {
            type_annot = parse_type();
        }

        consume(TokenType::Equal, "Expected '=' in variable declaration.");
        auto init_expr = parse_expression();
        consume(TokenType::Semicolon, "Expected ';' after let statement.");

        return std::make_unique<LetStmtAST>(loc, std::string(name_tok.lexeme), 
                                            std::move(type_annot), std::move(init_expr));
    }

    std::unique_ptr<StmtAST> parse_expression_statement() {
        SourceLocation loc = current_token().location;
        auto expr = parse_expression();
        consume(TokenType::Semicolon, "Expected ';' after expression statement.");
        return std::make_unique<ExprStmtAST>(loc, std::move(expr));
    }

    TypeNode parse_type() {
        if (match(TokenType::Identifier)) {
            std::string_view name = previous_token().lexeme;
            if (name == "Int")    return TypeNode(PrimitiveTypeKind::Int);
            if (name == "Float")  return TypeNode(PrimitiveTypeKind::Float);
            if (name == "String") return TypeNode(PrimitiveTypeKind::String);
            if (name == "Bool")   return TypeNode(PrimitiveTypeKind::Bool);
            if (name == "Void")   return TypeNode(PrimitiveTypeKind::Void);
            throw std::runtime_error("Unknown type identifier '" + std::string(name) + "'.");
        }
        throw std::runtime_error("Expected type name.");
    }

    // -------------------------------------------------------------------------
    // Pratt Expression Parsing
    // -------------------------------------------------------------------------
    static int get_infix_precedence(TokenType type) noexcept {
        switch (type) {
            case TokenType::PipePipe:           return 10; // ||
            case TokenType::AmpersandAmpersand: return 20; // &&
            case TokenType::EqualEqual:
            case TokenType::ExclamationEqual:   return 30; // ==, !=
            case TokenType::Less:
            case TokenType::LessEqual:
            case TokenType::Greater:
            case TokenType::GreaterEqual:       return 40; // <, <=, >, >=
            case TokenType::Plus:
            case TokenType::Minus:              return 50; // +, -
            case TokenType::Asterisk:
            case TokenType::Slash:
            case TokenType::Percent:            return 60; // *, /, %
            case TokenType::LeftParen:          return 70; // Function Call: f(...)
            default:                            return 0;
        }
    }

    std::unique_ptr<ExprAST> parse_expression(int min_precedence = 0) {
        auto left = parse_prefix();

        while (min_precedence < get_infix_precedence(current_token().type)) {
            left = parse_infix(std::move(left));
        }

        return left;
    }

    std::unique_ptr<ExprAST> parse_prefix() {
        SourceLocation loc = current_token().location;

        // 1. Unary Operators: -, !
        if (match(TokenType::Minus) || match(TokenType::Exclamation)) {
            TokenType op = previous_token().type;
            auto operand = parse_expression(60); // High unary binding power
            return std::make_unique<UnaryExprAST>(loc, op, std::move(operand));
        }

        // 2. Literals
        if (match(TokenType::Literal_Integer) || 
            match(TokenType::Literal_Float)   || 
            match(TokenType::Literal_String)  || 
            match(TokenType::Literal_Boolean)) {
            return std::make_unique<LiteralExprAST>(loc, previous_token().type, previous_token().lexeme);
        }

        // 3. Variables and Identifiers
        if (match(TokenType::Identifier)) {
            return std::make_unique<VariableExprAST>(loc, std::string(previous_token().lexeme));
        }

        // 4. Parenthesised Grouping: ( expr )
        if (match(TokenType::LeftParen)) {
            auto inner = parse_expression(0);
            consume(TokenType::RightParen, "Expected ')' after grouping expression.");
            return inner;
        }

        // 5. Block Expressions: { stmt* expr? }
        if (match(TokenType::LeftBrace)) {
            return parse_block_expression();
        }

        // 6. Conditional Expressions: if cond then expr else expr
        if (match(TokenType::Keyword_If)) {
            return parse_if_expression();
        }

        // 7. Lambda Expressions: fn (params) => body
        if (match(TokenType::Keyword_Fn)) {
            return parse_lambda_expression();
        }

        ++m_error_count;
        DiagnosticEngine::report_error(m_source_code, current_token().location, 
                                       "Expected expression, found '" + std::string(current_token().lexeme) + "'.");
        throw std::runtime_error("Syntax error: unexpected expression token.");
    }

    std::unique_ptr<ExprAST> parse_infix(std::unique_ptr<ExprAST> left) {
        SourceLocation loc = current_token().location;
        Token op_tok = advance();

        // Function Call Operator: callee(arg1, arg2)
        if (op_tok.type == TokenType::LeftParen) {
            std::vector<std::unique_ptr<ExprAST>> args;
            if (!check(TokenType::RightParen)) {
                do {
                    args.push_back(parse_expression(0));
                } while (match(TokenType::Comma));
            }
            consume(TokenType::RightParen, "Expected ')' after argument list.");
            return std::make_unique<CallExprAST>(loc, std::move(left), std::move(args));
        }

        // Standard Binary Arithmetic and Logical Operators
        int prec = get_infix_precedence(op_tok.type);
        auto right = parse_expression(prec); // Left-associative recursion

        return std::make_unique<BinaryExprAST>(loc, op_tok.type, std::move(left), std::move(right));
    }

    std::unique_ptr<ExprAST> parse_block_expression() {
        SourceLocation loc = previous_token().location;
        std::vector<std::unique_ptr<StmtAST>> stmts;
        std::unique_ptr<ExprAST> result_expr;

        while (!check(TokenType::RightBrace) && !is_at_end()) {
            if (check(TokenType::Keyword_Let)) {
                stmts.push_back(parse_statement());
            } else {
                // Peek ahead to check if this is an expression-statement (with ';') or a tail expression
                auto expr = parse_expression();
                if (match(TokenType::Semicolon)) {
                    stmts.push_back(std::make_unique<ExprStmtAST>(expr->location(), std::move(expr)));
                } else {
                    result_expr = std::move(expr);
                    break;
                }
            }
        }

        consume(TokenType::RightBrace, "Expected '}' to close block expression.");
        return std::make_unique<BlockExprAST>(loc, std::move(stmts), std::move(result_expr));
    }

    std::unique_ptr<ExprAST> parse_if_expression() {
        SourceLocation loc = previous_token().location;
        auto cond = parse_expression();
        consume(TokenType::Keyword_Then, "Expected 'then' after if condition.");
        auto then_b = parse_expression();
        consume(TokenType::Keyword_Else, "Expected 'else' in conditional expression.");
        auto else_b = parse_expression();

        return std::make_unique<IfExprAST>(loc, std::move(cond), std::move(then_b), std::move(else_b));
    }

    std::unique_ptr<ExprAST> parse_lambda_expression() {
        SourceLocation loc = previous_token().location;
        consume(TokenType::LeftParen, "Expected '(' after 'fn'.");

        std::vector<Parameter> params;
        if (!check(TokenType::RightParen)) {
            do {
                Token param_name = consume(TokenType::Identifier, "Expected parameter name.");
                consume(TokenType::Colon, "Expected ':' after parameter name.");
                TypeNode param_type = parse_type();
                params.push_back({std::string(param_name.lexeme), std::move(param_type), param_name.location});
            } while (match(TokenType::Comma));
        }

        consume(TokenType::RightParen, "Expected ')' after parameter list.");

        std::optional<TypeNode> ret_type;
        if (match(TokenType::Colon)) {
            ret_type = parse_type();
        }

        consume(TokenType::FatArrow, "Expected '=>' in function definition.");
        auto body = parse_expression();

        return std::make_unique<LambdaExprAST>(loc, std::move(params), std::move(ret_type), std::move(body));
    }
};

} // namespace Faberge

#endif // PARSER_HPP

Listing 9: Verification and AST Tree Visualisation (main.cpp)

In [ ]:
#include "../chapter-18/Lexer.hpp"
// #include "Parser.hpp"
// #include "ASTVisitor.hpp"
#include <iostream>

void test_parser(std::string_view title, std::string_view source_code) {
    using namespace Faberge;

    std::cout << "\n====================================================\n";
    std::cout << " TEST CASE: " << title << "\n";
    std::cout << "====================================================\n";
    std::cout << "SOURCE CODE:\n" << source_code << "\n----------------------------------------------------\n";

    // 1. Stage 1: Lexical Analysis
    Lexer lexer(source_code, "test.fab");
    auto tokens = lexer.tokenise();

    if (lexer.has_errors()) {
        std::cerr << "[Lexer Error]: Aborting parse due to scan errors.\n";
        return;
    }

    // 2. Stage 2: Syntactic Analysis
    Parser parser(std::move(tokens), source_code);
    auto ast = parser.parse_program();

    if (parser.has_errors()) {
        std::cerr << "\n[Parser Error]: Parsing failed with " << parser.error_count() << " error(s).\n\n";
        return;
    }

    // 3. Render AST
    std::cout << "ABSTRACT SYNTAX TREE (AST):\n";
    ASTPrinter printer;
    printer.print(*ast);
    std::cout << "----------------------------------------------------\n";
    std::cout << "[Success]: AST constructed cleanly.\n\n";
}

int main() {
    // Test 1: Complex Arithmetic and Operator Precedence
    constexpr std::string_view precedence_source = R"(
        let result = 1 + 2 * 3 == 7 && 10 > 5;
    )";
    test_parser("Arithmetic and Precedence Resolution", precedence_source);

    // Test 2: Recursive Functional Pipeline with Lambda and Block Expression
    constexpr std::string_view functional_source = R"(
        let factorial = fn (n: Int): Int => {
            if n <= 1 then 1 else n * factorial(n - 1)
        };

        let total = factorial(5) + 100;
    )";
    test_parser("Higher-Order Functional Pipeline", functional_source);

    // Test 3: Parser Error Diagnostics and Panic-Mode Recovery
    constexpr std::string_view broken_source = R"(
        let valid_a = 10;
        let broken_b = ; // Missing initialiser expression!
        let valid_c = 30 + 40;
    )";
    test_parser("Panic-Mode Error Recovery", broken_source);

    return 0;
}

### 19.7 Summary

In this second stage of our compiler project, we converted linear token
sequences into structured **Abstract Syntax Trees**:

- **Context-Free Grammars (Type-2)** provide the mathematical framework
  for defining recursively nested syntax.
- **Pratt Parsing (Top-Down Operator Precedence)** combines the
  readability of recursive descent with an $O(1)$ precedence table,
  solving complex operator precedence and associativity without deep
  call stacks.
- **The AST Polymorphic Hierarchy** represents language expressions
  cleanly using modern C++ `std::unique_ptr` ownership and
  double-dispatch `ASTVisitor` patterns.
- **Panic-Mode Error Recovery** guarantees that compiler front-ends
  synchronise gracefully after syntax errors rather than crashing midway
  through analysis.

### 19.8 Programming Challenges

#### Exercise 19.1: Right-Associative Exponentiation (`^`) in Pratt Parsing

**Context and Theory:**

Binary operators possess two orthogonal grammatical attributes:
**precedence** (which operator binds tighter) and **associativity** (how
operators of equal precedence group together).

Most arithmetic operators are **left-associative**, grouping expressions
from left to right:

$$a - b - c \equiv (a - b) - c$$

The exponentiation operator (`^`), however, is mathematically
**right-associative**, grouping expressions from right to left (see
fig. 16):

$$a ^ { b ^ {c} } \equiv a ^ { ( b ^ { c } ) }$$

In a Pratt parser, associativity is governed by the recursive binding
power passed into the right-hand expression call:

- **Left Associativity:** The right-hand operand is parsed with the
  operator’s exact precedence: `parse_expression(prec)`. A subsequent
  operator of identical precedence will fail the condition
  `prec < next_prec` and terminate the sub-expression.
- **Right Associativity:** The right-hand operand is parsed with a
  decremented precedence: `parse_expression(prec - 1)`. A subsequent
  operator of identical precedence satisfies the condition and continues
  binding recursively into the right child.

<figure id="fig:left-right-associative">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzMzhwdCIgaGVpZ2h0PSIxNDNwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzMzguMDAgMTQzLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDEz
OSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gbWludXMxIC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNz
PSJub2RlIj4KPHRpdGxlPm1pbnVzMTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIGN4PSI5OSIgY3k9Ii0xMTMiIHJ4PSIxMSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQt
YW5jaG9yPSJtaWRkbGUiIHg9Ijk5IiB5PSItMTEwLjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENo
YXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+KCYjNDU7KTwvdGV4dD4KPHRleHQg
dGV4dC1hbmNob3I9InN0YXJ0IiB4PSIwIiB5PSItMTI3IiBmb250LWZhbWlseT0iSW9zZXZrYSBD
aGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPkxlZnQmIzQ1O0Fzc29jaWF0aXZl
ICgmIzQ1Oyk6PC90ZXh0Pgo8L2c+CjwhLS0gbWludXMyIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNz
PSJub2RlIj4KPHRpdGxlPm1pbnVzMjwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIGN4PSIzOSIgY3k9Ii02MiIgcng9IjExIiByeT0iMTEiLz4KPHRleHQgdGV4dC1h
bmNob3I9Im1pZGRsZSIgeD0iMzkiIHk9Ii01OS41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFy
b24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPigmIzQ1Oyk8L3RleHQ+CjwvZz4KPCEt
LSBtaW51czEmIzQ1OyZndDttaW51czIgLS0+CjxnIGlkPSJlZGdlMSIgY2xhc3M9ImVkZ2UiPgo8
dGl0bGU+bWludXMxJiM0NTsmZ3Q7bWludXMyPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgZD0iTTkwLjY2LC0xMDUuMTlDODEuNTIsLTk3LjczIDY2LjY1LC04NS41OCA1
NS4yMSwtNzYuMjMiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50
cz0iNTcuMjMsLTczLjM3IDQ3LjI3LC02OS43NiA1Mi44LC03OC43OSA1Ny4yMywtNzMuMzciLz4K
PC9nPgo8IS0tIGMgLS0+CjxnIGlkPSJub2RlMyIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+YzwvdGl0
bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxMTkiIGN5PSItNjIi
IHJ4PSIxMSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjExOSIgeT0i
LTU5LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXpl
PSIxMC4wMCI+YzwvdGV4dD4KPC9nPgo8IS0tIG1pbnVzMSYjNDU7Jmd0O2MgLS0+CjxnIGlkPSJl
ZGdlMiIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+bWludXMxJiM0NTsmZ3Q7YzwvdGl0bGU+CjxwYXRo
IGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xMDIuODYsLTEwMi41NUMxMDUuMjQsLTk2
LjcyIDEwOC4zNywtODkuMDQgMTExLjI1LC04MS45OSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIg
c3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxMTQuNjIsLTgzIDExNS4xNiwtNzIuNDIgMTA4LjE0LC04
MC4zNSAxMTQuNjIsLTgzIi8+CjwvZz4KPCEtLSBhIC0tPgo8ZyBpZD0ibm9kZTQiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPmE8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNr
IiBjeD0iMzkiIGN5PSItMTEiIHJ4PSIxMSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjM5IiB5PSItOC41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBN
ZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPmE8L3RleHQ+CjwvZz4KPCEtLSBtaW51czImIzQ1OyZn
dDthIC0tPgo8ZyBpZD0iZWRnZTMiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPm1pbnVzMiYjNDU7Jmd0
O2E8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMzksLTUwLjYx
QzM5LC00NS4yMSAzOSwtMzguNDIgMzksLTMyLjA1Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBz
dHJva2U9ImJsYWNrIiBwb2ludHM9IjQyLjUsLTMyLjA0IDM5LC0yMi4wNCAzNS41LC0zMi4wNCA0
Mi41LC0zMi4wNCIvPgo8L2c+CjwhLS0gYiAtLT4KPGcgaWQ9Im5vZGU1IiBjbGFzcz0ibm9kZSI+
Cjx0aXRsZT5iPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9
IjExOSIgY3k9Ii0xMSIgcng9IjExIiByeT0iMTEiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRs
ZSIgeD0iMTE5IiB5PSItOC41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRp
dW0iIGZvbnQtc2l6ZT0iMTAuMDAiPmI8L3RleHQ+CjwvZz4KPCEtLSBtaW51czImIzQ1OyZndDti
IC0tPgo8ZyBpZD0iZWRnZTQiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPm1pbnVzMiYjNDU7Jmd0O2I8
L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNNDguMTgsLTU1LjM4
QzYxLjA4LC00Ny40OCA4NC44MywtMzIuOTMgMTAxLjI1LC0yMi44NyIvPgo8cG9seWdvbiBmaWxs
PSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxMDMuMjMsLTI1Ljc2IDEwOS45MywtMTcu
NTYgOTkuNTcsLTE5Ljc5IDEwMy4yMywtMjUuNzYiLz4KPC9nPgo8IS0tIGV4cDEgLS0+CjxnIGlk
PSJub2RlNiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+ZXhwMTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9
Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIyMTkiIGN5PSItMTEzIiByeD0iMTEiIHJ5PSIxMSIv
Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIyMTkiIHk9Ii0xMTAuNSIgZm9udC1mYW1p
bHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4oXik8L3Rl
eHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iMTA4IiB5PSItMTI3IiBmb250LWZhbWls
eT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPiAmIzE2MDtS
aWdodCYjNDU7QXNzb2NpYXRpdmUgKF4pOjwvdGV4dD4KPC9nPgo8IS0tIGExIC0tPgo8ZyBpZD0i
bm9kZTciIGNsYXNzPSJub2RlIj4KPHRpdGxlPmExPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9u
ZSIgc3Ryb2tlPSJibGFjayIgY3g9IjE5OSIgY3k9Ii02MiIgcng9IjExIiByeT0iMTEiLz4KPHRl
eHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTk5IiB5PSItNTkuNSIgZm9udC1mYW1pbHk9Iklv
c2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5hPC90ZXh0Pgo8L2c+
CjwhLS0gZXhwMSYjNDU7Jmd0O2ExIC0tPgo8ZyBpZD0iZWRnZTUiIGNsYXNzPSJlZGdlIj4KPHRp
dGxlPmV4cDEmIzQ1OyZndDthMTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIGQ9Ik0yMTUuMTQsLTEwMi41NUMyMTIuNzYsLTk2LjcyIDIwOS42MywtODkuMDQgMjA2Ljc1
LC04MS45OSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIy
MDkuODYsLTgwLjM1IDIwMi44NCwtNzIuNDIgMjAzLjM4LC04MyAyMDkuODYsLTgwLjM1Ii8+Cjwv
Zz4KPCEtLSBleHAyIC0tPgo8ZyBpZD0ibm9kZTgiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmV4cDI8
L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMjc5IiBjeT0i
LTYyIiByeD0iMTEiIHJ5PSIxMSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIyNzki
IHk9Ii01OS41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQt
c2l6ZT0iMTAuMDAiPiheKTwvdGV4dD4KPC9nPgo8IS0tIGV4cDEmIzQ1OyZndDtleHAyIC0tPgo8
ZyBpZD0iZWRnZTYiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmV4cDEmIzQ1OyZndDtleHAyPC90aXRs
ZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTIyNy4zNCwtMTA1LjE5QzIz
Ni40OCwtOTcuNzMgMjUxLjM1LC04NS41OCAyNjIuNzksLTc2LjIzIi8+Cjxwb2x5Z29uIGZpbGw9
ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjI2NS4yLC03OC43OSAyNzAuNzMsLTY5Ljc2
IDI2MC43NywtNzMuMzcgMjY1LjIsLTc4Ljc5Ii8+CjwvZz4KPCEtLSBiMSAtLT4KPGcgaWQ9Im5v
ZGU5IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5iMTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUi
IHN0cm9rZT0iYmxhY2siIGN4PSIyMzkiIGN5PSItMTEiIHJ4PSIxMSIgcnk9IjExIi8+Cjx0ZXh0
IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjIzOSIgeT0iLTguNSIgZm9udC1mYW1pbHk9Iklvc2V2
a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5iPC90ZXh0Pgo8L2c+Cjwh
LS0gZXhwMiYjNDU7Jmd0O2IxIC0tPgo8ZyBpZD0iZWRnZTciIGNsYXNzPSJlZGdlIj4KPHRpdGxl
PmV4cDImIzQ1OyZndDtiMTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2si
IGQ9Ik0yNzIuMzksLTUyLjkxQzI2Ni44NiwtNDYuMTMgMjU4LjgsLTM2LjI1IDI1MS45OCwtMjcu
OSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyNTQuNTIs
LTI1LjQ3IDI0NS40OCwtMTkuOTQgMjQ5LjA5LC0yOS45IDI1NC41MiwtMjUuNDciLz4KPC9nPgo8
IS0tIGMxIC0tPgo8ZyBpZD0ibm9kZTEwIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5jMTwvdGl0bGU+
CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIzMTkiIGN5PSItMTEiIHJ4
PSIxMSIgcnk9IjExIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjMxOSIgeT0iLTgu
NSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEw
LjAwIj5jPC90ZXh0Pgo8L2c+CjwhLS0gZXhwMiYjNDU7Jmd0O2MxIC0tPgo8ZyBpZD0iZWRnZTgi
IGNsYXNzPSJlZGdlIj4KPHRpdGxlPmV4cDImIzQ1OyZndDtjMTwvdGl0bGU+CjxwYXRoIGZpbGw9
Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0yODUuNjEsLTUyLjkxQzI5MS4xNCwtNDYuMTMgMjk5
LjIsLTM2LjI1IDMwNi4wMiwtMjcuOSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJi
bGFjayIgcG9pbnRzPSIzMDguOTEsLTI5LjkgMzEyLjUyLC0xOS45NCAzMDMuNDgsLTI1LjQ3IDMw
OC45MSwtMjkuOSIvPgo8L2c+CjwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 16: Left and Right Associativity</figcaption>
</figure>

**Specification:**

Extend Faberge’s Pratt expression parser to support the exponentiation
operator `TokenType::Caret` (`^`). Assign it higher precedence than
multiplication/division ($70$). Implement right-associative recursion in
`parse_infix()`, and update AST nodes to construct the proper binary
expression hierarchy.

``` cpp
class PrattParserExtension {
public:
    static int get_infix_precedence(TokenType type) noexcept {
        switch (type) {
            case TokenType::Caret:    return 70; // Exponentiation: right-associative
            case TokenType::Asterisk:
            case TokenType::Slash:    return 60; // Multiplicative
            case TokenType::Plus:
            case TokenType::Minus:    return 50; // Additive
            default:                  return 0;
        }
    }
    static bool is_right_associative(TokenType type) noexcept {
        return type == TokenType::Caret;
    }
    std::unique_ptr<ExprAST> parse_infix_operator(std::unique_ptr<ExprAST> left, Token op_tok) {
        int prec = get_infix_precedence(op_tok.type);
        int next_min_prec = is_right_associative(op_tok.type) ? (prec - 1) : prec;
        auto right = parse_expression(next_min_prec);
        return std::make_unique<BinaryExprAST>(op_tok.location, op_tok.type, std::move(left), std::move(right));
    }
};
```

**Verification Tasks:**

1.  **Right-Associativity Grouping:** Parse the expression `2 ^ 3 ^ 2`.
    Verify that the root AST node is a `BinaryExprAST(^)` whose left
    child is `Literal(2)` and whose right child is
    `BinaryExprAST(3 ^ 2)` (evaluating to $2^9 = 512$, not $8^2 = 64$).
2.  **Precedence over Multiplication:** Parse `2 * 3 ^ 2`. Verify that
    the root is `BinaryExprAST(*)` with left child `2` and right child
    `BinaryExprAST(3 ^ 2)`.
3.  **Parenthesised Override:** Parse `(2 ^ 3) ^ 2`. Verify that the
    grouping overrides right-associativity, producing a root with left
    child `BinaryExprAST(2 ^ 3)` and right child `Literal(2)`.

#### Exercise 19.2: Subscript Indexing and Array/List Literal AST Nodes

**Context and Theory:**

A functional language requires syntax for constructing sequential data
structures and indexing into them. In Faberge, array/list literals and
subscript index access are specified via:

$$\text{ListLiteral} \to \text{"["} \; (\text{Expr} \; (\text{","} \; \text{Expr})^*)^? \; \text{"]"}$$

$$\text{IndexAccess} \to \text{Expr} \; \text{"["} \; \text{Expr} \; \text{"]"}$$

In Pratt parsing, brackets serve two distinct semantic functions
depending on their syntactic context (see fig. 17):

1.  **Prefix Denotation (`nud`):** When `[` appears at the start of an
    expression, it acts as a list constructor literal, parsing
    comma-separated elements until `]`.
2.  **Infix Denotation (`led`):** When `[` appears immediately following
    an expression, it acts as a postfix/infix operator of high
    precedence, parsing the index expression and consuming the closing
    `]`.

<figure id="fig:subscript">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2NDIuNTI4IiBo
ZWlnaHQ9IjU5Ljg3MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA2NDIuNTI4IDU5Ljg3
MiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI2NDIu
NTI4IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjE3Ni4zNDYwMSIgeT0iMTAuNDc2IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSIxOTAuMzE0MDEiIHk9IjEwLjQ3NiIgPjEsPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjIxMS4yNjYiIHk9IjEwLjQ3NiIgPjIsPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIzMi4yMTgiIHk9IjEwLjQ3NiIgPjM8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjQ2LjE4NiIgeT0iMTAuNDc2IiA+XTwvdGV4
dD4KICA8bGluZSB4MT0iMjkzLjMyOCIgeTE9IjYuOTg0IiB4Mj0iMzE0LjI4IiB5Mj0iNi45ODQi
IGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzE0LjI4LDMuNDkyIDMy
MS4yNjQsNi45ODQgMzE0LjI4LDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMyOS45OTQwMiIgeT0iMTAuNDc2IiA+TGlzdExp
dGVyYWxFeHByQVNUPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE3Ni4z
NDYwMSIgeT0iMzguNDEyMDAzIiA+bWF0cml4W3Jvd11bY29sXTwvdGV4dD4KICA8bGluZSB4MT0i
MjkzLjMyOCIgeTE9IjM0LjkyMDAwMiIgeDI9IjMxNC4yOCIgeTI9IjM0LjkyMDAwMiIgY2xhc3M9
InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIzMTQuMjgsMzEuNDI4MDAxIDMyMS4y
NjQsMzQuOTIwMDAyIDMxNC4yOCwzOC40MTIwMDMiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjc0NiIgeT0iMTAuNDc2IiA+QXJyYXkg
TGl0ZXJhbCAoUHJlZml4KTo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MS43NDYiIHk9IjM4LjQxMjAwMyIgPkFycmF5IEluZGV4IChJbmZpeCk6PC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMyOS45OTQwMiIgeT0iMzguNDEyMDAzIiA+SW5kZXhF
eHByQVNUKEluZGV4RXhwckFTVChtYXRyaXgsIHJvdyksIGNvbCk8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 17: Prefix and Infix Denotation</figcaption>
</figure>

**Specification:**

Create `ListLiteralExprAST` and `IndexExprAST` classes inheriting from
`ExprAST`. Extend `parse_prefix()` to parse list literals upon
encountering `TokenType::LeftBracket`, and extend `parse_infix()` to
parse array subscripting with high precedence ($80$).

``` cpp
class ListLiteralExprAST : public ExprAST {
public:
    ListLiteralExprAST(SourceLocation loc, std::vector<std::unique_ptr<ExprAST>> elements)
        : ExprAST(loc), m_elements(std::move(elements)) {}
    const auto& elements() const noexcept { return m_elements; }
    void accept(ASTVisitor& visitor) const override;
private:
    std::vector<std::unique_ptr<ExprAST>> m_elements;
};

class IndexExprAST : public ExprAST {
public:
    IndexExprAST(SourceLocation loc, std::unique_ptr<ExprAST> target, std::unique_ptr<ExprAST> index)
        : ExprAST(loc), m_target(std::move(target)), m_index(std::move(index)) {}
    const ExprAST& target() const noexcept { return *m_target; }
    const ExprAST& index() const noexcept { return *m_index; }
    void accept(ASTVisitor& visitor) const override;
private:
    std::unique_ptr<ExprAST> m_target, m_index;
};
```

**Verification Tasks:**

1.  **Empty and Populated Literals:** Parse `[]` and
    `[1, 2 + 3, "four"]`. Verify that `ListLiteralExprAST` stores 0 and
    3 element subtrees respectively.
2.  **Chained Subscripting:** Parse `grid[x][y + 1]`. Verify that the
    root is an `IndexExprAST` whose target is another
    `IndexExprAST(grid, x)` and whose index is `BinaryExprAST(y + 1)`.
3.  **Unclosed Bracket Recovery:** Parse `arr[i + 1;`. Verify that the
    parser flags the missing `]` delimiter at the semicolon and triggers
    panic-mode synchronisation without crashing.

#### Exercise 19.3: Pattern Matching and Destructuring AST Grammar

**Context and Theory:**

Functional languages replace complex nested `if-else` cascades with
**Pattern Matching**. In Faberge, a `match` expression inspects a target
expression and dispatches to the first arm whose structural pattern
satisfies the value:

$$\text{MatchExpr} \to \text{"match"} \; \text{Expr} \; \text{"with"} \; \text{"\{"} \; \text{MatchArm}^* \; \text{"\}"}$$

$$\text{MatchArm} \to \text{Pattern} \; \text{"=>"} \; \text{Expr} \; \text{";"}^?$$

$$\text{Pattern} \to \text{Literal} \mid \text{Identifier} \mid \text{"\_"} \mid \text{"("} \; \text{Pattern} \; (\text{","} \; \text{Pattern})^* \; \text{")"}$$

A pattern can bind new variables in the arm’s local scope (e.g. `(x, y)`
extracts tuple components), match exact scalar literals
(e.g. `0 => "zero"`), or match anything via a wildcard `_`.

```
match point with {
    (0, 0) => "origin",
    (x, 0) => "x-axis",
    _      => "generic point"
}
```

**Specification:**

Define a `PatternAST` base class with concrete subclasses:
`LiteralPatternAST`, `VariablePatternAST`, `WildcardPatternAST`, and
`TuplePatternAST`. Implement a `MatchExprAST` node storing the target
expression and a list of `MatchArm` descriptors. Extend the parser with
`parse_match_expression()` and `parse_pattern()`.

``` cpp
struct MatchArm {
    std::unique_ptr<PatternAST> pattern;
    std::unique_ptr<ExprAST> body;
};

class MatchExprAST : public ExprAST {
public:
    MatchExprAST(SourceLocation loc, std::unique_ptr<ExprAST> target, std::vector<MatchArm> arms)
        : ExprAST(loc), m_target(std::move(target)), m_arms(std::move(arms)) {}
    const ExprAST& target() const noexcept { return *m_target; }
    const std::vector<MatchArm>& arms() const noexcept { return m_arms; }
    void accept(ASTVisitor& visitor) const override;
private:
    std::unique_ptr<ExprAST> m_target;
    std::vector<MatchArm> m_arms;
};
```

**Verification Tasks:**

1.  **Wildcard and Literal Arm Parsing:** Parse
    `match x with { 1 => "one", _ => "other" }`. Verify that the
    resulting `MatchExprAST` contains exactly two arms, with the second
    arm possessing a `WildcardPatternAST`.
2.  **Tuple Destructuring Pattern:** Parse
    `match pair with { (a, b) => a + b }`. Verify that the arm’s pattern
    parses as a `TuplePatternAST` containing two variable patterns (`a`
    and `b`).
3.  **Missing Fat Arrow Diagnostic:** Parse
    `match x with { 1 "error" }`. Verify that the parser detects the
    missing `=>` separator and reports a syntax error indicating the
    expected delimiter.

#### Exercise 19.4: Product Types: Heterogeneous Tuples

**Context and Theory:**

A **Tuple** represents an anonymous product type
$(v_1, v_2, \dots, v_n)$. In Faberge, tuples are declared using
comma-separated parenthesised expressions:

$$\text{TupleExpr} \to \text{"("} \; \text{Expr} \; \text{","} \; (\text{Expr} \; (\text{","} \; \text{Expr})^*)^? \; \text{")"} \mid \text{"("} \; \text{")"}$$

This construct introduces a classic parsing ambiguity with grouping
parentheses:

- `(x + 1)` is a grouped arithmetic expression containing a single inner
  AST node.
- `(x + 1,)` or `(x, y)` is a 2-tuple constructing a product value.
- `()` is a 0-tuple (the `Unit` or `Void` value).

The parser must disambiguate this at the opening parenthesis: parse the
first expression, and inspect the subsequent token. If followed by a
comma `,`, it transitions to tuple parsing; if followed by `)`, it
unwraps and returns the single grouped expression (see fig. 18).

<figure id="fig:parse-tuple">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1OTMuNjQiIGhl
aWdodD0iODcuODA4IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDU5My42NCA4Ny44MDgi
PgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5z
dmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13
aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJv
a2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0
ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBJ
b3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTRweDsK
fQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNw
YXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2
Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBm
aWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7
CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewog
IG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9u
ZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2Vk
X2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5k
X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5z
dmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9j
aXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1z
dGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5f
Y2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2Ig
LnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdf
b3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ci
IHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1h
cmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdv
biBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8
bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIi
IG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHln
b24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDgg
OCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3Jp
ZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0i
MiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9
Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJX
aWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAg
ICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNs
ZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94
PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0
PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5
PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwv
ZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNTkzLjY0
IiBoZWlnaHQ9Ijg3LjgwOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+VG9rZW48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iNTAuNjM0MDAzIiB5PSIxMC40NzYiID5TdHJlYW06PC90ZXh0PgogIDxsaW5l
IHgxPSIxNDYuNjY0IiB5MT0iNi45ODQiIHgyPSIxNjcuNjE2IiB5Mj0iNi45ODQiIGNsYXNzPSJz
b2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMTY3LjYxNiwzLjQ5MiAxNzQuNiw2Ljk4
NCAxNjcuNjE2LDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjE4My4zMyIgeT0iMTAuNDc2IiA+RXhwcjwvdGV4dD4KICA8bGlu
ZSB4MT0iMjE2LjUwNDAxIiB5MT0iNi45ODQiIHgyPSIyMzcuNDU2MDEiIHkyPSI2Ljk4NCIgY2xh
c3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyMzcuNDU2MDEsMy40OTIgMjQ0
LjQ0LDYuOTg0IDIzNy40NTYwMSwxMC40NzYiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8
bGluZSB4MT0iMjg2LjM0NCIgeTE9IjYuOTg0IiB4Mj0iMzA3LjI5NjAyIiB5Mj0iNi45ODQiIGNs
YXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzA3LjI5NjAyLDMuNDkyIDMx
NC4yOCw2Ljk4NCAzMDcuMjk2MDIsMTAuNDc2IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzIzLjAxIiB5PSIxMC40NzYiID5UdXBsZUV4
cHJBU1Q8L3RleHQ+CiAgPGxpbmUgeDE9IjE0Ni42NjQiIHkxPSIzNC45MjAwMDIiIHgyPSIxNjcu
NjE2IiB5Mj0iMzQuOTIwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2lu
dHM9IjE2Ny42MTYsMzEuNDI4MDAxIDE3NC42LDM0LjkyMDAwMiAxNjcuNjE2LDM4LjQxMjAwMyIg
Y2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjE4My4zMyIgeT0iMzguNDEyMDAzIiA+RXhwcjwvdGV4dD4KICA8bGluZSB4MT0iMjE2LjUwNDAx
IiB5MT0iMzQuOTIwMDAyIiB4Mj0iMjM3LjQ1NjAxIiB5Mj0iMzQuOTIwMDAyIiBjbGFzcz0ic29s
aWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjIzNy40NTYwMSwzMS40MjgwMDEgMjQ0LjQ0
LDM0LjkyMDAwMiAyMzcuNDU2MDEsMzguNDEyMDAzIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+
CiAgPGxpbmUgeDE9IjI4Ni4zNDQiIHkxPSIzNC45MjAwMDIiIHgyPSIzMDcuMjk2MDIiIHkyPSIz
NC45MjAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzA3LjI5
NjAyLDMxLjQyODAwMSAzMTQuMjgsMzQuOTIwMDAyIDMwNy4yOTYwMiwzOC40MTIwMDMiIGNsYXNz
PSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjMu
MDEiIHk9IjM4LjQxMjAwMyIgPkdyb3VwZWQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMzc4Ljg4MjAyIiB5PSIzOC40MTIwMDMiID5FeHByZXNzaW9uPC90ZXh0PgogIDxs
aW5lIHgxPSIxNDYuNjY0IiB5MT0iNjIuODU2MDAzIiB4Mj0iMjM3LjQ1NjAxIiB5Mj0iNjIuODU2
MDAzIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjIzNy40NTYwMSw1
OS4zNjQwMDIgMjQ0LjQ0LDYyLjg1NjAwMyAyMzcuNDU2MDEsNjYuMzQ4IiBjbGFzcz0iZmlsbGVk
Ij48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjI4Ni4zNDQiIHkxPSI2Mi44NTYwMDMiIHgyPSIzMDcu
Mjk2MDIiIHkyPSI2Mi44NTYwMDMiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBv
aW50cz0iMzA3LjI5NjAyLDU5LjM2NDAwMiAzMTQuMjgsNjIuODU2MDAzIDMwNy4yOTYwMiw2Ni4z
NDgiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIzMjMuMDEiIHk9IjY2LjM0OCIgPlR1cGxlRXhwckFTVDwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIxMTMuNDkwMDA1IiB5PSIxMC40NzYiID4mIzM5OygmIzM5Ozwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNTMuMTcwMDEiIHk9IjEwLjQ3
NiIgPiYjMzk7LCYjMzk7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQx
My44MDIiIHk9IjEwLjQ3NiIgPihNdWx0aS1lbGVtZW50IHByb2R1Y3QpPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjExMy40OTAwMDUiIHk9IjM4LjQxMjAwMyIgPiYjMzk7
KCYjMzk7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1My4xNzAwMSIg
eT0iMzguNDEyMDAzIiA+JiMzOTspJiMzOTs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iNDU1LjcwNjAyIiB5PSIzOC40MTIwMDMiID4oVW53cmFwcGVkIEV4cHJBU1QpPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjExMy40OTAwMDUiIHk9IjY2LjM0
OCIgPiYjMzk7KCYjMzk7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1
My4xNzAwMSIgeT0iNjYuMzQ4IiA+JiMzOTspJiMzOTs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iNDEzLjgwMiIgeT0iNjYuMzQ4IiA+KDAtdHVwbGUgLyBVbml0KTwvdGV4
dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 18: Correct Parsing of Tuples</figcaption>
</figure>

**Specification:**

Create a `TupleExprAST` class storing a
`std::vector<std::unique_ptr<ExprAST>>`. Refactor `parse_prefix()` when
consuming `TokenType::LeftParen` to distinguish cleanly between unit
literals, parenthesised expressions, and multi-element tuples.

``` cpp
class TupleExprAST : public ExprAST {
public:
    TupleExprAST(SourceLocation loc, std::vector<std::unique_ptr<ExprAST>> elements)
        : ExprAST(loc), m_elements(std::move(elements)) {}
    const auto& elements() const noexcept { return m_elements; }
    [[nodiscard]] size_t arity() const noexcept { return m_elements.size(); }
    void accept(ASTVisitor& visitor) const override;
private:
    std::vector<std::unique_ptr<ExprAST>> m_elements;
};
```

**Verification Tasks:**

1.  **Unit Literal Disambiguation:** Parse `()`. Verify that the parser
    emits a `TupleExprAST` with `arity() == 0`.
2.  **Grouping vs Tuple Invariant:** Parse `(42)` and `(42,)`. Verify
    that the former yields a raw `LiteralExprAST(42)` without tuple
    wrapping, while the latter yields a `TupleExprAST` of arity 1.
3.  **Nested Heterogeneous Products:** Parse `(1, "two", (3.0, true))`.
    Verify that the root tuple has arity 3, and its third element is a
    nested `TupleExprAST` of arity 2.

#### Exercise 19.5: Postfix Factorial and Unary Increment Operators

**Context and Theory:**

Standard unary operators (such as negative `-x` or logical negation
`!x`) are **prefix operators**, preceding their operand. Conversely,
operators such as postfix factorial ($n!$) or increment ($x\text{++}$)
are **postfix operators**, trailing their operand.

In Pratt parsing, postfix operators are handled as a special case of the
infix denotation (`led`, see fig. 19):

1.  A postfix operator appears after an expression: it is registered in
    the infix operator table with very high precedence ($85$).
2.  Unlike a binary infix operator (which consumes a right-hand
    operand), a postfix operator **consumes no further expressions**. It
    wraps the existing `left` AST node into a `UnaryExprAST` (or
    `PostfixExprAST`) and returns immediately.

<figure id="fig:parse-postfix">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MTYuODE2MDQi
IGhlaWdodD0iMTQzLjY4MDAxIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDUxNi44MTYw
NCAxNDMuNjgwMDEiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2Jv
YiBjaXJjbGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNr
OwogIHN0cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0
eTogMTsKICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsK
fQoKLnN2Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZv
bnQtZmFtaWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQt
c2l6ZTogMTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJh
eTogODsKfQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdf
ZmlsbGVkIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdi
b2IgLm5vZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
YXJyb3cgewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfYXJyb3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9t
YXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsK
fQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Np
cmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6
IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUg
ewogIG1hcmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7
Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtl
ciBpZD0iYXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2Vy
V2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgog
ICAgICA8cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9t
YXJrZXI+CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9
IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0
by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQg
MCwyIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmll
d0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhl
aWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0
IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZ
PSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQt
cmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmls
bGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2ly
Y2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJj
bGUgY3g9IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwv
bWFya2VyPgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3
aWR0aD0iNTE2LjgxNjA0IiBoZWlnaHQ9IjE0My42ODAwMSI+PC9yZWN0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+U3RyZWFtOjwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3MS41ODYiIHk9IjEwLjQ3NiIgPm48L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iODUuNTU0IiB5PSIxMC40NzYiID4h
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijk5LjUyMiIgeT0iMTAuNDc2
IiA+KzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMTMuNDkwMDA1IiB5
PSIxMC40NzYiID4xPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE1Ljcx
NDAwMSIgeT0iMTA4LjI1MjAxIiA+U3RlcDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI1MC42MzQwMDMiIHk9IjEwOC4yNTIwMSIgPjM6PC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjcxLjU4NiIgeT0iMTA4LjI1MjAxIiA+JiMzOTs8L3RleHQ+CiAg
PGxpbmUgeDE9Ijc2LjgyNDAwNSIgeTE9IjEwNC43NiIgeDI9IjgzLjgwOCIgeTI9IjEwNC43NiIg
Y2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iODUu
NTU0IiB5PSIxMDguMjUyMDEiID4mIzM5OzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI5OS41MjIiIHk9IjEwOC4yNTIwMSIgPmVuY291bnRlcmVkLjwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxOTAuMzE0MDEiIHk9IjEwOC4yNTIwMSIgPlByZWNl
ZGVuY2U8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjY3LjEzOCIgeT0i
MTA4LjI1MjAxIiA+NTA8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjg4
LjA5IiB5PSIxMDguMjUyMDEiID4mbHQ7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjMwMi4wNTgiIHk9IjEwOC4yNTIwMSIgPjg1LjwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSI3MS41ODYiIHk9IjEyMi4yMiIgPldyYXBzPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjExMy40OTAwMDUiIHk9IjEyMi4yMiIgPlBvc3RmaXhF
eHByQVNUPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIxOC4yNSIgeT0i
MTIyLjIyIiA+YW5kPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI0Ni4x
ODYiIHk9IjEyMi4yMiIgPnBhcnNlczwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIyOTUuMDc0IiB5PSIxMjIuMjIiID5SSFM8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMzIzLjAxIiB5PSIxMjIuMjIiID4mIzM5OzEmIzM5Oy48L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTUuNzE0MDAxIiB5PSIzOC40MTIwMDMiID5TdGVw
IDE6ICYjMzk7biYjMzk7IHBhcnNlZCBhcyBQcmltYXJ5IChudWQpPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjE1LjcxNDAwMSIgeT0iNjYuMzQ4IiA+U3RlcCAyOiAmIzM5
OyEmIzM5OyBlbmNvdW50ZXJlZC4gUHJlY2VkZW5jZSA4NSAmZ3Q7IDAuPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjcxLjU4NiIgeT0iODAuMzE2IiA+V3JhcHMgJiMzOTtu
JiMzOTsgaW50byBQb3N0Zml4RXhwckFTVChuLCAhKS4gQ29uc3VtZXMgbm8gcmlnaHQgb3BlcmFu
ZCE8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 19: Precedence-Based Parsing of Postfix
Operator</figcaption>
</figure>

**Specification:**

Implement a `PostfixExprAST` node. Register postfix operators
`TokenType::Exclamation` (when trailing an expression) and
`TokenType::PlusPlus` in the Pratt parser’s infix table. Implement the
zero-RHS consumption logic inside `parse_infix()`.

``` cpp
class PostfixExprAST : public ExprAST {
public:
    PostfixExprAST(SourceLocation loc, TokenType op, std::unique_ptr<ExprAST> operand)
        : ExprAST(loc), m_op(op), m_operand(std::move(operand)) {}
    TokenType op() const noexcept { return m_op; }
    const ExprAST& operand() const noexcept { return *m_operand; }
    void accept(ASTVisitor& visitor) const override;
private:
    TokenType m_op;
    std::unique_ptr<ExprAST> m_operand;
};
```

**Verification Tasks:**

1.  **Postfix vs Prefix Invariance:** Parse `!x!`. Verify that the root
    is a prefix `UnaryExprAST(!)` whose operand is a
    `PostfixExprAST(x, !)`.
2.  **Precedence over Binary Operators:** Parse `n! * 2`. Verify that
    the multiplication operator binds above the factorial, yielding
    `BinaryExprAST(PostfixExprAST(n, !), *, 2)`.
3.  **Chained Postfix Execution:** Parse `n!!`. Verify that the AST
    groups into `PostfixExprAST(PostfixExprAST(n, !), !)`.

#### Exercise 19.6: High-Fidelity JSON AST Serialiser (Visitor Engine)

**Context and Theory:**

Modern compiler toolchains require structural interchange formats for
language servers (LSP), static analysers, and downstream code emitters
(such as the Faberge C++ emitter).

The **Visitor Pattern** enables recursive AST traversal without mutating
node classes. A `JsonASTSerializer` visits every node in the AST and
generates a well-formed JSON object:

- `kind`: The concrete node type name (e.g. `"LetStmt"`,
  `"BinaryExpr"`).
- `location`: The starting line, column, and character offset.
- Node-specific attributes (e.g. `operator`, `variable_name`,
  `literal_value`).
- Nested child subtrees recursively serialised into JSON arrays or
  sub-objects.

To demonstrate, the JSON Serialisation of `LetStmtAST('x', Literal(10))`
should be:

``` json
 {
   "kind": "LetStmt",
   "name": "x",
   "loc": { "line": 1, "col": 1 },
   "init": { "kind": "Literal", "type": "Int", "value": "10" }
 }
```

**Specification:**

Implement a `JsonASTSerializer` class inheriting from `ASTVisitor`.
Provide an entry method `serialize(const ASTNode&)` returning a
formatted JSON string representing the entire AST hierarchy.

``` cpp
class JsonASTSerializer : public ASTVisitor {
public:
    std::string serialize(const ASTNode& node) {
        m_buffer.str("");
        m_buffer.clear();
        node.accept(*this);
        return m_buffer.str();
    }
    void visit(const LetStmtAST& node) override;
    void visit(const BinaryExprAST& node) override;
    void visit(const LiteralExprAST& node) override;
    // ... Implement remaining node visitors ...
private:
    std::ostringstream m_buffer;
    void emit_location(SourceLocation loc);
};
```

**Verification Tasks:**

1.  **Valid JSON Formatting:** Serialise a program containing variables,
    lambdas, and conditionals. Parse the resulting string with a
    standard JSON validator and assert zero parsing errors.
2.  **Location Accuracy:** Verify that the JSON output contains `"line"`
    and `"column"` keys matching the source coordinate of every
    expression.
3.  **Full Subtree Traversal:** Serialise a complex arithmetic
    expression tree `1 + 2 * 3`. Verify that the JSON reflects the
    binary tree structure with `"left"` and `"right"` child keys.

#### Exercise 19.7: Ternary Conditional Operator Integration (`? :`)

**Context and Theory:**

Many programming languages provide the ternary conditional operator as a
concise alternative to `if-then-else`:

$$\text{TernaryExpr} \to \text{Condition} \; \text{"?"} \; \text{ThenExpr} \; \text{":"} \; \text{ElseExpr}$$

The ternary operator is a **mixfix (ternary) operator** requiring two
separate delimiters (`?` and `:`). In Pratt parsing (as shown in
fig. 20):

1.  `?` is treated as an infix operator with low precedence ($25$,
    binding just above assignment).
2.  When `?` is encountered, the parser consumes the `then_branch`
    expression with precedence $0$ (allowing any nested expression).
3.  The parser strictly asserts and consumes the `:` delimiter.
4.  The `else_branch` expression is parsed with precedence $24$ (one
    less than the ternary precedence) to ensure **right-associativity**:

$$c_1 \mathbin{?} e_1 \mathbin{:} c_2 \mathbin{?} e_2 \mathbin{:} e_3 \equiv c_1 \mathbin{?} e_1 \mathbin{:} (c_2 \mathbin{?} e_2 \mathbin{:} e_3)$$

<figure id="fig:parse-ternary">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0MTIuMDU2IiBo
ZWlnaHQ9IjczLjg0MDAwNCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA0MTIuMDU2IDcz
Ljg0MDAwNCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNp
cmNsZSwgLnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAg
c3Ryb2tlLXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAx
OwogIHN0cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgou
c3ZnYm9iIHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1m
YW1pbHk6IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXpl
OiAxNHB4Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxs
OiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4
Owp9Cgouc3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxs
ZWQgewogIGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAu
bm9maWxsIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJv
dyB7CiAgbWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9h
cnJvdyB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtl
ZF9kaWFtb25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3Zn
Ym9iIC5lbmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgou
c3ZnYm9iIC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJs
KCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAg
bWFya2VyLXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9i
aWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1
cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlk
PSJhcnJvdyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxwb2x5Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtl
cj4KICAgIDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIg
cmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0
YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIi
PjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94
PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0
PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5
PSI0IiByPSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1h
cmtlciBpZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQi
IG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQi
PjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUi
IHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJr
ZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBj
eD0iNCIgY3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJr
ZXI+CiAgPC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRo
PSI0MTIuMDU2IiBoZWlnaHQ9IjczLjg0MDAwNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+U3RyZWFtOjwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2NC42MDIwMDUiIHk9IjEwLjQ3NiIgPmNvbmQ8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOTkuNTIyIiB5PSIxMC40NzYiID4/
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjExMy40OTAwMDUiIHk9IjEw
LjQ3NiIgPkE8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTQxLjQyNjAx
IiB5PSIxMC40NzYiID5CPC90ZXh0PgogIDxwb2x5Z29uIHBvaW50cz0iMTY3LjYxNiwzMS40Mjgw
MDEgMTc0LjYsMzQuOTIwMDAyIDE2Ny42MTYsMzguNDEyMDAzIiBjbGFzcz0iZmlsbGVkIj48L3Bv
bHlnb24+CiAgPHBvbHlnb24gcG9pbnRzPSIxNjcuNjE2LDQ1LjM5NiAxNzQuNiw0OC44ODggMTY3
LjYxNiw1Mi4zOCIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjEyNy40NTgiIHk9IjEwLjQ3NiIgPjo8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMTgzLjMzIiB5PSIzOC40MTIwMDMiID5SSFM8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTgzLjMzIiB5PSI1Mi4zOCIgPk1pZDwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2NC42MDIwMDUiIHk9IjUyLjM4IiA+TEhT
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIxMS4yNjYiIHk9IjM4LjQx
MjAwMyIgPihQYXJzZWQgYWZ0ZXIgJiMzOTs6JiMzOTspPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjIxMS4yNjYiIHk9IjUyLjM4IiA+KFBhcnNlZCBiZXR3ZWVuICYjMzk7
PyYjMzk7IGFuZCAmIzM5OzomIzM5Oyk8L3RleHQ+CiAgPGc+CiAgICA8bGluZSB4MT0iNjIuODU2
MDAzIiB5MT0iMTMuOTY4IiB4Mj0iNjkuODQwMDA0IiB5Mj0iMjcuOTM2IiBjbGFzcz0ic29saWQi
PjwvbGluZT4KICAgIDxsaW5lIHgxPSI2OS44NDAwMDQiIHkxPSIyNy45MzYiIHgyPSI4My44MDgi
IHkyPSIyNy45MzYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjkwLjc5MiIg
eTE9IjEzLjk2OCIgeDI9IjgzLjgwOCIgeTI9IjI3LjkzNiIgY2xhc3M9InNvbGlkIj48L2xpbmU+
CiAgPC9nPgogIDxnPgogICAgPGxpbmUgeDE9IjExNS4yMzYiIHkxPSIxMy45NjgiIHgyPSIxMTUu
MjM2IiB5Mj0iNDguODg4IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSIxMTUu
MjM2IiB5MT0iNDguODg4IiB4Mj0iMTY3LjYxNiIgeTI9IjQ4Ljg4OCIgY2xhc3M9InNvbGlkIj48
L2xpbmU+CiAgPC9nPgogIDxnPgogICAgPGxpbmUgeDE9IjE0My4xNzIiIHkxPSIxMy45NjgiIHgy
PSIxNDMuMTcyIiB5Mj0iMzQuOTIwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5l
IHgxPSIxNDMuMTcyIiB5MT0iMzQuOTIwMDAyIiB4Mj0iMTY3LjYxNiIgeTI9IjM0LjkyMDAwMiIg
Y2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 20: Correct Parsing of Ternary Operator</figcaption>
</figure>

**Specification:**

Implement a `TernaryExprAST` node class. Extend Faberge’s Pratt parser
to register `TokenType::Question` in the infix table. Parse the middle
expression, consume `TokenType::Colon`, and recursively parse the right
branch with right-associative binding power.

``` cpp
class TernaryExprAST : public ExprAST {
public:
    TernaryExprAST(SourceLocation loc, std::unique_ptr<ExprAST> cond,
                   std::unique_ptr<ExprAST> then_expr, std::unique_ptr<ExprAST> else_expr)
        : ExprAST(loc), m_condition(std::move(cond)), 
          m_then_expr(std::move(then_expr)), m_else_expr(std::move(else_expr)) {}
    const ExprAST& condition() const noexcept { return *m_condition; }
    const ExprAST& then_expr() const noexcept { return *m_then_expr; }
    const ExprAST& else_expr() const noexcept { return *m_else_expr; }
    void accept(ASTVisitor& visitor) const override;
private:
    std::unique_ptr<ExprAST> m_condition, m_then_expr, m_else_expr;
};
```

**Verification Tasks:**

1.  **Standard Ternary Parsing:** Parse `x > 0 ? 1 : -1`. Verify that
    the resulting `TernaryExprAST` accurately separates condition,
    then-expression, and else-expression.
2.  **Right-Associativity Chaining:** Parse `a ? 1 : b ? 2 : 3`. Verify
    that the root else-branch contains another nested
    `TernaryExprAST(b ? 2 : 3)`.
3.  **Missing Colon Syntax Recovery:** Parse `a ? 1 ;`. Verify that the
    parser flags the missing `:` delimiter at the semicolon, records a
    syntax diagnostic, and recovers gracefully.

#### Exercise 19.8: AST Linting: Redundant Grouping Elimination

**Context and Theory:**

Developers frequently introduce redundant grouping parentheses, either
out of habit or uncertainty over operator precedence:

```
let result = ((x * y)) + (z);
```

While harmless to evaluation semantics, redundant groupings clutter AST
structures, inflate memory usage, and introduce noise into downstream
code emitters.

An **AST Linter Pass** traverses the tree and identifies redundant
parentheses:

1.  **Duplicate Enclosure:** Any expression directly wrapped in multiple
    consecutive grouping parentheses `((e))`.
2.  **Natural Precedence Enclosure:** A sub-expression enclosed in
    parentheses whose operator naturally binds tighter than its parent
    operator (e.g. `(a * b) + c`, where `*` already has higher
    precedence than `+`).
3.  **Essential Grouping:** Parentheses that actively override
    precedence (e.g. `(a + b) * c`) are non-redundant and must be
    preserved.

- **Redundant:** `(a * b) + c` → Naturally binds as: `a * b + c`
- **Essential:** `(a + b) * c` → Overrides natural precedence! Must NOT
  remove!

**Specification:**

Create an `ASTLintVisitor` class that traverses expressions, tracks
parent and child operator precedences, and reports diagnostics with
`FixItHint` replacements pointing out redundant parentheses.

``` cpp
class ASTLintVisitor : public ASTVisitor {
public:
    void check(const ASTNode& root);
    size_t redundant_parentheses_count() const noexcept { return m_redundant_count; }
    void visit(const BinaryExprAST& node) override;
    // ... Implement visitor methods ...
private:
    size_t m_redundant_count{0};
    void inspect_child_precedence(TokenType parent_op, const ExprAST& child, bool is_right_child);
};
```

**Verification Tasks:**

1.  **Duplicate Parenthesis Detection:** Lint `let x = (((42)));`.
    Verify that the linter reports exactly two redundant grouping
    warnings with fix-it deletions.
2.  **Natural Precedence Warning:** Lint `(a * b) + c`. Verify that the
    linter flags the parentheses around `a * b` as redundant.
3.  **Essential Parenthesis Preservation:** Lint `(a + b) * c`. Verify
    that the linter identifies the grouping as essential and emits zero
    warnings.

#### Exercise 19.9: High-Performance Arena Allocator for AST Nodes

**Context and Theory:**

In a standard compiler architecture, every AST node is individually
allocated on the free store via `std::make_unique<Node>()`. For a
$100,000$-line codebase containing millions of AST nodes, this incurs
massive performance penalties:

- High heap allocation overhead (`malloc`/`free` book-keeping).
- Pointer fragmentation across physical RAM, resulting in CPU cache
  misses during tree traversals.

Production compilers (such as Clang and rustc) employ **Arena (Bump)
Allocators** (see fig. 21):

1.  Memory is pre-allocated in large contiguous memory regions
    (e.g. $4\text{ MB}$ chunks).
2.  Creating a node involves advancing a bump pointer by `sizeof(Node)`
    aligned to 8-byte boundaries ($O(1)$ allocation).
3.  Individual nodes are never destructed one-by-one. When the
    compilation pass completes, the entire arena memory pool is
    deallocated or reset in a single operation ($O(1)$ teardown).

<figure id="fig:arena-ast">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MDkuODMyIiBo
ZWlnaHQ9IjU5Ljg3MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1MDkuODMyIDU5Ljg3
MiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI1MDku
ODMyIiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+QXJlbmE8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iNTAuNjM0MDAzIiB5PSIxMC40NzYiID5NZW1vcnk8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOTkuNTIyIiB5PSIxMC40NzYiID5CbG9jazwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNDEuNDI2MDEiIHk9IjEwLjQ3NiIg
Pls0PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE2Mi4zNzgiIHk9IjEw
LjQ3NiIgPk1CXTo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOC43MzAw
MDA1IiB5PSIzOC40MTIwMDMiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjIyLjY5OCIgeT0iMzguNDEyMDAzIiA+Tm9kZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSI1Ny42MTgiIHk9IjM4LjQxMjAwMyIgPjE8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzEuNTg2IiB5PSIzOC40MTIwMDMiID5dWzwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5Mi41MzgiIHk9IjM4LjQxMjAwMyIgPk5vZGU8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTI3LjQ1OCIgeT0iMzguNDEy
MDAzIiA+MjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNDEuNDI2MDEi
IHk9IjM4LjQxMjAwMyIgPl1bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjE2Mi4zNzgiIHk9IjM4LjQxMjAwMyIgPk5vZGU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMTk3LjI5OCIgeT0iMzguNDEyMDAzIiA+MzwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIyMTEuMjY2IiB5PSIzOC40MTIwMDMiID5dPC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIyNS4yMzQwMSIgeT0iMzguNDEyMDAzIiA+Li4u
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1My4xNzAwMSIgeT0iMzgu
NDEyMDAzIiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNjcuMTM4
IiB5PSIzOC40MTIwMDMiID5CdW1wPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjMwMi4wNTgiIHk9IjM4LjQxMjAwMyIgPlBvaW50ZXI8L3RleHQ+CiAgPGxpbmUgeDE9IjM1
Ni4xODQwMiIgeTE9IjM0LjkyMDAwMiIgeDI9IjQwNS4wNzIwMiIgeTI9IjM0LjkyMDAwMiIgY2xh
c3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSI0MDUuMDcyMDIsMzEuNDI4MDAx
IDQxMi4wNTYsMzQuOTIwMDAyIDQwNS4wNzIwMiwzOC40MTIwMDMiIGNsYXNzPSJmaWxsZWQiPjwv
cG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MjAuNzg2IiB5PSIzOC40
MTIwMDMiID5GcmVlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ1NS43
MDYwMiIgeT0iMzguNDEyMDAzIiA+U3BhY2U8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iNDk3LjYxMDAyIiB5PSIzOC40MTIwMDMiID5dPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 21: AST Nodes Arena Allocator</figcaption>
</figure>

**Specification:**

Implement a custom `ASTArenaAllocator` supporting typed allocations via
placement `new`. Integrate it into the Faberge parser to allocate AST
nodes contiguously without individual heap calls.

``` cpp
class ASTArenaAllocator {
public:
    explicit ASTArenaAllocator(size_t chunk_size = 4096 * 1024);
    ~ASTArenaAllocator();

    template <typename T, typename... Args>
    T* allocate(Args&&... args) {
        size_t bytes = sizeof(T);
        size_t aligned_size = (bytes + 7) & ~7; // 8-byte alignment
        if (m_current_offset + aligned_size > m_chunk_size) allocate_new_chunk();
        void* ptr = m_current_chunk + m_current_offset;
        m_current_offset += aligned_size;
        return new (ptr) T(std::forward<Args>(args)...);
    }
    void reset();
private:
    size_t m_chunk_size;
    size_t m_current_offset{0};
    char* m_current_chunk{nullptr};
    std::vector<char*> m_allocated_chunks;
    void allocate_new_chunk();
};
```

**Verification Tasks:**

1.  **Contiguous Address Verification:** Allocate three consecutive
    `LiteralExprAST` nodes. Verify that the difference between their
    memory addresses matches their 8-byte aligned `sizeof` footprint.
2.  **Chunk Growth across Capacity:** Set chunk size to $1024$ bytes.
    Allocate $500$ nodes. Verify that the arena transparently allocates
    additional chunks without memory corruption or segmentation faults.
3.  **Instantaneous Teardown:** Measure the deallocation duration of
    $1,000,000$ AST nodes. Verify that resetting the arena executes in
    sub-millisecond time.

#### Exercise 19.10: Syntactic Error Recovery: Error Nodes and Synchronisation

**Context and Theory:**

A primitive compiler aborts upon encountering its first syntax error. A
production compiler must continue parsing after an error to discover
subsequent issues in a single pass, enabling IDE features like
autocompletion and diagnostics.

To recover from syntax errors without crashing or cascading false
errors, modern parsers use **Error Nodes** (see fig. 22):

1.  When parsing an expression fails, rather than throwing an exception
    that unwinds the entire call stack, the parser logs a diagnostic and
    constructs an `ErrorExprAST` node.
2.  The `ErrorExprAST` node is inserted directly into the AST,
    preserving the hierarchy of surrounding statements.
3.  The parser synchronises by advancing past unrecognised tokens until
    reaching an element in its **Synchronisation Set** (e.g. `;`, `}`,
    or keywords `let`, `fn`, `if`).

<figure id="fig:error-nodes">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MjMuOCIgaGVp
Z2h0PSIxMTUuNzQ0IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDUyMy44IDExNS43NDQi
PgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5z
dmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13
aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJv
a2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0
ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBJ
b3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTRweDsK
fQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNw
YXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2
Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBm
aWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7
CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewog
IG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9u
ZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2Vk
X2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5k
X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAu
c3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5z
dmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9j
aXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1z
dGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5f
Y2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2Ig
LnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdf
b3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ci
IHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1h
cmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdv
biBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8
bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIi
IG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHln
b24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDgg
OCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3Jp
ZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0i
MiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9
Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJX
aWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAg
ICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNs
ZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94
PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0
PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5
PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwv
ZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNTIzLjgi
IGhlaWdodD0iMTE1Ljc0NCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+U291cmNlOjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSI3OC41NyIgeT0iMTAuNDc2IiA+bGV0PC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjEwNi41MDYwMDQiIHk9IjEwLjQ3NiIgPmE8L3RleHQ+CiAgPGxp
bmUgeDE9IjExOC43MjgwMDQiIHkxPSI1LjIzOCIgeDI9IjEyNS43MTIwMDYiIHkyPSI1LjIzOCIg
Y2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPGxpbmUgeDE9IjExOC43MjgwMDQiIHkxPSI4LjczMDAw
MDUiIHgyPSIxMjUuNzEyMDA2IiB5Mj0iOC43MzAwMDA1IiBjbGFzcz0ic29saWQiPjwvbGluZT4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMzQuNDQyIiB5PSIxMC40NzYiID4xMDs8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTc2LjM0NjAxIiB5PSIxMC40
NzYiID5sZXQ8L3RleHQ+CiAgPGxpbmUgeDE9IjE5Mi4wNjAwMSIgeTE9IjI0LjQ0NCIgeDI9IjE5
Mi4wNjAwMSIgeTI9IjQ1LjM5NiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMTYyLjM3OCIgeT0iNjYuMzQ4IiA+U3ludGF4PC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIwNC4yODIwMSIgeT0iMTAuNDc2IiA+YjwvdGV4
dD4KICA8bGluZSB4MT0iMjE2LjUwNDAxIiB5MT0iNS4yMzgiIHgyPSIyMjMuNDg4IiB5Mj0iNS4y
MzgiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxsaW5lIHgxPSIyMTYuNTA0MDEiIHkxPSI4Ljcz
MDAwMDUiIHgyPSIyMjMuNDg4IiB5Mj0iOC43MzAwMDA1IiBjbGFzcz0ic29saWQiPjwvbGluZT4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMzIuMjE4IiB5PSIxMC40NzYiID4qPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI0Ni4xODYiIHk9IjEwLjQ3NiIg
Pjs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjc0LjEyMiIgeT0iMTAu
NDc2IiA+bGV0PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMwMi4wNTgi
IHk9IjEwLjQ3NiIgPmM8L3RleHQ+CiAgPGxpbmUgeDE9IjMxNC4yOCIgeTE9IjUuMjM4IiB4Mj0i
MzIxLjI2NCIgeTI9IjUuMjM4IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8bGluZSB4MT0iMzE0
LjI4IiB5MT0iOC43MzAwMDA1IiB4Mj0iMzIxLjI2NCIgeTI9IjguNzMwMDAwNSIgY2xhc3M9InNv
bGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzI5Ljk5NDAyIiB5
PSIxMC40NzYiID4yMDs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOC43
MzAwMDA1IiB5PSI2Ni4zNDgiID5QYXJzZXI6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjIxMS4yNjYiIHk9IjY2LjM0OCIgPkVycm9yITwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSI4LjczMDAwMDUiIHk9Ijk0LjI4NDAwNCIgPkFTVDo8L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzguNTciIHk9Ijk0LjI4NDAwNCIgPltM
ZXRTdG10OjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNDguNDEiIHk9
Ijk0LjI4NDAwNCIgPmFdPC90ZXh0PgogIDxsaW5lIHgxPSIxNjcuNjE2IiB5MT0iOTAuNzkyIiB4
Mj0iMTgxLjU4NCIgeTI9IjkwLjc5MiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24g
cG9pbnRzPSIxODEuNTg0LDg3LjMgMTg4LjU2ODAxLDkwLjc5MiAxODEuNTg0LDk0LjI4NDAwNCIg
Y2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjE5Ny4yOTgiIHk9Ijk0LjI4NDAwNCIgPltMZXRTdG10OjwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSIyNjcuMTM4IiB5PSI5NC4yODQwMDQiID5iLDwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyODguMDkiIHk9Ijk0LjI4NDAwNCIgPkluaXQ6PC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMyOS45OTQwMiIgeT0iOTQuMjg0
MDA0IiA+RXJyb3JOb2RlXTwvdGV4dD4KICA8bGluZSB4MT0iNDA1LjA3MjAyIiB5MT0iOTAuNzky
IiB4Mj0iNDE5LjA0IiB5Mj0iOTAuNzkyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdv
biBwb2ludHM9IjQxOS4wNCw4Ny4zIDQyNi4wMjQwMiw5MC43OTIgNDE5LjA0LDk0LjI4NDAwNCIg
Y2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjQzNC43NTQiIHk9Ijk0LjI4NDAwNCIgPltMZXRTdG10OjwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSI1MDQuNTk0MDIiIHk9Ijk0LjI4NDAwNCIgPmNdPC90ZXh0Pgo8L3N2
Zz4K
" alt="" />
<figcaption>Figure 22: Syntactic Error Recovery with Error
Nodes</figcaption>
</figure>

**Specification:**

Define an `ErrorExprAST` class. Refactor `Parser::parse_expression()`
and statement parsing routines: upon encountering a malformed
expression, emit a diagnostic, construct an `ErrorExprAST`, synchronise
to the next statement delimiter, and resume normal parsing.

``` cpp
class ErrorExprAST : public ExprAST {
public:
    explicit ErrorExprAST(SourceLocation loc) : ExprAST(loc) {}
    void accept(ASTVisitor& visitor) const override;
};

class ResilientParser {
public:
    std::unique_ptr<ExprAST> parse_expression_resilient(int min_prec) {
        try {
            return parse_expression(min_prec);
        } catch (const std::exception& e) {
            SourceLocation err_loc = current_token().location;
            synchronise_to_expression_boundary();
            return std::make_unique<ErrorExprAST>(err_loc);
        }
    }
private:
    void synchronise_to_expression_boundary();
};
```

**Verification Tasks:**

1.  **Multi-Error Discovery:** Parse a file containing three separate
    syntax errors across three distinct function bodies. Verify that the
    parser reports all three errors and successfully parses the final
    valid statement.
2.  **Error Node AST Retention:** Inspect the generated AST of a file
    containing a malformed variable initialiser. Verify that the
    statement list contains an entry with an `ErrorExprAST` child node.
3.  **Infinite Loop Immunity:** Feed a stream consisting of entirely
    invalid punctuation `!@#$%^&*()`. Verify that the synchronisation
    loop consumes tokens toward `EOF` without locking into an infinite
    loop.

#### Exercise 19.11: Precedence Climbing vs Pratt Parsing

**Context and Theory:**

**Precedence Climbing** is an algorithmically compact alternative to
Pratt parsing. While Pratt parsing models operator rules through
separate prefix (`nud`) and infix (`led`) denotation dispatch tables,
Precedence Climbing formulates binary expression parsing as an iterative
loop driven by operator binding queries (see fig. 23):

$$\text{Climb}(\text{min\_prec}):$$

1.  Parse primary/prefix operand $L$.
2.  While current token is binary operator with
    $\text{prec} \ge \text{min\_prec}$:
    - Fetch precedence $p$ and associativity of operator.
    - Compute next precedence threshold:
      $p' = (\text{is\_right\_assoc}) \mathbin{?} p \mathbin{:} p + 1$.
    - Recursively parse right-hand operand:
      $R \leftarrow \text{Climb}(p')$.
    - Update $L \leftarrow \text{BinaryExpr}(L, \text{op}, R)$.
3.  Return $L$.

Both algorithms produce isomorphic Abstract Syntax Trees.

<figure id="fig:precedence-climbing">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI1NzJwdCIgaGVpZ2h0PSI0NHB0Igogdmlld0JveD0iMC4wMCAwLjAw
IDU3Mi4wMSA0NC4wMCIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIiB4bWxuczp4
bGluaz0iaHR0cDovL3d3dy53My5vcmcvMTk5OS94bGluayI+CjxnIGlkPSJncmFwaDAiIGNsYXNz
PSJncmFwaCIgdHJhbnNmb3JtPSJzY2FsZSgxIDEpIHJvdGF0ZSgwKSB0cmFuc2xhdGUoNCA0MCki
Pgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gcHJpbWFyeSAtLT4KPGcgaWQ9Im5vZGUxIiBjbGFzcz0i
bm9kZSI+Cjx0aXRsZT5wcmltYXJ5PC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tl
PSJibGFjayIgY3g9IjQxLjgiIGN5PSItMTgiIHJ4PSI0MS41OSIgcnk9IjE4Ii8+Cjx0ZXh0IHRl
eHQtYW5jaG9yPSJtaWRkbGUiIHg9IjQxLjgiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iSW9zZXZr
YSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlsgUHJpbWFyeSBMIF08L3Rl
eHQ+CjwvZz4KPCEtLSB3aGlsZSAtLT4KPGcgaWQ9Im5vZGUyIiBjbGFzcz0ibm9kZSI+Cjx0aXRs
ZT53aGlsZTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIx
OTAuMzYiIGN5PSItMTgiIHJ4PSI4NC41NCIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjE5MC4zNiIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBN
b25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+V2hpbGUgKG9wLnByZWMgJmd0Oz0gbWluX3By
ZWMpPC90ZXh0Pgo8L2c+CjwhLS0gcHJpbWFyeSYjNDU7Jmd0O3doaWxlIC0tPgo8ZyBpZD0iZWRn
ZTEiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnByaW1hcnkmIzQ1OyZndDt3aGlsZTwvdGl0bGU+Cjxw
YXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik04My42MiwtMThDODcuNDIsLTE4IDkx
LjM2LC0xOCA5NS40LC0xOCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIg
cG9pbnRzPSI5NS41OSwtMjEuNSAxMDUuNTksLTE4IDk1LjU5LC0xNC41IDk1LjU5LC0yMS41Ii8+
CjwvZz4KPCEtLSBjbGltYiAtLT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5j
bGltYjwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIzNjEu
ODgiIGN5PSItMTgiIHJ4PSI2NC41MSIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRk
bGUiIHg9IjM2MS44OCIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25v
IE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+UiA9IENsaW1iKG5leHRfcHJlYyk8L3RleHQ+Cjwv
Zz4KPCEtLSB3aGlsZSYjNDU7Jmd0O2NsaW1iIC0tPgo8ZyBpZD0iZWRnZTIiIGNsYXNzPSJlZGdl
Ij4KPHRpdGxlPndoaWxlJiM0NTsmZ3Q7Y2xpbWI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBz
dHJva2U9ImJsYWNrIiBkPSJNMjc1LjYyLC0xOEMyNzkuMzEsLTE4IDI4Mi45OSwtMTggMjg2LjY1
LC0xOCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyODYu
ODcsLTIxLjUgMjk2Ljg3LC0xOCAyODYuODcsLTE0LjUgMjg2Ljg3LC0yMS41Ii8+CjwvZz4KPCEt
LSBsbm9kZSAtLT4KPGcgaWQ9Im5vZGU0IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5sbm9kZTwvdGl0
bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSI1MDYuMzIiIGN5PSIt
MTgiIHJ4PSI1Ny44OCIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjUw
Ni4zMiIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIg
Zm9udC1zaXplPSIxMC4wMCI+TCA9IE5vZGUoTCwgb3AsIFIpPC90ZXh0Pgo8L2c+CjwhLS0gY2xp
bWImIzQ1OyZndDtsbm9kZSAtLT4KPGcgaWQ9ImVkZ2UzIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5j
bGltYiYjNDU7Jmd0O2xub2RlPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFj
ayIgZD0iTTQyNy4wMiwtMThDNDMwLjY3LC0xOCA0MzQuMzUsLTE4IDQzOC4wMiwtMTgiLz4KPHBv
bHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iNDM4LjI4LC0yMS41IDQ0
OC4yOCwtMTggNDM4LjI4LC0xNC41IDQzOC4yOCwtMjEuNSIvPgo8L2c+CjwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 23: Precedence Climbing Loop</figcaption>
</figure>

**Specification:**

Implement a standalone `PrecedenceClimbingParser` class. Parse Faberge
expression tokens using the precedence climbing algorithm, and verify
that the emitted AST matches the Pratt parser’s output node-for-node.

``` cpp
class PrecedenceClimbingParser {
public:
    explicit PrecedenceClimbingParser(std::vector<Token> tokens);
    std::unique_ptr<ExprAST> parse_expression(int min_prec = 1) {
        auto left = parse_primary();
        while (is_binary_op(current_token()) && get_precedence(current_token()) >= min_prec) {
            Token op = advance();
            int p = get_precedence(op);
            int next_p = is_right_assoc(op) ? p : p + 1;
            auto right = parse_expression(next_p);
            left = std::make_unique<BinaryExprAST>(op.location, op.type, std::move(left), std::move(right));
        }
        return left;
    }
private:
    std::unique_ptr<ExprAST> parse_primary();
};
```

**Verification Tasks:**

1.  **Tree Isomorphism:** Parse `1 + 2 * 3 - 4 / 5` using both the Pratt
    parser and the Precedence Climbing parser. Compare both trees and
    assert structural identity.
2.  **Associativity Verification:** Parse a sequence of
    right-associative operators `a ^ b ^ c`. Verify that Precedence
    Climbing constructs an identical right-skewed AST.
3.  **Performance Profiling:** Parse $50,000$ synthetic expressions
    across both implementations. Compare execution times and verify that
    both complete within comparable performance bands.

#### Exercise 19.12: Automated Grammar Mutation and Parser Fuzzing Engine

**Context and Theory:**

Compilers must maintain **Parsing Robustness**: no token
sequence—regardless of grammatical invalidity, structural corruption, or
nesting depth—should cause the parser to crash, trigger undefined
behaviour, or exhaust the OS stack via unbounded recursion.

An automated **Parser Fuzzing Engine** subjects the grammar parser to
adversarial testing:

- **Token Deletion:** Randomly dropping essential structural tokens
  (e.g. omitting `then` from an `if` expression, or dropping closing
  `}`).
- **Token Insertion:** Injecting arbitrary keywords or punctuation into
  expression sequences (e.g. `1 + let * 4`).
- **Pathological Nesting:** Emitting millions of opening delimiters
  `(((( ...` to test recursion depth limits.

<figure id="fig:parser-fuzzer">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI1NDhwdCIgaGVpZ2h0PSI2NnB0Igogdmlld0JveD0iMC4wMCAwLjAw
IDU0Ny43NiA2NS45OCIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIiB4bWxuczp4
bGluaz0iaHR0cDovL3d3dy53My5vcmcvMTk5OS94bGluayI+CjxnIGlkPSJncmFwaDAiIGNsYXNz
PSJncmFwaCIgdHJhbnNmb3JtPSJzY2FsZSgxIDEpIHJvdGF0ZSgwKSB0cmFuc2xhdGUoNCA2MS45
OCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gdmFsaWQgLS0+CjxnIGlkPSJub2RlMSIgY2xhc3M9
Im5vZGUiPgo8dGl0bGU+dmFsaWQ8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9
ImJsYWNrIiBjeD0iNDguMDgiIGN5PSItMjguOTkiIHJ4PSI0OC4xNyIgcnk9IjIxLjQzIi8+Cjx0
ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjQ4LjA4IiB5PSItMzEuOTkiIGZvbnQtZmFtaWx5
PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+VmFsaWQ8L3Rl
eHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjQ4LjA4IiB5PSItMjAuOTkiIGZvbnQt
ZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+QVNU
IFRva2VuczwvdGV4dD4KPC9nPgo8IS0tIG11dGF0b3IgLS0+CjxnIGlkPSJub2RlMiIgY2xhc3M9
Im5vZGUiPgo8dGl0bGU+bXV0YXRvcjwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIGN4PSIxOTcuMzYiIGN5PSItMjguOTkiIHJ4PSI3OS4zOSIgcnk9IjIxLjQzIi8+
Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE5Ny4zNiIgeT0iLTMxLjk5IiBmb250LWZh
bWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlsgR3Jh
bW1hciBNdXRhdG9yIF08L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE5Ny4z
NiIgeT0iLTIwLjk5IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZv
bnQtc2l6ZT0iMTAuMDAiPihEcm9wLCBJbnNlcnQsIFN3YXApPC90ZXh0Pgo8L2c+CjwhLS0gdmFs
aWQmIzQ1OyZndDttdXRhdG9yIC0tPgo8ZyBpZD0iZWRnZTEiIGNsYXNzPSJlZGdlIj4KPHRpdGxl
PnZhbGlkJiM0NTsmZ3Q7bXV0YXRvcjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGQ9Ik05Ni40OSwtMjguOTlDMTAwLjE2LC0yOC45OSAxMDMuOTIsLTI4Ljk5IDEwNy43
NCwtMjguOTkiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0i
MTA3Ljk1LC0zMi40OSAxMTcuOTUsLTI4Ljk5IDEwNy45NSwtMjUuNDkgMTA3Ljk1LC0zMi40OSIv
Pgo8L2c+CjwhLS0gbXV0YXRlZCAtLT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRs
ZT5tdXRhdGVkPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9
IjMzOC4xNiIgY3k9Ii0yOC45OSIgcng9IjM5LjciIHJ5PSIyOC45OCIvPgo8dGV4dCB0ZXh0LWFu
Y2hvcj0ibWlkZGxlIiB4PSIzMzguMTYiIHk9Ii0zNy40OSIgZm9udC1mYW1pbHk9Iklvc2V2a2Eg
Q2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5NdXRhdGVkPC90ZXh0Pgo8dGV4
dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIzMzguMTYiIHk9Ii0yNi40OSIgZm9udC1mYW1pbHk9
Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5Ub2tlbjwvdGV4
dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMzM4LjE2IiB5PSItMTUuNDkiIGZvbnQt
ZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+U3Ry
ZWFtPC90ZXh0Pgo8L2c+CjwhLS0gbXV0YXRvciYjNDU7Jmd0O211dGF0ZWQgLS0+CjxnIGlkPSJl
ZGdlMiIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+bXV0YXRvciYjNDU7Jmd0O211dGF0ZWQ8L3RpdGxl
Pgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMjc2LjY4LC0yOC45OUMyODAu
NTUsLTI4Ljk5IDI4NC4zOCwtMjguOTkgMjg4LjEzLC0yOC45OSIvPgo8cG9seWdvbiBmaWxsPSJi
bGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyODguNDEsLTMyLjQ5IDI5OC40MSwtMjguOTkg
Mjg4LjQxLC0yNS40OSAyODguNDEsLTMyLjQ5Ii8+CjwvZz4KPCEtLSBwYXJzZXIgLS0+CjxnIGlk
PSJub2RlNCIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+cGFyc2VyPC90aXRsZT4KPGVsbGlwc2UgZmls
bD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjQ2OS43NiIgY3k9Ii0yOC45OSIgcng9IjcwLjAx
IiByeT0iMjEuNDMiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNDY5Ljc2IiB5PSIt
MzEuOTkiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXpl
PSIxMC4wMCI+WyBGYWJlcmdlIFBhcnNlciBdPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIiB4PSI0NjkuNzYiIHk9Ii0yMC45OSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1v
bm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj4oTXVzdCBOT1QgQ3Jhc2ghKTwvdGV4dD4KPC9n
Pgo8IS0tIG11dGF0ZWQmIzQ1OyZndDtwYXJzZXIgLS0+CjxnIGlkPSJlZGdlMyIgY2xhc3M9ImVk
Z2UiPgo8dGl0bGU+bXV0YXRlZCYjNDU7Jmd0O3BhcnNlcjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5v
bmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0zNzguMDQsLTI4Ljk5QzM4MS42OSwtMjguOTkgMzg1LjQ2
LC0yOC45OSAzODkuMzEsLTI4Ljk5Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJs
YWNrIiBwb2ludHM9IjM4OS42MiwtMzIuNDkgMzk5LjYyLC0yOC45OSAzODkuNjIsLTI1LjQ5IDM4
OS42MiwtMzIuNDkiLz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 24: Parser Fuzzing Engine</figcaption>
</figure>

**Specification:**

Construct a `ParserFuzzEngine` class. The harness takes valid Faberge
programs, applies stochastic grammatical mutations (token drops,
delimiter swaps, operator insertions), passes the corrupted tokens to
the `Parser`, and asserts that the parser synchronises or reports errors
cleanly without halting abnormally.

``` cpp
class ParserFuzzEngine {
public:
    struct FuzzStats {
        size_t programs_tested{0};
        size_t errors_handled{0};
        bool survived{true};
    };
    static FuzzStats run_grammar_fuzz(std::string_view seed_source, size_t iterations) {
        FuzzStats stats;
        std::mt19937_64 rng(42);
        Lexer base_lexer(seed_source);
        auto base_tokens = base_lexer.tokenise();

        for (size_t i = 0; i < iterations; ++i) {
            auto mutated = mutate_tokens(base_tokens, rng);
            Parser parser(std::move(mutated), seed_source);
            auto ast = parser.parse_program(); // Must NEVER segfault or throw unhandled exceptions
            stats.errors_handled += parser.error_count();
            ++stats.programs_tested;
        }
        return stats;
    }
private:
    static std::vector<Token> mutate_tokens(std::vector<Token> tokens, std::mt19937_64& rng);
};
```

**Verification Tasks:**

1.  **Random Mutation Resistance:** Subject the parser to $10,000$
    randomized token mutations of a complex recursive Faberge script.
    Verify that the parser halts with clean error reporting in $100\%$
    of test runs.
2.  **Pathological Recursion Clamping:** Generate an adversarial token
    stream consisting of $10,000$ nested opening parentheses `((((...`.
    Verify that the parser catches the maximum recursion limit and halts
    gracefully rather than overflowing the stack.
3.  **Delimiter Mismatch Recovery:** Randomly replace closing
    parentheses with brackets or braces (e.g. `(1 + 2]`). Verify that
    the parser identifies delimiter mismatches and continues processing
    subsequent lines.

# Chapter 20: Check

> *“A type system is a tractable syntactic method for proving the
> absence of certain program behaviours by classifying phrases according
> to the kinds of values they compute.”* — Benjamin C. Pierce, Types and
> Programming Languages (2002)

------------------------------------------------------------------------

### 20.1 Semantic Analysis: Beyond Syntactic Structure

In Chapters 18 and 19, we built the compiler’s syntax engine. The
lexical scanner converted source characters into tokens, and the
recursive descent parser organised those tokens into an Abstract Syntax
Tree (AST).

However, structural validity does not imply semantic correctness.
Consider the following syntactically flawless **Faberge** snippet:

```
let compute = fn (x: Int): Int => {
    let message = "Total score: ";
    let flag = true;
    (message * flag) + undefined_variable(x)
};
```

If you feed this text into our Chapter 19 parser, it constructs an AST
without raising a single error. The grammar productions are satisfied:
`message * flag` is a syntactically valid `BinaryExprAST`, and
`undefined_variable(x)` is a valid `CallExprAST`.

Yet the program is nonsensical:

1.  **Type Incompatibility:** What does it mean to multiply a `String`
    by a `Bool`?
2.  **Undeclared Symbols:** The identifier `undefined_variable` was
    never declared in this scope or any enclosing scope.
3.  **Return Type Violation:** The function signature promises to return
    an `Int`, but its body evaluates to an illegal compound expression.

<figure id="fig:semantic-analysis">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzNjhwdCIgaGVpZ2h0PSIxOTZwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzNjguMDAgMTk2LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDE5
MikiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gYXN0IC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPmFzdDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIHBvaW50cz0iMzYwLC0xODggMCwtMTg4IDAsLTE0NSAzNjAsLTE0NSAzNjAsLTE4OCIvPgo8
dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii0xNzUuNSIgZm9udC1mYW1pbHk9Iklv
c2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5BU1QgKEFic3RyYWN0
IFN5bnRheCBUcmVlIGZyb20gQ2hhcHRlciAxOSk8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJz
dGFydCIgeD0iOCIgeT0iLTE1Mi41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBN
ZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlByb2dyYW1BU1QgJiM0NTsmZ3Q7IExldFN0bXRBU1Qg
JiM0NTsmZ3Q7IExhbWJkYUV4cHJBU1QgJiM0NTsmZ3Q7IEJsb2NrRXhwckFTVCAuLi48L3RleHQ+
CjwvZz4KPCEtLSBzZW1hbnRpYyAtLT4KPGcgaWQ9Im5vZGUyIiBjbGFzcz0ibm9kZSI+Cjx0aXRs
ZT5zZW1hbnRpYzwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBv
aW50cz0iMzYwLC0xMjMgMCwtMTIzIDAsLTU4IDM2MCwtNTggMzYwLC0xMjMiLz4KPHRleHQgdGV4
dC1hbmNob3I9InN0YXJ0IiB4PSI4IiB5PSItMTEwIiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFy
b24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlNUQUdFIDM6IFNlbWFudGljIEFuYWx5
c2lzIGFuZCBTdGF0aWMgVHlwZSBDaGVja2luZyAoVGhpcyBDaHBhdGVyKTwvdGV4dD4KPHRleHQg
dGV4dC1hbmNob3I9InN0YXJ0IiB4PSI4IiB5PSItOTkiIGZvbnQtZmFtaWx5PSJJb3NldmthIENo
YXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+MS4gU2NvcGUgUmVzb2x1dGlvbiAo
QmluZCBpZGVudGlmaWVycyB0byBTeW1ib2wgVGFibGUgZW50cmllcyk8L3RleHQ+Cjx0ZXh0IHRl
eHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTg4IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFy
b24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPjIuIFN0YXRpYyBUeXBlIENoZWNraW5n
IChTeW50aGVzaXNlIGFuZCB2ZXJpZnkgb3BlcmFuZCB0eXBlcyk8L3RleHQ+Cjx0ZXh0IHRleHQt
YW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTc3IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24g
TW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPjMuIENvbnRyb2wmIzQ1O0Zsb3cgYW5kIFJl
dHVybiBWZXJpZmljYXRpb248L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIg
eT0iLTY2IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6
ZT0iMTAuMDAiPjQuIERlY29yYXRlIEFTVCB3aXRoIFZlcmlmaWVkIFR5cGVzPC90ZXh0Pgo8L2c+
CjwhLS0gYXN0JiM0NTsmZ3Q7c2VtYW50aWMgLS0+CjxnIGlkPSJlZGdlMSIgY2xhc3M9ImVkZ2Ui
Pgo8dGl0bGU+YXN0JiM0NTsmZ3Q7c2VtYW50aWM8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBz
dHJva2U9ImJsYWNrIiBkPSJNMTgwLC0xNDQuODlDMTgwLC0xNDEuMTkgMTgwLC0xMzcuMjIgMTgw
LC0xMzMuMTgiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0i
MTgzLjUsLTEzMy4wOCAxODAsLTEyMy4wOCAxNzYuNSwtMTMzLjA4IDE4My41LC0xMzMuMDgiLz4K
PC9nPgo8IS0tIGNvZGUgLS0+CjxnIGlkPSJub2RlMyIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+Y29k
ZTwvdGl0bGU+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE4MCIgeT0iLTE1LjUiIGZv
bnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+
RGVjb3JhdGVkIC8gVmVyaWZpZWQgVHlwZWQgQVNUIChSZWFkeSBmb3IgQ29kZSBHZW5lcmF0aW9u
KTwvdGV4dD4KPC9nPgo8IS0tIHNlbWFudGljJiM0NTsmZ3Q7Y29kZSAtLT4KPGcgaWQ9ImVkZ2Uy
IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5zZW1hbnRpYyYjNDU7Jmd0O2NvZGU8L3RpdGxlPgo8cGF0
aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTgwLC01Ny45NkMxODAsLTU0LjExIDE4
MCwtNTAuMTkgMTgwLC00Ni40Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNr
IiBwb2ludHM9IjE4My41LC00Ni4yMiAxODAsLTM2LjIyIDE3Ni41LC00Ni4yMiAxODMuNSwtNDYu
MjIiLz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 25: Detailed Overview of Semantic
Analysis</figcaption>
</figure>

**Semantic Analysis** is the compiler pass that enforces the contextual
and mathematical rules of the language (see fig. 25). In this chapter,
we build **Stage 3** of our compiler project: a **Static Type Checker
and Semantic Analyser** in Modern C++20.

### 20.2 Lexical Scoping and the Scoped Symbol Table

To check whether an identifier is legal, the compiler must track every
variable, parameter, and function declaration in existence at any point
during program execution. This tracking structure is the **Symbol
Table** (or **Environment**).

#### 20.2.1 Lexical Scoping Rules

Faberge—like C++, TypeScript, Rust, and Scheme—uses **Lexical (Static)
Scoping**. In a lexically scoped language:

1.  The scope of an identifier is determined entirely at compile time by
    its textual position in the source code.
2.  Inner blocks inherit all bindings declared in enclosing outer
    blocks.
3.  **Variable Shadowing:** An inner scope may declare a variable with
    the exact same name as an outer variable. The inner declaration
    “shadows” (hides) the outer declaration for the duration of the
    inner block, without destroying the outer variable.
4.  Outer scopes cannot see variables declared inside nested inner
    scopes.

See fig. 26 for an illustration of lexical scoping lookup.

<figure id="fig:scopes">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIyMCAyMCA4NjAgMjMwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyMzAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0ic2NvcGVTaGFkb3ciIHg9Ii01JSIg
eT0iLTUlIiB3aWR0aD0iMTEwJSIgaGVpZ2h0PSIxMjAlIj4KICAgICAgPGZlRHJvcFNoYWRvdyBk
eD0iMCIgZHk9IjMiIHN0ZERldmlhdGlvbj0iMyIgZmxvb2QtY29sb3I9InZhcigtLXN2Zy1mZyki
IGZsb29kLW9wYWNpdHk9IjAuNSIvPgogICAgPC9maWx0ZXI+CiAgPC9kZWZzPgogIDwhLS0gR2xv
YmFsIFNjb3BlIEJveCAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSg0MCwgMzApIj4KICAg
IDxyZWN0IHg9IjAiIHk9IjAiIHdpZHRoPSIyNDAiIGhlaWdodD0iMjAwIiByeD0iOCIgZmlsbD0i
dmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZh
cigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgZmlsdGVyPSJ1cmwoI3Njb3BlU2hhZG93KSIv
PgogICAgPHRleHQgeD0iMTIwIiB5PSIyMyIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXpl
PSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlNjb3BlIDA6IEds
b2JhbCBFbnZpcm9ubWVudDwvdGV4dD4KICAgIDwhLS0gR2xvYmFsIFN5bWJvbHMgLS0+CiAgICA8
dGV4dCB4PSIyMCIgeT0iNjUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiPnBp
IDogRmxvYXQ8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iOTAiIGZpbGw9InZhcigtLXN2Zy1m
ZykiIGZvbnQtc2l6ZT0iMTQiPmdsb2JhbF9saW1pdCA6IEludDwvdGV4dD4KICAgIDx0ZXh0IHg9
IjIwIiB5PSIxMTUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiPmZhY3Rvcmlh
bCA6IChJbnQgLSZndDsgSW50KTwvdGV4dD4KICAgIDxsaW5lIHgxPSIyMCIgeTE9IjEzNSIgeDI9
IjIyMCIgeTI9IjEzNSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIv
PgogICAgPHRleHQgeD0iMjAiIHk9IjE2MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXpl
PSIxNCI+UGFyZW50OiBudWxscHRyIChSb290KTwvdGV4dD4KICA8L2c+CiAgPCEtLSBBcnJvdyBm
cm9tIEZ1bmMgU2NvcGUgdG8gR2xvYmFsIFNjb3BlIC0tPgogIDxwYXRoIGQ9Ik0gMzMwIDE0MCBM
IDI5NSAxNDAiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyLjUiIGZpbGw9
Im5vbmUiLz4KICA8cG9seWdvbiBwb2ludHM9IjI5MCwxNDAgMjk4LDEzNSAyOTgsMTQ1IiBmaWxs
PSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iMzEwIiB5PSIxMzAiIGZpbGw9InZhcigtLXN2
Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPnBhcmVudDwvdGV4dD4K
ICA8IS0tIEZ1bmN0aW9uIFNjb3BlIEJveCAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSgz
NDAsIDMwKSI+CiAgICA8cmVjdCB4PSIwIiB5PSIwIiB3aWR0aD0iMjQwIiBoZWlnaHQ9IjIwMCIg
cng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2sp
KSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGZpbHRlcj0idXJsKCNz
Y29wZVNoYWRvdykiLz4KICAgIDx0ZXh0IHg9IjEyMCIgeT0iMjMiIGZpbGw9InZhcigtLXN2Zy1m
ZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxl
Ij5TY29wZSAxOiBGdW5jdGlvbiAnY2FsY3VsYXRlJzwvdGV4dD4KICAgIDwhLS0gRnVuY3Rpb24g
U3ltYm9scyAtLT4KICAgIDx0ZXh0IHg9IjIwIiB5PSI2NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIg
Zm9udC1zaXplPSIxNCI+eCA6IEludCAoUGFyYW0pPC90ZXh0PgogICAgPHRleHQgeD0iMjAiIHk9
IjkwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij55IDogRmxvYXQgKFBhcmFt
KTwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSIxMTUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZv
bnQtc2l6ZT0iMTQiPm11bHRpcGxpZXIgOiBJbnQgPSAxMDwvdGV4dD4KICAgIDxsaW5lIHgxPSIy
MCIgeTE9IjEzNSIgeDI9IjIyMCIgeTI9IjEzNSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJv
a2Utd2lkdGg9IjEuNSIvPgogICAgPHRleHQgeD0iMjAiIHk9IjE2MCIgZmlsbD0idmFyKC0tc3Zn
LWZnKSIgZm9udC1zaXplPSIxNCI+SW5oZXJpdHM6IHBpLCBmYWN0b3JpYWw8L3RleHQ+CiAgPC9n
PgogIDwhLS0gQXJyb3cgZnJvbSBCbG9jayBTY29wZSB0byBGdW5jIFNjb3BlIC0tPgogIDxwYXRo
IGQ9Ik0gNjMwIDE0MCBMIDU5NSAxNDAiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdp
ZHRoPSIyLjUiIGZpbGw9Im5vbmUiLz4KICA8cG9seWdvbiBwb2ludHM9IjU5MCwxNDAgNTk4LDEz
NSA1OTgsMTQ1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iNjEwIiB5PSIxMzAi
IGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRkbGUi
PnBhcmVudDwvdGV4dD4KICA8IS0tIEJsb2NrIFNjb3BlIEJveCAtLT4KICA8ZyB0cmFuc2Zvcm09
InRyYW5zbGF0ZSg2NDAsIDMwKSI+CiAgICA8cmVjdCB4PSIwIiB5PSIwIiB3aWR0aD0iMjMwIiBo
ZWlnaHQ9IjIwMCIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50
LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGZp
bHRlcj0idXJsKCNzY29wZVNoYWRvdykiLz4KICAgIDx0ZXh0IHg9IjExNSIgeT0iMjMiIGZpbGw9
InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFu
Y2hvcj0ibWlkZGxlIj5TY29wZSAyOiBOZXN0ZWQgQmxvY2sgeyAuLi4gfTwvdGV4dD4KICAgIDwh
LS0gQmxvY2sgU3ltYm9scyAtLT4KICAgIDx0ZXh0IHg9IjIwIiB5PSI2NSIgZmlsbD0idmFyKC0t
c3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiPnggOiBTdHJpbmcgPSAi
c2hhZG93IjwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSI5MCIgZmlsbD0idmFyKC0tc3ZnLWZn
KSIgZm9udC1zaXplPSIxNCI+dGVtcF9vZmZzZXQgOiBJbnQgPSA0MjwvdGV4dD4KICAgIDxsaW5l
IHgxPSIyMCIgeTE9IjEzNSIgeDI9IjIxMCIgeTI9IjEzNSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcp
IiBzdHJva2Utd2lkdGg9IjEuNSIvPgogICAgPHRleHQgeD0iMjAiIHk9IjE2MCIgZmlsbD0idmFy
KC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCI+Tm90ZTogJ3gnIHNoYWRvd3M8L3RleHQ+CiAgICA8
dGV4dCB4PSIyMCIgeT0iMTc4IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5T
Y29wZSAxJ3MgcGFyYW1ldGVyIHghPC90ZXh0PgogIDwvZz4KPC9zdmc+
" alt="Figure 26: Identifier Lookup in Nested Scopes" />
<figcaption>Figure 26: Identifier Lookup in Nested Scopes</figcaption>
</figure>

#### 20.2.2 The Cactus-Stack (Spaghetti-Stack) Architecture

To implement lexical scoping, we organise our `Environment` as a
**Cactus Stack** (or spaghetti stack):

- Each `Environment` node contains a hash map of identifier bindings
  (`std::unordered_map<std::string, Symbol>`) and an enclosing parent
  pointer (`std::shared_ptr<Environment> parent`).
- **Insertion (`insert`):** Adds a binding exclusively to the current,
  innermost scope.
- **Lookup (`lookup`):** Searches the current scope. If found, returns
  the symbol. If not found, recursively traverses the `parent` pointer
  up the tree until either the symbol is located or the root (`nullptr`)
  is reached.

### 20.3 Type Systems: Theory and Verification

A **Type System** is a tractable syntactic method for proving the
absence of certain program behaviours by classifying phrases according
to the kinds of values they compute.

#### 20.3.1 Why Types Matter: Soundness and Milner’s Dictum

In 1978, Turing Award laureate Robin Milner established the core theorem
of modern programming language theory:

$$\text{Type Safety} = \text{Progress} + \text{Preservation}$$

1.  **Progress:** A well-typed expression is never “stuck”. It is either
    already a final value or can take a deterministic computational
    evaluation step.
2.  **Preservation (Subject Reduction):** If a well-typed expression $e$
    of type $\tau$ evaluates to $e'$, then $e'$ also has type $\tau$.

Together, these principles prove that a program passing our type checker
will never invoke undefined behaviour, corrupt memory layouts, or
attempt illegal machine operations at runtime.

#### 20.3.2 Bidirectional Type Checking

In compiler engineering, type checkers broadly operate in two modes (see
fig. 27):

1.  **Type Synthesis / Inference ($\Gamma \vdash e \Rightarrow \tau$):**
    “Given this expression $e$ and context $\Gamma$, what type does it
    produce?” (Bottom-up deduction).
2.  **Type Checking / Verification
    ($\Gamma \vdash e \Leftarrow \tau$):** “Verify that this expression
    $e$ conforms to the expected type $\tau$.” (Top-down verification).

<figure id="fig:bidirectional">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA4MDAgMTkwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxOTAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0iZlNoYWRvdyIgeD0iLTUlIiB5PSIt
NSUiIHdpZHRoPSIxMTAlIiBoZWlnaHQ9IjEyMCUiPgogICAgICA8ZmVEcm9wU2hhZG93IGR4PSIw
IiBkeT0iMiIgc3RkRGV2aWF0aW9uPSIzIiBmbG9vZC1jb2xvcj0idmFyKC0tc3ZnLWZnKSIgZmxv
b2Qtb3BhY2l0eT0iMC41Ii8+CiAgICA8L2ZpbHRlcj4KICA8L2RlZnM+CiAgPCEtLSBMZWZ0IEJv
eDogU3ludGhlc2lzIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDYwLCAzNSkiPgogICAg
PHJlY3QgeD0iMCIgeT0iMCIgd2lkdGg9IjM0MCIgaGVpZ2h0PSIxNTAiIHJ4PSI4IiBmaWxsPSJ2
YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFy
KC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIiBmaWx0ZXI9InVybCgjZlNoYWRvdykiLz4KICAg
IDx0ZXh0IHg9IjE3MCIgeT0iMjMiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtZmFtaWx5PSJt
b25vc3BhY2UiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0i
bWlkZGxlIj5UeXBlIFN5bnRoZXNpczogzpMg4oqiIGUg4oeSIM+EPC90ZXh0PgogICAgPHRleHQg
eD0iMjAiIHk9IjY1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LWZhbWlseT0ic2Fucy1zZXJp
ZiIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiPkJvdHRvbS1VcCBEZWR1Y3Rpb24g
KEluZmVyZW5jZSk6PC90ZXh0PgogICAgPHRleHQgeD0iMjAiIHk9IjkwIiBmaWxsPSJ2YXIoLS1z
dmctZmcpIiBmb250LWZhbWlseT0ibW9ub3NwYWNlIiBmb250LXNpemU9IjE0Ij5MaXRlcmFsOiA0
MiAgICAgICAgICAgIOKHkiBJbnQ8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iMTEwIiBmaWxs
PSJ2YXIoLS1zdmctZmcpIiBmb250LWZhbWlseT0ibW9ub3NwYWNlIiBmb250LXNpemU9IjE0Ij5M
aXRlcmFsOiAzLjE0ICAgICAgICAgIOKHkiBGbG9hdDwvdGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5
PSIxMzAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtZmFtaWx5PSJtb25vc3BhY2UiIGZvbnQt
c2l6ZT0iMTQiPkJpbmFyeTogKDEwICsgMjApICAgICAg4oeSIEludDwvdGV4dD4KICA8L2c+CiAg
PCEtLSBSaWdodCBCb3g6IENoZWNraW5nIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDQ4
MCwgMzUpIj4KICAgIDxyZWN0IHg9IjAiIHk9IjAiIHdpZHRoPSIzNDAiIGhlaWdodD0iMTUwIiBy
eD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykp
IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgZmlsdGVyPSJ1cmwoI2ZT
aGFkb3cpIi8+CiAgICA8dGV4dCB4PSIxNzAiIHk9IjIzIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBm
b250LWZhbWlseT0ibW9ub3NwYWNlIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIg
dGV4dC1hbmNob3I9Im1pZGRsZSI+VHlwZSBWZXJpZmljYXRpb246IM6TIOKKoiBlIOKHkCDPhDwv
dGV4dD4KICAgIDx0ZXh0IHg9IjIwIiB5PSI2NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1m
YW1pbHk9InNhbnMtc2VyaWYiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIj5Ub3At
RG93biBFeHBlY3RhdGlvbiBNYXRjaGluZzo8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iOTAi
IGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtZmFtaWx5PSJtb25vc3BhY2UiIGZvbnQtc2l6ZT0i
MTQiPmxldCBjb3VudDogSW50ID0gZXhwciAgIOKHkCBFeHBlY3QgSW50PC90ZXh0PgogICAgPHRl
eHQgeD0iMjAiIHk9IjExMCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1mYW1pbHk9Im1vbm9z
cGFjZSIgZm9udC1zaXplPSIxNCI+aWYgKGNvbmRpdGlvbikgLi4uICAgICAg4oeQIEV4cGVjdCBC
b29sPC90ZXh0PgogICAgPHRleHQgeD0iMjAiIHk9IjEzMCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIg
Zm9udC1mYW1pbHk9Im1vbm9zcGFjZSIgZm9udC1zaXplPSIxNCI+Zm4gKC4uLik6IFN0cmluZyA9
PiAuLi4g4oeQIEV4cGVjdCBTdHJpbmc8L3RleHQ+CiAgPC9nPgo8L3N2Zz4=
" alt="Figure 27: Bottom-Up and Top-Down Type Checking" />
<figcaption>Figure 27: Bottom-Up and Top-Down Type Checking</figcaption>
</figure>

### 20.4 Formal Type Judgements for Faberge

In programming language theory, type systems are specified as
mathematical inference rules using **Natural Deduction** notation:

$$\frac{\text{Premise}_1 \quad \text{Premise}_2 \quad \dots}{\text{Conclusion}}$$

Where $\Gamma$ denotes the **Type Environment** (our scoped symbol table
mapping variable identifiers $x$ to their declared types $\tau$).

#### 20.4.1 Variables and Literals

$$\frac{x : \tau \in \Gamma}{\Gamma \vdash x : \tau} \quad [\text{T-Var}] \qquad \frac{}{\Gamma \vdash n : \text{Int}} \quad [\text{T-Int}] \qquad \frac{}{\Gamma \vdash s : \text{String}} \quad [\text{T-Str}]$$

#### 20.4.2 Binary Arithmetic Operators ($+, -, *, /, \%$)

Arithmetic operators demand identical numeric operands on both sides and
synthesise the same numeric type:

$$\frac{\Gamma \vdash e_1 : \text{Int} \quad \Gamma \vdash e_2 : \text{Int}}{\Gamma \vdash e_1 + e_2 : \text{Int}} \quad [\text{T-AddInt}] \qquad \frac{\Gamma \vdash e_1 : \text{Float} \quad \Gamma \vdash e_2 : \text{Float}}{\Gamma \vdash e_1 + e_2 : \text{Float}} \quad [\text{T-AddFloat}]$$

#### 20.4.3 Comparison Operators ($==, !=, <, <=, >, >=$)

Comparisons require compatible operand types and always evaluate to
`Bool`:

$$\frac{\Gamma \vdash e_1 : \tau \quad \Gamma \vdash e_2 : \tau}{\Gamma \vdash e_1 == e_2 : \text{Bool}} \quad [\text{T-Eq}] \qquad \frac{\Gamma \vdash e_1 : \tau \quad \Gamma \vdash e_2 : \tau \quad \tau \in \{\text{Int}, \text{Float}\}}{\Gamma \vdash e_1 < e_2 : \text{Bool}} \quad [\text{T-Lt}]$$

#### 20.4.4 Conditional Expressions (`if-then-else`)

In a pure functional language, `if` is an expression that yields a
value. Thus:

1.  The condition expression must evaluate to `Bool`.
2.  Both the `then` branch and the `else` branch must evaluate to the
    **exact same type $\tau$**.

$$\frac{\Gamma \vdash e_{\text{cond}} : \text{Bool} \quad \Gamma \vdash e_{\text{then}} : \tau \quad \Gamma \vdash e_{\text{else}} : \tau}{\Gamma \vdash (\text{if } e_{\text{cond}} \text{ then } e_{\text{then}} \text{ else } e_{\text{else}}) : \tau} \quad [\text{T-If}]$$

#### 20.4.5 Lambda Functions and Function Application (Calls)

A lambda abstraction introduces its parameters into a new child
environment $\Gamma'$:

$$\frac{\Gamma, x_1 : \tau_1, \dots, x_k : \tau_k \vdash e_{\text{body}} : \tau_{\text{ret}}}{\Gamma \vdash (\text{fn } (x_1 : \tau_1, \dots, x_k : \tau_k) \Rightarrow e_{\text{body}}) : ((\tau_1, \dots, \tau_k) \to \tau_{\text{ret}})} \quad [\text{T-Lambda}]$$

A function call verifies that the callee is a function type, the
argument count matches the parameter arity, and each argument type
satisfies its corresponding parameter:

$$\frac{\Gamma \vdash e_{\text{callee}} : ((\tau_1, \dots, \tau_k) \to \tau_{\text{ret}}) \quad \Gamma \vdash a_1 : \tau_1 \quad \dots \quad \Gamma \vdash a_k : \tau_k}{\Gamma \vdash e_{\text{callee}}(a_1, \dots, a_k) : \tau_{\text{ret}}} \quad [\text{T-Call}]$$

### 20.5 Experimental Implementation in Modern C++: Semantic Analyser Architecture

#### 20.5.1 Architecture Outline

Let us architect our Semantic Analyser using clean C++20 abstractions:

- `Type`: A robust value-semantic class representing atomic types
  (`Int`, `Float`, `String`, `Bool`, `Void`), compound function
  signatures `(T1, T2 -> Ret)`, and error states.
- `SymbolTable`: A RAII-friendly lexical environment maintaining scoped
  identifier bindings.
- `TypeChecker`: An AST traversal pass implementing `ASTVisitor` that
  checks types, binds symbols, and decorates every AST node with its
  verified type.

<figure id="fig:semantic-engine">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NDQuNzUyIiBo
ZWlnaHQ9IjI0MS40NTYwMSIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1NDQuNzUyIDI0
MS40NTYwMSI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNp
cmNsZSwgLnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAg
c3Ryb2tlLXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAx
OwogIHN0cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgou
c3ZnYm9iIHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1m
YW1pbHk6IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXpl
OiAxNHB4Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxs
OiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4
Owp9Cgouc3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxs
ZWQgewogIGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAu
bm9maWxsIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJv
dyB7CiAgbWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9h
cnJvdyB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtl
ZF9kaWFtb25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3Zn
Ym9iIC5lbmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgou
c3ZnYm9iIC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJs
KCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAg
bWFya2VyLXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9i
aWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1
cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlk
PSJhcnJvdyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxwb2x5Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtl
cj4KICAgIDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIg
cmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0
YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIi
PjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94
PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0
PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5
PSI0IiByPSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1h
cmtlciBpZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQi
IG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQi
PjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUi
IHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJr
ZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBj
eD0iNCIgY3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJr
ZXI+CiAgPC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRo
PSI1NDQuNzUyIiBoZWlnaHQ9IjI0MS40NTYwMSI+PC9yZWN0PgogIDxyZWN0IHg9IjEwLjQ3NiIg
eT0iMTMyLjY5NiIgd2lkdGg9IjIwOS41MiIgaGVpZ2h0PSI4My44MDgwMTQiIGNsYXNzPSJzb2xp
ZCBub2ZpbGwiIHJ4PSIwIj48L3JlY3Q+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
NDMuNjUiIHk9IjE1MC4xNTYiID5TY29wZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI4NS41NTQiIHk9IjE1MC4xNTYiID5SZXNvbHV0aW9uPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjI5LjY4MjAwMSIgeT0iMTY0LjEyNDAxIiA+LSBwdXNoX3Nj
b3BlKCk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjkuNjgyMDAxIiB5
PSIxNzguMDkyMDEiID4tIHBvcF9zY29wZSgpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjI5LjY4MjAwMSIgeT0iMTkyLjA2MDAxIiA+LSBpbnNlcnRfc3ltYm9sKG5hbWUs
IHQpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI5LjY4MjAwMSIgeT0i
MjA2LjAyOCIgPi0gbG9va3VwX3N5bWJvbChuYW1lKTwvdGV4dD4KICA8cmVjdCB4PSIzMTAuNzg4
IiB5PSIxMzIuNjk2IiB3aWR0aD0iMjIzLjQ4OCIgaGVpZ2h0PSI4My44MDgwMTQiIGNsYXNzPSJz
b2xpZCBub2ZpbGwiIHJ4PSIwIj48L3JlY3Q+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMzM2Ljk3OCIgeT0iMTUwLjE1NiIgPlR5cGU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMzcxLjg5OCIgeT0iMTUwLjE1NiIgPkluZmVyZW5jZTwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NDEuNzM4IiB5PSIxNTAuMTU2IiA+YW5kPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ2OS42NzQiIHk9IjE1MC4xNTYiID5D
aGVjazwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjkuOTk0MDIiIHk9
IjE2NC4xMjQwMSIgPi0gc3ludGhlc2lzZV90eXBlKGV4cHIpPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjMyOS45OTQwMiIgeT0iMTc4LjA5MjAxIiA+LSBjaGVja190eXBl
KGV4cHIsIGV4cGVjdCk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzI5
Ljk5NDAyIiB5PSIxOTIuMDYwMDEiID4tIHZlcmlmeV9jYWxsX2FyaXR5KCk8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzI5Ljk5NDAyIiB5PSIyMDYuMDI4IiA+LSBjaGVj
a19icmFuY2hfZXF1YWxpdHkoKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIxOTAuMzE0MDEiIHk9IjI0LjQ0NCIgPlNlbWFudGljPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjI1My4xNzAwMSIgeT0iMjQuNDQ0IiA+QW5hbHlzZXI8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzE2LjAyNiIgeT0iMjQuNDQ0IiA+RW5naW5l
PC90ZXh0PgogIDxsaW5lIHgxPSIxMTUuMjM2IiB5MT0iOTcuNzc2IiB4Mj0iMTE1LjIzNiIgeTI9
IjExNS4yMzYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxsaW5lIHgxPSI0MTUuNTQ4IiB5MT0i
OTcuNzc2IiB4Mj0iNDE1LjU0OCIgeTI9IjExNS4yMzYiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgog
IDxwb2x5Z29uIHBvaW50cz0iMTExLjc0NCwxMTUuMjM2IDExOC43MjgwMDQsMTE1LjIzNiAxMTUu
MjM2LDEyNS43MTIwMDYiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8cG9seWdvbiBwb2lu
dHM9IjQxMi4wNTYsMTE1LjIzNiA0MTkuMDQsMTE1LjIzNiA0MTUuNTQ4LDEyNS43MTIwMDYiIGNs
YXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIy
OS42ODIwMDEiIHk9IjUyLjM4IiA+U3ltYm9sIFRhYmxlOiBzdGQ6OnNoYXJlZF9wdHImbHQ7RW52
aXJvbm1lbnQmZ3Q7IG1fY3VycmVudF9lbnY8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMjkuNjgyMDAxIiB5PSI2Ni4zNDgiID5UeXBlIERlY29yYXRvcjogc3RkOjp1bm9y
ZGVyZWRfbWFwJmx0O2NvbnN0IEV4cHJBU1QqLCBUeXBlJmd0OyBtX25vZGVfdHlwZXM8L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjkuNjgyMDAxIiB5PSI4MC4zMTYiID5E
aWFnbm9zdGljczogRGlhZ25vc3RpY0VuZ2luZSAoQU5TSSBlcnJvciBjYXJldHMpPC90ZXh0Pgog
IDxnPgogICAgPGxpbmUgeDE9IjEwLjQ3NiIgeTE9IjYuOTg0IiB4Mj0iNTM0LjI3NiIgeTI9IjYu
OTg0IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgxPSIxMC40NzYiIHkxPSI2Ljk4
NCIgeDI9IjEwLjQ3NiIgeTI9IjkwLjc5MiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGlu
ZSB4MT0iNTM0LjI3NiIgeTE9IjYuOTg0IiB4Mj0iNTM0LjI3NiIgeTI9IjkwLjc5MiIgY2xhc3M9
InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMTAuNDc2IiB5MT0iMzQuOTIwMDAyIiB4Mj0i
NTM0LjI3NiIgeTI9IjM0LjkyMDAwMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4
MT0iMTAuNDc2IiB5MT0iOTAuNzkyIiB4Mj0iNTM0LjI3NiIgeTI9IjkwLjc5MiIgY2xhc3M9InNv
bGlkIj48L2xpbmU+CiAgPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 28: Facilities of the Semantic Analyser</figcaption>
</figure>

#### 20.5.2 Implementation

The listings below implement the architecture described in this chapter:

- lst. 10 implments a strongly-typed class to hold the types supported
  by the Faberge compiler.
- lst. 11 creates the class which will hold named and typed symbols to
  allow for lexical scoping.
- lst. 12 implements class `TypeChecker`. It walks the AST, performs
  bottom-up type synthesis and top-down checking, verifies constraints,
  and records type diagnostics.
- lst. 13 verifies the semantic analyser against both valid functional
  programs and intentionally malformed test cases to exercise the
  compiler’s diagnostic error reporting.

Listing 10: Strong Value-Semantic Types (Type.hpp)

In [ ]:
#ifndef TYPE_HPP
#define TYPE_HPP

#include <string>
#include <vector>
#include <memory>
#include <sstream>
#include <algorithm>

namespace Faberge {

enum class PrimitiveKind {
    Int,
    Float,
    String,
    Bool,
    Void,
    Error // Sentinel for poisonous error propagation
};

class Type {
public:
    enum class Kind { Primitive, Function };

    Type() : m_kind(Kind::Primitive), m_primitive_kind(PrimitiveKind::Void) {}
    explicit Type(PrimitiveKind pk) : m_kind(Kind::Primitive), m_primitive_kind(pk) {}

    static Type make_int()    { return Type(PrimitiveKind::Int); }
    static Type make_float()  { return Type(PrimitiveKind::Float); }
    static Type make_string() { return Type(PrimitiveKind::String); }
    static Type make_bool()   { return Type(PrimitiveKind::Bool); }
    static Type make_void()   { return Type(PrimitiveKind::Void); }
    static Type make_error()  { return Type(PrimitiveKind::Error); }

    static Type make_function(std::vector<Type> param_types, Type return_type) {
        Type t(Kind::Function);
        t.m_param_types = std::move(param_types);
        t.m_return_type = std::make_shared<Type>(std::move(return_type));
        return t;
    }

    [[nodiscard]] Kind kind() const noexcept { return m_kind; }
    [[nodiscard]] PrimitiveKind primitive_kind() const noexcept { return m_primitive_kind; }
    [[nodiscard]] bool is_error() const noexcept { 
        return m_kind == Kind::Primitive && m_primitive_kind == PrimitiveKind::Error; 
    }
    [[nodiscard]] bool is_numeric() const noexcept {
        return m_kind == Kind::Primitive && 
               (m_primitive_kind == PrimitiveKind::Int || m_primitive_kind == PrimitiveKind::Float);
    }

    [[nodiscard]] const std::vector<Type>& param_types() const noexcept { return m_param_types; }
    [[nodiscard]] const Type& return_type() const { return *m_return_type; }

    // Strict structural type equality
    bool operator==(const Type& other) const {
        if (is_error() || other.is_error()) return true; // Suppress cascading error spam
        if (m_kind != other.m_kind) return false;

        if (m_kind == Kind::Primitive) {
            return m_primitive_kind == other.m_primitive_kind;
        }

        if (m_param_types.size() != other.m_param_types.size()) return false;
        for (size_t i = 0; i < m_param_types.size(); ++i) {
            if (m_param_types[i] != other.m_param_types[i]) return false;
        }
        return *m_return_type == *other.m_return_type;
    }

    bool operator!=(const Type& other) const { return !(*this == other); }

    [[nodiscard]] std::string to_string() const {
        if (m_kind == Kind::Primitive) {
            switch (m_primitive_kind) {
                case PrimitiveKind::Int:    return "Int";
                case PrimitiveKind::Float:  return "Float";
                case PrimitiveKind::String: return "String";
                case PrimitiveKind::Bool:   return "Bool";
                case PrimitiveKind::Void:   return "Void";
                case PrimitiveKind::Error:  return "<type-error>";
            }
        }

        std::ostringstream ss;
        ss << "(";
        for (size_t i = 0; i < m_param_types.size(); ++i) {
            ss << m_param_types[i].to_string();
            if (i + 1 < m_param_types.size()) ss << ", ";
        }
        ss << " -> " << m_return_type->to_string() << ")";
        return ss.str();
    }

private:
    explicit Type(Kind k) : m_kind(k) {}
    Kind m_kind;
    PrimitiveKind m_primitive_kind{PrimitiveKind::Void};
    std::vector<Type> m_param_types;
    std::shared_ptr<Type> m_return_type;
};

} // namespace Faberge

#endif // TYPE_HPP

Listing 11: Scoped Environment Tree (SymbolTable.hpp)

In [ ]:
#ifndef SYMBOL_TABLE_HPP
#define SYMBOL_TABLE_HPP

// #include "Type.hpp"
#include "../chapter-18/SourceLocation.hpp"
#include <string>
#include <unordered_map>
#include <memory>
#include <optional>
#include <stdexcept>

namespace Faberge {

struct Symbol {
    std::string name;
    Type type;
    SourceLocation location;
    bool is_function{false};
};

class Environment : public std::enable_shared_from_this<Environment> {
public:
    explicit Environment(std::shared_ptr<Environment> parent = nullptr)
        : m_parent(std::move(parent)) {}

    static std::shared_ptr<Environment> create(std::shared_ptr<Environment> parent = nullptr) {
        return std::make_shared<Environment>(std::move(parent));
    }

    [[nodiscard]] std::shared_ptr<Environment> parent() const noexcept { return m_parent; }

    bool define(std::string name, Type type, SourceLocation loc, bool is_fn = false) {
        if (m_bindings.contains(name)) {
            return false; // Duplicate symbol declaration in the EXACT same scope
        }
        m_bindings.emplace(name, Symbol{name, std::move(type), std::move(loc), is_fn});
        return true;
    }

    [[nodiscard]] std::optional<Symbol> lookup(const std::string& name) const {
        auto it = m_bindings.find(name);
        if (it != m_bindings.end()) {
            return it->second;
        }
        if (m_parent) {
            return m_parent->lookup(name);
        }
        return std::nullopt;
    }

private:
    std::shared_ptr<Environment> m_parent;
    std::unordered_map<std::string, Symbol> m_bindings;
};

} // namespace Faberge

#endif // SYMBOL_TABLE_HPP

Listing 12: Type Checker Engine (TypeChecker.hpp)

In [ ]:
#ifndef TYPE_CHECKER_HPP
#define TYPE_CHECKER_HPP

#include "../chapter-19/AST.hpp"
#include "../chapter-19/ASTVisitor.hpp"
// #include "SymbolTable.hpp"
#include "../chapter-18/Diagnostic.hpp"
#include <unordered_map>

namespace Faberge {

class TypeChecker : public ASTVisitor {
public:
    explicit TypeChecker(std::string_view source_code)
        : m_source_code(source_code), 
          m_current_env(Environment::create(nullptr)) {}

    bool check(const ProgramAST& program) {
        m_error_count = 0;
        program.accept(*this);
        return m_error_count == 0;
    }

    [[nodiscard]] size_t error_count() const noexcept { return m_error_count; }

    [[nodiscard]] Type get_node_type(const ExprAST& expr) const {
        auto it = m_node_types.find(&expr);
        if (it != m_node_types.end()) return it->second;
        return Type::make_error();
    }

    // -------------------------------------------------------------------------
    // Visitor Dispatch Implementations
    // -------------------------------------------------------------------------

    void visit(const ProgramAST& node) override {
        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }
    }

    void visit(const LetStmtAST& node) override {
        // Pre-bind function identifier in environment to support recursive self-references
        const auto* lambda = dynamic_cast<const LambdaExprAST*>(&node.initialiser());
        bool prebound = false;
        Type prebound_type = Type::make_error();

        if (node.type_annotation().has_value()) {
            prebound_type = convert_ast_type(*node.type_annotation());
            prebound = true;
        } else if (lambda && lambda->return_type().has_value()) {
            std::vector<Type> param_types;
            for (const auto& p : lambda->parameters()) {
                param_types.push_back(convert_ast_type(p.type));
            }
            Type ret_type = convert_ast_type(*lambda->return_type());
            prebound_type = Type::make_function(std::move(param_types), ret_type);
            prebound = true;
        }

        if (prebound) {
            if (!m_current_env->define(node.name(), prebound_type, node.location(), lambda != nullptr)) {
                report_type_error(node.location(), 
                                  "Redeclaration of identifier '" + node.name() + "' in the same scope.");
            }
        }

        node.initialiser().accept(*this);
        Type init_type = get_node_type(node.initialiser());

        Type var_type = init_type;

        // If explicit type annotation exists, verify compatibility
        if (node.type_annotation().has_value()) {
            Type declared_type = convert_ast_type(*node.type_annotation());
            if (init_type != declared_type && !init_type.is_error()) {
                report_type_error(node.initialiser().location(),
                                  "Type mismatch in variable declaration: expected '" + 
                                  declared_type.to_string() + "', got '" + 
                                  init_type.to_string() + "'.");
            }
            var_type = declared_type;
        }

        if (!prebound) {
            if (!m_current_env->define(node.name(), var_type, node.location())) {
                report_type_error(node.location(), 
                                  "Redeclaration of identifier '" + node.name() + "' in the same scope.");
            }
        }
    }

    void visit(const ExprStmtAST& node) override {
        node.expression().accept(*this);
    }

    void visit(const LiteralExprAST& node) override {
        Type t = Type::make_error();
        switch (node.token_type()) {
            case TokenType::Literal_Integer: t = Type::make_int(); break;
            case TokenType::Literal_Float:   t = Type::make_float(); break;
            case TokenType::Literal_String:  t = Type::make_string(); break;
            case TokenType::Literal_Boolean: t = Type::make_bool(); break;
            default: break;
        }
        set_node_type(node, t);
    }

    void visit(const VariableExprAST& node) override {
        auto sym = m_current_env->lookup(node.name());
        if (sym == std::nullopt) {
            report_type_error(node.location(), 
                              "Use of undeclared identifier '" + node.name() + "'.");
            set_node_type(node, Type::make_error());
            return;
        }
        set_node_type(node, sym->type);
    }

    void visit(const UnaryExprAST& node) override {
        node.operand().accept(*this);
        Type operand_type = get_node_type(node.operand());

        if (node.op() == TokenType::Minus) {
            if (!operand_type.is_numeric() && !operand_type.is_error()) {
                report_type_error(node.location(), 
                                  "Unary '-' operator requires numeric operand, found '" + 
                                  operand_type.to_string() + "'.");
                set_node_type(node, Type::make_error());
                return;
            }
            set_node_type(node, operand_type);
            return;
        }

        if (node.op() == TokenType::Exclamation) {
            if (operand_type != Type::make_bool() && !operand_type.is_error()) {
                report_type_error(node.location(), 
                                  "Logical '!' operator requires Bool operand, found '" + 
                                  operand_type.to_string() + "'.");
                set_node_type(node, Type::make_error());
                return;
            }
            set_node_type(node, Type::make_bool());
            return;
        }

        set_node_type(node, Type::make_error());
    }

    void visit(const BinaryExprAST& node) override {
        node.left().accept(*this);
        node.right().accept(*this);

        Type left_t = get_node_type(node.left());
        Type right_t = get_node_type(node.right());

        if (left_t.is_error() || right_t.is_error()) {
            set_node_type(node, Type::make_error());
            return;
        }

        // 1. Arithmetic Operators (+, -, *, /, %)
        if (node.op() == TokenType::Plus || node.op() == TokenType::Minus ||
            node.op() == TokenType::Asterisk || node.op() == TokenType::Slash || 
            node.op() == TokenType::Percent) {
            
            // Allow string concatenation on '+'
            if (node.op() == TokenType::Plus && left_t == Type::make_string() && right_t == Type::make_string()) {
                set_node_type(node, Type::make_string());
                return;
            }

            if (!left_t.is_numeric() || !right_t.is_numeric() || left_t != right_t) {
                report_type_error(node.location(), 
                                  "Binary operator '" + std::string(Token::to_string(node.op())) + 
                                  "' expects matching numeric operands, found '" + 
                                  left_t.to_string() + "' and '" + right_t.to_string() + "'.");
                set_node_type(node, Type::make_error());
                return;
            }

            set_node_type(node, left_t);
            return;
        }

        // 2. Comparison Operators (==, !=, <, <=, >, >=)
        if (node.op() == TokenType::EqualEqual || node.op() == TokenType::ExclamationEqual) {
            if (left_t != right_t) {
                report_type_error(node.location(), 
                                  "Equality operator requires identical operand types, found '" + 
                                  left_t.to_string() + "' and '" + right_t.to_string() + "'.");
            }
            set_node_type(node, Type::make_bool());
            return;
        }

        if (node.op() == TokenType::Less || node.op() == TokenType::LessEqual ||
            node.op() == TokenType::Greater || node.op() == TokenType::GreaterEqual) {
            if (!left_t.is_numeric() || !right_t.is_numeric() || left_t != right_t) {
                report_type_error(node.location(), 
                                  "Relational comparison requires matching numeric types, found '" + 
                                  left_t.to_string() + "' and '" + right_t.to_string() + "'.");
            }
            set_node_type(node, Type::make_bool());
            return;
        }

        // 3. Logical Operators (&&, ||)
        if (node.op() == TokenType::AmpersandAmpersand || node.op() == TokenType::PipePipe) {
            if (left_t != Type::make_bool() || right_t != Type::make_bool()) {
                report_type_error(node.location(), 
                                  "Logical operator requires Bool operands, found '" + 
                                  left_t.to_string() + "' and '" + right_t.to_string() + "'.");
            }
            set_node_type(node, Type::make_bool());
            return;
        }

        set_node_type(node, Type::make_error());
    }

    void visit(const IfExprAST& node) override {
        node.condition().accept(*this);
        Type cond_t = get_node_type(node.condition());

        if (cond_t != Type::make_bool() && !cond_t.is_error()) {
            report_type_error(node.condition().location(), 
                              "If condition must evaluate to Bool, found '" + cond_t.to_string() + "'.");
        }

        node.then_branch().accept(*this);
        node.else_branch().accept(*this);

        Type then_t = get_node_type(node.then_branch());
        Type else_t = get_node_type(node.else_branch());

        if (then_t != else_t && !then_t.is_error() && !else_t.is_error()) {
            report_type_error(node.location(), 
                              "Branches of 'if' expression must have matching types. Then branch is '" + 
                              then_t.to_string() + "', Else branch is '" + else_t.to_string() + "'.");
            set_node_type(node, Type::make_error());
            return;
        }

        set_node_type(node, then_t);
    }

    void visit(const CallExprAST& node) override {
        node.callee().accept(*this);
        Type callee_t = get_node_type(node.callee());

        if (callee_t.is_error()) {
            set_node_type(node, Type::make_error());
            return;
        }

        if (callee_t.kind() != Type::Kind::Function) {
            report_type_error(node.location(), 
                              "Attempted to call non-function expression of type '" + 
                              callee_t.to_string() + "'.");
            set_node_type(node, Type::make_error());
            return;
        }

        if (node.arguments().size() != callee_t.param_types().size()) {
            report_type_error(node.location(), 
                              "Function expects " + std::to_string(callee_t.param_types().size()) + 
                              " arguments, but received " + std::to_string(node.arguments().size()) + ".");
            set_node_type(node, callee_t.return_type());
            return;
        }

        for (size_t i = 0; i < node.arguments().size(); ++i) {
            node.arguments()[i]->accept(*this);
            Type arg_t = get_node_type(*node.arguments()[i]);
            Type expected_param_t = callee_t.param_types()[i];

            if (arg_t != expected_param_t && !arg_t.is_error()) {
                report_type_error(node.arguments()[i]->location(), 
                                  "Argument " + std::to_string(i + 1) + " type mismatch: expected '" + 
                                  expected_param_t.to_string() + "', got '" + arg_t.to_string() + "'.");
            }
        }

        set_node_type(node, callee_t.return_type());
    }

    void visit(const BlockExprAST& node) override {
        push_scope();

        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }

        Type block_result_t = Type::make_void();
        if (node.result_expr()) {
            node.result_expr()->accept(*this);
            block_result_t = get_node_type(*node.result_expr());
        }

        pop_scope();
        set_node_type(node, block_result_t);
    }

    void visit(const LambdaExprAST& node) override {
        std::vector<Type> param_types;
        
        // Construct child environment for function parameters
        auto fn_env = Environment::create(m_current_env);

        for (const auto& p : node.parameters()) {
            Type pt = convert_ast_type(p.type);
            param_types.push_back(pt);
            fn_env->define(p.name, pt, p.location);
        }

        // Switch to function environment to evaluate body
        auto prev_env = m_current_env;
        m_current_env = fn_env;

        node.body().accept(*this);
        Type body_t = get_node_type(node.body());

        m_current_env = prev_env;

        Type return_t = body_t;
        if (node.return_type().has_value()) {
            Type declared_ret = convert_ast_type(*node.return_type());
            if (body_t != declared_ret && !body_t.is_error()) {
                report_type_error(node.body().location(), 
                                  "Function body type mismatch: expected return type '" + 
                                  declared_ret.to_string() + "', but body evaluates to '" + 
                                  body_t.to_string() + "'.");
            }
            return_t = declared_ret;
        }

        Type fn_type = Type::make_function(std::move(param_types), return_t);
        set_node_type(node, fn_type);
    }

private:
    std::string_view m_source_code;
    std::shared_ptr<Environment> m_current_env;
    std::unordered_map<const ExprAST*, Type> m_node_types;
    size_t m_error_count{0};

    void push_scope() {
        m_current_env = Environment::create(m_current_env);
    }

    void pop_scope() {
        if (m_current_env->parent()) {
            m_current_env = m_current_env->parent();
        }
    }

    void set_node_type(const ExprAST& expr, Type t) {
        m_node_types[&expr] = std::move(t);
    }

    void report_type_error(SourceLocation loc, std::string_view msg) {
        ++m_error_count;
        DiagnosticEngine::report_error(m_source_code, loc, msg);
    }

    static Type convert_ast_type(const TypeNode& ast_type) {
        switch (ast_type.kind()) {
            case TypeNode::Kind::Primitive:
                switch (ast_type.primitive_kind()) {
                    case PrimitiveTypeKind::Int:    return Type::make_int();
                    case PrimitiveTypeKind::Float:  return Type::make_float();
                    case PrimitiveTypeKind::String: return Type::make_string();
                    case PrimitiveTypeKind::Bool:   return Type::make_bool();
                    case PrimitiveTypeKind::Void:   return Type::make_void();
                }
                break;
            case TypeNode::Kind::Function:
                // Extendable for higher-order type annotations
                return Type::make_error();
            case TypeNode::Kind::List:
                return Type::make_error();
        }
        return Type::make_error();
    }
};

} // namespace Faberge

#endif // TYPE_CHECKER_HPP

Listing 13: Verification and Negative Diagnostics Showcase

In [ ]:
#include "../chapter-18/Lexer.hpp"
#include "../chapter-19/Parser.hpp"
// #include "TypeChecker.hpp"
#include <iostream>

void run_semantic_test(std::string_view title, std::string_view source_code) {
    using namespace Faberge;

    std::cout << "\n====================================================\n";
    std::cout << " TEST CASE: " << title << "\n";
    std::cout << "====================================================\n";
    std::cout << "SOURCE CODE:\n" << source_code << "\n----------------------------------------------------\n";

    // Stage 1: Lexical Analysis
    Lexer lexer(source_code, "test.fab");
    auto tokens = lexer.tokenise();
    if (lexer.has_errors()) {
        std::cerr << "[Abort]: Lexer errors encountered.\n";
        return;
    }

    // Stage 2: Syntactic Analysis (Parser)
    Parser parser(std::move(tokens), source_code);
    auto ast = parser.parse_program();
    if (parser.has_errors()) {
        std::cerr << "[Abort]: Parser errors encountered.\n";
        return;
    }

    // Stage 3: Semantic Analysis and Type Checking
    TypeChecker checker(source_code);
    bool passed = checker.check(*ast);

    if (passed) {
        std::cout << "\033[1;32m[PASS]: Static Semantic and Type Verification Succeeded (0 Errors).\033[0m\n";
    } else {
        std::cout << "\033[1;31m[FAIL]: Semantic Analysis Failed with " 
                  << checker.error_count() << " error(s).\033[0m\n";
    }
}

int main() {
    // 1. Valid Higher-Order Recursive Program
    constexpr std::string_view valid_source = R"(
        let add = fn (a: Int, b: Int): Int => a + b;

        let factorial = fn (n: Int): Int => {
            if n <= 1 then 1 else n * factorial(n - 1)
        };

        let main = fn (): Int => {
            let x = 10;
            let result = factorial(x);
            add(result, 42)
        };
    )";
    run_semantic_test("Valid Recursive Functional Program", valid_source);

    // 2. Intentional Semantic Violations
    constexpr std::string_view invalid_source = R"(
        let bad_math = 10 + "hello"; // Error 1: Type mismatch in addition

        let bad_branch = if true then 42 else "divergent string"; // Error 2: Branch type mismatch

        let bad_call = fn (x: Int): Int => {
            let local_flag = true;
            undeclared_function(x, local_flag) // Error 3: Undeclared identifier
        };

        let return_mismatch = fn (a: Int): Int => {
            "this is not an integer" // Error 4: Return type mismatch
        };
    )";
    //run_semantic_test("Negative Semantic Tests (Intentional Errors)", invalid_source);

    return 0;
}

### 20.6 Summary

In this third stage of our compiler project, we transitioned from
structural syntax to semantic meaning and type safety:

- **Semantic Analysis** enforces logical invariants that context-free
  grammars cannot express, verifying identifier existence, visibility,
  and operand type compatibility.
- **The Scoped Symbol Table (Cactus Stack)** mirrors lexical scoping
  hierarchies, supporting nested child environments and variable
  shadowing.
- **Milner’s Dictum** proves that a sound static type system prevents
  well-typed programs from entering undefined or illegal runtime states.
- **Bidirectional Type Checking** combines bottom-up synthesis with
  top-down expected-type checking to deduce and verify expression types
  cleanly.

### 20.7 Programming Challenges

#### Exercise 20.1: Immutable vs Mutable Binding Enforcement (`let` vs `mut`)

**Context and Theory:**

In modern systems and functional languages, immutability is the default
state of values. In Faberge, a binding declared via `let` is immutable:
once assigned an initial value, its memory cannot be reassigned. To
allow mutation, the developer must explicitly annotate the binding with
the `mut` keyword:

$$\text{let } x = 10; \quad x = 20; \quad \implies \text{Semantic Error (Mutation of Immutable Binding)}$$

$$\text{let mut } y = 10; \quad y = 20; \quad \implies \text{Valid (Mutable Binding)}$$

The semantic analyser enforces two invariants during assignment
expressions ($x = e$):

1.  **Mutability Invariant:** The target symbol $x$ must be declared
    with `is_mutable = true` in the active scope environment $\Gamma$.
2.  **Type Invariance:** The type of the assigned expression $e$ must
    match the declared type of $x$:

$$\frac{x : \tau \in \Gamma \quad \text{is\_mutable}(x) = \text{true} \quad \Gamma \vdash e : \tau}{\Gamma \vdash (x = e) : \tau} \quad [\text{T-Assign}]$$

<figure id="fig:mutable">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MTQuNTkyMDQi
IGhlaWdodD0iNTkuODcyIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDYxNC41OTIwNCA1
OS44NzIiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTog
MTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsK
fQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVk
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5v
ZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7
Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1h
cmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmln
X29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJs
KCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0i
YXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+
CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJl
Zlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFy
dC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48
L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0i
MCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0i
NyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0i
NCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJr
ZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBt
YXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJz
ZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48
L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2
aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2Vy
SGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9
IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2Vy
PgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0i
NjE0LjU5MjA0IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+WzwvdGV4dD4KICA8bGluZSB4MT0iMzQ5
LjIiIHkxPSI2Ljk4NCIgeDI9IjM3MC4xNTIiIHkyPSI2Ljk4NCIgY2xhc3M9InNvbGlkIj48L2xp
bmU+CiAgPHBvbHlnb24gcG9pbnRzPSIzNzAuMTUyLDMuNDkyIDM3Ny4xMzYwMiw2Ljk4NCAzNzAu
MTUyLDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjM4NS44NjYiIHk9IjEwLjQ3NiIgPk11dGF0aW9uPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ0OC43MjIwMiIgeT0iMTAuNDc2IiA+QXR0ZW1wdDwv
dGV4dD4KICA8bGluZSB4MT0iNTAyLjg0ODAyIiB5MT0iNi45ODQiIHgyPSI1MjMuOCIgeTI9IjYu
OTg0IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjUyMy44LDMuNDky
IDUzMC43ODQsNi45ODQgNTIzLjgsMTAuNDc2IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTM5LjUxNDA0IiB5PSIxMC40NzYiID5UeXBl
RXJyb3IhPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjguNzMwMDAwNSIg
eT0iMzguNDEyMDAzIiA+WzwvdGV4dD4KICA8bGluZSB4MT0iMzQ5LjIiIHkxPSIzNC45MjAwMDIi
IHgyPSIzNzAuMTUyIiB5Mj0iMzQuOTIwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9s
eWdvbiBwb2ludHM9IjM3MC4xNTIsMzEuNDI4MDAxIDM3Ny4xMzYwMiwzNC45MjAwMDIgMzcwLjE1
MiwzOC40MTIwMDMiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIzODUuODY2IiB5PSIzOC40MTIwMDMiID5NdXRhdGlvbjwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NDguNzIyMDIiIHk9IjM4LjQxMjAwMyIgPkF0
dGVtcHQ8L3RleHQ+CiAgPGxpbmUgeDE9IjUwMi44NDgwMiIgeTE9IjM0LjkyMDAwMiIgeDI9IjUy
My44IiB5Mj0iMzQuOTIwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2lu
dHM9IjUyMy44LDMxLjQyODAwMSA1MzAuNzg0LDM0LjkyMDAwMiA1MjMuOCwzOC40MTIwMDMiIGNs
YXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1
MzkuNTE0MDQiIHk9IjM4LjQxMjAwMyIgPk9LPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjIyLjY5OCIgeT0iMTAuNDc2IiA+U3ltYm9sOiAmIzM5O3gmIzM5OyB8IFR5cGU6
IEludCB8IGlzX211dGFibGU6IGZhbHNlIF08L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMjIuNjk4IiB5PSIzOC40MTIwMDMiID5TeW1ib2w6ICYjMzk7eSYjMzk7IHwgVHlw
ZTogSW50IHwgaXNfbXV0YWJsZTogdHJ1ZSAgXTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 29: Symbol Table Entry</figcaption>
</figure>

**Specification:**

Extend the `Symbol` descriptor with a `bool is_mutable` flag. Implement
a new AST node
`AssignExprAST(SourceLocation, std::string name, std::unique_ptr<ExprAST> value)`.
In `TypeChecker`, implement the assignment visitor: verify that the
identifier exists, assert that it is mutable, and check that the
assigned expression matches the variable’s type.

``` cpp
struct Symbol {
    std::string name;
    Type type;
    SourceLocation location;
    bool is_mutable{false};
};

class TypeCheckerExtension {
public:
    void check_assignment(const std::string& name, const ExprAST& val_expr, SourceLocation loc) {
        auto sym = m_current_env->lookup(name);
        if (!sym) {
            report_error(loc, "Assignment to undeclared identifier '" + name + "'.");
            return;
        }
        if (!sym->is_mutable) {
            report_error(loc, "Cannot reassign to immutable binding '" + name + 
                              "'. Declared with 'let'; did you mean 'let mut'?");
        }
        Type val_type = synthesise_type(val_expr);
        if (val_type != sym->type && !val_type.is_error()) {
            report_error(loc, "Type mismatch in assignment to '" + name + "': expected '" + 
                              sym->type.to_string() + "', got '" + val_type.to_string() + "'.");
        }
    }
};
```

**Verification Tasks:**

1.  **Immutable Reassignment Rejection:** Attempt to reassign an
    immutable variable: `let x = 10; x = 20;`. Verify that the type
    checker rejects the program with an immutability violation error.
2.  **Valid Mutable Mutation:** Check
    `let mut count = 0; count = count + 1;`. Verify that the assignment
    passes with zero errors and synthesises type `Int`.
3.  **Mutable Type Mismatch:** Attempt to assign a string to a mutable
    integer: `let mut y = 42; y = "forty-two";`. Verify that the
    analyser catches the type mismatch and preserves the variable’s
    original integer type.

#### Exercise 20.2: First-Class Homogeneous List and Subscript Type Verification

**Context and Theory:**

In Faberge, sequential collections are represented as **Homogeneous
Lists** $[e_1, e_2, \dots, e_k]$, where every element shares an
identical type $\tau$. The collection itself has the compound parametric
type $\text{List}\langle\tau\rangle$.

The type checker enforces two formal deduction rules (see fig. 30):

1.  **List Construction:** Every element expression in the literal must
    unify with a common type $\tau$. If the list is empty `[]`, its type
    must be inferred from surrounding contextual expectations (e.g. an
    explicit type annotation):

$$\frac{\Gamma \vdash e_1 : \tau \quad \Gamma \vdash e_2 : \tau \quad \dots \quad \Gamma \vdash e_k : \tau}{\Gamma \vdash [e_1, e_2, \dots, e_k] : \text{List}\langle\tau\rangle} \quad [\text{T-List}]$$

1.  **Subscript Indexing:** The collection target must evaluate to
    $\text{List}\langle\tau\rangle$, and the subscript index must
    evaluate strictly to `Int`:

$$\frac{\Gamma \vdash e_{\text{target}} : \text{List}\langle\tau\rangle \quad \Gamma \vdash e_{\text{index}} : \text{Int}}{\Gamma \vdash e_{\text{target}}[e_{\text{index}}] : \tau} \quad [\text{T-Index}]$$

<figure id="fig:type-checker">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NTguNzIwMDMi
IGhlaWdodD0iODcuODA4IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDU1OC43MjAwMyA4
Ny44MDgiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTog
MTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsK
fQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVk
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5v
ZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7
Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1h
cmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmln
X29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJs
KCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0i
YXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+
CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJl
Zlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFy
dC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48
L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0i
MCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0i
NyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0i
NCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJr
ZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBt
YXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJz
ZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48
L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2
aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2Vy
SGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9
IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2Vy
PgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0i
NTU4LjcyMDAzIiBoZWlnaHQ9Ijg3LjgwOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+TGl0ZXJhbDo8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzEuNTg2IiB5PSIxMC40NzYiID5bPC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijg1LjU1NCIgeT0iMTAuNDc2IiA+MTAsPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjExMy40OTAwMDUiIHk9IjEwLjQ3NiIg
PjIwLDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNDEuNDI2MDEiIHk9
IjEwLjQ3NiIgPjMwPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE2Mi4z
NzgiIHk9IjEwLjQ3NiIgPl08L3RleHQ+CiAgPGxpbmUgeDE9IjI0NC40NCIgeTE9IjYuOTg0IiB4
Mj0iMjY1LjM5MiIgeTI9IjYuOTg0IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBw
b2ludHM9IjI2NS4zOTIsMy40OTIgMjcyLjM3Niw2Ljk4NCAyNjUuMzkyLDEwLjQ3NiIgY2xhc3M9
ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI4MS4x
MDYwMiIgeT0iMTAuNDc2IiA+SW5mZXJyZWQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMzQzLjk2MiIgeT0iMTAuNDc2IiA+VHlwZTo8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMzg1Ljg2NiIgeT0iMTAuNDc2IiA+TGlzdCZsdDtJbnQmZ3Q7PC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjguNzMwMDAwNSIgeT0iMzguNDEy
MDAzIiA+SW5kZXg6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjcxLjU4
NiIgeT0iMzguNDEyMDAzIiA+bnVtYmVyc1swXTwvdGV4dD4KICA8bGluZSB4MT0iMjQ0LjQ0IiB5
MT0iMzQuOTIwMDAyIiB4Mj0iMjY1LjM5MiIgeTI9IjM0LjkyMDAwMiIgY2xhc3M9InNvbGlkIj48
L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyNjUuMzkyLDMxLjQyODAwMSAyNzIuMzc2LDM0Ljky
MDAwMiAyNjUuMzkyLDM4LjQxMjAwMyIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI4MS4xMDYwMiIgeT0iMzguNDEyMDAzIiA+U3ludGhl
c2lzZWQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzY0LjkxNCIgeT0i
MzguNDEyMDAzIiA+VHlwZTo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
NDA2LjgxODAyIiB5PSIzOC40MTIwMDMiID5JbnQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iOC43MzAwMDA1IiB5PSI2Ni4zNDgiID5JbGxlZ2FsOjwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3MS41ODYiIHk9IjY2LjM0OCIgPls8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iODUuNTU0IiB5PSI2Ni4zNDgiID4xMCw8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTEzLjQ5MDAwNSIgeT0iNjYuMzQ4
IiA+JiMzOTtzdHJpbmcmIzM5Oyw8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMTgzLjMzIiB5PSI2Ni4zNDgiID50cnVlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjIxOC4yNSIgeT0iNjYuMzQ4IiA+XTwvdGV4dD4KICA8bGluZSB4MT0iMjQ0LjQ0
IiB5MT0iNjIuODU2MDAzIiB4Mj0iMjY1LjM5MiIgeTI9IjYyLjg1NjAwMyIgY2xhc3M9InNvbGlk
Ij48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyNjUuMzkyLDU5LjM2NDAwMiAyNzIuMzc2LDYy
Ljg1NjAwMyAyNjUuMzkyLDY2LjM0OCIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI4MS4xMDYwMiIgeT0iNjYuMzQ4IiA+U2VtYW50aWM8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzQzLjk2MiIgeT0iNjYuMzQ4
IiA+RXJyb3I6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM5Mi44NSIg
eT0iNjYuMzQ4IiA+SGV0ZXJvZ2VuZW91czwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSI0OTAuNjI2IiB5PSI2Ni4zNDgiID5FbGVtZW50cyE8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 30: List and Subscript Type Verification</figcaption>
</figure>

**Specification:**

Extend Faberge’s `Type` class with `Kind::List` containing an inner
element `std::shared_ptr<Type>`. Implement visitor methods in
`TypeChecker` for `ListLiteralExprAST` and `IndexExprAST`, enforcing
element type homogeneity and integer-only indexing.

``` cpp
class ListTypeChecker {
public:
    Type check_list_literal(const ListLiteralExprAST& node) {
        if (node.elements().empty()) return Type::make_list(Type::make_unknown());
        Type element_type = synthesise_type(*node.elements()[0]);
        for (size_t i = 1; i < node.elements().size(); ++i) {
            Type t = synthesise_type(*node.elements()[i]);
            if (t != element_type && !t.is_error()) {
                report_error(node.elements()[i]->location(), 
                             "List element type mismatch: expected '" + element_type.to_string() + 
                             "', got '" + t.to_string() + "'.");
            }
        }
        return Type::make_list(element_type);
    }
};
```

**Verification Tasks:**

1.  **Homogeneous List Deduction:** Check `let primes = [2, 3, 5, 7];`.
    Verify that the variable `primes` is entered into the symbol table
    with type `List<Int>`.
2.  **Heterogeneous List Rejection:** Check
    `let mixed = [1, "two", 3];`. Verify that the type checker flags the
    second element as a type mismatch.
3.  **Non-Integer Index Rejection:** Check
    `let arr = [10, 20]; let v = arr["first"];`. Verify that the
    analyser rejects the string index, stating that subscript operations
    demand an `Int`.

#### Exercise 20.3: Heterogeneous Product Types (Tuples) and Arity Equivalence

**Context and Theory:**

In contrast to homogeneous lists, a **Tuple** is a heterogeneous product
type whose components can each possess a distinct type:

$$\tau_{\text{tuple}} = (\tau_1 \times \tau_2 \times \dots \times \tau_k)$$

Tuple types obey **Structural Equivalence**. Two tuple types are
identical if and only if their arities match and their component types
are pairwise identical:

$$(\tau_1, \dots, \tau_n) \equiv (\sigma_1, \dots, \sigma_m) \iff n = m \land \forall i \in [1, n]: \tau_i \equiv \sigma_i$$

Tuple component extraction using dot-indexing (e.g. `pair.0`, `pair.1`)
must be checked statically against the tuple’s compile-time arity $k$.
Attempting to access an index out of bounds ($\text{idx} \ge k$) is an
illegal semantic error (see fig. 31).

<figure id="fig:tuple-equivalence">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2NzcuNDQ4IiBo
ZWlnaHQ9Ijg3LjgwOCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA2NzcuNDQ4IDg3Ljgw
OCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI2Nzcu
NDQ4IiBoZWlnaHQ9Ijg3LjgwOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjE0MS40MjYwMSIgeT0iMTAuNDc2IiA+RmFiZXJnZTwvdGV4dD4KICA8bGluZSB4MT0iMjU4
LjQwODAyIiB5MT0iNi45ODQiIHgyPSIyNzkuMzYwMDIiIHkyPSI2Ljk4NCIgY2xhc3M9InNvbGlk
Ij48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyNzkuMzYwMDIsMy40OTIgMjg2LjM0NCw2Ljk4
NCAyNzkuMzYwMDIsMTAuNDc2IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9
IjI1OC40MDgwMiIgeTE9IjM0LjkyMDAwMiIgeDI9IjI3OS4zNjAwMiIgeTI9IjM0LjkyMDAwMiIg
Y2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyNzkuMzYwMDIsMzEuNDI4
MDAxIDI4Ni4zNDQsMzQuOTIwMDAyIDI3OS4zNjAwMiwzOC40MTIwMDMiIGNsYXNzPSJmaWxsZWQi
PjwvcG9seWdvbj4KICA8bGluZSB4MT0iMjU4LjQwODAyIiB5MT0iNjIuODU2MDAzIiB4Mj0iMjc5
LjM2MDAyIiB5Mj0iNjIuODU2MDAzIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBw
b2ludHM9IjI3OS4zNjAwMiw1OS4zNjQwMDIgMjg2LjM0NCw2Mi44NTYwMDMgMjc5LjM2MDAyLDY2
LjM0OCIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+RXhwcmVzc2lvbjogKDQyLCA8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTkwLjMxNDAxIiB5PSIxMC40NzYiID4sIHRy
dWUpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI5NS4wNzQiIHk9IjEw
LjQ3NiIgPlR5cGU6IChJbnQgKiBTdHJpbmcgKiBCb29sKTwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSI4LjczMDAwMDUiIHk9IjM4LjQxMjAwMyIgPkFjY2VzczogICAgIHQu
MDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOTUuMDc0IiB5PSIzOC40
MTIwMDMiID5UeXBlOiBJbnQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
OC43MzAwMDA1IiB5PSI2Ni4zNDgiID5BY2Nlc3M6ICAgICB0LjM8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjk1LjA3NCIgeT0iNjYuMzQ4IiA+Q29tcGlsZS1UaW1lIEVy
cm9yOiBJbmRleCAzIG91dCBvZiBib3VuZHMgZm9yIGFyaXR5IDMhPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 31: Structural Equivalence in Tuples</figcaption>
</figure>

**Specification:**

Extend `Type` with `Kind::Tuple` holding `std::vector<Type>`. Implement
type checking for `TupleExprAST` and a member-access expression
`TupleAccessExprAST(SourceLocation, std::unique_ptr<ExprAST> target, size_t index)`.

``` cpp
class TupleTypeChecker {
public:
    Type check_tuple_access(const ExprAST& target, size_t index, SourceLocation loc) {
        Type target_t = synthesise_type(target);
        if (target_t.kind() != Type::Kind::Tuple) {
            report_error(loc, "Cannot perform tuple indexing on non-tuple type '" + 
                              target_t.to_string() + "'.");
            return Type::make_error();
        }
        if (index >= target_t.tuple_element_types().size()) {
            report_error(loc, "Tuple index " + std::to_string(index) + " out of bounds for tuple of arity " + 
                              std::to_string(target_t.tuple_element_types().size()) + ".");
            return Type::make_error();
        }
        return target_t.tuple_element_types()[index];
    }
};
```

**Verification Tasks:**

1.  **Tuple Synthesis:** Check `let record = (101, "Alice", 3.14);`.
    Verify that `record` synthesises type `(Int, String, Float)`.
2.  **Static Index Projection:** Check `let name = (101, "Alice").1;`.
    Verify that `name` is statically verified as type `String`.
3.  **Out-of-Bounds Rejection:** Check `let invalid = (10, 20).5;`.
    Verify that the checker flags an index out-of-bounds error at
    compile time.

#### Exercise 20.4: Recursive Function Self-Binding and Mutual Recursion

**Context and Theory:**

In a functional language, functions can reference themselves
recursively:

```
let factorial = fn (n: Int): Int => {
    if n <= 1 then 1 else n * factorial(n - 1)
};
```

Under a naive sequential symbol table, evaluating the lambda body fails
with an `Undeclared Identifier` error because `factorial` has not yet
been bound in the enclosing environment!

Furthermore, **Mutual Recursion** occurs when function $f$ invokes $g$,
and $g$ invokes $f$:

```
let is_even = fn (n: Int): Bool => if n == 0 then true else is_odd(n - 1);
let is_odd  = fn (n: Int): Bool => if n == 0 then false else is_even(n - 1);
```

To resolve this, the semantic analyser must execute a **Two-Pass
Environment Setup** over program blocks (see fig. 32):

- **Pass 1 (Hoisting / Signature Registration):** Traverse all top-level
  `LetStmtAST` nodes. If a statement binds a function with explicit
  parameter and return types, pre-declare the function symbol in the
  environment *before* checking function bodies.
- **Pass 2 (Body Verification):** Traverse the function bodies. All
  function names are now mutually visible.

<figure id="fig:two-pass-environment">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0MTIuMDU2IiBo
ZWlnaHQ9IjU5Ljg3MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA0MTIuMDU2IDU5Ljg3
MiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI0MTIu
MDU2IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+UGFzczwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSI0My42NSIgeT0iMTAuNDc2IiA+MTo8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iNjQuNjAyMDA1IiB5PSIxMC40NzYiID5FbnZpcm9ubWVudDwvdGV4
dD4KICA8cG9seWdvbiBwb2ludHM9IjE1My42NDgwMSwzLjQ5MiAxNDYuNjY0LDYuOTg0IDE1My42
NDgwMSwxMC40NzYiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGluZSB4MT0iMTUzLjY0
ODAxIiB5MT0iNi45ODQiIHgyPSIxNjcuNjE2IiB5Mj0iNi45ODQiIGNsYXNzPSJzb2xpZCI+PC9s
aW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE3Ni4zNDYwMSIgeT0iMTAuNDc2
IiA+UmVnaXN0ZXI8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNjQuNjAy
MDA1IiB5PSIyNC40NDQiID5FbnZpcm9ubWVudDwvdGV4dD4KICA8cG9seWdvbiBwb2ludHM9IjE1
My42NDgwMSwxNy40NjAwMDEgMTQ2LjY2NCwyMC45NTIgMTUzLjY0ODAxLDI0LjQ0NCIgY2xhc3M9
ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSIxNTMuNjQ4MDEiIHkxPSIyMC45NTIiIHgy
PSIxNjcuNjE2IiB5Mj0iMjAuOTUyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIxNzYuMzQ2MDEiIHk9IjI0LjQ0NCIgPlJlZ2lzdGVyPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjY0LjYwMjAwNSIgeT0iMzguNDEyMDAz
IiA+VHlwZWNoZWNrPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEzNC40
NDIiIHk9IjM4LjQxMjAwMyIgPmJvZGllczwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIxODMuMzMiIHk9IjM4LjQxMjAwMyIgPndpdGg8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iMjE4LjI1IiB5PSIzOC40MTIwMDMiID5jb21wbGV0ZTwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI4LjczMDAwMDUiIHk9IjM4LjQxMjAwMyIg
PlBhc3M8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDMuNjUiIHk9IjM4
LjQxMjAwMyIgPjI6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI4MS4x
MDYwMiIgeT0iMzguNDEyMDAzIiA+bXV0dWFsPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjMyOS45OTQwMiIgeT0iMzguNDEyMDAzIiA+dmlzaWJpbGl0eSE8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjM5LjIwMjAxIiB5PSIxMC40NzYiID5pc19l
dmVuIDogKEludCAtJmd0OyBCb29sKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIyMzkuMjAyMDEiIHk9IjI0LjQ0NCIgPmlzX29kZCAgOiAoSW50IC0mZ3Q7IEJvb2wpPC90
ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 32: Two-Pass Environment Setup</figcaption>
</figure>

**Specification:**

Refactor `TypeChecker::visit(const ProgramAST&)` to perform two-pass
environment population. Pre-register all declared function signatures in
Pass 1, and verify function bodies in Pass 2.

``` cpp
void TypeChecker::visit(const ProgramAST& node) {
    // Pass 1: Forward declaration hoisting
    for (const auto& stmt : node.statements()) {
        if (const auto* let_stmt = dynamic_cast<const LetStmtAST*>(stmt.get())) {
            if (let_stmt->type_annotation().has_value()) {
                Type declared_t = convert_ast_type(*let_stmt->type_annotation());
                m_current_env->define(let_stmt->name(), declared_t, let_stmt->location());
            }
        }
    }
    // Pass 2: Full semantic verification
    for (const auto& stmt : node.statements()) {
        stmt->accept(*this);
    }
}
```

**Verification Tasks:**

1.  **Direct Recursion Verification:** Check the recursive `factorial`
    implementation. Verify that the self-referential call
    `factorial(n - 1)` resolves cleanly without diagnostic errors.
2.  **Mutual Recursion Verification:** Check the mutually recursive
    `is_even` and `is_odd` functions. Verify that both forward and
    backward function calls resolve correctly.
3.  **Non-Function Recursion Guard:** Check `let x = x + 1;` where `x`
    has not been defined previously. Verify that the checker flags an
    undeclared identifier error rather than allowing a variable to bind
    to itself.

#### Exercise 20.5: Hindley-Milner Type Inference and Unification (Algorithm W)

**Context and Theory:**

Writing explicit type annotations on every variable and lambda parameter
can be tedious. The **Hindley-Milner (HM)** type system deduces the most
general (principal) type for every expression without requiring
developer annotations:

$$\text{Identity Function: } \text{let } id = \text{fn } (x) \Rightarrow x \implies id : \forall \alpha. (\alpha \to \alpha)$$

Type inference operates by:

1.  Assigning fresh **Type Variables** ($\alpha, \beta, \dots$) to
    unannotated parameters and expressions.
2.  Generating type equality equations from operations (e.g. $x + 1$
    implies $\alpha_x = \text{Int}$).
3.  Solving equations via **Robinson’s First-Order Unification
    Algorithm**:

$$\text{unify}(\tau_1, \tau_2) \implies \text{Substitution } \theta$$

A critical safety invariant is the **Occurs Check** (see fig. 33): a
type variable $\alpha$ cannot unify with a compound type containing
$\alpha$ (e.g. $\alpha \sim (\alpha \to \beta)$). Failing this check
indicates an infinite recursive type and must be rejected.

<figure id="fig:occurs-check">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0NjAuOTQ0IiBo
ZWlnaHQ9IjU5Ljg3MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA0NjAuOTQ0IDU5Ljg3
MiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI0NjAu
OTQ0IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjguNzMwMDAwNSIgeT0iMzguNDEyMDAzIiA+UmVzdWx0OjwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSI2NC42MDIwMDUiIHk9IjM4LjQxMjAwMyIgPkN5Y2xlPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEwNi41MDYwMDQiIHk9IjM4LjQxMjAw
MyIgPmRldGVjdGVkITwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNzYu
MzQ2MDEiIHk9IjM4LjQxMjAwMyIgPkFib3J0PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjIxOC4yNSIgeT0iMzguNDEyMDAzIiA+d2l0aDwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIyNTMuMTcwMDEiIHk9IjM4LjQxMjAwMyIgPkluZmluaXRlPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMxNi4wMjYiIHk9IjM4LjQxMjAw
MyIgPlR5cGU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzUwLjk0NiIg
eT0iMzguNDEyMDAzIiA+RXJyb3IhPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+RXF1YXRpb246IGFscGhhID0gKGFscGhhIC0mZ3Q7
IGJldGEpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjguNzMwMDAwNSIg
eT0iMjQuNDQ0IiA+T2NjdXJzIENoZWNrOiBEb2VzICYjMzk7YWxwaGEmIzM5OyBhcHBlYXIgaW5z
aWRlICYjMzk7KGFscGhhIC0mZ3Q7IGJldGEpJiMzOTs/IFlFUyE8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 33: Implementing the Occurs Check for Type
Inference</figcaption>
</figure>

**Specification:**

Implement a `TypeVariable` representation and a substitution environment
`std::unordered_map<uint64_t, Type>`. Construct a `TypeUnification`
class providing `unify(Type a, Type b)` with an occurs-check validator.

``` cpp
class TypeUnification {
public:
    static bool unify(Type& a, Type& b, std::unordered_map<uint64_t, Type>& subst) {
        Type t1 = resolve(a, subst);
        Type t2 = resolve(b, subst);
        if (t1 == t2) return true;
        if (t1.is_type_var()) return bind_var(t1.var_id(), t2, subst);
        if (t2.is_type_var()) return bind_var(t2.var_id(), t1, subst);
        if (t1.kind() == Type::Kind::Function && t2.kind() == Type::Kind::Function) {
            if (t1.param_types().size() != t2.param_types().size()) return false;
            for (size_t i = 0; i < t1.param_types().size(); ++i) {
                if (!unify(t1.param_types()[i], t2.param_types()[i], subst)) return false;
            }
            return unify(t1.return_type(), t2.return_type(), subst);
        }
        return false;
    }
private:
    static bool occurs_in(uint64_t var_id, const Type& target);
    static bool bind_var(uint64_t id, const Type& target, std::unordered_map<uint64_t, Type>& s);
    static Type resolve(const Type& t, const std::unordered_map<uint64_t, Type>& s);
};
```

**Verification Tasks:**

1.  **Simple Variable Deduction:** Unify a type variable $\alpha$ with
    `Int`. Verify that `resolve(alpha)` returns `Type::make_int()`.
2.  **Function Signature Synthesis:** Infer the type of
    `let double = fn (x) => x * 2;`. Verify that the parameter variable
    $\alpha$ unifies with `Int` and that the overall signature infers as
    `(Int -> Int)`.
3.  **Occurs Check Cycle Rejection:** Attempt to unify $\alpha$ with
    $(\alpha \to \text{Int})$. Verify that the occurs check triggers and
    rejects the self-referential cycle.

#### Exercise 20.6: Exhaustive Pattern Matching and Dead Arm Analysis

**Context and Theory:**

In a robust functional language, pattern matching over sum types,
booleans, and tuples must satisfy two semantic properties:

1.  **Exhaustiveness:** The match arms must cover every possible value
    in the domain. An incomplete match (e.g. matching `true` but
    omitting `false`) can cause a runtime crash.
2.  **Non-Redundancy (Dead Arm Detection):** No match arm should be
    unreachable. A match arm is dead if all values matching its pattern
    are already covered by preceding arms.

A wildcard `_` or bare identifier pattern matches any value
unconditionally. Therefore:

- If a match statement contains a wildcard arm, it is guaranteed to be
  **exhaustive**.
- Any arm positioned *after* an unconditional wildcard is completely
  **unreachable**.

```
 match flag with {
     true => 1,
     _    => 0,
     false => 2 // <=== DEAD ARM! Already shadowed by wildcard '_'!
 }
```

**Specification:**

Implement an `ExhaustivenessChecker` class that analyses `MatchExprAST`
nodes. Verify that boolean domains match both `true` and `false` (or
include a wildcard), and emit warning diagnostics for dead arms
positioned after wildcards.

``` cpp
class ExhaustivenessChecker {
public:
    static void verify_match(const MatchExprAST& match_node, Type target_type, 
                             DiagnosticReporter& reporter) {
        bool saw_wildcard = false;
        bool saw_true = false, saw_false = false;

        for (const auto& arm : match_node.arms()) {
            if (saw_wildcard) {
                reporter.report_warning(arm.pattern->location(), 
                                        "Unreachable match arm: preceding wildcard pattern already matches all values.");
            }
            if (arm.pattern->is_wildcard() || arm.pattern->is_variable()) {
                saw_wildcard = true;
            } else if (target_type == Type::make_bool()) {
                if (arm.pattern->matches_bool_literal(true)) saw_true = true;
                if (arm.pattern->matches_bool_literal(false)) saw_false = true;
            }
        }
        if (!saw_wildcard && target_type == Type::make_bool() && (!saw_true || !saw_false)) {
            reporter.report_error(match_node.location(), 
                                  "Non-exhaustive pattern match: missing case for '" + 
                                  std::string(!saw_true ? "true" : "false") + "'.");
        }
    }
};
```

**Verification Tasks:**

1.  **Non-Exhaustive Boolean Rejection:** Check
    `match is_ready with { true => 1 };`. Verify that the analyser
    rejects the expression, reporting that the `false` case is missing.
2.  **Wildcard Exhaustiveness:** Check
    `match is_ready with { true => 1, _ => 0 };`. Verify that the match
    passes with zero errors.
3.  **Dead Arm Warning:** Check
    `match x with { _ => "all", 42 => "forty-two" };`. Verify that the
    compiler reports a dead-arm warning pointing directly to `42`.

#### Exercise 20.7: Definite Assignment and Control-Flow Reachability

**Context and Theory:**

Reading uninitialised variables produces undefined behaviour in compiled
native code. Faberge enforces **Definite Assignment**: every variable
must be guaranteed to hold a valid value before any read access.

To enforce this, the semantic analyser tracks variable state across
branches:

- When a conditional `if cond then e1 else e2` executes, a variable $x$
  is definitely assigned after the conditional if and only if it is
  assigned in **both** the `then` branch and the `else` branch:

$$\text{Assigned}_{\text{after}}(x) = \text{Assigned}_{\text{then}}(x) \land \text{Assigned}_{\text{else}}(x)$$

If a variable is initialised in only one branch, it remains in an
uninitialised state following the conditional block (as shown in
fig. 34).

<figure id="fig:definite-assignment">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIyNDJwdCIgaGVpZ2h0PSIxNjBwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAyNDIuMzEgMTYwLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDE1
NikiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gbGV0IC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPmxldDwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIGN4PSIxMDYuNTYiIGN5PSItMTM0IiByeD0iNDYuNTEiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0
LWFuY2hvcj0ibWlkZGxlIiB4PSIxMDYuNTYiIHk9Ii0xMzEuNSIgZm9udC1mYW1pbHk9Iklvc2V2
a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5sZXQgbXV0IHg6IEludDs8
L3RleHQ+CjwvZz4KPCEtLSB0aGVuIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRp
dGxlPnRoZW48L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0i
NDMuNTYiIGN5PSItNzYiIHJ4PSI0My42MiIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjQzLjU2IiB5PSItNzMuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1v
bm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj50aGVuOiB4ID0gMTA7PC90ZXh0Pgo8L2c+Cjwh
LS0gbGV0JiM0NTsmZ3Q7dGhlbiAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRs
ZT5sZXQmIzQ1OyZndDt0aGVuPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFj
ayIgZD0iTTg4LjY4LC0xMTcuMUM4Mi41MSwtMTExLjYyIDc1LjQ3LC0xMDUuMzcgNjguODIsLTk5
LjQ2Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjcxLjA3
LC05Ni43NyA2MS4yNywtOTIuNzQgNjYuNDIsLTEwMiA3MS4wNywtOTYuNzciLz4KPC9nPgo8IS0t
IGVsc2UgLS0+CjxnIGlkPSJub2RlMyIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+ZWxzZTwvdGl0bGU+
CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxNjkuNTYiIGN5PSItNzYi
IHJ4PSI2NC41MSIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE2OS41
NiIgeT0iLTczLjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9u
dC1zaXplPSIxMC4wMCI+ZWxzZTogLy8geCB1bmFzc2lnbmVkPC90ZXh0Pgo8L2c+CjwhLS0gbGV0
JiM0NTsmZ3Q7ZWxzZSAtLT4KPGcgaWQ9ImVkZ2UyIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5sZXQm
IzQ1OyZndDtlbHNlPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0i
TTEyNC40NSwtMTE3LjFDMTMwLjI4LC0xMTEuOTIgMTM2Ljg4LC0xMDYuMDUgMTQzLjIsLTEwMC40
MyIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxNDUuODIs
LTEwMi43OCAxNTAuOTcsLTkzLjUyIDE0MS4xNywtOTcuNTUgMTQ1LjgyLC0xMDIuNzgiLz4KPC9n
Pgo8IS0tIHJlYWQgLS0+CjxnIGlkPSJub2RlNCIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+cmVhZDwv
dGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxMDYuNTYiIGN5
PSItMTgiIHJ4PSIyNyIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjEw
Ni41NiIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIg
Zm9udC1zaXplPSIxMC4wMCI+cmVhZCB4PC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQi
IHg9IjcuNTYiIHk9Ii0zNiIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVt
IiBmb250LXNpemU9IjEwLjAwIj5TZW1hbnRpYyBFcnJvciE8L3RleHQ+Cjx0ZXh0IHRleHQtYW5j
aG9yPSJzdGFydCIgeD0iNy41NiIgeT0iLTI1IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24g
TW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPiYjMzk7eCYjMzk7IGlzIG5vdDwvdGV4dD4K
PHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSI3LjU2IiB5PSItMTQiIGZvbnQtZmFtaWx5PSJJ
b3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+YXNzaWduZWQgb248
L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iNy41NiIgeT0iLTMiIGZvbnQtZmFt
aWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+YWxsIHBh
dGhzITwvdGV4dD4KPC9nPgo8IS0tIHRoZW4mIzQ1OyZndDtyZWFkIC0tPgo8ZyBpZD0iZWRnZTMi
IGNsYXNzPSJlZGdlIj4KPHRpdGxlPnRoZW4mIzQ1OyZndDtyZWFkPC90aXRsZT4KPHBhdGggZmls
bD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTYxLjExLC01OS40QzY3Ljk1LC01My4zMiA3NS45
LC00Ni4yNSA4My4yMiwtMzkuNzUiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxh
Y2siIHBvaW50cz0iODUuNjgsLTQyLjI1IDkwLjgzLC0zMi45OSA4MS4wMywtMzcuMDEgODUuNjgs
LTQyLjI1Ii8+CjwvZz4KPCEtLSBlbHNlJiM0NTsmZ3Q7cmVhZCAtLT4KPGcgaWQ9ImVkZ2U0IiBj
bGFzcz0iZWRnZSI+Cjx0aXRsZT5lbHNlJiM0NTsmZ3Q7cmVhZDwvdGl0bGU+CjxwYXRoIGZpbGw9
Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xNTEsLTU4LjVDMTQ0LjQzLC01Mi42NiAxMzYuOTQs
LTQ2IDEzMC4wMSwtMzkuODUiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2si
IHBvaW50cz0iMTMyLjAyLC0zNi45NSAxMjIuMjMsLTMyLjkyIDEyNy4zNywtNDIuMTggMTMyLjAy
LC0zNi45NSIvPgo8L2c+CjwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 34: Ensuring Definite Assignment (All
Paths)</figcaption>
</figure>

**Specification:**

Construct a `DefiniteAssignmentValidator` pass. Track variable
initialisation flags across control-flow splits (`if-then-else` and
block expressions). Raise semantic errors whenever an identifier is read
while in an unassigned state.

``` cpp
class DefiniteAssignmentValidator : public ASTVisitor {
public:
    void check_function_body(const ExprAST& body);
private:
    struct VarState { bool is_assigned{false}; };
    std::unordered_map<std::string, VarState> m_current_state;

    void visit(const IfExprAST& node) override {
        node.condition().accept(*this);
        auto state_before = m_current_state;
        node.then_branch().accept(*this);
        auto state_after_then = m_current_state;
        m_current_state = state_before;
        node.else_branch().accept(*this);
        auto state_after_else = m_current_state;
        // Intersection: must be assigned in BOTH branches
        for (auto& [var_name, var_state] : m_current_state) {
            var_state.is_assigned = state_after_then[var_name].is_assigned && 
                                    state_after_else[var_name].is_assigned;
        }
    }
};
```

**Verification Tasks:**

1.  **Unassigned Read Rejection:** Check
    `let mut a: Int; let b = a + 1;`. Verify that the checker flags `a`
    as an uninitialised read.
2.  **Branch Intersection Invariance:** Declare uninitialised
    `let mut x: Int;`. Assign `x = 10;` in the `then` branch, but leave
    `else` empty. Attempt to read `x` after the `if`. Verify that the
    read is rejected.
3.  **Symmetric Assignment Conformance:** Assign `x = 10;` in `then`,
    and `x = 20;` in `else`. Verify that reading `x` after the
    conditional succeeds with zero errors.

#### Exercise 20.8: Dead Code and Unreachable Expression Pruning

**Context and Theory:**

Dead code represents statements or expressions that can never be
executed under any control-flow path. Detecting dead code prevents
developer oversights and enables the compiler to prune dead branches
before target C++ emission.

Dead code arises primarily through:

1.  **Post-Return Statements:** Statements appearing in a block
    immediately after an unconditional `return` expression.
2.  **Statically Known Predicates:** Conditional expressions whose
    condition is a compile-time constant literal:
    - `if true then e1 else e2` $\implies e_2$ is dead code.
    - `if false then e1 else e2` $\implies e_1$ is dead code.

```
 {
     return 42;
     let dead_x = 100; // <=== UNREACHABLE CODE!
 }
```

**Specification:**

Create an `ASTDeadCodePruner` pass inheriting from `ASTVisitor`.
Identify statements appearing after returns and branches governed by
boolean literals. Emit diagnostic warnings and prune dead subtrees from
the AST.

``` cpp
class ASTDeadCodePruner : public ASTVisitor {
public:
    bool prune(std::unique_ptr<ExprAST>& expr_node) {
        if (auto* if_node = dynamic_cast<IfExprAST*>(expr_node.get())) {
            if (auto* lit = dynamic_cast<const LiteralExprAST*>(&if_node->condition())) {
                if (lit->token_type() == TokenType::Literal_Boolean) {
                    bool cond = (lit->value() == "true");
                    // Prune and promote active branch directly!
                    expr_node = cond ? std::move(if_node->extract_then()) 
                                     : std::move(if_node->extract_else());
                    return true;
                }
            }
        }
        return false;
    }
};
```

**Verification Tasks:**

1.  **Post-Return Warning:** Check `{ return 1; let x = 2; x }`. Verify
    that the analyser flags `let x = 2;` as unreachable dead code.
2.  **Constant Branch Pruning:** Run the pruner on
    `if true then 10 else 20`. Verify that the `IfExprAST` node is
    completely replaced by `LiteralExprAST(10)`.
3.  **Nested Dead Block Identification:** Test an outer `if false`
    guarding a complex block containing loops and declarations. Verify
    that all nested statements are identified as dead.

#### Exercise 20.9: Implicit Numeric Widening and Type Coercion

**Context and Theory:**

Strict static typing rejects mixed-mode arithmetic: adding an `Int` to a
`Float` fails because their bit-level representations differ (two’s
complement integer vs IEEE-754 floating-point exponent/significand).

However, many languages permit **Safe Non-Lossy Widening** (see
fig. 35):

$$\text{Int} \subset \text{Float}$$

When evaluating binary arithmetic between an `Int` and a `Float`, the
semantic analyser:

1.  Recognises the mixed operands as compatible under widening coercion.
2.  Injects a synthetic `CastExprAST(operand, Float)` node into the AST
    around the integer operand.
3.  Synthesises the resulting expression type as `Float`.

Narrowing coercions (`Float` to `Int`) lose precision and are strictly
rejected.

<figure id="fig:safe-widening">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1OTMuNjQiIGhl
aWdodD0iMTE1Ljc0NCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1OTMuNjQgMTE1Ljc0
NCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI1OTMu
NjQiIGhlaWdodD0iMTE1Ljc0NCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+T3JpZ2luYWw8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iNzEuNTg2IiB5PSIxMC40NzYiID5BU1Q6PC90ZXh0PgogIDxsaW5l
IHgxPSIzMTAuNzg4IiB5MT0iMjQuNDQ0IiB4Mj0iMzEwLjc4OCIgeTI9IjQ1LjM5NiIgY2xhc3M9
InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOC43MzAwMDA1
IiB5PSI2Ni4zNDgiID5Db2VyY2lvbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSI3MS41ODYiIHk9IjY2LjM0OCIgPlBhc3M6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjguNzMwMDAwNSIgeT0iOTQuMjg0MDA0IiA+UmVzdWx0aW5nPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijc4LjU3IiB5PSI5NC4yODQwMDQiID5UeXBl
OjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNDEuNDI2MDEiIHk9Ijk0
LjI4NDAwNCIgPkZsb2F0PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE0
MS40MjYwMSIgeT0iMTAuNDc2IiA+QmluYXJ5RXhwcigrKSBbIExlZnQ6IEludCgxMCksIFJpZ2h0
OiBGbG9hdCgzLjE0KSBdPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE0
MS40MjYwMSIgeT0iNjYuMzQ4IiA+QmluYXJ5RXhwcigrKSBbIExlZnQ6IENhc3QmbHQ7RmxvYXQm
Z3Q7KEludCgxMCkpLCBSaWdodDogRmxvYXQoMy4xNCkgXTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 35: Safe Non-Lossy Widening</figcaption>
</figure>

**Specification:**

Implement a `can_coerce(Type from, Type to)` predicate. Extend
`TypeChecker::visit(const BinaryExprAST&)`: if an arithmetic operand
pair is `(Int, Float)` or `(Float, Int)`, rewrite the AST node to insert
a `CastExprAST` node wrapping the integer operand and assign the result
type `Float`.

``` cpp
class NumericCoercionEngine {
public:
    static bool try_coerce_binary(std::unique_ptr<ExprAST>& left, Type& left_t,
                                  std::unique_ptr<ExprAST>& right, Type& right_t) {
        if (left_t == Type::make_int() && right_t == Type::make_float()) {
            left = std::make_unique<CastExprAST>(left->location(), std::move(left), Type::make_float());
            left_t = Type::make_float();
            return true;
        }
        if (left_t == Type::make_float() && right_t == Type::make_int()) {
            right = std::make_unique<CastExprAST>(right->location(), std::move(right), Type::make_float());
            right_t = Type::make_float();
            return true;
        }
        return false;
    }
};
```

**Verification Tasks:**

1.  **Left-Operand Widening:** Check `let res = 10 + 3.14;`. Verify that
    the left operand is wrapped with a cast to `Float` and that `res`
    infers type `Float`.
2.  **Right-Operand Widening:** Check `let res = 2.5 * 4;`. Verify that
    the right operand is coerced and that the expression evaluates to
    `Float`.
3.  **Narrowing Rejection:** Attempt to assign a float to an explicitly
    annotated integer: `let x: Int = 3.14;`. Verify that the coercion
    engine refuses to implicitly narrow `Float` to `Int`.

#### Exercise 20.10: Unused Symbol and Variable Shadowing Warnings

**Context and Theory:**

High-quality compilers act as static analysis linters, warning
developers about suspicious code patterns that are syntactically legal
but indicative of logic bugs:

1.  **Unused Identifiers:** A variable is declared and initialised, but
    its identifier is never read within its scope.
2.  **Variable Shadowing:** An identifier in an inner scope shares the
    exact name of an active identifier in an outer enclosing scope. This
    can lead to subtle bugs where an author mistakenly assumes they are
    modifying the outer variable.

To support this without false positives, modern compilers allow a
leading underscore prefix (e.g. `_temp` or `_unused`) to signal
intentional non-use.

```
 let x = 10;
 {
     let x = 20; // WARNING: Declaration of 'x' shadows outer binding at line 1!
 }
 // WARNING: Variable 'x' declared at line 1 is never read!
```

**Specification:**

Add a `size_t read_count` to the `Symbol` structure in `SymbolTable`.
Increment the counter whenever `lookup()` is called during expression
checking. When exiting a scope (`pop_scope()`), inspect all bindings and
report unused variables (skipping identifiers prefixed with `_`).

``` cpp
class LinterEnvironment : public Environment {
public:
    void check_unused_symbols(DiagnosticReporter& reporter) {
        for (const auto& [name, sym] : m_bindings) {
            if (sym.read_count == 0 && !name.starts_with('_')) {
                reporter.report_warning(sym.location, 
                                        "Unused variable '" + name + "'. Prefix with '_' if intentional.");
            }
        }
    }
};
```

**Verification Tasks:**

1.  **Unused Variable Detection:** Check
    `let total = 100; let rate = 0.05; return rate;`. Verify that the
    compiler reports a warning for `total` and zero warnings for `rate`.
2.  **Underscore Suppression:** Check `let _placeholder = 42;`. Verify
    that the leading underscore suppresses the unused variable warning.
3.  **Shadowing Detection:** Check
    `let x = 10; { let x = 20; return x; }`. Verify that the inner
    declaration triggers a variable-shadowing warning pointing back to
    the line coordinate of the outer `x`.

#### Exercise 20.11: Function Overload Resolution and Candidate Ranking

**Context and Theory:**

Many programming languages support **Function Overloading**: multiple
functions sharing the same name but possessing different parameter
signatures:

```
let print = fn (x: Int): Void => ... ;
let print = fn (s: String): Void => ... ;
```

When a call expression `print(arg)` is analysed, the type checker must
resolve the call (see fig. 36):

1.  **Candidate Gathering:** Retrieve all functions matching the callee
    identifier in scope $\Gamma$.
2.  **Arity Filtering:** Discard candidates whose parameter count does
    not match argument count.
3.  **Type Compatibility Matching:** Discard candidates whose parameter
    types cannot accept the argument types (accounting for exact matches
    vs widened coercions).
4.  **Best-Match Ranking:** Rank surviving candidates. An exact type
    match scores higher than a coerced match. If two candidates tie with
    identical scores, raise an **Ambiguous Call Error**.

<figure id="fig:function-overloading">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0NTMuOTYwMDIi
IGhlaWdodD0iMTI5LjcxMiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA0NTMuOTYwMDIg
MTI5LjcxMiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNp
cmNsZSwgLnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAg
c3Ryb2tlLXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAx
OwogIHN0cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgou
c3ZnYm9iIHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1m
YW1pbHk6IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXpl
OiAxNHB4Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxs
OiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4
Owp9Cgouc3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxs
ZWQgewogIGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAu
bm9maWxsIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJv
dyB7CiAgbWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9h
cnJvdyB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtl
ZF9kaWFtb25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3Zn
Ym9iIC5lbmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgou
c3ZnYm9iIC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJs
KCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAg
bWFya2VyLXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9i
aWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1
cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlk
PSJhcnJvdyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxwb2x5Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtl
cj4KICAgIDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIg
cmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0
YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIi
PjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94
PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0
PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5
PSI0IiByPSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1h
cmtlciBpZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQi
IG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQi
PjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUi
IHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJr
ZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBj
eD0iNCIgY3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJr
ZXI+CiAgPC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRo
PSI0NTMuOTYwMDIiIGhlaWdodD0iMTI5LjcxMiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+Q2FuZGlkYXRlczogWzFdIHByaW50
KEludCksIFsyXSBwcmludChGbG9hdCksIFszXSBwcmludChTdHJpbmcpPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjguNzMwMDAwNSIgeT0iMzguNDEyMDAzIiA+Q2FsbDog
ICAgICAgcHJpbnQoNDIpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijgu
NzMwMDAwNSIgeT0iNjYuMzQ4IiA+UmFua2luZzogICAgQ2FuZGlkYXRlIFsxXSAoRXhhY3QgTWF0
Y2g6IFNjb3JlIDApPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkyLjUz
OCIgeT0iODAuMzE2IiA+Q2FuZGlkYXRlIFsyXSAoQ29lcmNpb24gTWF0Y2g6IFNjb3JlIDEpPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjguNzMwMDAwNSIgeT0iMTA4LjI1
MjAxIiA+V2lubmVyOiAgICAgQ2FuZGlkYXRlIFsxXSBzZWxlY3RlZCE8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 36: Candidate Selection in Function
Overloading</figcaption>
</figure>

**Specification:**

Construct an `OverloadResolver` class. Store overloaded symbols in the
environment as `std::vector<Symbol>`. Implement candidate scoring and
best-match selection during `CallExprAST` verification.

``` cpp
class OverloadResolver {
public:
    static Symbol resolve_call(const std::string& name, const std::vector<Type>& arg_types,
                               const std::vector<Symbol>& candidates, SourceLocation loc) {
        std::vector<std::pair<int, Symbol>> viable;
        for (const auto& cand : candidates) {
            int score = score_candidate(cand.type.param_types(), arg_types);
            if (score >= 0) viable.emplace_back(score, cand);
        }
        if (viable.empty()) throw std::runtime_error("No matching overload for '" + name + "'.");
        std::sort(viable.begin(), viable.end(), [](auto& a, auto& b) { return a.first < b.first; });
        if (viable.size() > 1 && viable[0].first == viable[1].first) {
            throw std::runtime_error("Ambiguous call to overloaded function '" + name + "'.");
        }
        return viable[0].second;
    }
private:
    static int score_candidate(const std::vector<Type>& params, const std::vector<Type>& args);
};
```

**Verification Tasks:**

1.  **Exact Overload Selection:** Register `f(Int)` and `f(String)`.
    Check `f(10)`. Verify that the resolver binds the call directly to
    the `f(Int)` overload.
2.  **Widened Overload Selection:** Register `g(Float)` and `g(String)`.
    Check `g(10)`. Verify that `10` is coerced to `Float` and binds to
    `g(Float)`.
3.  **Ambiguity Rejection:** Register `h(Int, Float)` and
    `h(Float, Int)`. Check `h(10, 20)`. Verify that the resolver detects
    an ambiguous tie and raises a compile-time diagnostic.

#### Exercise 20.12: Static Escape and Closure Capture Lifetime Analysis

**Context and Theory:**

In a functional language, a lambda can reference variables declared in
its enclosing scope (**Free Variables**):

```
let make_multiplier = fn (factor: Int): (Int -> Int) => {
    fn (x: Int): Int => x * factor // 'factor' is captured from outer scope!
};
```

Because our compiler targets **C++**, the lifetime of captured variables
is critical:

- If a lambda is evaluated only locally, it can capture variables by
  reference (`[&]`).
- If a lambda **escapes** its defining function (e.g. by being returned
  or assigned to a global), capturing local stack variables by reference
  creates a dangling pointer when the outer frame is destroyed.

The semantic analyser must compute the **Capture Set** for every lambda:

$$\text{Captured}(\lambda) = \text{FreeVars}(\lambda) \cap \text{EnclosingLocals}$$

Lambdas that escape their enclosing scope (as shown in fig. 37) must be
flagged so the downstream C++ emitter captures variables by value
(`[=]`) or inside heap-allocated `std::shared_ptr` wrapper.

<figure id="fig:return-inner">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1ODYuNjU2IiBo
ZWlnaHQ9IjczLjg0MDAwNCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1ODYuNjU2IDcz
Ljg0MDAwNCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNp
cmNsZSwgLnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAg
c3Ryb2tlLXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAx
OwogIHN0cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgou
c3ZnYm9iIHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1m
YW1pbHk6IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXpl
OiAxNHB4Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxs
OiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4
Owp9Cgouc3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxs
ZWQgewogIGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAu
bm9maWxsIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJv
dyB7CiAgbWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9h
cnJvdyB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtl
ZF9kaWFtb25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3Zn
Ym9iIC5lbmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgou
c3ZnYm9iIC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJs
KCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAg
bWFya2VyLXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9i
aWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1
cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlk
PSJhcnJvdyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxwb2x5Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtl
cj4KICAgIDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIg
cmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0
YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIi
PjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94
PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0
PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5
PSI0IiByPSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1h
cmtlciBpZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQi
IG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQi
PjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUi
IHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJr
ZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBj
eD0iNCIgY3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJr
ZXI+CiAgPC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRo
PSI1ODYuNjU2IiBoZWlnaHQ9IjczLjg0MDAwNCI+PC9yZWN0PgogIDxwb2x5Z29uIHBvaW50cz0i
NDguODg4LDMxLjQyODAwMSA1NS44NzIsMzQuOTIwMDAyIDQ4Ljg4OCwzOC40MTIwMDMiIGNsYXNz
PSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2NC42
MDIwMDUiIHk9IjM4LjQxMjAwMyIgPklubmVyPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjEwNi41MDYwMDQiIHk9IjM4LjQxMjAwMyIgPkxhbWJkYTo8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTYyLjM3OCIgeT0iMzguNDEyMDAzIiA+UmVmZXJl
bmNlczwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMzkuMjAyMDEiIHk9
IjM4LjQxMjAwMyIgPiYjMzk7ZmFjdG9yJiMzOTs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iNjQuNjAyMDA1IiB5PSI1Mi4zOCIgPklzPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9Ijg1LjU1NCIgeT0iNTIuMzgiID5Jbm5lcjwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMjcuNDU4IiB5PSI1Mi4zOCIgPkxhbWJkYTwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNzYuMzQ2MDEiIHk9IjUyLjM4IiA+
cmV0dXJuZWQ/PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI0Ni4xODYi
IHk9IjUyLjM4IiA+WUVTPC90ZXh0PgogIDxsaW5lIHgxPSIyNzIuMzc2IiB5MT0iNDguODg4IiB4
Mj0iMjg2LjM0NCIgeTI9IjQ4Ljg4OCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24g
cG9pbnRzPSIyODYuMzQ0LDQ1LjM5NiAyOTMuMzI4LDQ4Ljg4OCAyODYuMzQ0LDUyLjM4IiBjbGFz
cz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzAy
LjA1OCIgeT0iNTIuMzgiID4mIzM5O2ZhY3RvciYjMzk7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjM2NC45MTQiIHk9IjUyLjM4IiA+RVNDQVBFUyE8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDI3Ljc3MDAyIiB5PSI1Mi4zOCIgPk11c3Q8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDYyLjY5IiB5PSI1Mi4zOCIgPmNh
cHR1cmU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTE4LjU2MiIgeT0i
NTIuMzgiID5ieTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1MzkuNTE0
MDQiIHk9IjUyLjM4IiA+dmFsdWUhPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+U2NvcGU6IG1ha2VfbXVsdGlwbGllciAoTG9jYWw6
IGZhY3Rvcik8L3RleHQ+CiAgPGc+CiAgICA8bGluZSB4MT0iMjQuNDQ0IiB5MT0iMTMuOTY4IiB4
Mj0iMjQuNDQ0IiB5Mj0iNTUuODcyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxsaW5lIHgx
PSIyNC40NDQiIHkxPSIzNC45MjAwMDIiIHgyPSI0OC44ODgiIHkyPSIzNC45MjAwMDIiIGNsYXNz
PSJzb2xpZCI+PC9saW5lPgogIDwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 37: Inner Lambda Escape</figcaption>
</figure>

**Specification:**

Create an `EscapeAnalysisVisitor` class inheriting from `ASTVisitor`.
For every `LambdaExprAST`, compute its free variables, identify whether
the lambda escapes its declaring stack frame, and record its capture
list for the C++ code generator.

``` cpp
class EscapeAnalysisVisitor : public ASTVisitor {
public:
    struct LambdaMetadata {
        bool escapes_local_frame{false};
        std::unordered_set<std::string> captured_variables;
    };
    void analyse(const ProgramAST& root);
    const LambdaMetadata& get_metadata(const LambdaExprAST* lambda) const;
private:
    std::vector<std::unordered_set<std::string>> m_scope_stack;
    std::unordered_map<const LambdaExprAST*, LambdaMetadata> m_lambda_table;
    void visit(const VariableExprAST& node) override;
    void visit(const LambdaExprAST& node) override;
};
```

**Verification Tasks:**

1.  **Local Free Variable Capture:** Check a lambda accessing an outer
    variable `x`. Verify that `x` is correctly added to the lambda’s
    capture set.
2.  **Escaping Lambda Identification:** Check a function returning an
    inner closure referencing a parameter. Verify that the inner lambda
    is marked with `escapes_local_frame = true`.
3.  **Pure Function Invariance:** Check a lambda referencing only its
    own parameters and global constants. Verify that its capture set is
    empty (`size() == 0`).

# Chapter 21: Emit

> *“Much of my work has come from being lazy. I didn’t like writing raw
> machine code, and so I started work on a programming system that could
> write the instructions for me.”*  
> — John Backus, inventor of FORTRAN and BNF, Turing Award Lecture
> (1977)

------------------------------------------------------------------------

### 21.1 The Final Frontier: Target Code Generation

We have arrived at the fourth and final stage of our compiler project.
Having spent a lot of effort engineering the earlier parts of the
compiler, writing the code for actually emitting (outputting) the target
C++ source code may seem like an anticlimax! It is a truism that the
ultimate goal was always code transformation, and not code analysis, and
here we finally realize the fruits of our labours.

Over the preceding three chapters, we engineered the compiler’s
front-end and middle-end:

1.  **Chapter 18 (Lexer):** Scanned raw source text into discrete
    `Token` streams.
2.  **Chapter 19 (Parser):** Transformed linear token sequences into
    structured Abstract Syntax Tree (AST) hierarchies using Pratt
    operator-precedence parsing.
3.  **Chapter 20 (Check):** Evaluated lexical scopes via cactus-stack
    symbol tables and statically proved semantic correctness using
    bidirectional type judgements.

<figure id="fig:steps-to-generation">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzNjhwdCIgaGVpZ2h0PSIzMzNwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzNjguMDAgMzMzLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDMy
OSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gc291cmNlIC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNz
PSJub2RlIj4KPHRpdGxlPnNvdXJjZTwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIHBvaW50cz0iMzYwLC0zMjUgMCwtMzI1IDAsLTI4MiAzNjAsLTI4MiAzNjAsLTMy
NSIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii0zMTIuNSIgZm9udC1mYW1p
bHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5GYWJlcmdl
IFNvdXJjZSBDb2RlICgmcXVvdDtwcm9ncmFtLmZhYiZxdW90Oyk6PC90ZXh0Pgo8dGV4dCB0ZXh0
LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii0yODkuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hh
cm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5sZXQgZG91YmxlID0gZm4gKHg6IElu
dCk6IEludCA9Jmd0OyB4ICogMjs8L3RleHQ+CjwvZz4KPCEtLSB0b2tlbnMgLS0+CjxnIGlkPSJu
b2RlMiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+dG9rZW5zPC90aXRsZT4KPHBvbHlnb24gZmlsbD0i
bm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzNjAsLTI1NSAwLC0yNTUgMCwtMjMzIDM2MCwt
MjMzIDM2MCwtMjU1Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTI0MS41
IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAu
MDAiPlRva2VuIFN0cmVhbSAoc3RkOjp2ZWN0b3ImbHQ7VG9rZW4mZ3Q7KTwvdGV4dD4KPC9nPgo8
IS0tIHNvdXJjZSYjNDU7Jmd0O3Rva2VucyAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+
Cjx0aXRsZT5zb3VyY2UmIzQ1OyZndDt0b2tlbnM8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBz
dHJva2U9ImJsYWNrIiBkPSJNMTgwLC0yODEuNzRDMTgwLC0yNzYuMzkgMTgwLC0yNzAuNjQgMTgw
LC0yNjUuMzIiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0i
MTgzLjUsLTI2NS4yNSAxODAsLTI1NS4yNSAxNzYuNSwtMjY1LjI1IDE4My41LC0yNjUuMjUiLz4K
PHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSIxODAiIHk9Ii0yNjYiIGZvbnQtZmFtaWx5PSJJ
b3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+ICYjMTYwO1N0YWdl
IDE6IExleGljYWwgQW5hbHlzaXM8L3RleHQ+CjwvZz4KPCEtLSBhc3QgLS0+CjxnIGlkPSJub2Rl
MyIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+YXN0PC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIg
c3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzNjAsLTIwNiAwLC0yMDYgMCwtMTg0IDM2MCwtMTg0IDM2
MCwtMjA2Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTE5Mi41IiBmb250
LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPkFi
c3RyYWN0IFN5bnRheCBUcmVlIChBU1ROb2RlIEhpZXJhcmNoeSk8L3RleHQ+CjwvZz4KPCEtLSB0
b2tlbnMmIzQ1OyZndDthc3QgLS0+CjxnIGlkPSJlZGdlMiIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+
dG9rZW5zJiM0NTsmZ3Q7YXN0PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFj
ayIgZD0iTTE4MCwtMjMyLjU3QzE4MCwtMjI3Ljc2IDE4MCwtMjIxLjg3IDE4MCwtMjE2LjI0Ii8+
Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE4My41LC0yMTYu
MDggMTgwLC0yMDYuMDggMTc2LjUsLTIxNi4wOCAxODMuNSwtMjE2LjA4Ii8+Cjx0ZXh0IHRleHQt
YW5jaG9yPSJzdGFydCIgeD0iMTgwIiB5PSItMjE3IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFy
b24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPiAmIzE2MDtTdGFnZSAyOiBTeW50YWN0
aWMgQW5hbHlzaXM8L3RleHQ+CjwvZz4KPCEtLSB2ZXJpZmllZCAtLT4KPGcgaWQ9Im5vZGU0IiBj
bGFzcz0ibm9kZSI+Cjx0aXRsZT52ZXJpZmllZDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUi
IHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMzYwLC0xNTcgMCwtMTU3IDAsLTEzNSAzNjAsLTEzNSAz
NjAsLTE1NyIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii0xNDMuNSIgZm9u
dC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5E
ZWNvcmF0ZWQgYW5kIFZlcmlmaWVkIFR5cGVkIEFTVDwvdGV4dD4KPC9nPgo8IS0tIGFzdCYjNDU7
Jmd0O3ZlcmlmaWVkIC0tPgo8ZyBpZD0iZWRnZTMiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmFzdCYj
NDU7Jmd0O3ZlcmlmaWVkPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
ZD0iTTE4MCwtMTgzLjU3QzE4MCwtMTc4Ljc2IDE4MCwtMTcyLjg3IDE4MCwtMTY3LjI0Ii8+Cjxw
b2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE4My41LC0xNjcuMDgg
MTgwLC0xNTcuMDggMTc2LjUsLTE2Ny4wOCAxODMuNSwtMTY3LjA4Ii8+Cjx0ZXh0IHRleHQtYW5j
aG9yPSJzdGFydCIgeD0iMTgwIiB5PSItMTY4IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24g
TW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPiAmIzE2MDtTdGFnZSAzOiBTZW1hbnRpYyBB
bmFseXNpczwvdGV4dD4KPC9nPgo8IS0tIGVtaXQgLS0+CjxnIGlkPSJub2RlNSIgY2xhc3M9Im5v
ZGUiPgo8dGl0bGU+ZW1pdDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIHBvaW50cz0iMzYwLC05NyAwLC05NyAwLC0zOSAzNjAsLTM5IDM2MCwtOTciLz4KPHRleHQg
dGV4dC1hbmNob3I9InN0YXJ0IiB4PSI4IiB5PSItODIiIGZvbnQtZmFtaWx5PSJJb3NldmthIENo
YXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+VGhlIENvZGUgRW1pdHRlciAoQVNU
IFZpc2l0b3IpPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjgiIHk9Ii03MSIg
Zm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAw
Ij4mIzQ1OyBMb3dlcnMgZnVuY3Rpb25hbCBBU1Qgbm9kZXMgdG8gdGFyZ2V0IGxhbmd1YWdlIGNv
bnN0cnVjdHM8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTYwIiBm
b250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAi
PiYjNDU7IEVtaXRzIGlkaW9tYXRpYywgc3RyaWN0bHkgdHlwZWQgQysrMjMgc291cmNlIGNvZGU8
L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iOCIgeT0iLTQ5IiBmb250LWZhbWls
eT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPiYjNDU7IFBy
ZXNlcnZlcyBsZXhpY2FsIHNjb3BlLCBleHByZXNzaW9uIHNlbWFudGljcywgYW5kIGZvcm1hdHRp
bmc8L3RleHQ+CjwvZz4KPCEtLSB2ZXJpZmllZCYjNDU7Jmd0O2VtaXQgLS0+CjxnIGlkPSJlZGdl
NCIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+dmVyaWZpZWQmIzQ1OyZndDtlbWl0PC90aXRsZT4KPHBh
dGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE4MCwtMTM0Ljc1QzE4MCwtMTI3LjUy
IDE4MCwtMTE3LjM0IDE4MCwtMTA3LjE0Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9
ImJsYWNrIiBwb2ludHM9IjE4My41LC0xMDcuMDUgMTgwLC05Ny4wNSAxNzYuNSwtMTA3LjA1IDE4
My41LC0xMDcuMDUiLz4KPHRleHQgdGV4dC1hbmNob3I9InN0YXJ0IiB4PSIxODAiIHk9Ii0xMTki
IGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4w
MCI+ICYjMTYwO1N0YWdlIDQ6IENvZGUgR2VuZXJhdGlvbjwvdGV4dD4KPHRleHQgdGV4dC1hbmNo
b3I9InN0YXJ0IiB4PSIxODAiIHk9Ii0xMDgiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBN
b25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+ICYjMTYwOyhUaGlzIENoYXB0ZXIpPC90ZXh0
Pgo8L2c+CjwhLS0gY29kZSAtLT4KPGcgaWQ9Im5vZGU2IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5j
b2RlPC90aXRsZT4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSItOC41IiBm
b250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAi
PlRhcmdldCBDKysgQXJ0aWZhY3QgKCZxdW90O3Byb2dyYW0uY3BwJnF1b3Q7KTwvdGV4dD4KPC9n
Pgo8IS0tIGVtaXQmIzQ1OyZndDtjb2RlIC0tPgo8ZyBpZD0iZWRnZTUiIGNsYXNzPSJlZGdlIj4K
PHRpdGxlPmVtaXQmIzQ1OyZndDtjb2RlPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tl
PSJibGFjayIgZD0iTTE4MCwtMzguODZDMTgwLC0zNi43MyAxODAsLTM0LjYxIDE4MCwtMzIuNTQi
Lz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTgzLjUsLTMy
LjMxIDE4MCwtMjIuMzEgMTc2LjUsLTMyLjMxIDE4My41LC0zMi4zMSIvPgo8L2c+CjwvZz4KPC9z
dmc+Cg==
" alt="" />
<figcaption>Figure 38: Stages toward C++ Code Emission</figcaption>
</figure>

The final task of a compiler is **Target Code Generation (Emission)**.
The code generator traverses the verified, decorated AST and synthesises
an equivalent program in a target language.

#### 21.1.1 Transpilation and High-Level Target Emission

Compilers target a wide variety of execution substrates:

- **Machine Code / Assembly:** Direct CPU instructions (x86_64, ARM64,
  RISC-V) via back-ends like LLVM.
- **Virtual Machine Bytecode:** Stack-based or register-based bytecode
  (Java Virtual Machine bytecode, WebAssembly, Common Intermediate
  Language).
- **High-Level Target Code (Source-to-Source / Transpilation):**
  Translating one high-level programming language into another (e.g.,
  TypeScript to JavaScript via `tsc`, C++ to JavaScript/Wasm via
  Emscripten, Elm to JavaScript, Nim to C).

Transpiling our functional language **Faberge** into **C++23** provides
an ideal balance of pedagogical clarity and operational power. Modern
C++ possesses first-class lambda expressions, strict static typing, and
runs almost everywhere.

### 21.2 Semantic Mapping: Functional Faberge to C++

To generate valid C++, we must define an exact structural and semantic
mapping for every node in the **Faberge** Abstract Syntax Tree.

<figure>
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIyMCAyMCA4ODAgMzMwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIzMzAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0iYm94U2hhZG93IiB4PSItNSUiIHk9
Ii01JSIgd2lkdGg9IjExMCUiIGhlaWdodD0iMTIwJSI+CiAgICAgIDxmZURyb3BTaGFkb3cgZHg9
IjAiIGR5PSIzIiBzdGREZXZpYXRpb249IjMiIGZsb29kLWNvbG9yPSJ2YXIoLS1zdmctZmcpIiBm
bG9vZC1vcGFjaXR5PSIwLjUiLz4KICAgIDwvZmlsdGVyPgogIDwvZGVmcz4KICA8IS0tIExlZnQ6
IEZhYmVyZ2UgU291cmNlIENvbnN0cnVjdHMgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUo
NDAsIDMwKSI+CiAgICA8cmVjdCB4PSIwIiB5PSIwIiB3aWR0aD0iMzgwIiBoZWlnaHQ9IjMwMCIg
cng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2sp
KSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGZpbHRlcj0idXJsKCNi
b3hTaGFkb3cpIi8+CiAgICA8dGV4dCB4PSIxOTAiIHk9IjIzIiBmaWxsPSJ2YXIoLS1zdmctZmcp
IiBmb250LXNpemU9IjE2IiBmb250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+
RmFiZXJnZSBGdW5jdGlvbmFsIEFTVCBOb2RlPC90ZXh0PgogICAgPCEtLSBOb2RlIDE6IExldCBC
aW5kaW5nIC0tPgogICAgPHRleHQgeD0iMjAiIHk9IjY1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBm
b250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCI+TGV0U3RtdEFTVDwvdGV4dD4KICAgIDx0
ZXh0IHg9IjIwIiB5PSI4MiIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCI+bGV0
IGxpbWl0OiBJbnQgPSAxMDA7PC90ZXh0PgogICAgPCEtLSBOb2RlIDI6IExhbWJkYSAvIEZ1bmN0
aW9uIC0tPgogICAgPHRleHQgeD0iMjAiIHk9IjExNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9u
dC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiPkxhbWJkYUV4cHJBU1Q8L3RleHQ+CiAgICA8
dGV4dCB4PSIyMCIgeT0iMTMyIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5m
biAoeDogSW50LCB5OiBJbnQpOiBJbnQgPSZndDsgeCArIHk8L3RleHQ+CiAgICA8IS0tIE5vZGUg
MzogSWYgRXhwcmVzc2lvbiAtLT4KICAgIDx0ZXh0IHg9IjIwIiB5PSIxNjUiIGZpbGw9InZhcigt
LXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIj5JZkV4cHJBU1QgKEV4
cHJlc3Npb24gRm9ybSk8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iMTgyIiBmaWxsPSJ2YXIo
LS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5pZiBuICZsdDs9IDEgdGhlbiAxIGVsc2UgbiAqIGZh
YyhuIC0gMSk8L3RleHQ+CiAgICA8IS0tIE5vZGUgNDogQmxvY2sgRXhwcmVzc2lvbiAtLT4KICAg
IDx0ZXh0IHg9IjIwIiB5PSIyMTUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQi
IGZvbnQtd2VpZ2h0PSJib2xkIj5CbG9ja0V4cHJBU1QgKFNjb3BpbmcgJmFtcDsgVmFsdWUpPC90
ZXh0PgogICAgPHRleHQgeD0iMjAiIHk9IjIzMiIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1z
aXplPSIxNCI+eyBsZXQgdCA9IHggKiAyOyB0ICsgMTAgfTwvdGV4dD4KICAgIDwhLS0gTm9kZSA1
OiBQcmltaXRpdmUgVHlwZXMgLS0+CiAgICA8dGV4dCB4PSIyMCIgeT0iMjY1IiBmaWxsPSJ2YXIo
LS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCI+VHlwZU5vZGU8L3Rl
eHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iMjgyIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNp
emU9IjE0Ij5JbnQgfCBGbG9hdCB8IFN0cmluZyB8IEJvb2wgfCBWb2lkPC90ZXh0PgogIDwvZz4K
ICA8IS0tIFRyYW5zZm9ybWF0aW9uIEFycm93cyAtLT4KICA8ZyBzdHJva2U9InZhcigtLXN2Zy1m
ZykiIHN0cm9rZS13aWR0aD0iMi41IiBmaWxsPSJub25lIj4KICAgIDxwYXRoIGQ9Ik0gNDMwIDEw
MCBMIDQ5MCAxMDAiLz4KICAgIDxwb2x5Z29uIHBvaW50cz0iNDkwLDEwMCA0ODAsOTUgNDgwLDEw
NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHBhdGggZD0iTSA0MzAgMTUwIEwgNDkwIDE1
MCIvPgogICAgPHBvbHlnb24gcG9pbnRzPSI0OTAsMTUwIDQ4MCwxNDUgNDgwLDE1NSIgZmlsbD0i
dmFyKC0tc3ZnLWZnKSIvPgogICAgPHBhdGggZD0iTSA0MzAgMjAwIEwgNDkwIDIwMCIvPgogICAg
PHBvbHlnb24gcG9pbnRzPSI0OTAsMjAwIDQ4MCwxOTUgNDgwLDIwNSIgZmlsbD0idmFyKC0tc3Zn
LWZnKSIvPgogICAgPHBhdGggZD0iTSA0MzAgMjUwIEwgNDkwIDI1MCIvPgogICAgPHBvbHlnb24g
cG9pbnRzPSI0OTAsMjUwIDQ4MCwyNDUgNDgwLDI1NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIvPgog
ICAgPHBhdGggZD0iTSA0MzAgMzAwIEwgNDkwIDMwMCIvPgogICAgPHBvbHlnb24gcG9pbnRzPSI0
OTAsMzAwIDQ4MCwyOTUgNDgwLDMwNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIvPgogIDwvZz4KICA8
IS0tIFJpZ2h0OiBUYXJnZXQgQysrIENvbnN0cnVjdHMgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFu
c2xhdGUoNTAwLCAzMCkiPgogICAgPHJlY3QgeD0iMCIgeT0iMCIgd2lkdGg9IjM4MCIgaGVpZ2h0
PSIzMDAiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZh
bGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIiBmaWx0ZXI9
InVybCgjYm94U2hhZG93KSIvPgogICAgPHJlY3QgeD0iMCIgeT0iMCIgd2lkdGg9IjM4MCIgaGVp
Z2h0PSIzNSIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEt
ZmFsbGJhY2spKSIvPgogICAgPHRleHQgeD0iMTkwIiB5PSIyMyIgZmlsbD0idmFyKC0tc3ZnLWZn
KSIgZm9udC1zaXplPSIxNiIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUi
PlRhcmdldCBDKysgT3V0cHV0PC90ZXh0PgogICAgPCEtLSBUUyAxIC0tPgogICAgPHRleHQgeD0i
MjAiIHk9IjY1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdo
dD0iYm9sZCI+Y29uc3QgVmFyaWFibGUgRGVjbGFyYXRpb248L3RleHQ+CiAgICA8dGV4dCB4PSIy
MCIgeT0iODIiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiPmNvbnN0IGludCBs
aW1pdCA9IDEwMDs8L3RleHQ+CiAgICA8IS0tIFRTIDIgLS0+CiAgICA8dGV4dCB4PSIyMCIgeT0i
MTE1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9s
ZCI+VHlwZWQgTW9kZXJuIEMrKyBMYW1iZGE8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iMTMy
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5bJmFtcDtdKGludCB4LCBpbnQg
eSkgLT4gaW50IHsgcmV0dXJuICh4ICsgeSk7IH08L3RleHQ+CiAgICA8IS0tIFRTIDMgLS0+CiAg
ICA8dGV4dCB4PSIyMCIgeT0iMTY1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0
IiBmb250LXdlaWdodD0iYm9sZCI+VGVybmFyeSBDb25kaXRpb25hbCBPcGVyYXRvcjwvdGV4dD4K
ICAgIDx0ZXh0IHg9IjIwIiB5PSIxODIiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0i
MTQiPihuICZsdDs9IDEgPyAxIDogKG4gKiBmYWMobiAtIDEpKSk8L3RleHQ+CiAgICA8IS0tIFRT
IDQgLS0+CiAgICA8dGV4dCB4PSIyMCIgeT0iMjE1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250
LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCI+SW1tZWRpYXRlbHkgSW52b2tlZCBMYW1iZGEg
RXhwcmVzc2lvbiAoSUlMRSk8L3RleHQ+CiAgICA8dGV4dCB4PSIyMCIgeT0iMjMyIiBmaWxsPSJ2
YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5bJmFtcDtdeyBjb25zdCBpbnQgdCA9IHggKiAy
OyByZXR1cm4gKHQgKyAxMCk7IH0oKTwvdGV4dD4KICAgIDwhLS0gVFMgNSAtLT4KICAgIDx0ZXh0
IHg9IjIwIiB5PSIyNjUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGZvbnQt
d2VpZ2h0PSJib2xkIj5DKysgUHJpbWl0aXZlIE1hcHBpbmc8L3RleHQ+CiAgICA8dGV4dCB4PSIy
MCIgeT0iMjgyIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5pbnQgfCBkb3Vi
bGUgfCBzdGQ6OnN0cmluZyB8IGJvb2wgfCB2b2lkPC90ZXh0PgogIDwvZz4KPC9zdmc+
" alt="Mapping Faberge to C++" />
<figcaption>Mapping Faberge to C++</figcaption>
</figure>

#### 21.2.1 Type Translation

C++ posesses types which match directly to the source Faberge types, as
shown in tbl. 3.

| Faberge Type | C++ Target Type | Semantic Meaning |
|------------------------|------------------------|------------------------|
| `Int` | `long long` | Numeric value (integral domain, 64-bit) |
| `Float` | `double` | Numeric value (floating-point domain) |
| `String` | `std::string` | UTF-8 character sequence |
| `Bool` | `bool` | Logical truth value (`true` / `false`) |
| `Void` | `void` | Absence of value |
| `(T1, T2 -> Ret)` | Use `auto` to match `[&](T1 p1, T2 p2) -> Ret{...}` | First-class function closure (Modern C++ lambda, `[&]` captures any enclosing scopes) |

Table 3: Faberge Types {#tbl:target-types}

#### 21.2.2 Expression vs Statement Discrepancy: The IIFE Pattern

In functional languages like Faberge, **everything is an expression**. A
block `{ let a = 1; let b = 2; a + b }` has a value: the result of its
trailing expression.

In C++, however, curly-brace code blocks `{ ... }` are strictly
**statements** and do not evaluate to values. We cannot write:

``` cpp
// ILLEGAL C++:
const long long x = { const long long a = 1; a + 10; };
```

To bridge this semantic gap, the code emitter wraps Faberge block
expressions inside **Immediately Invoked Function Expressions (IIFEs)**
(also called IILEs, or Immediately Invoked Lambda Expressions in Modern
C++):

``` cpp
// VALID C++ emitted by our compiler:
const long long x = [&]{
    const a = 1;
    return a + 10;
}();
```

This pattern preserves:

1.  **Lexical Scope Isolation:** Variables declared inside the block
    cannot leak to the enclosing outer scope.
2.  **Expression Composability:** The block can appear inline anywhere
    an expression is expected (e.g., inside function arguments, ternary
    branches, or arithmetic operations).

### 21.3 Architecture of the C++ Code Emitter

Let us design the code emitter subsystem using modern C++20 patterns:

- **Visitor Dispatch:** Inherits from `ASTVisitor` to walk the AST in
  preorder and postorder passes.
- **Clean Indentation Buffer:** Manages indentation depth (`indent()`,
  `dedent()`) and formats readable, production-grade C++ with consistent
  spacing.
- **Expression Parenthesisation:** Guarantees arithmetic precedence by
  wrapping complex sub-expressions in grouping parentheses where
  necessary.
- **Entry-Point Synthesis:** Automatically emits runtime driver code to
  execute the `main` function and output results to the console.

<figure>
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIzMCAyMCA4MjAgMjEwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyMTAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0iZVNoYWRvdyIgeD0iLTUlIiB5PSIt
NSUiIHdpZHRoPSIxMTAlIiBoZWlnaHQ9IjEyMCUiPgogICAgICA8ZmVEcm9wU2hhZG93IGR4PSIw
IiBkeT0iMiIgc3RkRGV2aWF0aW9uPSIzIiBmbG9vZC1jb2xvcj0iIzAwMCIgZmxvb2Qtb3BhY2l0
eT0iMC40Ii8+CiAgICA8L2ZpbHRlcj4KICA8L2RlZnM+CiAgPCEtLSBNYWluIFBpcGVsaW5lIEJv
eCAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSg0MCwgMzApIj4KICAgIDxyZWN0IHg9IjAi
IHk9IjAiIHdpZHRoPSI4MDAiIGhlaWdodD0iMTkwIiByeD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRp
bnQtMiwgdmFyKC0tc3ZnLXRpbnQtMi1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZyki
IHN0cm9rZS13aWR0aD0iMiIgZmlsdGVyPSJ1cmwoI2VTaGFkb3cpIi8+CiAgICA8dGV4dCB4PSI0
MDAiIHk9IjMwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE3IiBmb250LXdlaWdo
dD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+QysrMjMgQ29kZSBFbWl0dGVyIEJ1ZmZlciBN
YW5hZ2VtZW50PC90ZXh0PgogICAgPCEtLSBTdWItYm94ZXMgLS0+CiAgICA8IS0tIEluZGVudGF0
aW9uIENvbnRyb2xsZXIgLS0+CiAgICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSgzMCwgNjApIj4K
ICAgICAgPHJlY3QgeD0iMCIgeT0iMCIgd2lkdGg9IjIyMCIgaGVpZ2h0PSIxMDAiIHJ4PSI2IiBm
aWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9r
ZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICAgICAgPHRleHQgeD0iMTEw
IiB5PSIyNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9udC13ZWlnaHQ9
ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPkluZGVudGF0aW9uIEVuZ2luZTwvdGV4dD4KICAg
ICAgPHRleHQgeD0iMjAiIHk9IjUwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0
Ij5pbmRlbnRfbGV2ZWxfIDogaW50PC90ZXh0PgogICAgICA8dGV4dCB4PSIyMCIgeT0iNzAiIGZp
bGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiPmluZGVudCgpICAgPSZndDsgKytsZXZl
bDwvdGV4dD4KICAgICAgPHRleHQgeD0iMjAiIHk9IjkwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBm
b250LXNpemU9IjE0Ij5kZWRlbnQoKSAgID0mZ3Q7IC0tbGV2ZWw8L3RleHQ+CiAgICA8L2c+CiAg
ICA8IS0tIEV4cHJlc3Npb24gU3RyZWFtIC0tPgogICAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUo
MjkwLCA2MCkiPgogICAgICA8cmVjdCB4PSIwIiB5PSIwIiB3aWR0aD0iMjIwIiBoZWlnaHQ9IjEw
MCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJh
Y2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgogICAgICA8
dGV4dCB4PSIxMTAiIHk9IjI1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE2IiBm
b250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+RXhwcmVzc2lvbiBGb3JtYXR0
ZXI8L3RleHQ+CiAgICAgIDx0ZXh0IHg9IjIwIiB5PSI1MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIg
Zm9udC1zaXplPSIxNCI+c3RkOjpvc3RyaW5nc3RyZWFtIGJ1ZmZlcl88L3RleHQ+CiAgICAgIDx0
ZXh0IHg9IjIwIiB5PSI3MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCI+VGVy
bmFyeSBjb25kaXRpb25hbCBsb2dpYzwvdGV4dD4KICAgICAgPHRleHQgeD0iMjAiIHk9IjkwIiBm
aWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5JSUxFIGJsb2NrIGVtaXNzaW9uPC90
ZXh0PgogICAgPC9nPgogICAgPCEtLSBGaWxlIE91dHB1dCBTdWJzeXN0ZW0gLS0+CiAgICA8ZyB0
cmFuc2Zvcm09InRyYW5zbGF0ZSg1NTAsIDYwKSI+CiAgICAgIDxyZWN0IHg9IjAiIHk9IjAiIHdp
ZHRoPSIyMjAiIGhlaWdodD0iMTAwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFy
KC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13
aWR0aD0iMS41Ii8+CiAgICAgIDx0ZXh0IHg9IjExMCIgeT0iMjUiIGZpbGw9InZhcigtLXN2Zy1m
ZykiIGZvbnQtc2l6ZT0iMTYiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxl
Ij5FbWl0dGVkIFRhcmdldCBBcnRpZmFjdDwvdGV4dD4KICAgICAgPHRleHQgeD0iMjAiIHk9IjUw
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0Ij5IZWFkZXIgYW5kIFByZWFtYmxl
PC90ZXh0PgogICAgICA8dGV4dCB4PSIyMCIgeT0iNzAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZv
bnQtc2l6ZT0iMTQiPlN5bnRoZXNpc2VkIG1haW4oKSBkcml2ZXI8L3RleHQ+CiAgICAgIDx0ZXh0
IHg9IjIwIiB5PSI5MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCI+RmlsZSBz
dHJlYW06IG91dC5jcHA8L3RleHQ+CiAgICA8L2c+CiAgPC9nPgo8L3N2Zz4=
" alt="C++ Code Output Formatter" />
<figcaption>C++ Code Output Formatter</figcaption>
</figure>

### 21.4 Experimental Implementation in C++: The C++ Code Generator

The listings below complete the Faberge compiler (with reference to all
of the previous C++ code for Faberge):

- lst. 14 implements the complete `CppEmitter` class. It walks the AST,
  emits formatted code, and translates types into valid C++ syntax.
- lst. 15 unites all four pipeline stages—**Lexer**, **Parser**,
  **TypeChecker**, and **CppEmitter**—into a single compiler driver (see
  fig. 39).

<figure id="fig:complete-compiler">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NDQuNzUyIiBo
ZWlnaHQ9IjE1Ny42NDgwMSIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1NDQuNzUyIDE1
Ny42NDgwMSI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNp
cmNsZSwgLnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAg
c3Ryb2tlLXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAx
OwogIHN0cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgou
c3ZnYm9iIHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1m
YW1pbHk6IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXpl
OiAxNHB4Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxs
OiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4
Owp9Cgouc3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxs
ZWQgewogIGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAu
bm9maWxsIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJv
dyB7CiAgbWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9h
cnJvdyB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtl
ZF9kaWFtb25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFy
dF9tYXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3Zn
Ym9iIC5lbmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgou
c3ZnYm9iIC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJs
KCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAg
bWFya2VyLXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9i
aWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1
cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlk
PSJhcnJvdyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxwb2x5Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtl
cj4KICAgIDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIg
cmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0
YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIi
PjwvcG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94
PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0
PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5
PSI0IiByPSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1h
cmtlciBpZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQi
IG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQi
PjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUi
IHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJr
ZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBj
eD0iNCIgY3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJr
ZXI+CiAgPC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRo
PSI1NDQuNzUyIiBoZWlnaHQ9IjE1Ny42NDgwMSI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjE5MC4zMTQwMSIgeT0iMjQuNDQ0IiA+RmFiZXJnZTwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNDYuMTg2IiB5PSIyNC40NDQiID5Db21waWxlcjwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMDkuMDQyMDIiIHk9IjI0LjQ0
NCIgPkRyaXZlcjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOS42ODIw
MDEiIHk9IjUyLjM4IiA+MS48L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MjkuNjgyMDAxIiB5PSI2Ni4zNDgiID4yLjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIyOS42ODIwMDEiIHk9IjgwLjMxNiIgPjMuPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjI5LjY4MjAwMSIgeT0iOTQuMjg0MDA0IiA+NC48L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjkuNjgyMDAxIiB5PSIxMDguMjUyMDEiID41Ljwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOS42ODIwMDEiIHk9IjEyMi4y
MiIgPjYuPC90ZXh0PgogIDxsaW5lIHgxPSIyMzcuNDU2MDEiIHkxPSI2Mi44NTYwMDMiIHgyPSIy
NTEuNDI0MDEiIHkyPSI2Mi44NTYwMDMiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29u
IHBvaW50cz0iMjUxLjQyNDAxLDU5LjM2NDAwMiAyNTguNDA4MDIsNjIuODU2MDAzIDI1MS40MjQw
MSw2Ni4zNDgiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGluZSB4MT0iMjM3LjQ1NjAx
IiB5MT0iNzYuODI0MDA1IiB4Mj0iMjUxLjQyNDAxIiB5Mj0iNzYuODI0MDA1IiBjbGFzcz0ic29s
aWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjI1MS40MjQwMSw3My4zMzIgMjU4LjQwODAy
LDc2LjgyNDAwNSAyNTEuNDI0MDEsODAuMzE2IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAg
PGxpbmUgeDE9IjIzNy40NTYwMSIgeTE9IjkwLjc5MiIgeDI9IjI1MS40MjQwMSIgeTI9IjkwLjc5
MiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyNTEuNDI0MDEsODcu
MyAyNTguNDA4MDIsOTAuNzkyIDI1MS40MjQwMSw5NC4yODQwMDQiIGNsYXNzPSJmaWxsZWQiPjwv
cG9seWdvbj4KICA8bGluZSB4MT0iMjM3LjQ1NjAxIiB5MT0iMTA0Ljc2IiB4Mj0iMjUxLjQyNDAx
IiB5Mj0iMTA0Ljc2IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjI1
MS40MjQwMSwxMDEuMjY4MDA1IDI1OC40MDgwMiwxMDQuNzYgMjUxLjQyNDAxLDEwOC4yNTIwMSIg
Y2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjI2Ny4xMzgiIHk9Ijk0LjI4NDAwNCIgPlByb3ZlbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIzMTYuMDI2IiB5PSI5NC4yODQwMDQiID5UeXBlPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM1MC45NDYiIHk9Ijk0LjI4NDAwNCIgPlNhZmV0eTwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNjcuMTM4IiB5PSIxMDguMjUy
MDEiID5Gb3JtYXR0ZWQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzM2
Ljk3OCIgeT0iMTA4LjI1MjAxIiA+T3V0cHV0PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjM4NS44NjYiIHk9IjEwOC4yNTIwMSIgPkNvZGU8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTAuNjM0MDAzIiB5PSI1Mi4zOCIgPlJlYWQgU291cmNlIEZp
bGUgKC5mYWIpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUwLjYzNDAw
MyIgeT0iNjYuMzQ4IiA+TGV4ZXI6OnRva2VuaXNlKCk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iMjY3LjEzOCIgeT0iNjYuMzQ4IiA+c3RkOjp2ZWN0b3ImbHQ7VG9rZW4m
Z3Q7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUwLjYzNDAwMyIgeT0i
ODAuMzE2IiA+UGFyc2VyOjpwYXJzZV9wcm9ncmFtKCk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iMjY3LjEzOCIgeT0iODAuMzE2IiA+c3RkOjp1bmlxdWVfcHRyJmx0O1By
b2dyYW1BU1QmZ3Q7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjUwLjYz
NDAwMyIgeT0iOTQuMjg0MDA0IiA+VHlwZUNoZWNrZXI6OmNoZWNrKCk8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTAuNjM0MDAzIiB5PSIxMDguMjUyMDEiID5DcHBFbWl0
dGVyOjplbWl0KCk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTAuNjM0
MDAzIiB5PSIxMjIuMjIiID5Xcml0ZSBPdXRwdXQgRmlsZSAoLmNwcCk8L3RleHQ+CiAgPGc+CiAg
ICA8bGluZSB4MT0iMTAuNDc2IiB5MT0iNi45ODQiIHgyPSI1MzQuMjc2IiB5Mj0iNi45ODQiIGNs
YXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjEwLjQ3NiIgeTE9IjYuOTg0IiB4Mj0i
MTAuNDc2IiB5Mj0iMTMyLjY5NiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0i
NTM0LjI3NiIgeTE9IjYuOTg0IiB4Mj0iNTM0LjI3NiIgeTI9IjEzMi42OTYiIGNsYXNzPSJzb2xp
ZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjEwLjQ3NiIgeTE9IjM0LjkyMDAwMiIgeDI9IjUzNC4y
NzYiIHkyPSIzNC45MjAwMDIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjEw
LjQ3NiIgeTE9IjEzMi42OTYiIHgyPSI1MzQuMjc2IiB5Mj0iMTMyLjY5NiIgY2xhc3M9InNvbGlk
Ij48L2xpbmU+CiAgPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 39: Completed Compiler Driver</figcaption>
</figure>

Listing 14: AST Visitor to Emit C++ Code (Emitter.hpp)

In [ ]:
#ifndef EMITTER_HPP
#define EMITTER_HPP

#include "../chapter-19/AST.hpp"
#include "../chapter-19/ASTVisitor.hpp"
#include <sstream>
#include <string>
#include <string_view>
#include <iostream>

namespace Faberge {

class CppEmitter : public ASTVisitor {
public:
    CppEmitter() = default;

    std::string generate(const ProgramAST& program) {
        m_buffer.str("");
        m_buffer.clear();
        m_indent_level = 0;

        // 1. Emit Compiler Header & Preamble
        emit_line("// =============================================================================");
        emit_line("// Generated automatically by the Faberge Compiler (Stage 4 Target Code Emitter)");
        emit_line("// Target: Modern C++23");
        emit_line("// =============================================================================\n");
        emit_line("#include <iostream>");
        emit_line("#include <any>");
        emit_line("#include <vector>");
        emit_line("#include <string>\n");
        emit_line("int main() {");
        indent();

        // 2. Traverse AST and Emit Code
        program.accept(*this);

        // 3. Emit Main Entry-Point Execution Driver
        if (m_has_main) {
            emit_line("\nstd::cout << \"[Faberge Runtime] Program terminated with output: \" << main_() << '\\n';\n");
        }
        emit_line("return 0;");
        dedent();
        emit_line("}");

        return m_buffer.str();
    }

    // -------------------------------------------------------------------------
    // AST Visitor Implementations
    // -------------------------------------------------------------------------

    void visit(const ProgramAST& node) override {
        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }
    }

    void visit(const LetStmtAST& node) override {
        emit_indent();
        m_buffer << "const ";

        if (node.type_annotation().has_value()) {
            m_buffer << map_type(*node.type_annotation()) << " ";
        }
        else {
            m_buffer << "auto ";
        }

        if (node.name() == "main") {
            m_buffer << "main_ = ";
            m_has_main = true;
        }
        else {
            m_buffer << node.name() << " = ";
        }

        std::string prev_self = m_self;
        m_self = node.name();

        node.initialiser().accept(*this);
        m_buffer << ";\n";

        m_self = prev_self;
    }

    void visit(const ExprStmtAST& node) override {
        emit_indent();
        node.expression().accept(*this);
        m_buffer << ";\n";
    }

    void visit(const LiteralExprAST& node) override {
        if (node.token_type() == TokenType::Literal_String) {
            m_buffer << "\"" << escape_string(node.value()) << "\"";
        } else {
            m_buffer << node.value();
        }
    }

    void visit(const VariableExprAST& node) override {
        if (node.name() == m_self) {
            m_buffer << "self_";
        }
        else {
            m_buffer << node.name();
        }
    }

    void visit(const UnaryExprAST& node) override {
        m_buffer << "(";
        if (node.op() == TokenType::Minus) m_buffer << "-";
        else if (node.op() == TokenType::Exclamation) m_buffer << "!";
        
        node.operand().accept(*this);
        m_buffer << ")";
    }

    void visit(const BinaryExprAST& node) override {
        m_buffer << "(";
        node.left().accept(*this);
        m_buffer << " " << map_operator(node.op()) << " ";
        node.right().accept(*this);
        m_buffer << ")";
    }

    void visit(const IfExprAST& node) override {
        // Translate functional if-then-else to C++ ternary expression
        m_buffer << "(";
        node.condition().accept(*this);
        m_buffer << " ? ";
        node.then_branch().accept(*this);
        m_buffer << " : ";
        node.else_branch().accept(*this);
        m_buffer << ")";
    }

    void visit(const CallExprAST& node) override {
        node.callee().accept(*this);
        m_buffer << "(";
        for (size_t i = 0; i < node.arguments().size(); ++i) {
            node.arguments()[i]->accept(*this);
            if (i + 1 < node.arguments().size()) {
                m_buffer << ", ";
            }
        }
        m_buffer << ")";
    }

    void visit(const BlockExprAST& node) override {
        // Emit as an Immediately Invoked Lambda Expression (IILE)
        m_buffer << "([&]{\n";
        indent();

        for (const auto& stmt : node.statements()) {
            stmt->accept(*this);
        }

        if (node.result_expr()) {
            emit_indent();
            m_buffer << "return ";
            node.result_expr()->accept(*this);
            m_buffer << ";\n";
        }

        dedent();
        emit_indent();
        m_buffer << "})()";
    }

    void visit(const LambdaExprAST& node) override {
        m_buffer << "[&](this auto self_";
        for (size_t i = 0; i < node.parameters().size(); ++i) {
            m_buffer << ", " << map_type(node.parameters()[i].type)
                << " " << node.parameters()[i].name;
        }
        m_buffer << ")";

        if (node.return_type().has_value()) {
            m_buffer << " -> " << map_type(*node.return_type()) << " ";
        }

        m_buffer << "{return ";        
        node.body().accept(*this);
        m_buffer << ";}";
    }

private:
    std::ostringstream m_buffer;
    std::string m_self;
    bool m_has_main{false};
    int m_indent_level{0};

    void indent() noexcept { ++m_indent_level; }
    void dedent() noexcept { if (m_indent_level > 0) --m_indent_level; }

    void emit_indent() {
        for (int i = 0; i < m_indent_level; ++i) {
            m_buffer << "    ";
        }
    }

    void emit_line(std::string_view line) {
        emit_indent();
        m_buffer << line << "\n";
    }

    static std::string map_type(const TypeNode& type) {
        switch (type.kind()) {
            case TypeNode::Kind::Primitive:
                switch (type.primitive_kind()) {
                    case PrimitiveTypeKind::Int:    return "long long";
                    case PrimitiveTypeKind::Float:  return "double";
                    case PrimitiveTypeKind::String: return "std::string";
                    case PrimitiveTypeKind::Bool:   return "bool";
                    case PrimitiveTypeKind::Void:   return "void";
                }
                break;
            case TypeNode::Kind::Function:
                // Recursive mapping for higher-order function types
                return "Function";
            case TypeNode::Kind::List:
                return "std::vector<std::any>";
        }
        return "std::any";
    }

    static std::string_view map_operator(TokenType op) noexcept {
        switch (op) {
            case TokenType::Plus:               return "+";
            case TokenType::Minus:              return "-";
            case TokenType::Asterisk:           return "*";
            case TokenType::Slash:              return "/";
            case TokenType::Percent:            return "%";
            case TokenType::EqualEqual:         return "==";
            case TokenType::ExclamationEqual:   return "!=";
            case TokenType::Less:               return "<";
            case TokenType::LessEqual:          return "<=";
            case TokenType::Greater:            return ">";
            case TokenType::GreaterEqual:       return ">=";
            case TokenType::AmpersandAmpersand: return "&&";
            case TokenType::PipePipe:           return "||";
            default:                            return "/* ? */";
        }
    }

    static std::string escape_string(std::string_view raw) {
        std::string out;
        for (char c : raw) {
            if (c == '"') out += "\\\"";
            else if (c == '\\') out += "\\\\";
            else if (c == '\n') out += "\\n";
            else if (c == '\t') out += "\\t";
            else if (c == '\r') out += "\\r";
            else out += c;
        }
        return out;
    }
};

} // namespace Faberge

#endif // EMITTER_HPP

Listing 15: Complete Integrated Compiler (main.cpp)

In [ ]:
#include "../chapter-18/Lexer.hpp"
#include "../chapter-19/Parser.hpp"
#include "../chapter-20/TypeChecker.hpp"
// #include "Emitter.hpp"

#include <iostream>
#include <fstream>
#include <sstream>

namespace Faberge {

class Compiler {
public:
    static bool compile_source(std::string_view source_code, 
                               std::string_view filename, 
                               std::ostream& output_stream) {
        std::cout << "\033[1;34m[Faberge Compiler]\033[0m Compiling '" << filename << "'...\n";

        // STAGE 1: Lexical Analysis
        Lexer lexer(source_code, filename);
        auto tokens = lexer.tokenise();
        if (lexer.has_errors()) {
            std::cerr << "\033[1;31m[Compilation Aborted]\033[0m Lexical errors detected.\n";
            return false;
        }
        std::cout << "  ✓ Stage 1: Lexical Analysis completed (" << tokens.size() << " tokens).\n";

        // STAGE 2: Syntactic Analysis (Parser)
        Parser parser(std::move(tokens), source_code);
        auto ast = parser.parse_program();
        if (parser.has_errors()) {
            std::cerr << "\033[1;31m[Compilation Aborted]\033[0m Syntax errors detected.\n";
            return false;
        }
        std::cout << "  ✓ Stage 2: Syntactic Analysis completed (AST constructed).\n";

        // STAGE 3: Semantic Analysis and Static Type Checking
        TypeChecker checker(source_code);
        if (!checker.check(*ast)) {
            std::cerr << "\033[1;31m[Compilation Aborted]\033[0m Semantic type verification failed.\n";
            return false;
        }
        std::cout << "  ✓ Stage 3: Semantic Analysis completed (Type Safety Proved).\n";

        // STAGE 4: Code Generation (C++ Target Emission)
        CppEmitter emitter;
        std::string cpp_code = emitter.generate(*ast);
        std::cout << "  ✓ Stage 4: Target Code Generation completed.\n";

        output_stream << cpp_code;
        return true;
    }
};

} // namespace Faberge

int main() {
    // A complete, expressive functional Faberge program
    constexpr std::string_view faberge_source = R"(
        // --- Mathematical Helper Functions ---
        let square = fn (x: Int): Int => x * x;

        let add = fn (a: Int, b: Int): Int => a + b;

        // Recursive Factorial Implementation
        let factorial = fn (n: Int): Int => {
            if n <= 1 then 
                1 
            else 
                n * factorial(n - 1)
        };

        // Recursive Fibonacci Implementation
        let fibonacci = fn (n: Int): Int => {
            if n <= 0 then
                0
            else if n == 1 then
                1
            else
                fibonacci(n - 1) + fibonacci(n - 2)
        };

        // Main Program Entry Point
        let main = fn (): Int => {
            let num = 5;
            let fac5 = factorial(num);
            let fib7 = fibonacci(7);
            
            // Block Expression with Lexical Scoping
            let combined = {
                let multiplier = 10;
                let squared = square(fac5);
                (squared / multiplier) + fib7
            };

            combined
        };
    )";

    std::cout << "====================================================\n";
    std::cout << "      The Faberge Compiler (Complete Demo)    \n";
    std::cout << "====================================================\n\n";

    std::ostringstream generated_cpp;
    bool success = Faberge::Compiler::compile_source(faberge_source, "sample.fab", generated_cpp);

    if (success) {
        std::cout << "\n\033[1;32m[SUCCESS] Compilation Succeeded!\033[0m\n";
        std::cout << "====================================================\n";
        std::cout << "EMITTED C++ CODE:\n";
        std::cout << "====================================================\n";
        std::cout << generated_cpp.str() << "\n";
        std::cout << "====================================================\n";
    } else {
        std::cout << "\n\033[1;31m[FAILURE] Compilation Failed.\033[0m\n";
        return 1;
    }

    return 0;
}

### 21.6 Verification: Executing Emitted C++

When our compiler executes the driver above, the **C++ Emitter**
synthesises the following clean, typed C++23 output:

``` cpp
// =============================================================================
// Generated automatically by the Faberge Compiler (Stage 4 Target Code Emitter)
// Target: Modern C++23
// =============================================================================

#include <iostream>
#include <string>

int main() {
    const auto square = [&](this auto self_, long long x) -> long long {return (x * x);};
    const auto add = [&](this auto self_, long long a, long long b) -> long long {return (a + b);};
    const auto factorial = [&](this auto self_, long long n) -> long long {return ([&]{
        return ((n <= 1) ? 1 : (n * self_((n - 1))));
    })();};
    const auto fibonacci = [&](this auto self_, long long n) -> long long {return ([&]{
        return ((n <= 0) ? 0 : ((n == 1) ? 1 : (self_((n - 1)) + self_((n - 2)))));
    })();};
    const auto main_ = [&](this auto self_) -> long long {return ([&]{
        const auto num = 5;
        const auto fac5 = factorial(num);
        const auto fib7 = fibonacci(7);
        const auto combined = ([&]{
            const auto multiplier = 10;
            const auto squared = square(fac5);
            return ((squared / multiplier) + fib7);
        })();
        return combined;
    })();};
    std::cout << "[Faberge Runtime] Program terminated with output: " << main_() << '\n';

    return 0;
}
```

When run, this program outputs:

```
[Faberge Runtime] Program terminated with output: 1453
```

#### 21.6.1 Mathematical Trace Verification

Let us verify the runtime output step-by-step:

1.  `factorial(5)` $= 5 \times 4 \times 3 \times 2 \times 1 = 120$.
2.  `fibonacci(7)` $= 13$.
3.  `square(fac5)` $= 120^2 = 14,400$.
4.  `squared / multiplier` $= 14,400 / 10 = 1,440$.
5.  `combined` $= 1,440 + 13 = 1,453$.

The emitted code executes with exact mathematical precision, confirming
the end-to-end correctness of our entire compiler pipeline.

### 21.7 Summary

In this chapter we have achieved the following:

- The Visitor Pattern has been used to create a class which generates
  valid C++23 code from the type-checked Abstract Syntax Tree.
- Equivalent types are used in the C++ code to that in the Faberge
  source, and this guarantees that the target program is syntactically
  valid. -A primitive, but fully functional, compiler driver reads a
  valid Faberge program from a string and outputs a valid C++ program
  which can then be compiled into an executable.

With the completion of **Stage 4: Code Generation**, our four-stage
compiler is fully realized.

### 21.8 Programming Challenges

#### Exercise 21.1: Tail-Call Optimisation (TCO) to Iterative Loops

**Context and Theory:**

Recursive algorithms (such as accumulator-based factorial or list
processing) can exhaust the physical operating system call stack when
recursing over large inputs, causing a runtime segmentation fault:

$$\text{Stack Frame Depth} = O(N)$$

While some C++ compilers perform tail-call elimination under `-O2` or
`-O3`, it is not guaranteed by the C++ standard. The Faberge compiler
can enforce **Guaranteed Tail-Call Elimination** at the code generation
level:

1.  Identify recursive calls located strictly in **Tail Position** (the
    final expression evaluated before function return).
2.  Rewrite the tail-recursive function into an iterative C++
    `while (true)` loop:
3.  Update parameter variables in-place and continue looping, reducing
    call-stack consumption to $O(1)$ memory.

Faberge Tail-Recursive AST:

```
let fac = fn (n, acc) =>
    if n <= 1 then acc
    else fac(n - 1, n * acc)
```

Emitted C++ Loop:

``` cpp
const auto fac = [](int64_t n, int64_t acc) {
    while (true) {
        if (n <= 1) return acc;
        acc = n * acc; n = n - 1;
    }
};
```

**Specification:**

Implement an AST analysis pass in `CppEmitter` that detects
tail-recursive lambda functions. Emit an iterative C++ `while (true)`
loop where recursive calls are lowered into parameter mutations and
`continue` statements.

``` cpp
class TailCallOptimizer {
public:
    static bool is_tail_call(const CallExprAST& call, std::string_view current_fn);
    static void emit_tco_loop(const LambdaExprAST& lambda, std::string_view fn_name, 
                             std::ostringstream& out) {
        out << "[";
        emit_parameters_as_mutable_locals(lambda.parameters(), out);
        out << "]() mutable {\n    while (true) {\n";
        emit_body_as_tco_branch(lambda.body(), fn_name, out);
        out << "    }\n}()";
    }
};
```

**Verification Tasks:**

1.  **Stack Overflow Immunity:** Emit a tail-recursive accumulator sum
    function calculating $\sum_{i=1}^{10^7} i$. Run the compiled binary
    and verify that it completes without stack overflow.
2.  **Non-Tail Preservation:** Verify that non-tail recursive functions
    (e.g. `n * fac(n - 1)`) are left untouched as standard function
    calls.
3.  **Multi-Parameter Mutation Order:** Check a function where arguments
    swap positions: `gcd(b, a % b)`. Verify that the emitter generates
    temporary variables during reassignment to prevent clobbering:
    `auto __t0 = b; auto __t1 = a % b; a = __t0; b = __t1; continue;`.

#### Exercise 21.2: Lowering Heterogeneous Tuples to `std::tuple`

**Context and Theory:**

Faberge heterogeneous product types (tuples) must be mapped to
efficient, cache-friendly native structures. In Modern C++, the
canonical representation is `std::tuple`.

The emitter maps tuple operations as follows:

- **Construction:**
  $(e_1, e_2, \dots, e_k) \implies \text{std::make\_tuple}(e_1, e_2, \dots, e_k)$.
- **Component Extraction:** $t.i \implies \text{std::get}<i>(t)$.
- **Destructuring:** Lowering pattern-matched tuple variables into C++17
  **Structured Bindings**:

$$\text{let } (a, b) = \text{pair}; \implies \text{const auto } [a, b] = \text{pair};$$

This lowering guarantees that tuples are stored contiguously in stack
memory without dynamic heap allocations. This allows for compiling the
Faberge code:

```
let p = (42, "Faberge", true); let id = p.0;
```

To C++ target:

``` cpp
const auto p = std::make_tuple(42, std::string("Faberge"), true);
const auto id = std::get<0>(p);
```

**Specification:**

Extend `CppEmitter` to lower `TupleExprAST` and `TupleAccessExprAST`.
Generate the required `#include <tuple>` header directive and emit clean
`std::make_tuple` and `std::get` function calls.

``` cpp
void CppEmitter::visit(const TupleExprAST& node) {
    buffer_ << "std::make_tuple(";
    for (size_t i = 0; i < node.elements().size(); ++i) {
        node.elements()[i]->accept(*this);
        if (i + 1 < node.elements().size()) buffer_ << ", ";
    }
    buffer_ << ")";
}

void CppEmitter::emit_tuple_access(const ExprAST& target, size_t index) {
    buffer_ << "std::get<" << index << ">(";
    target.accept(*this);
    buffer_ << ")";
}
```

**Verification Tasks:**

1.  **Tuple Instantiation and Access:** Emit
    `let t = (10, 3.14); let x = t.0; let y = t.1;`. Compile the output
    and verify that `x` is `int64_t(10)` and `y` is `double(3.14)`.
2.  **Structured Binding Emission:** Emit a destructuring let-binding:
    `let (name, age) = get_user();`. Verify that the emitted C++ matches
    `const auto [name, age] = get_user();`.
3.  **Zero-Tuple / Unit Handling:** Verify that the empty tuple `()`
    emits `std::monostate{}` or `std::tuple<>{}`.

#### Exercise 21.3: Lowering Homogeneous Lists to `std::vector`

**Context and Theory:**

Faberge homogeneous list collections map naturally to modern C++
`std::vector<T>`.

The code generator must translate operations cleanly:

- **List Construction:**
  $[e_1, e_2, \dots, e_k] \implies \text{std::vector}\langle T\rangle\{e_1, e_2, \dots, e_k\}$.
- **Empty Lists:** Requires explicit type specialization based on
  semantic analysis: `std::vector<int64_t>{}`.
- **Subscript Access:** Translates $arr[i]$ into safe, bounds-checked
  element extraction (`arr.at(static_cast<size_t>(i))`) or unchecked
  direct indexing (`arr[static_cast<size_t>(i)]`).

Because Faberge integers are signed 64-bit (`int64_t`) and C++ container
indices are unsigned (`size_t`), the emitter must insert static casts to
prevent compiler signed/unsigned comparison warnings. This allows for
compiling the Faberge code:

```
[ 10, 20, 30 ][0]
```

To C++ target:

``` cpp
std::vector<int64_t>{10, 20, 30}.at(static_cast<size_t>(0));
```

**Specification:**

Extend `CppEmitter` to handle `ListLiteralExprAST` and `IndexExprAST`.
Retrieve the verified element type from the semantic analysis table,
emit the explicit `std::vector<T>` constructor, and generate static
casts on index access expressions.

``` cpp
void CppEmitter::visit(const ListLiteralExprAST& node) {
    Type elem_t = type_table_.get_type(node).list_element_type();
    buffer_ << "std::vector<" << map_cpp_type(elem_t) << ">{";
    for (size_t i = 0; i < node.elements().size(); ++i) {
        node.elements()[i]->accept(*this);
        if (i + 1 < node.elements().size()) buffer_ << ", ";
    }
    buffer_ << "}";
}

void CppEmitter::visit(const IndexExprAST& node) {
    node.target().accept(*this);
    buffer_ << ".at(static_cast<size_t>(";
    node.index().accept(*this);
    buffer_ << "))";
}
```

**Verification Tasks:**

1.  **Vector Literal Compilation:** Emit `let xs = [1, 2, 3, 4];`.
    Verify that the emitted C++ compiles cleanly without type inference
    warnings.
2.  **Bounds-Checked Indexing:** Emit `let v = xs[2];`. Compile and run
    the binary; assert that `v == 3`.
3.  **Empty List Type Specialization:** Emit `let empty: [Float] = [];`.
    Verify that the emitter generates `std::vector<double>{}` rather
    than untyped `{}`.

#### Exercise 21.4: AST Constant Folding and `constexpr` Emission

**Context and Theory:**

If an expression consists entirely of compile-time constants
(e.g. `10 + 20 * 3`), evaluating it at runtime wastes CPU cycles.

The Faberge compiler performs **Constant Folding** as an AST-to-AST
optimization pass before code generation:

1.  Traverse the AST in postorder.
2.  If both children of a `BinaryExprAST` are `LiteralExprAST` nodes,
    evaluate the operation at compile time.
3.  Replace the binary operator node with a single computed
    `LiteralExprAST`.
4.  Variables initialized with compile-time pure expressions are emitted
    using C++ **`constexpr`** rather than `const`, shifting execution to
    C++ compile-time evaluation.

- **Before Folding:** `BinaryExpr(+) [ 10, BinaryExpr(*) [ 20, 3 ] ]`
- **After Folding:** `LiteralExprAST(70)`
- **Emitted C++:** `constexpr int64_t total = 70;`

**Specification:**

Implement a `ConstantFoldingPass` that folds arithmetic and boolean
operations. Extend `CppEmitter` to emit `constexpr` for variables marked
as compile-time constants by the optimizer.

``` cpp
class ConstantFoldingPass : public ASTVisitor {
public:
    std::unique_ptr<ExprAST> fold(std::unique_ptr<ExprAST> expr) {
        if (auto* bin = dynamic_cast<BinaryExprAST*>(expr.get())) {
            auto left_folded = fold(bin->extract_left());
            auto right_folded = fold(bin->extract_right());
            if (is_literal(left_folded) && is_literal(right_folded)) {
                return evaluate_constant(bin->op(), *left_folded, *right_folded);
            }
            return std::make_unique<BinaryExprAST>(bin->location(), bin->op(), 
                                                  std::move(left_folded), std::move(right_folded));
        }
        return expr;
    }
};
```

**Verification Tasks:**

1.  **Arithmetic Folding:** Run the constant folder on
    `100 * 2 + 50 / 2`. Verify that the generated C++ emits
    `constexpr int64_t x = 225;`.
2.  **Boolean Logic Folding:** Fold `true && false || true`. Verify that
    the output emits `constexpr bool flag = true;`.
3.  **Partial Folding Safety:** Fold `x + (10 * 0)`. Verify that
    non-constant subtrees are preserved and that the AST simplifies to
    `(x + 0)`.

#### Exercise 21.5: Lowering Pattern Matching to C++ `if-else` Cascades

**Context and Theory:**

Faberge’s `match expr with { ... }` construct allows structural matching
against literals, tuples, and wildcards. Because standard C++ does not
possess native pattern matching, the code generator lowers `match` into
an IIFE containing a sequence of **`if-else` branches**.

For each arm:

- **Literal Patterns:** Lowered into equality checks:
  `if (target == literal) { return body; }`.
- **Tuple Destructuring:** Lowered into temporary structured bindings
  with guard checks:
  `if (auto [a, b] = target; a == 0) { return body; }`.
- **Wildcard Pattern (`_`):** Lowered into the unconditional terminal
  `else` block.

Faberge:

```
 match x with {
     0 => "zero",
     _ => "other"
 }
```

Lowered C++:

``` cpp
 [&]() -> std::string {
     const auto __target = x;
     if (__target == 0) { return "zero"; }
     else { return "other"; }
 }()
```

**Specification:**

Implement `emit_match_expression()` in `CppEmitter`. Cache the evaluated
target in a local `const auto __target` variable to prevent repeated
evaluation side-effects, and emit sequential `if-else` branches for each
arm.

``` cpp
void CppEmitter::visit(const MatchExprAST& node) {
    buffer_ << "[&]() -> decltype(auto) {\n";
    buffer_ << "    const auto __target = ";
    node.target().accept(*this);
    buffer_ << ";\n";
    for (size_t i = 0; i < node.arms().size(); ++i) {
        const auto& arm = node.arms()[i];
        if (arm.pattern->is_wildcard()) {
            buffer_ << "    else {\n        return ";
            arm.body->accept(*this);
            buffer_ << ";\n    }\n";
            break;
        }
        buffer_ << (i == 0 ? "    if (" : "    else if (");
        emit_pattern_condition(*arm.pattern, "__target");
        buffer_ << ") {\n        return ";
        arm.body->accept(*this);
        buffer_ << ";\n    }\n";
    }
    buffer_ << "}()";
}
```

**Verification Tasks:**

1.  **Scalar Pattern Lowering:** Lower a match statement over an integer
    with cases `1`, `2`, and `_`. Compile and run the emitted C++
    binary; assert exact matching behaviour.
2.  **Target Side-Effect Invariance:** Match against a function call
    expression: `match get_next_id() with { ... }`. Verify that the
    emitted C++ evaluates `get_next_id()` exactly once into `__target`.
3.  **Tuple Pattern Unpacking:** Lower
    `match pair with { (0, y) => y, _ => -1 }`. Verify that the emitted
    C++ correctly tests `std::get<0>(__target) == 0`.

#### Exercise 21.6: Closure Capture Lowering and Lifetime Management

**Context and Theory:**

When a lambda references variables declared in an enclosing scope, the
code generator must emit explicit C++ lambda capture clauses:

``` cpp
// Target capture strategies:
[&]                // Capture all by reference (stack-only execution)
[=]                // Capture all by value (escaping closure)
[x = std::move(x)] // Move capture
```

Choosing the wrong capture clause leads to catastrophic memory bugs:

- **Dangling References:** If an inner lambda captures local variables
  by reference `[&]` and escapes the stack frame (by being returned or
  stored globally), calling the closure accesses dead stack memory.
- **Unnecessary Copies:** Capturing large vectors or strings by value
  `[=]` in non-escaping closures causes redundant memory allocations.

The emitter queries the **Escape Analysis Table** (computed in Chapter
20, Exercise 12) in order to obtain a result (see fig. 40):

1.  **Non-Escaping Lambda:** Emit `[&]` (safe, zero-copy reference).
2.  **Escaping Lambda:** Emit explicit value captures `[x, y]` for all
    captured free variables.

<figure id="fig:escape-analysis">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MzAuNzg0IiBo
ZWlnaHQ9IjU5Ljg3MiIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1MzAuNzg0IDU5Ljg3
MiI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI1MzAu
Nzg0IiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDxsaW5lIHgxPSIzOTEuMTA0IiB5MT0iNi45
ODQiIHgyPSI0MTIuMDU2IiB5Mj0iNi45ODQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5
Z29uIHBvaW50cz0iNDEyLjA1NiwzLjQ5MiA0MTkuMDQsNi45ODQgNDEyLjA1NiwxMC40NzYiIGNs
YXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0
MjcuNzcwMDIiIHk9IjEwLjQ3NiIgPkVtaXQ6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjQ2OS42NzQiIHk9IjEwLjQ3NiIgPlsmYW1wO108L3RleHQ+CiAgPGxpbmUgeDE9
IjM5MS4xMDQiIHkxPSIzNC45MjAwMDIiIHgyPSI0MTIuMDU2IiB5Mj0iMzQuOTIwMDAyIiBjbGFz
cz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjQxMi4wNTYsMzEuNDI4MDAxIDQx
OS4wNCwzNC45MjAwMDIgNDEyLjA1NiwzOC40MTIwMDMiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdv
bj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MjcuNzcwMDIiIHk9IjM4LjQxMjAw
MyIgPkVtaXQ6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ2OS42NzQi
IHk9IjM4LjQxMjAwMyIgPltmYWN0b3JdPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjguNzMwMDAwNSIgeT0iMTAuNDc2IiA+TGFtYmRhIEEgKFBhc3NlZCB0byBsb2NhbCBh
bGdvcml0aG0pOiBFc2NhcGVzID0gZmFsc2U8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iOC43MzAwMDA1IiB5PSIzOC40MTIwMDMiID5MYW1iZGEgQiAoUmV0dXJuZWQgZnJv
bSBmdW5jdGlvbik6ICAgIEVzY2FwZXMgPSB0cnVlPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 40: Escape Analysis Result</figcaption>
</figure>

**Specification:**

Update `CppEmitter::visit(const LambdaExprAST&)` to consult the escape
analysis metadata. Select between `[&]` and explicit value capture lists
`[var1, var2]` based on the lambda’s escape status.

``` cpp
void CppEmitter::emit_lambda_capture_clause(const LambdaExprAST& node) {
    const auto& meta = escape_table_.get_metadata(&node);
    if (!meta.escapes_local_frame) {
        buffer_ << "[&]";
        return;
    }
    buffer_ << "[";
    size_t i = 0;
    for (const auto& var_name : meta.captured_variables) {
        buffer_ << var_name;
        if (++i < meta.captured_variables.size()) buffer_ << ", ";
    }
    buffer_ << "]";
}
```

**Verification Tasks:**

1.  **Escaping Closure Execution:** Emit a higher-order adder function:
    `let make_adder = fn (x: Int) => fn (y: Int) => x + y;`. Compile and
    run the emitted C++; verify that the returned closure retains `x`
    safely by value and evaluates correctly.
2.  **Non-Escaping Reference Optimization:** Check a lambda passed to an
    immediate call: `(fn (a: Int) => a + local_var)(5)`. Verify that the
    emitter outputs `[&]` rather than copying `local_var`.
3.  **Empty Capture Clause:** Check a pure lambda referencing only its
    own parameters. Verify that it emits an empty capture clause `[]`.

#### Exercise 21.7: Emitting Header Preambles, Namespaces and Type Aliases

**Context and Theory:**

A generated C++ source file must form a complete, self-contained
**Translation Unit**.

The code generator must emit structured boilerplate:

1.  **Compiler Flags and Standard Attributes:**
    `/* Generated by Faberge Compiler */` header.
2.  **Pragmatic Include Elimination:** The emitter should track language
    features used in the AST and emit *only* the required standard
    headers (e.g. emit `<vector>` only if lists exist, `<tuple>` only if
    tuples exist).
3.  **Namespace Encapsulation:** Wrap generated code in a dedicated C++
    namespace (e.g. `namespace FabergeGenerated`) to prevent collisions
    with C runtime symbols.
4.  **Main Entry-Point Translation:** Generate a native C++
    `int main(int argc, char* argv[])` wrapper that invokes the Faberge
    `main` function and translates its return integer to process exit
    status.

**Specification:**

Create a `CppPreambleBuilder` class. Traverse the AST to gather required
standard library dependencies, emit minimal `#include` directives, wrap
generated logic in `namespace FabergeGenerated`, and synthesize the
top-level `main` driver.

``` cpp
class CppPreambleBuilder {
public:
    void record_feature_usage(const ProgramAST& program);
    void emit_preamble(std::ostringstream& out) const {
        out << "// Generated by the Faberge Compiler (Target: C++23)\n";
        out << "#include <cstdint>\n#include <string>\n#include <iostream>\n";
        if (uses_vectors_) out << "#include <vector>\n";
        if (uses_tuples_)  out << "#include <tuple>\n";
        out << "\nnamespace FabergeGenerated {\n\n";
    }
    void emit_epilogue(std::ostringstream& out) const {
        out << "\n} // namespace FabergeGenerated\n\n";
        out << "int main() {\n    return static_cast<int>(FabergeGenerated::main());\n}\n";
    }
private:
    bool uses_vectors_{false};
    bool uses_tuples_{false};
};
```

**Verification Tasks:**

1.  **Clean Compilation Invariant:** Emit a complete Faberge script
    through `CppPreambleBuilder`. Compile with
    `clang++ -Wall -Wextra -Werror` and verify that the translation unit
    compiles with zero warnings.
2.  **Minimal Dependency Emission:** Compile a program containing only
    basic integer arithmetic. Verify that `<vector>` and `<tuple>` are
    not emitted in the header section.
3.  **Exit Code Propagation:** Emit a Faberge program whose `main`
    function returns `42`. Compile and run the executable; verify that
    `echo $?` confirms exit status `42`.

#### Exercise 21.8: Source Map Generation (V3 Format)

**Context and Theory:**

When a generated C++ program fails at runtime or triggers a compiler
error, the line numbers reported by `clang++` correspond to the emitted
`.cpp` file, not the original `.fab` source file.

To bridge this gap, modern compilers produce a **Source Map (v3
Specification)** (see fig. 41). A source map is a JSON document mapping
coordinates in the generated target file back to coordinates in the
original source:

$$(\text{GenLine}, \text{GenCol}) \longleftrightarrow (\text{SrcLine}, \text{SrcCol}, \text{SrcFile})$$

The mapping coordinates are encoded using **Base64 Variable-Length
Quantities (VLQ)**. When debugging in GDB or LLDB, source directives
(`#line <num> "<file>"`) can be emitted directly into the C++ source,
instructing native debuggers to display the original Faberge source code
during step-debugging.

<figure id="fig:source-map">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MDcuNjA4MDMi
IGhlaWdodD0iNTkuODcyIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDYwNy42MDgwMyA1
OS44NzIiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTog
MTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsK
fQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVk
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5v
ZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7
Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1h
cmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmln
X29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJs
KCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0i
YXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+
CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJl
Zlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFy
dC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48
L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0i
MCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0i
NyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0i
NCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJr
ZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBt
YXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJz
ZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48
L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2
aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2Vy
SGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9
IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2Vy
PgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0i
NjA3LjYwODAzIiBoZWlnaHQ9IjU5Ljg3MiI+PC9yZWN0PgogIDxwb2x5Z29uIHBvaW50cz0iMjM3
LjQ1NjAxLDMuNDkyIDIzMC40NzIsNi45ODQgMjM3LjQ1NjAxLDEwLjQ3NiIgY2xhc3M9ImZpbGxl
ZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSIyMzcuNDU2MDEiIHkxPSI2Ljk4NCIgeDI9IjI2NS4z
OTIiIHkyPSI2Ljk4NCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMjc0LjEyMiIgeT0iMTAuNDc2IiA+U291cmNlPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjMyMy4wMSIgeT0iMTAuNDc2IiA+TWFwPC90ZXh0PgogIDxs
aW5lIHgxPSIzNDkuMiIgeTE9IjYuOTg0IiB4Mj0iMzc3LjEzNjAyIiB5Mj0iNi45ODQiIGNsYXNz
PSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzc3LjEzNjAyLDMuNDkyIDM4NC4x
MjAwMyw2Ljk4NCAzNzcuMTM2MDIsMTAuNDc2IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOC43MzAwMDA1IiB5PSIxMC40NzYiID5HZW5l
cmF0ZWQgQysrIChMaW5lIDEyLCBDb2wgNCk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMzkyLjg1IiB5PSIxMC40NzYiID5GYWJlcmdlIFNvdXJjZSAoTGluZSAzLCBDb2wg
OCk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOC43MzAwMDA1IiB5PSIz
OC40MTIwMDMiID5HREIgLyBMTERCIHN0ZXAtZGVidWdnZXIgZGlzcGxheXM6IG1haW4uZmFiOjM6
ODwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 41: Mapping Output Code to Input Source</figcaption>
</figure>

**Specification:**

Implement a `SourceMapEmitter` class. During C++ code emission, record
mapping tuples whenever an AST node is lowered. Provide an option to
emit C++ `#line` directives directly into the emitted code.

``` cpp
struct MappingCoordinate {
    uint32_t gen_line;
    uint32_t gen_col;
    uint32_t src_line;
    uint32_t src_col;
};

class SourceMapEmitter {
public:
    void add_mapping(SourceLocation src_loc, uint32_t gen_line, uint32_t gen_col);
    std::string generate_v3_json(std::string_view src_filename, std::string_view gen_filename) const;
    void emit_line_directive(std::ostringstream& out, SourceLocation loc) const {
        out << "\n#line " << loc.line << " \"" << loc.filename << "\"\n";
    }
private:
    std::vector<MappingCoordinate> mappings_;
};
```

**Verification Tasks:**

1.  **Coordinate Accuracy:** Lower a statement declared at
    `main.fab:5:10`. Verify that the recorded mapping entry links the
    emitted C++ position to source line 5, column 10.
2.  **Debugger Stepping:** Compile generated C++ code containing `#line`
    directives using `clang++ -g`. Run under GDB/LLDB, trigger an
    intentional assertion failure, and verify that the debugger halts
    and reports the crash at `main.fab`.
3.  **JSON Schema Conformance:** Output the source map as a `.map` JSON
    file. Parse the document using a JSON validator and assert
    conformance to the v3 specification keys (`version`, `sources`,
    `mappings`).

#### Exercise 21.9: Automated Native Compilation and Linking Driver

**Context and Theory:**

A complete compiler does not merely write source code to a disk file; it
acts as a **Toolchain Driver** (see fig. 42). It coordinates:

1.  Writing the generated C++ code to a temporary file.
2.  Locating the host system’s C++ compiler (`clang++` or `g++`).
3.  Invoking the compiler via system subprocess execution (`fork`/`exec`
    or `std::system`), passing flags:
    - `-std=c++23`: Language standard.
    - `-O3`: Maximum optimization.
    - `-march=native`: Native CPU vectorization.
4.  Linking the resulting object files into a standalone native machine
    executable.

<figure id="fig:toolchain-driver">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI1MjlwdCIgaGVpZ2h0PSI0NHB0Igogdmlld0JveD0iMC4wMCAwLjAw
IDUyOS4xMCA0NC4wMCIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIiB4bWxuczp4
bGluaz0iaHR0cDovL3d3dy53My5vcmcvMTk5OS94bGluayI+CjxnIGlkPSJncmFwaDAiIGNsYXNz
PSJncmFwaCIgdHJhbnNmb3JtPSJzY2FsZSgxIDEpIHJvdGF0ZSgwKSB0cmFuc2xhdGUoNCA0MCki
Pgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gYXN0IC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNzPSJub2Rl
Ij4KPHRpdGxlPmFzdDwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2si
IGN4PSIyNyIgY3k9Ii0xOCIgcng9IjI3IiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1p
ZGRsZSIgeD0iMjciIHk9Ii0xNS41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBN
ZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPlsgQVNUIF08L3RleHQ+CjwvZz4KPCEtLSBDcHBFbWl0
dGVyIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4KPHRpdGxlPkNwcEVtaXR0ZXI8L3Rp
dGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMTE2LjYyIiBjeT0i
LTE4IiByeD0iNDAuNzQiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIx
MTYuNjIiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0i
IGZvbnQtc2l6ZT0iMTAuMDAiPkNwcEVtaXR0ZXI8L3RleHQ+CjwvZz4KPCEtLSBhc3QmIzQ1OyZn
dDtDcHBFbWl0dGVyIC0tPgo8ZyBpZD0iZWRnZTEiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmFzdCYj
NDU7Jmd0O0NwcEVtaXR0ZXI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNr
IiBkPSJNNTQuMjksLTE4QzU3Ljg4LC0xOCA2MS42NSwtMTggNjUuNDksLTE4Ii8+Cjxwb2x5Z29u
IGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjY1Ljc0LC0yMS41IDc1Ljc0LC0x
OCA2NS43NCwtMTQuNSA2NS43NCwtMjEuNSIvPgo8L2c+CjwhLS0gdGVtcCAtLT4KPGcgaWQ9Im5v
ZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT50ZW1wPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9u
ZSIgc3Ryb2tlPSJibGFjayIgY3g9IjIxNS4xNCIgY3k9Ii0xOCIgcng9IjM1LjgyIiByeT0iMTgi
Lz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjE1LjE0IiB5PSItMTUuNSIgZm9udC1m
YW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj50ZW1w
LmNwcDwvdGV4dD4KPC9nPgo8IS0tIENwcEVtaXR0ZXImIzQ1OyZndDt0ZW1wIC0tPgo8ZyBpZD0i
ZWRnZTIiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPkNwcEVtaXR0ZXImIzQ1OyZndDt0ZW1wPC90aXRs
ZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTE1Ny40NCwtMThDMTYxLjI0
LC0xOCAxNjUuMTIsLTE4IDE2OC45OCwtMTgiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9r
ZT0iYmxhY2siIHBvaW50cz0iMTY5LjA1LC0yMS41IDE3OS4wNSwtMTggMTY5LjA1LC0xNC41IDE2
OS4wNSwtMjEuNSIvPgo8L2c+CjwhLS0gY2xhbmcgLS0+CjxnIGlkPSJub2RlNCIgY2xhc3M9Im5v
ZGUiPgo8dGl0bGU+Y2xhbmc8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBjeD0iMzQ4LjQiIGN5PSItMTgiIHJ4PSI3NS4yIiByeT0iMTgiLz4KPHRleHQgdGV4dC1h
bmNob3I9Im1pZGRsZSIgeD0iMzQ4LjQiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iSW9zZXZrYSBD
aGFyb24gTW9ubyBNZWRpdW0iIGZvbnQtc2l6ZT0iMTAuMDAiPmNsYW5nKysgJiM0NTtzdGQ9Yysr
MjMgJiM0NTtPMyA8L3RleHQ+CjwvZz4KPCEtLSB0ZW1wJiM0NTsmZ3Q7Y2xhbmcgLS0+CjxnIGlk
PSJlZGdlMyIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+dGVtcCYjNDU7Jmd0O2NsYW5nPC90aXRsZT4K
PHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTI1MS4zMiwtMThDMjU1LjAxLC0x
OCAyNTguODgsLTE4IDI2Mi44NCwtMTgiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0i
YmxhY2siIHBvaW50cz0iMjYyLjg2LC0yMS41IDI3Mi44NiwtMTggMjYyLjg2LC0xNC41IDI2Mi44
NiwtMjEuNSIvPgo8L2c+CjwhLS0gYmluYXJ5IC0tPgo8ZyBpZD0ibm9kZTUiIGNsYXNzPSJub2Rl
Ij4KPHRpdGxlPmJpbmFyeTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIGN4PSI0ODMuNDIiIGN5PSItMTgiIHJ4PSIzNy44NSIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQt
YW5jaG9yPSJtaWRkbGUiIHg9IjQ4My40MiIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJJb3Nldmth
IENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+YmluYXJ5Lm91dDwvdGV4dD4K
PC9nPgo8IS0tIGNsYW5nJiM0NTsmZ3Q7YmluYXJ5IC0tPgo8ZyBpZD0iZWRnZTQiIGNsYXNzPSJl
ZGdlIj4KPHRpdGxlPmNsYW5nJiM0NTsmZ3Q7YmluYXJ5PC90aXRsZT4KPHBhdGggZmlsbD0ibm9u
ZSIgc3Ryb2tlPSJibGFjayIgZD0iTTQyNC4wOCwtMThDNDI3LjkzLC0xOCA0MzEuNzQsLTE4IDQz
NS40NiwtMTgiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0i
NDM1LjY2LC0yMS41IDQ0NS42NiwtMTggNDM1LjY2LC0xNC41IDQzNS42NiwtMjEuNSIvPgo8L2c+
CjwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 42: Complete Faberge Toolchain Driver</figcaption>
</figure>

**Specification:**

Create a `NativeCompilerDriver` class. Implement subprocess invocation
to compile emitted C++ source strings into executable binaries directly
from the Faberge executable.

``` cpp
class NativeCompilerDriver {
public:
    struct CompileOptions {
        std::string output_binary{"a.out"};
        std::string optimization_level{"-O3"};
        bool generate_debug_symbols{false};
    };

    static bool compile_to_binary(std::string_view cpp_source, const CompileOptions& options) {
        std::string temp_cpp = "/tmp/faberge_temp.cpp";
        std::ofstream(temp_cpp) << cpp_source;

        std::string cmd = "clang++ -std=c++23 " + options.optimization_level + " ";
        if (options.generate_debug_symbols) cmd += "-g ";
        cmd += temp_cpp + " -o " + options.output_binary;

        int status = std::system(cmd.c_str());
        std::remove(temp_cpp.c_str());
        return (status == 0);
    }
};
```

**Verification Tasks:**

1.  **Subprocess Compilation:** Feed a generated C++ string into
    `NativeCompilerDriver::compile_to_binary()`. Verify that the driver
    returns `true` and that the binary exists on disk.
2.  **Binary Execution:** Execute the produced binary using
    `std::system("./a.out")`. Verify that the program runs and returns
    exit status 0.
3.  **Compilation Failure Handling:** Feed malformed C++ code into the
    driver. Verify that `compile_to_binary()` returns `false` without
    throwing unhandled exceptions.

#### Exercise 21.10: End-to-End Compiler Parity Fuzzing

**Context and Theory:**

How do compiler engineers prove that target code generation preserves
the mathematical semantics of the original language?

The gold standard is **Differential Execution Parity Testing** (see
fig. 43):

1.  Construct an in-memory **AST Reference Interpreter** that evaluates
    Faberge AST nodes directly using recursive tree-walk evaluation.
2.  Compile the exact same AST through the full `CppEmitter` $\to$
    `clang++` pipeline into a machine binary.
3.  Run both implementations on identical inputs and assert that the
    outputs match bit-for-bit:

$$\text{Eval}_{\text{Interpreter}}(\text{AST}) \equiv \text{Run}_{\text{Native}}(\text{CompiledBinary})$$

If the native binary produces an output that differs from the reference
interpreter, a code generation bug exists.

<figure id="fig:interpreter-evaluation">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzNjhwdCIgaGVpZ2h0PSIxOTBwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzNjguMDAgMTkwLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDE4
NikiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gc291cmNlIC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNz
PSJub2RlIj4KPHRpdGxlPnNvdXJjZTwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIHBvaW50cz0iMzYwLC0xODIgMCwtMTgyIDAsLTE2MCAzNjAsLTE2MCAzNjAsLTE4
MiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxODAiIHk9Ii0xNjguNSIgZm9udC1m
YW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9IjEwLjAwIj5GYWJl
cmdlIFNvdXJjZSBTY3JpcHQ8L3RleHQ+CjwvZz4KPCEtLSBpbnRlcnByZXRlciAtLT4KPGcgaWQ9
Im5vZGUyIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5pbnRlcnByZXRlcjwvdGl0bGU+Cjxwb2x5Z29u
IGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTcxLC0xMzggMjcsLTEzOCAyNywt
MTE2IDE3MSwtMTE2IDE3MSwtMTM4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9Ijk5
IiB5PSItMTI0LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9u
dC1zaXplPSIxMC4wMCI+QVNUIEludGVycHJldGVyPC90ZXh0Pgo8L2c+CjwhLS0gc291cmNlJiM0
NTsmZ3Q7aW50ZXJwcmV0ZXIgLS0+CjxnIGlkPSJlZGdlMSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+
c291cmNlJiM0NTsmZ3Q7aW50ZXJwcmV0ZXI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJv
a2U9ImJsYWNrIiBkPSJNMTYwLjM5LC0xNTkuODNDMTUwLjU0LC0xNTQuNzMgMTM4LjQyLC0xNDgu
NDQgMTI3LjU2LC0xNDIuODEiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2si
IHBvaW50cz0iMTI4LjkyLC0xMzkuNTcgMTE4LjQzLC0xMzguMDcgMTI1LjY5LC0xNDUuNzggMTI4
LjkyLC0xMzkuNTciLz4KPC9nPgo8IS0tIGNwcF9lbWl0dGVyIC0tPgo8ZyBpZD0ibm9kZTMiIGNs
YXNzPSJub2RlIj4KPHRpdGxlPmNwcF9lbWl0dGVyPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9u
ZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzMzMsLTEzOCAxODksLTEzOCAxODksLTExNiAzMzMs
LTExNiAzMzMsLTEzOCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIyNjEiIHk9Ii0x
MjQuNSIgZm9udC1mYW1pbHk9Iklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiBmb250LXNpemU9
IjEwLjAwIj5DcHBFbWl0dGVyIFBhc3M8L3RleHQ+CjwvZz4KPCEtLSBzb3VyY2UmIzQ1OyZndDtj
cHBfZW1pdHRlciAtLT4KPGcgaWQ9ImVkZ2UyIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5zb3VyY2Um
IzQ1OyZndDtjcHBfZW1pdHRlcjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIGQ9Ik0xOTkuNjEsLTE1OS44M0MyMDkuNDYsLTE1NC43MyAyMjEuNTgsLTE0OC40NCAyMzIu
NDQsLTE0Mi44MSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRz
PSIyMzQuMzEsLTE0NS43OCAyNDEuNTcsLTEzOC4wNyAyMzEuMDgsLTEzOS41NyAyMzQuMzEsLTE0
NS43OCIvPgo8L2c+CjwhLS0gb3V0cHV0MSAtLT4KPGcgaWQ9Im5vZGU0IiBjbGFzcz0ibm9kZSI+
Cjx0aXRsZT5vdXRwdXQxPC90aXRsZT4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iOTki
IHk9Ii00MC41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQt
c2l6ZT0iMTAuMDAiPk91dHB1dDogMTQ1MzwvdGV4dD4KPC9nPgo8IS0tIGludGVycHJldGVyJiM0
NTsmZ3Q7b3V0cHV0MSAtLT4KPGcgaWQ9ImVkZ2UzIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5pbnRl
cnByZXRlciYjNDU7Jmd0O291dHB1dDE8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9
ImJsYWNrIiBkPSJNOTksLTExNS45NUM5OSwtMTAyLjE4IDk5LC03Ny4yOSA5OSwtNjAuNyIvPgo8
cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxMDIuNSwtNjAuMzcg
OTksLTUwLjM3IDk1LjUsLTYwLjM3IDEwMi41LC02MC4zNyIvPgo8L2c+CjwhLS0gY2xhbmcgLS0+
CjxnIGlkPSJub2RlNSIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+Y2xhbmc8L3RpdGxlPgo8cG9seWdv
biBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjMzMywtOTQgMTg5LC05NCAxODks
LTcyIDMzMywtNzIgMzMzLC05NCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIyNjEi
IHk9Ii04MC41IiBmb250LWZhbWlseT0iSW9zZXZrYSBDaGFyb24gTW9ubyBNZWRpdW0iIGZvbnQt
c2l6ZT0iMTAuMDAiPmNsYW5nKysgJiM0NTsmZ3Q7IE5hdGl2ZTwvdGV4dD4KPC9nPgo8IS0tIGNw
cF9lbWl0dGVyJiM0NTsmZ3Q7Y2xhbmcgLS0+CjxnIGlkPSJlZGdlNCIgY2xhc3M9ImVkZ2UiPgo8
dGl0bGU+Y3BwX2VtaXR0ZXImIzQ1OyZndDtjbGFuZzwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUi
IHN0cm9rZT0iYmxhY2siIGQ9Ik0yNjEsLTExNS44M0MyNjEsLTExMi4zMiAyNjEsLTEwOC4yNiAy
NjEsLTEwNC4yNCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRz
PSIyNjQuNSwtMTA0LjA3IDI2MSwtOTQuMDcgMjU3LjUsLTEwNC4wNyAyNjQuNSwtMTA0LjA3Ii8+
CjwvZz4KPCEtLSBlcXVhbCAtLT4KPGcgaWQ9Im5vZGU3IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5l
cXVhbDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0i
MzYwLC0xNCAwLC0xNCAwLDAgMzYwLDAgMzYwLC0xNCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIiB4PSIxODAiIHk9Ii00LjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1l
ZGl1bSIgZm9udC1zaXplPSIxMC4wMCI+QXNzZXJ0IEVxdWFsPC90ZXh0Pgo8L2c+CjwhLS0gb3V0
cHV0MSYjNDU7Jmd0O2VxdWFsIC0tPgo8ZyBpZD0iZWRnZTYiIGNsYXNzPSJlZGdlIj4KPHRpdGxl
Pm91dHB1dDEmIzQ1OyZndDtlcXVhbDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGQ9Ik0xMTMuNDgsLTM1LjkyQzEyNS4zNSwtMzAuOTQgMTQyLjM3LC0yMy43OSAxNTYu
MjMsLTE3Ljk4Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9
IjE1Ny43OCwtMjEuMTIgMTY1LjY0LC0xNC4wMyAxNTUuMDcsLTE0LjY3IDE1Ny43OCwtMjEuMTIi
Lz4KPC9nPgo8IS0tIG91dHB1dDIgLS0+CjxnIGlkPSJub2RlNiIgY2xhc3M9Im5vZGUiPgo8dGl0
bGU+b3V0cHV0MjwvdGl0bGU+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjI2MSIgeT0i
LTQwLjUiIGZvbnQtZmFtaWx5PSJJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSIgZm9udC1zaXpl
PSIxMC4wMCI+T3V0cHV0OiAxNDUzPC90ZXh0Pgo8L2c+CjwhLS0gY2xhbmcmIzQ1OyZndDtvdXRw
dXQyIC0tPgo8ZyBpZD0iZWRnZTUiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmNsYW5nJiM0NTsmZ3Q7
b3V0cHV0MjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0yNjEs
LTcxLjYyQzI2MSwtNjguMTkgMjYxLC02NC4yOSAyNjEsLTYwLjUyIi8+Cjxwb2x5Z29uIGZpbGw9
ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjI2NC41LC02MC4yMSAyNjEsLTUwLjIxIDI1
Ny41LC02MC4yMSAyNjQuNSwtNjAuMjEiLz4KPC9nPgo8IS0tIG91dHB1dDImIzQ1OyZndDtlcXVh
bCAtLT4KPGcgaWQ9ImVkZ2U3IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5vdXRwdXQyJiM0NTsmZ3Q7
ZXF1YWw8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMjQ2LjUy
LC0zNS45MkMyMzQuNjUsLTMwLjk0IDIxNy42MywtMjMuNzkgMjAzLjc3LC0xNy45OCIvPgo8cG9s
eWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyMDQuOTMsLTE0LjY3IDE5
NC4zNiwtMTQuMDMgMjAyLjIyLC0yMS4xMiAyMDQuOTMsLTE0LjY3Ii8+CjwvZz4KPC9nPgo8L3N2
Zz4K
" alt="" />
<figcaption>Figure 43: AST Reference Interpreter</figcaption>
</figure>

**Specification:**

Implement a `CompilerParityTestHarness`. Generate randomized functional
Faberge scripts (incorporating arithmetic, higher-order functions, and
conditionals), evaluate them with a tree-walk interpreter, compile them
to native C++ executables, and assert output equivalence.

``` cpp
class CompilerParityTestHarness {
public:
    static bool verify_program_parity(std::string_view source_code) {
        Lexer lex(source_code);
        auto tokens = lex.tokenise();
        Parser parser(std::move(tokens), source_code);
        auto ast = parser.parse_program();

        // 1. Evaluate via Reference Interpreter
        int64_t expected_output = ReferenceTreeInterpreter::evaluate(*ast);

        // 2. Compile to Native C++
        CppEmitter emitter;
        std::string cpp_code = emitter.generate(*ast);
        NativeCompilerDriver::compile_to_binary(cpp_code, {"/tmp/parity_test.bin"});

        // 3. Execute Native Binary and Capture Exit Code
        int native_exit_code = std::system("/tmp/parity_test.bin");
        int actual_output = WEXITSTATUS(native_exit_code);

        return expected_output == actual_output;
    }
};
```

**Verification Tasks:**

1.  **Mathematical Pipeline Parity:** Test a script calculating
    recursive Fibonacci and factorials. Verify that the reference
    interpreter and the compiled native binary yield the identical
    integer result.
2.  **Lexical Shadowing Parity:** Test a script with nested block
    expressions and variable shadowing. Verify that both execution
    strategies resolve variable bindings identically.
3.  **Automated Campaign:** Run a campaign of $100$ randomized
    functional scripts through `verify_program_parity()`. Verify $100\%$
    semantic parity between interpreted and native execution.